#**ONE RUN MODEL (ORM)**

# Installations and importations

L'obiettivo attuale è di cercare di unificare i vari modelli che abbiamo già sviluppato, uno per ogni piattaforma, in un unico modello che, dato in ingresso un brevetto, riesca a classificare la categoria di appartenza di tale campione

Installiamo le dipendenze che ci servono

In [ ]:
!pip install openpyxl
!pip install tensorflow
!pip install keras
!pip install jax
!pip install jaxlib
!pip install imblearn
!pip install iterative-stratification

Importiamo le libreria che ci servono per questo script di apprendimento

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from itertools import combinations
import seaborn as sns
import sklearn as sk
import tensorflow as tf
import keras as ks
from sklearn.preprocessing import MultiLabelBinarizer
import nltk
import joblib
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from collections import Counter
from sklearn.utils import shuffle
from sklearn.feature_extraction.text import TfidfVectorizer
import re
from scipy.sparse import hstack
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.multiclass import OneVsRestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import precision_recall_fscore_support
import numpy as np
import pandas as pd
import pickle
from sklearn.metrics import precision_recall_fscore_support, fbeta_score, jaccard_score
from sklearn.preprocessing import MultiLabelBinarizer
import numpy as np
import pandas as pd
import pickle
import numpy as np
import matplotlib.pyplot as plt
import jax.numpy as jnp
import jax
import time
from sklearn.preprocessing import OneHotEncoder
import jax.numpy as jnp
from jax.nn import softmax
from google.colab import files
from sklearn.metrics import precision_recall_fscore_support, fbeta_score
from scipy.sparse import hstack
import copy
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_recall_fscore_support
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

# 2024 DATASET

## Carichiamo il dataset di brevetti 2024_ATG_SORV manual

In [ ]:
uploaded = files.upload()

Saving 2024_ATG_SORV manual.xlsx to 2024_ATG_SORV manual.xlsx


Diamo una occhiata al dataset ...

In [ ]:
df_2024_manual = pd.read_excel("2024_ATG_SORV manual.xlsx")

In [ ]:
df_2024_manual.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5264 entries, 0 to 5263
Data columns (total 48 columns):
 #   Column                                     Non-Null Count  Dtype  
---  ------                                     --------------  -----  
 0   Number                                     5264 non-null   int64  
 1   Manually assigned PLATFORM                 5264 non-null   object 
 2   Predicted Neural Network
PLATFORM          4843 non-null   object 
 3   Publication Number                         5264 non-null   object 
 4   Patent Type                                5264 non-null   object 
 5   Title (Translated)(English)                5264 non-null   object 
 6   Abstract (Translated)(English)             5264 non-null   object 
 7   Independent Claims (Translated)(English)   5264 non-null   object 
 8   IPC                                        5264 non-null   object 
 9   CPC                                        5264 non-null   object 
 10  Application domain      

In [ ]:
key_col = "Publication Number"
label_col = "Manually assigned PLATFORM"

duplicates = df_2024_manual[df_2024_manual.duplicated(subset=[key_col], keep=False)]

summary = duplicates.groupby(key_col)[label_col].nunique()

print("Duplicati con stessa label:", (summary == 1).sum())
print("Duplicati con label diverse:", (summary > 1).sum())

Duplicati con stessa label: 0
Duplicati con label diverse: 0


Prepara il dataset all'unione sucessiva

Funzione di normalizzazione delle labels

In [ ]:
def normalize_platform_label(label):
    if pd.isna(label):
        return None

    label = str(label).upper()
    label = re.sub(r"GAS\s+BOILER", "GASBOILER", label)
    label = label.replace("/", " ")
    label = re.sub(r"\s+", " ", label).strip()

    # Convert 'NON' and 'RELEVANT' to 'NOT_RELEVANT'
    if label == "NON" or label == "RELEVANT":
        return "NOT_RELEVANT"

    return label if label != "" else None

Definiamo le colonne target

In [ ]:
selected_columns = [
    "Manually assigned PLATFORM",
    "Title (Translated)(English)",
    "Abstract (Translated)(English)",
    "Independent Claims (Translated)(English)",
    "IPC",
    "CPC",
    "Application domain",
    "Technology topic",
    "Technical Problem",
    "Technical Field",
    "Std. Technical Benefit Phrase",
    "Publication Number"
]

In [ ]:
for col in selected_columns:
    if col not in df_2024_manual.columns:
        df_2024_manual[col] = ""

In [ ]:
df_2024_manual["Publication Number"] = (
    df_2024_manual["Publication Number"]
    .astype(str)
    .str.strip()
    .str.upper()
)

df_2024_manual["Manually assigned PLATFORM"] = (
    df_2024_manual["Manually assigned PLATFORM"]
    .apply(normalize_platform_label)
)


In [ ]:
df_2024_manual = df_2024_manual[
    df_2024_manual["Publication Number"].notna() &
    (df_2024_manual["Publication Number"] != "") &
    (df_2024_manual["Publication Number"] != "NAN") &
    df_2024_manual["Manually assigned PLATFORM"].notna()
].copy()


In [ ]:
df_2024_manual = df_2024_manual.drop_duplicates(
    subset=["Publication Number", "Manually assigned PLATFORM"]
).copy()


Aggregazione delle labels

In [ ]:
def combine_labels(series):
    labels = []
    for value in series.dropna():
        for token in str(value).split():
            if token not in labels:
                labels.append(token)
    return " ".join(labels)


def first_non_empty(series):
    for value in series:
        if pd.notna(value) and str(value).strip() != "":
            return value
    return ""


In [ ]:
agg_map_2024 = {
    "Manually assigned PLATFORM": combine_labels,
    "Title (Translated)(English)": first_non_empty,
    "Abstract (Translated)(English)": first_non_empty,
    "Independent Claims (Translated)(English)": first_non_empty,
    "IPC": first_non_empty,
    "CPC": first_non_empty,
    "Application domain": first_non_empty,
    "Technology topic": first_non_empty,
    "Technical Problem": first_non_empty,
    "Technical Field": first_non_empty,
    "Std. Technical Benefit Phrase": first_non_empty,
}


In [ ]:
df_2024_agg = (
    df_2024_manual[selected_columns]
    .groupby("Publication Number", as_index=False)
    .agg(agg_map_2024)
)

In [ ]:
print("2024 aggregated shape:", df_2024_agg.shape)
df_2024_agg.head()

2024 aggregated shape: (5264, 12)


,Publication Number,Manually assigned PLATFORM,Title (Translated)(English),Abstract (Translated)(English),Independent Claims (Translated)(English),IPC,CPC,Application domain,Technology topic,Technical Problem,Technical Field,Std. Technical Benefit Phrase
0,AT526453A2,HHP,Wall duct unit and external wall grille with a...,Wall feed-through unit (100) for a connection ...,-,F24F13/02 | F24F13/08,F24F13/0272 | F24F13/082 | F24F2007/0025 | F24...,Ducting arrangements|Mechanical apparatus|Ligh...,Home appliance|Mechanics|Waste gas|Heat pump|H...,-,-,-
1,AU2023203051B2,ESWH HPWH,Foaming method and device for water tank of wa...,The present invention relates to the technical...,1. A foaming method for a water tank of water ...,F24H1/18 | B29C44/18,B29C44/02 | B29C44/35,Water heaters,Foaming agent|Injection port|Heat conservation...,The existing foaming method for water tanks in...,-,-
2,AU2023206174A1,ESWH,Water heater sensor,Abstract A temperature sensing device 10 for a...,1. A water heater including:_x000D_\na water s...,F24H15/225 | F24H9/20 | F24H15/37 | G01K1/14 |...,F24H15/225 | F24H9/20 | G05D23/1931 | F24H15/3...,Thermometer details|Thermometers using electri...,Sensing data|Control cell|Process engineering|...,Water heaters with storage tanks experience te...,-,-
3,AU2024203072A1,ESWH,Program executed on mobile communication termi...,[Problem] To provide a technology for pairing ...,1. A program executed on a mobile communicatio...,H04Q9/00 | F24H1/00 | F24H15/45 | G05B19/02 | ...,H04W12/06 | H04W76/10 | H04W12/50,Telemetry/telecontrol selection arrangements|T...,Data matching|Telecommunications|Display devic...,In scenarios where multiple water heaters are ...,-,-
4,BR202022015721U2,GSWH,UNIFIED FILTERING DEVICE WITH OVER PRESSURE RE...,Unified filtering device with overpressure rel...,-,F24H9/00 | F24H9/20 | B01D35/02,F24H9/00 | F24H9/20 | F24H9/0042 | B01D35/02,Fluid heaters|Filtration separation,Environmental engineering|Patent application|F...,Existing gas water heaters have complex design...,-,-


## Carichiamo il dataset di brevetti 2024_ATG_SORV neural

In [ ]:
uploaded = files.upload()

Saving 2024_ATG_SORV neural.xlsx to 2024_ATG_SORV neural.xlsx


In [ ]:
df_2024_neural = pd.read_excel("2024_ATG_SORV neural.xlsx")

In [ ]:
df_2024_neural.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8683 entries, 0 to 8682
Data columns (total 47 columns):
 #   Column                                     Non-Null Count  Dtype  
---  ------                                     --------------  -----  
 0   Platform                                   8683 non-null   object 
 1   Number                                     8683 non-null   int64  
 2   Publication Number                         8683 non-null   object 
 3   Patent Type                                8683 non-null   object 
 4   Title (Translated)(English)                8683 non-null   object 
 5   Abstract (Translated)(English)             8683 non-null   object 
 6   Independent Claims (Translated)(English)   8683 non-null   object 
 7   IPC                                        8683 non-null   object 
 8   CPC                                        8683 non-null   object 
 9   Application domain                         8683 non-null   object 
 10  Technology topic        

In [ ]:
num_duplicates = df_2024_neural.duplicated().sum()
print(f"Numero di duplicati nel df_2024_neural: {num_duplicates}")

# Per visualizzare le righe duplicate (opzionale)
# duplicates_rows = df_2024_neural[df_2024_neural.duplicated(keep=False)]
# print("Righe duplicate:")
# print(duplicates_rows.head())

Numero di duplicati nel df_2024_neural: 0


In [ ]:
# Create a modified list of selected columns for df_2024_neural,
# replacing "Manually assigned PLATFORM" with "Platform"
selected_columns_neural = [
    "Platform",
    "Title (Translated)(English)",
    "Abstract (Translated)(English)",
    "Independent Claims (Translated)(English)",
    "IPC",
    "CPC",
    "Application domain",
    "Technology topic",
    "Technical Problem",
    "Technical Field",
    "Std. Technical Benefit Phrase",
    "Publication Number"
]

# Ensure all columns in selected_columns_neural exist in df_2024_neural
# and add them if they don't, filling with empty strings
for col in selected_columns_neural:
    if col not in df_2024_neural.columns:
        df_2024_neural[col] = ""

# Clean "Publication Number" and normalize "Platform"
df_2024_neural["Publication Number"] = (
    df_2024_neural["Publication Number"]
    .astype(str)
    .str.strip()
    .str.upper()
)

df_2024_neural["Platform"] = (
    df_2024_neural["Platform"]
    .apply(normalize_platform_label)
)

# Remove rows with null or empty Publication Number or Platform
df_2024_neural = df_2024_neural[
    df_2024_neural["Publication Number"].notna() &
    (df_2024_neural["Publication Number"] != "") &
    (df_2024_neural["Publication Number"] != "NAN") &
    df_2024_neural["Platform"].notna()
].copy()

# Drop duplicates based on "Publication Number" and "Platform"
df_2024_neural = df_2024_neural.drop_duplicates(
    subset=["Publication Number", "Platform"]
).copy()

# Define aggregation map for df_2024_neural
agg_map_neural = {
    "Platform": combine_labels,
    "Title (Translated)(English)": first_non_empty,
    "Abstract (Translated)(English)": first_non_empty,
    "Independent Claims (Translated)(English)": first_non_empty,
    "IPC": first_non_empty,
    "CPC": first_non_empty,
    "Application domain": first_non_empty,
    "Technology topic": first_non_empty,
    "Technical Problem": first_non_empty,
    "Technical Field": first_non_empty,
    "Std. Technical Benefit Phrase": first_non_empty,
}

# Aggregate df_2024_neural
df_2024_neural_agg = (
    df_2024_neural[selected_columns_neural]
    .groupby("Publication Number", as_index=False)
    .agg(agg_map_neural)
)

print("df_2024_neural aggregated shape:", df_2024_neural_agg.shape)
df_2024_neural_agg.head()


df_2024_neural aggregated shape: (6325, 12)


,Publication Number,Platform,Title (Translated)(English),Abstract (Translated)(English),Independent Claims (Translated)(English),IPC,CPC,Application domain,Technology topic,Technical Problem,Technical Field,Std. Technical Benefit Phrase
0,AT526453A2,HHP,Wall duct unit and external wall grille with a...,Wall feed-through unit (100) for a connection ...,-,F24F13/02 | F24F13/08,F24F13/0272 | F24F13/082 | F24F2007/0025 | F24...,Ducting arrangements|Mechanical apparatus|Ligh...,Home appliance|Mechanics|Waste gas|Heat pump|H...,-,-,-
1,AU2023200424B2,ESWH,"A system, apparatus and method for efficient u...",A solar photovoltaic (PV) water heating system...,1. A water heating device comprising:_x000D_\n...,H02S10/20 | F24D19/10 | H02J3/28,F24H1/201 | F24H1/18 | H02S40/44 | H02S10/12 |...,Solar heating energy|Photovoltaic monitoring|B...,Collector device|Alternating current|Heating s...,-,-,-
2,AU2023203051B2,ESWH HPWH,Foaming method and device for water tank of wa...,The present invention relates to the technical...,1. A foaming method for a water tank of water ...,F24H1/18 | B29C44/18,B29C44/02 | B29C44/35,Water heaters,Foaming agent|Injection port|Heat conservation...,The existing foaming method for water tanks in...,-,-
3,AU2023206174A1,ESWH,Water heater sensor,Abstract A temperature sensing device 10 for a...,1. A water heater including:_x000D_\na water s...,F24H15/225 | F24H9/20 | F24H15/37 | G01K1/14 |...,F24H15/225 | F24H9/20 | G05D23/1931 | F24H15/3...,Thermometer details|Thermometers using electri...,Sensing data|Control cell|Process engineering|...,Water heaters with storage tanks experience te...,-,-
4,AU2023270312A1,HHP,Hot water storing-type hot water supplying system,[Abstract] [Problem] It is an object of the pr...,[Claim 1]_x000D_\nA hot water storing-type hot...,F24D19/10 | F16K17/04 | F16K17/08 | F16K17/168...,-,Equalizing valves|Safety valves|Space heating ...,Thermodynamics|Heat losses|Thermal water|Mecha...,-,-,-


## Fondiamo i due dataset df_2024_manual e df_2024_neural

In [ ]:
df_2024_neural_renamed = df_2024_neural_agg.rename(columns={'Platform': 'Manually assigned PLATFORM'})

# Identifica i brevetti unici in df_2024_neural_renamed che non sono in df_2024_agg
unique_in_neural = df_2024_neural_renamed[~df_2024_neural_renamed['Publication Number'].isin(df_2024_agg['Publication Number'])]

# Assegna la label 'No Relevant' a questi brevetti, unendo 'NO' e 'RELEVANT' in 'NOT_RELEVANT'
unique_in_neural_labelled = unique_in_neural.copy()
unique_in_neural_labelled['Manually assigned PLATFORM'] = 'NOT_RELEVANT'

# Concatena i due dataset
df_combined_2024 = pd.concat([df_2024_agg, unique_in_neural_labelled], ignore_index=True)

print(f"Shape di df_2024_manual aggregato: {df_2024_agg.shape}")
print(f"Shape di df_2024_neural aggregato: {df_2024_neural_agg.shape}")
print(f"Numero di brevetti unici da df_2024_neural aggiunti: {len(unique_in_neural_labelled)}")
print(f"Shape del dataset combinato: {df_combined_2024.shape}")

df_combined_2024.head()

# Calculate samples per class for df_combined_2024
# Multi-label binarization
y_raw_combined = df_combined_2024['Manually assigned PLATFORM']

# For MultiLabelBinarizer to correctly identify 'NOT_RELEVANT' as a single class,
# we can simply use the default split() as 'NOT_RELEVANT' is a single word.
y_split_combined = y_raw_combined.apply(lambda x: x.split())

mlb_combined = MultiLabelBinarizer()
y_multi_combined = mlb_combined.fit_transform(y_split_combined)

num_classes_combined = len(mlb_combined.classes_)

print(f"\nNumero di classi reali in df_combined_2024: {num_classes_combined}")
print(f"Classi in df_combined_2024: {mlb_combined.classes_}")

# Distribuzione delle classi
class_counts_combined = y_multi_combined.sum(axis=0)

class_distribution_combined = pd.DataFrame({
    "Classe": mlb_combined.classes_,
    "Numero di sample": class_counts_combined
}).sort_values(by="Numero di sample", ascending=False)

print("\nDistribuzione delle classi in df_combined_2024:")
print(class_distribution_combined)

Shape di df_2024_manual aggregato: (5264, 12)
Shape di df_2024_neural aggregato: (6325, 12)
Numero di brevetti unici da df_2024_neural aggiunti: 1061
Shape del dataset combinato: (6325, 12)

Numero di classi reali in df_combined_2024: 9
Classi in df_combined_2024: ['EIWH' 'ESWH' 'GAHP' 'GASBOILER' 'GIWH' 'GSWH' 'HHP' 'HPWH'
 'NOT_RELEVANT']

Distribuzione delle classi in df_combined_2024:
         Classe  Numero di sample
6           HHP              3564
8  NOT_RELEVANT              1061
7          HPWH               574
1          ESWH               556
5          GSWH               528
3     GASBOILER               314
4          GIWH               216
0          EIWH                95
2          GAHP                65


In [ ]:
df_combined_2024.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6325 entries, 0 to 6324
Data columns (total 12 columns):
 #   Column                                    Non-Null Count  Dtype 
---  ------                                    --------------  ----- 
 0   Publication Number                        6325 non-null   object
 1   Manually assigned PLATFORM                6325 non-null   object
 2   Title (Translated)(English)               6325 non-null   object
 3   Abstract (Translated)(English)            6325 non-null   object
 4   Independent Claims (Translated)(English)  6325 non-null   object
 5   IPC                                       6325 non-null   object
 6   CPC                                       6325 non-null   object
 7   Application domain                        6325 non-null   object
 8   Technology topic                          6325 non-null   object
 9   Technical Problem                         6325 non-null   object
 10  Technical Field                           6325 n

In [ ]:
key_col_combined = "Publication Number"
label_col_combined = "Manually assigned PLATFORM"

# Identifica i brevetti duplicati basandosi solo su 'Publication Number'
duplicates_combined = df_combined_2024[df_combined_2024.duplicated(subset=[key_col_combined], keep=False)]

# Se non ci sono duplicati, stampa un messaggio appropriato
if duplicates_combined.empty:
    print("Nessun duplicato trovato lungo la colonna 'Publication Number' nel df_combined_2024.")
else:
    print(f"Trovati {len(duplicates_combined)} righe duplicate basate su 'Publication Number'.")
    print("Dettagli dei duplicati (prime 5 righe):")
    display(duplicates_combined.head())

    # Raggruppa i duplicati e conta le etichette uniche
    summary_duplicates_combined = duplicates_combined.groupby(key_col_combined)[label_col_combined].nunique()

    print("\nRiepilogo dei duplicati rispetto alla 'Manually assigned PLATFORM':")
    print("Duplicati con stessa label:", (summary_duplicates_combined == 1).sum())
    print("Duplicati con label diverse:", (summary_duplicates_combined > 1).sum())

Nessun duplicato trovato lungo la colonna 'Publication Number' nel df_combined_2024.


# Dataset 2025

## Trattiamo il dataset 2025_ATG_SORV_pubbl_raw.xlsx

In [ ]:
uploaded = files.upload()

Saving 2025_ATG_pubbl_raw.XLSX to 2025_ATG_pubbl_raw.XLSX


Diamo un'occhiata al dataset ...

In [ ]:
df_2025_raw = pd.read_excel("2025_ATG_pubbl_raw.XLSX")

In [ ]:
df_2025_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9335 entries, 0 to 9334
Data columns (total 25 columns):
 #   Column                                     Non-Null Count  Dtype 
---  ------                                     --------------  ----- 
 0   Number                                     9335 non-null   int64 
 1   source pltf                                9335 non-null   object
 2   Publication Number                         9335 non-null   object
 3   text merger                                9335 non-null   object
 4   Patent Type                                9335 non-null   object
 5   Grouped Assignee                           9335 non-null   object
 6   Standardized Original Assignee(Applicant)  9335 non-null   object
 7   Title (Translated)(English)                9335 non-null   object
 8   Abstract (Translated)(English)             9335 non-null   object
 9   Independent Claims (Translated)(English)   9335 non-null   object
 10  IPC                                 

In [ ]:
df_2025_raw.head()

,Number,source pltf,Publication Number,text merger,Patent Type,Grouped Assignee,Standardized Original Assignee(Applicant),Title (Translated)(English),Abstract (Translated)(English),Independent Claims (Translated)(English),...,Earliest Priority Date,Application Date,Publication Date,Application Number,Simple Legal Status,Authority,INPADOC Family,INPADOC Family Application Count,Priority Country/Region,Patent Valuation
0,320,GAS BOILER,AU2024259712A1,Process for ammonia synthesis using a reaction...,Applications,SIEMENS,SIEMENS AG,Process for ammonia synthesis using a reaction...,Process for ammonia synthesis using a reaction...,1. A process for ammonia synthesis with a reac...,...,2023-11-02,2024-11-04,2025-05-22,AU2024259712,Pending,Australia,AU2024259712A1 | EP4549385A1 | US20250145481A1,3,EP,"$ 13,000"
1,3502,HHP,AU2025201295A1,Air conditionerAn air conditioner includes a r...,Applications,DAIKIN,DAIKIN INDUSTRIES LTD,Air conditioner,An air conditioner includes a rotary shaft mem...,[Claim 1] An air conditioner including:_...,...,2024-03-29,2025-02-24,2025-10-16,AU2025201295,Pending,Australia,AU2025201295A1 | JP2025153143A | JP7828004B2,2,JP,"$ 3,700"
2,910,ESWH,BG67821B1,Vertical boiler with bottom water supplyThe ve...,Patents,TESY,TESY OOD,Vertical boiler with bottom water supply,The vertical boiler with bottom water supply a...,-,...,2023-12-13,2023-12-13,2025-12-15,BG113825,Active,Bulgaria,BG113825A | BG67821B1,1,-,"$ 5,800"
3,399,EIWH,BR102024024922A2,GAS WATER HEATER--Gaseous fuel burner-Existing...,Applications,MIDEA,WUHU MIDEA KITCHEN & BATH APPLIANCES MFG CO LT...,GAS WATER HEATER,-,-,...,2024-06-21,2024-11-28,2025-12-23,BR102024024922,Pending,Brazil,AU2024227730A1 | AU2024227730B2 | BR1020240249...,4,CN,"$ 13,000"
4,942,ESWH,BR102024024922A2,GAS WATER HEATER--Gaseous fuel burner-Existing...,Applications,MIDEA,WUHU MIDEA KITCHEN & BATH APPLIANCES MFG CO LT...,GAS WATER HEATER,-,-,...,2024-06-21,2024-11-28,2025-12-23,BR102024024922,Pending,Brazil,AU2024227730A1 | AU2024227730B2 | BR1020240249...,4,CN,"$ 13,000"


Controllo i duplicati

In [ ]:
duplicates = df_2025_raw[df_2025_raw.duplicated(subset=["Publication Number"], keep=False)]
print(duplicates)

      Number source pltf Publication Number  \
3        399        EIWH   BR102024024922A2   
4        942        ESWH   BR102024024922A2   
5       1251        HPWH   BR102024024922A2   
6        400        EIWH   BR102024027003A2   
7        943        ESWH   BR102024027003A2   
...      ...         ...                ...   
9327    1259        HPWH     WO2025263172A1   
9328     398        GAHP     WO2025263247A1   
9329    4481         HHP     WO2025263247A1   
9333     397        GAHP     WO2025263492A1   
9334    4480         HHP     WO2025263492A1   

                                            text merger   Patent Type  \
3     GAS WATER HEATER--Gaseous fuel burner-Existing...  Applications   
4     GAS WATER HEATER--Gaseous fuel burner-Existing...  Applications   
5     GAS WATER HEATER--Gaseous fuel burner-Existing...  Applications   
6     Housing for combustion chamber and gas water h...  Applications   
7     Housing for combustion chamber and gas water h...  Applications 

In [ ]:
num_duplicates = df_2025_raw.duplicated(subset=["Publication Number"]).sum()
print("Numero di duplicati:", num_duplicates)

Numero di duplicati: 2613


In [ ]:
key_col = "Publication Number"
label_col = "source pltf"

duplicates = df_2025_raw[df_2025_raw.duplicated(subset=[key_col], keep=False)]

summary = duplicates.groupby(key_col)[label_col].nunique()

print("Duplicati con stessa label:", (summary == 1).sum())
print("Duplicati con label diverse:", (summary > 1).sum())

Duplicati con stessa label: 0
Duplicati con label diverse: 1593


Rinomina le colonne del 2025 per renderle compatibili

In [ ]:
rename_2025_to_2024 = {
    "PLATFORM": "Manually assigned PLATFORM",
    "source pltf": "Manually assigned PLATFORM",
    "Piattaforma di Estrazione/ Destinazione": "Manually assigned PLATFORM",

    "PUBLICATION NUMBER": "Publication Number",
    "Pubblication Number": "Publication Number",

    "TITLE": "Title (Translated)(English)",
    "ABSTRACT": "Abstract (Translated)(English)",
    "INDEPENDENT CLAIMS": "Independent Claims (Translated)(English)",

    "PATENT TYPE": "Patent Type",
    "Technical Benefit Phrase": "Std. Technical Benefit Phrase",
    "Technical Field": "Technical Field",
    "Technical Problem": "Technical Problem",
}

In [ ]:
df_2025_raw = df_2025_raw.rename(columns=rename_2025_to_2024).copy()

In [ ]:
print(df_2025_raw.columns.tolist())

['Number', 'Manually assigned PLATFORM', 'Publication Number', 'text merger', 'Patent Type', 'Grouped Assignee', 'Standardized Original Assignee(Applicant)', 'Title (Translated)(English)', 'Abstract (Translated)(English)', 'Independent Claims (Translated)(English)', 'IPC', 'Application domain', 'Technology topic', 'Technical Problem', 'Std. Technical Problem Phrase', 'Earliest Priority Date', 'Application Date', 'Publication Date', 'Application Number', 'Simple Legal Status', 'Authority', 'INPADOC Family', 'INPADOC Family Application Count', 'Priority Country/Region', 'Patent Valuation']


Crea eventuali colonne mancanti nel 2025

In [ ]:
for col in selected_columns:
    if col not in df_2025_raw.columns:
        df_2025_raw[col] = ""

Pulizia di Publication Number e Label

In [ ]:
df_2025_raw["Publication Number"] = (
    df_2025_raw["Publication Number"]
    .astype(str)
    .str.strip()
    .str.upper()
)

df_2025_raw["Manually assigned PLATFORM"] = (
    df_2025_raw["Manually assigned PLATFORM"]
    .apply(normalize_platform_label)
)

In [ ]:
df_2025_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9335 entries, 0 to 9334
Data columns (total 28 columns):
 #   Column                                     Non-Null Count  Dtype 
---  ------                                     --------------  ----- 
 0   Number                                     9335 non-null   int64 
 1   Manually assigned PLATFORM                 9335 non-null   object
 2   Publication Number                         9335 non-null   object
 3   text merger                                9335 non-null   object
 4   Patent Type                                9335 non-null   object
 5   Grouped Assignee                           9335 non-null   object
 6   Standardized Original Assignee(Applicant)  9335 non-null   object
 7   Title (Translated)(English)                9335 non-null   object
 8   Abstract (Translated)(English)             9335 non-null   object
 9   Independent Claims (Translated)(English)   9335 non-null   object
 10  IPC                                 

In [ ]:
df_2025_raw = df_2025_raw.reset_index(drop=True)
df_2025_raw = df_2025_raw[
    df_2025_raw["Publication Number"].notna() &
    (df_2025_raw["Publication Number"] != "") &
    (df_2025_raw["Publication Number"] != "NAN") &
    df_2025_raw["Manually assigned PLATFORM"].notna()
].copy()

In [ ]:
df_2025_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9335 entries, 0 to 9334
Data columns (total 28 columns):
 #   Column                                     Non-Null Count  Dtype 
---  ------                                     --------------  ----- 
 0   Number                                     9335 non-null   int64 
 1   Manually assigned PLATFORM                 9335 non-null   object
 2   Publication Number                         9335 non-null   object
 3   text merger                                9335 non-null   object
 4   Patent Type                                9335 non-null   object
 5   Grouped Assignee                           9335 non-null   object
 6   Standardized Original Assignee(Applicant)  9335 non-null   object
 7   Title (Translated)(English)                9335 non-null   object
 8   Abstract (Translated)(English)             9335 non-null   object
 9   Independent Claims (Translated)(English)   9335 non-null   object
 10  IPC                                 

Rimuovi duplicati perfetti nel 2025

In [ ]:
df_2025_raw = df_2025_raw.drop_duplicates(
    subset=["Publication Number", "Manually assigned PLATFORM"]
).copy()

In [ ]:
df_2025_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9335 entries, 0 to 9334
Data columns (total 28 columns):
 #   Column                                     Non-Null Count  Dtype 
---  ------                                     --------------  ----- 
 0   Number                                     9335 non-null   int64 
 1   Manually assigned PLATFORM                 9335 non-null   object
 2   Publication Number                         9335 non-null   object
 3   text merger                                9335 non-null   object
 4   Patent Type                                9335 non-null   object
 5   Grouped Assignee                           9335 non-null   object
 6   Standardized Original Assignee(Applicant)  9335 non-null   object
 7   Title (Translated)(English)                9335 non-null   object
 8   Abstract (Translated)(English)             9335 non-null   object
 9   Independent Claims (Translated)(English)   9335 non-null   object
 10  IPC                                 

Aggrega il 2025 in formato multi-label

In [ ]:
agg_map_2025 = {
    "Manually assigned PLATFORM": combine_labels,
    "Title (Translated)(English)": first_non_empty,
    "Abstract (Translated)(English)": first_non_empty,
    "Independent Claims (Translated)(English)": first_non_empty,
    "IPC": first_non_empty,
    "CPC": first_non_empty,
    "Application domain": first_non_empty,
    "Technology topic": first_non_empty,
    "Technical Problem": first_non_empty,
    "Technical Field": first_non_empty,
    "Std. Technical Benefit Phrase": first_non_empty,
}


In [ ]:
if df_2025_raw.empty:
    # Define the expected columns for the aggregated DataFrame
    output_columns = ["Publication Number"] + list(agg_map_2025.keys())
    df_2025_raw_agg = pd.DataFrame(columns=output_columns)
else:
    df_2025_raw_agg = (
        df_2025_raw[selected_columns]
        .groupby("Publication Number", as_index=False)
        .agg(agg_map_2025)
    )

In [ ]:
df_2025_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9335 entries, 0 to 9334
Data columns (total 28 columns):
 #   Column                                     Non-Null Count  Dtype 
---  ------                                     --------------  ----- 
 0   Number                                     9335 non-null   int64 
 1   Manually assigned PLATFORM                 9335 non-null   object
 2   Publication Number                         9335 non-null   object
 3   text merger                                9335 non-null   object
 4   Patent Type                                9335 non-null   object
 5   Grouped Assignee                           9335 non-null   object
 6   Standardized Original Assignee(Applicant)  9335 non-null   object
 7   Title (Translated)(English)                9335 non-null   object
 8   Abstract (Translated)(English)             9335 non-null   object
 9   Independent Claims (Translated)(English)   9335 non-null   object
 10  IPC                                 

In [ ]:
print("2025 raw cleaned shape:", df_2025_raw.shape)
print("2025 aggregated shape:", df_2025_raw_agg.shape)
df_2025_raw_agg.head()

2025 raw cleaned shape: (9335, 28)
2025 aggregated shape: (6722, 12)


,Publication Number,Manually assigned PLATFORM,Title (Translated)(English),Abstract (Translated)(English),Independent Claims (Translated)(English),IPC,CPC,Application domain,Technology topic,Technical Problem,Technical Field,Std. Technical Benefit Phrase
0,AU2024259712A1,GASBOILER,Process for ammonia synthesis using a reaction...,Process for ammonia synthesis using a reaction...,1. A process for ammonia synthesis with a reac...,C01C1/04 | B01D5/00 | B01J8/02 | B01J23/755 | ...,,Additive manufacturing apparatus|Vapor condens...,Ptru catalyst|Nitrogen gas|Reagent|Chemical en...,Current ammonia synthesis processes are ineffi...,,
1,AU2025201295A1,HHP,Air conditioner,An air conditioner includes a rotary shaft mem...,[Claim 1] An air conditioner including:_...,F24F1/028 | F24F13/14 | F24F13/20,,Lighting and heating apparatus|Heating and ven...,Rotational axis|Air conditioning|Physics,Existing air conditioners suffer from air leak...,,
2,BG67821B1,ESWH,Vertical boiler with bottom water supply,The vertical boiler with bottom water supply a...,-,F28D20/00 | F24H1/18 | F24H1/20 | F24H9/13,,Mechanical apparatus|Heat storage plants|Water...,Water storage|Mechanical engineering|Water pip...,-,,
3,BR102024024922A2,EIWH ESWH HPWH,GAS WATER HEATER,-,-,F23D14/10,,Gaseous fuel burner,-,Existing gas water heaters face the issue of e...,,
4,BR102024027003A2,EIWH ESWH GIWH GWSH HPWH,Housing for combustion chamber and gas water h...,-,-,F24H1/10,,Water heaters,Thermodynamics|Combustion chamber|Mechanical e...,Existing gas water heaters face issues with hi...,,


## Trattiamo il dataset 2025_ATG_SORV_pubbl.xlsx

In [ ]:
uploaded = files.upload()

Saving 2025_ATG_pubbl.xlsx to 2025_ATG_pubbl.xlsx


In [ ]:
df_2025_pubbl = pd.read_excel("2025_ATG_pubbl.xlsx")

In [ ]:
df_2025_pubbl.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5903 entries, 0 to 5902
Data columns (total 48 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   #                             5903 non-null   int64  
 1   PLATFORM                      5902 non-null   object 
 2   ESWH                          5903 non-null   int64  
 3   GSWH                          5903 non-null   int64  
 4   EIWH                          5903 non-null   int64  
 5   GIWH                          5903 non-null   int64  
 6   GAS BOILER                    5903 non-null   int64  
 7   HPWH                          5903 non-null   int64  
 8   HHP                           5903 non-null   int64  
 9   GAHP                          5903 non-null   int64  
 10  TAGS                          5903 non-null   object 
 11  MAIN TAG (Gemini)             16 non-null     object 
 12  SECONDARY TAGs (Gemini)       16 non-null     object 
 13  Key

In [ ]:
df_2025_pubbl = df_2025_pubbl.rename(columns=rename_2025_to_2024).copy()

for col in selected_columns:
    if col not in df_2025_pubbl.columns:
        df_2025_pubbl[col] = ""

df_2025_pubbl["Publication Number"] = (
    df_2025_pubbl["Publication Number"]
    .astype(str)
    .str.strip()
    .str.upper()
)

df_2025_pubbl.info()

df_2025_pubbl["Manually assigned PLATFORM"] = (
    df_2025_pubbl["Manually assigned PLATFORM"]
    .apply(normalize_platform_label)
)

df_2025_pubbl.info()

df_2025_pubbl = df_2025_pubbl.reset_index(drop=True)
df_2025_pubbl = df_2025_pubbl[
    df_2025_pubbl["Publication Number"].notna() &
    (df_2025_pubbl["Publication Number"] != "") &
    (df_2025_pubbl["Publication Number"] != "NAN") &
    df_2025_pubbl["Manually assigned PLATFORM"].notna()
].copy()


df_2025_pubbl.info()

df_2025_pubbl = df_2025_pubbl.drop_duplicates(
    subset=["Publication Number", "Manually assigned PLATFORM"]
).copy()


df_2025_pubbl.info()

if df_2025_pubbl.empty:
    output_columns = ["Publication Number"] + list(agg_map_2025.keys())
    df_2025_pubbl_agg = pd.DataFrame(columns=output_columns)
else:
    df_2025_pubbl_agg = (
        df_2025_pubbl[selected_columns]
        .groupby("Publication Number", as_index=False)
        .agg(agg_map_2025)
    )

df_2025_pubbl.info()

print("2025 pubbl cleaned shape:", df_2025_pubbl.shape)
print("2025 pubbl aggregated shape:", df_2025_pubbl_agg.shape)
df_2025_pubbl_agg.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5903 entries, 0 to 5902
Data columns (total 50 columns):
 #   Column                                    Non-Null Count  Dtype  
---  ------                                    --------------  -----  
 0   #                                         5903 non-null   int64  
 1   Manually assigned PLATFORM                5902 non-null   object 
 2   ESWH                                      5903 non-null   int64  
 3   GSWH                                      5903 non-null   int64  
 4   EIWH                                      5903 non-null   int64  
 5   GIWH                                      5903 non-null   int64  
 6   GAS BOILER                                5903 non-null   int64  
 7   HPWH                                      5903 non-null   int64  
 8   HHP                                       5903 non-null   int64  
 9   GAHP                                      5903 non-null   int64  
 10  TAGS                                

,Publication Number,Manually assigned PLATFORM,Title (Translated)(English),Abstract (Translated)(English),Independent Claims (Translated)(English),IPC,CPC,Application domain,Technology topic,Technical Problem,Technical Field,Std. Technical Benefit Phrase
0,AU2025201295A1,HHP,Air conditioner,An air conditioner includes a rotary shaft mem...,[Claim 1] An air conditioner including:_...,F24F1/028 | F24F13/14 | F24F13/20,,Lighting and heating apparatus|Heating and ven...,Rotational axis|Air conditioning|Physics,Existing air conditioners suffer from air leak...,,minimize condensation|Reduce air leakage|Guara...
1,BG67821B1,ESWH,Vertical boiler with bottom water supply,The vertical boiler with bottom water supply a...,-,F28D20/00 | F24H1/18 | F24H1/20 | F24H9/13,,Mechanical apparatus|Heat storage plants|Water...,Water storage|Mechanical engineering|Water pip...,-,,-
2,BR102024027003A2,GSWH,Housing for combustion chamber and gas water h...,-,-,F24H1/10,,Water heaters,Thermodynamics|Combustion chamber|Mechanical e...,Existing gas water heaters face issues with hi...,,avoid high water temperature|reduce surface te...
3,CA3243316A1,HHP,"Connection component, housing assembly, and ai...","Disclosed are a connection component, a housin...",1. A connection component for an air handling ...,F24F13/00,,Lighting and heating apparatus|Space heating a...,Air treatment|Mechanical engineering|Physics,Existing connection components for air handlin...,,Easy to assemble|Improve sealing efficiency|re...
4,CA3256449A1,GSWH,Gas exhausting device and gas water heater,-,-,F24H1/00 | F01N1/24 | F16F15/04 | F23J11/00 | ...,,-,-,-,,-


## Fondiamo i due dataset df_2025_pubbl e df_2025_neural

In [ ]:
df_2025_pubbl_renamed = df_2025_pubbl_agg.copy()

# Identifica i brevetti unici in df_2025_raw_agg che non sono in df_2025_pubbl_renamed
unique_in_raw = df_2025_raw_agg[~df_2025_raw_agg['Publication Number'].isin(df_2025_pubbl_renamed['Publication Number'])]

# Assegna la label 'No Relevant' a questi brevetti, unendo 'NO' e 'RELEVANT' in 'NOT_RELEVANT'
unique_in_raw_labelled = unique_in_raw.copy()
unique_in_raw_labelled['Manually assigned PLATFORM'] = 'NOT_RELEVANT'

# Concatena i due dataset (prima df_2025_pubbl_renamed per mantenere l'ordine come suggerito dall'operazione iniziale invertita)
df_combined_2025 = pd.concat([df_2025_pubbl_renamed, unique_in_raw_labelled], ignore_index=True)

# Start of added modification
# Define a local normalization function for the combined labels
def local_normalize_combined_platform_label(label_string):
    if pd.isna(label_string):
        return None

    # Process the string for general cleaning first
    temp_label_string = str(label_string).upper()
    temp_label_string = re.sub(r"GAS\s+BOILER", "GASBOILER", temp_label_string)
    temp_label_string = temp_label_string.replace("/", " ")
    temp_label_string = re.sub(r"\s+", " ", temp_label_string).strip()

    # Split into individual parts and normalize each
    parts = temp_label_string.split(' ')
    normalized_parts = []
    for part in parts:
        if part == "NON" or part == "RELEVANT":
            if "NOT_RELEVANT" not in normalized_parts: # Ensure 'NOT_RELEVANT' is only added once
                normalized_parts.append("NOT_RELEVANT")
        elif part != "": # Only add if not empty after processing
            if part not in normalized_parts: # Ensure uniqueness within this entry
                normalized_parts.append(part)

    return " ".join(normalized_parts) if normalized_parts else None

# Apply the local normalization to the combined DataFrame
df_combined_2025['Manually assigned PLATFORM'] = df_combined_2025['Manually assigned PLATFORM'].apply(local_normalize_combined_platform_label)
# End of added modification


print(f"Shape di df_2025_raw aggregato: {df_2025_raw_agg.shape}")
print(f"Shape di df_2025_pubbl aggregato: {df_2025_pubbl_agg.shape}")
print(f"Numero di brevetti unici da df_2025_raw aggiunti: {len(unique_in_raw_labelled)}")
print(f"Shape del dataset combinato: {df_combined_2025.shape}")

df_combined_2025.head()

# Calculate samples per class for df_combined_2025
# Multi-label binarization
y_raw_combined_2025 = df_combined_2025['Manually assigned PLATFORM']

# For MultiLabelBinarizer to correctly identify 'NOT_RELEVANT' as a single class,
# we can simply use the default split() as 'NOT_RELEVANT' is a single word.
y_split_combined_2025 = y_raw_combined_2025.apply(lambda x: x.split())

mlb_combined_2025 = MultiLabelBinarizer()
y_multi_combined_2025 = mlb_combined_2025.fit_transform(y_split_combined_2025)

num_classes_combined_2025 = len(mlb_combined_2025.classes_)

print(f"\nNumero di classi reali in df_combined_2025: {num_classes_combined_2025}")
print(f"Classi in df_combined_2025: {mlb_combined_2025.classes_}")

# Distribuzione delle classi
class_counts_combined_2025 = y_multi_combined_2025.sum(axis=0)

class_distribution_combined_2025 = pd.DataFrame({
    "Classe": mlb_combined_2025.classes_,
    "Numero di sample": class_counts_combined_2025
}).sort_values(by="Numero di sample", ascending=False)

print("\nDistribuzione delle classi in df_combined_2025:")
print(class_distribution_combined_2025)

Shape di df_2025_raw aggregato: (6722, 12)
Shape di df_2025_pubbl aggregato: (5902, 12)
Numero di brevetti unici da df_2025_raw aggiunti: 840
Shape del dataset combinato: (6742, 12)

Numero di classi reali in df_combined_2025: 9
Classi in df_combined_2025: ['EIWH' 'ESWH' 'GAHP' 'GASBOILER' 'GIWH' 'GSWH' 'HHP' 'HPWH'
 'NOT_RELEVANT']

Distribuzione delle classi in df_combined_2025:
         Classe  Numero di sample
6           HHP              4167
8  NOT_RELEVANT               842
7          HPWH               735
1          ESWH               600
5          GSWH               497
3     GASBOILER               277
0          EIWH                82
4          GIWH                75
2          GAHP                37


In [ ]:
df_combined_2025.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6742 entries, 0 to 6741
Data columns (total 12 columns):
 #   Column                                    Non-Null Count  Dtype 
---  ------                                    --------------  ----- 
 0   Publication Number                        6742 non-null   object
 1   Manually assigned PLATFORM                6742 non-null   object
 2   Title (Translated)(English)               6742 non-null   object
 3   Abstract (Translated)(English)            6742 non-null   object
 4   Independent Claims (Translated)(English)  6742 non-null   object
 5   IPC                                       6742 non-null   object
 6   CPC                                       6742 non-null   object
 7   Application domain                        6742 non-null   object
 8   Technology topic                          6742 non-null   object
 9   Technical Problem                         6742 non-null   object
 10  Technical Field                           6742 n

# DB_ONE_DEFINITIVO.xlsx

Importiamo più dati relativi agli anni ancora passati (DB ONE DEFINITIVO)

In [ ]:
uploaded = files.upload()

Saving DB_ONE_PREDICTED_UFFICIALE_checked_no TAC.xlsx to DB_ONE_PREDICTED_UFFICIALE_checked_no TAC.xlsx


In [ ]:
df_ONE = pd.read_excel("/content/DB_ONE_PREDICTED_UFFICIALE_checked_no TAC.xlsx")

In [ ]:
df_ONE.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26084 entries, 0 to 26083
Data columns (total 67 columns):
 #   Column                                     Non-Null Count  Dtype  
---  ------                                     --------------  -----  
 0   Piattaforma di Assegnazione                26084 non-null  object 
 1   ANNO                                       26082 non-null  object 
 2   Number                                     26082 non-null  object 
 3   Publication Number                         26084 non-null  object 
 4   Grouped Assignee                           26084 non-null  object 
 5   Patent Type                                7119 non-null   object 
 6   RELEVANT (Y/N)                             26084 non-null  object 
 7   Water heater                               26082 non-null  float64
 8   Water heating                              26082 non-null  float64
 9   Provenienza EWH/GWH                        26082 non-null  float64
 10  Provenienza EIWH      

Controllo innanzitutto se ci sono duplicati tra i 3 datasets excel (eventuali duplicati andrebbero eliminati) e duplicati interni nei 3 diversi datasets

In [ ]:
df_combined_2024 = df_combined_2024.rename(columns={
    "Publication Number": "Publication Number"
})

df_combined_2025 = df_combined_2025.rename(columns={
    "PUBLICATION NUMBER": "Publication Number"
})

df_ONE = df_ONE.rename(columns={
    "Publication Number": "Publication Number"
})


In [ ]:
for name, df in {
    "2024": df_combined_2024,
    "2025": df_combined_2025,
    "DB_ONE": df_ONE
}.items():
    print(name)
    print("Duplicati interni:", df["Publication Number"].duplicated().sum())
    print()


2024
Duplicati interni: 0

2025
Duplicati interni: 0

DB_ONE
Duplicati interni: 8906



Lavoriamo con la colonna RELEVANT (Y/N)

In [ ]:
print(f"Shape of df_DB_ONE before RELEVANT (Y/N) processing: {df_ONE.shape}")

# We will work on the already cleaned df_DB_ONE from the previous cell's processing
df_processing_relevance = df_ONE.copy()

# Identify Publication Numbers that have at least one 'Y'
pub_numbers_with_Y = df_processing_relevance[df_processing_relevance['RELEVANT (Y/N)'] == 'Y']['Publication Number'].unique()

# Separate patents into two groups based on the 'RELEVANT (Y/N)' logic:
# Group 1: Patents that have at least one 'Y' entry (keep all instances of these patents)
df_group1_relevant = df_processing_relevance[df_processing_relevance['Publication Number'].isin(pub_numbers_with_Y)].copy()

# Group 2: Patents that *only* have 'N' entries for a given Publication Number.
# This means patents whose 'Publication Number' are NOT in pub_numbers_with_Y
df_group2_N_only = df_processing_relevance[~df_processing_relevance['Publication Number'].isin(pub_numbers_with_Y)].copy()

# For Group 2, keep only one instance per Publication Number and set 'Piattaforma di Assegnazione' to 'NOT_RELEVANT'
# The 'RELEVANT (Y/N)' for these will inherently be 'N', but let's explicitly set it.
df_group2_processed = df_group2_N_only.drop_duplicates(subset=['Publication Number'], keep='first').copy()
df_group2_processed['Piattaforma di Assegnazione'] = 'NOT_RELEVANT'
df_group2_processed['RELEVANT (Y/N)'] = 'N'

# Concatenate the two processed groups. This intermediate DataFrame now reflects the RELEVANT (Y/N) logic.
df_intermediate_processed = pd.concat([df_group1_relevant, df_group2_processed], ignore_index=True)

print(f"Shape of intermediate DataFrame after RELEVANT (Y/N) logic: {df_intermediate_processed.shape}")

# Now, re-apply the multi-label aggregation structure based on selected_columns_db_one
# This ensures the final df_DB_ONE_aggregated has the correct columns and multi-labels

# Define the columns that should be selected for the final aggregation
selected_columns_db_one = [
    "Publication Number",
    "Piattaforma di Assegnazione",
    "ANNO",
    "Number",
    "Patent Type",
    "RELEVANT (Y/N)",
    "Title (Translated)(English)",
    "Abstract (Translated)(English)",
    "Independent Claims (Translated)(English)",
    "IPC",
    "CPC",
    "Application domain",
    "Technology topic",
]

# Define the aggregation map using globally available functions and selected_columns_db_one
agg_map_final_db_one = {
    "Piattaforma di Assegnazione": combine_labels,
    "ANNO": first_non_empty,
    "Number": first_non_empty,
    "Patent Type": first_non_empty,
    "RELEVANT (Y/N)": first_non_empty, # This column also needs aggregation
    "Title (Translated)(English)": first_non_empty,
    "Abstract (Translated)(English)": first_non_empty,
    "Independent Claims (Translated)(English)": first_non_empty,
    "IPC": first_non_empty,
    "CPC": first_non_empty,
    "Application domain": first_non_empty,
    "Technology topic": first_non_empty,
}

# Ensure all columns needed for aggregation exist in the intermediate DataFrame
for col in selected_columns_db_one:
    if col not in df_intermediate_processed.columns:
        df_intermediate_processed[col] = ""

# Perform the final aggregation using the defined map
df_DB_ONE_aggregated = (
    df_intermediate_processed[selected_columns_db_one]
    .groupby("Publication Number", as_index=False)
    .agg(agg_map_final_db_one)
)

print(f"Final shape of df_DB_ONE_aggregated after combined processing: {df_DB_ONE_aggregated.shape}")
print(df_DB_ONE_aggregated.head())

Shape of df_DB_ONE before RELEVANT (Y/N) processing: (26084, 67)
Shape of intermediate DataFrame after RELEVANT (Y/N) logic: (23340, 67)
Final shape of df_DB_ONE_aggregated after combined processing: (17178, 13)
  Publication Number Piattaforma di Assegnazione  ANNO Number   Patent Type  \
0          AT16972U1                NOT_RELEVANT  2021   1835                 
1         AT523245A3                NOT_RELEVANT  2021   1032  Applications   
2         AT526185A2                    HHP HPWH  2023    396                 
3         AT526194A1                    HHP HPWH  2023    395                 
4     AU2020205360A1                NOT_RELEVANT  2021    269  Applications   

  RELEVANT (Y/N)                        Title (Translated)(English)  \
0              N                                                  -   
1              N          Heizgerät, insbesondere Gasbrennwertgerät   
2              Y  Wanddurchführungseinheit und Wanddurchführungs...   
3              Y             

Ci sono molti duplicati interni in DB_ONE_DEFINITIVO.xlsx, ma con diverse labels, perciò andiamo a creare un datasets multilabels

In [ ]:
# Copia di lavoro per la raffinazione
df_DB_ONE_refined = df_DB_ONE_aggregated.copy()

# Normalizza Publication Number (già fatto, ma per robustezza)
df_DB_ONE_refined["Publication Number"] = (
    df_DB_ONE_refined["Publication Number"]
    .astype(str)
    .str.strip()
    .str.upper()
)

# Funzione di normalizzazione per stringhe multi-label già combinate
def normalize_combined_platform_label(label_string):
    if pd.isna(label_string):
        return None

    # Split the combined string into individual labels
    parts = str(label_string).upper().split(' ')
    normalized_parts = []
    for part in parts:
        part = re.sub(r"GAS\s+BOILER", "GASBOILER", part)
        part = part.replace("/", " ")
        part = re.sub(r"\s+", " ", part).strip()

        # Convert 'NON' and 'RELEVANT' to 'NOT_RELEVANT'
        if part == "NON" or part == "RELEVANT":
            if "NOT_RELEVANT" not in normalized_parts: # Ensure 'NOT_RELEVANT' is only added once
                normalized_parts.append("NOT_RELEVANT")
        elif part != "":
            if part not in normalized_parts: # Ensure uniqueness within this entry
                normalized_parts.append(part)

    return " ".join(normalized_parts) if normalized_parts else None

# Applica la normalizzazione alle label combinate
df_DB_ONE_refined["Piattaforma di Assegnazione"] = (
    df_DB_ONE_refined["Piattaforma di Assegnazione"]
    .apply(normalize_combined_platform_label)
)

# Rimuove righe senza Publication Number o senza label (dopo la normalizzazione)
df_DB_ONE_refined = df_DB_ONE_refined[
    (df_DB_ONE_refined["Publication Number"].notna()) &
    (df_DB_ONE_refined["Publication Number"] != "") &
    (df_DB_ONE_refined["Publication Number"] != "NAN") &
    (df_DB_ONE_refined["Piattaforma di Assegnazione"].notna()) &
    (df_DB_ONE_refined["Piattaforma di Assegnazione"] != "") # Assicurarsi che le stringhe vuote siano rimosse
].copy()

# Elimina duplicati perfetti che potrebbero essere emersi dalla normalizzazione
df_DB_ONE_refined = df_DB_ONE_refined.drop_duplicates(
    subset=["Publication Number", "Piattaforma di Assegnazione"]
).copy()

# Aggiorna df_DB_ONE_aggregated con i dati raffinati
df_DB_ONE_aggregated = df_DB_ONE_refined.copy()

print("Shape dopo la raffinazione:", df_DB_ONE_aggregated.shape)
df_DB_ONE_aggregated.head()


Shape dopo la raffinazione: (17178, 13)


,Publication Number,Piattaforma di Assegnazione,ANNO,Number,Patent Type,RELEVANT (Y/N),Title (Translated)(English),Abstract (Translated)(English),Independent Claims (Translated)(English),IPC,CPC,Application domain,Technology topic
0,AT16972U1,NOT_RELEVANT,2021,1835,,N,-,Method for changing the configuration of a pro...,1. Claims 1. A method for changing the configu...,G05B19/418 | G06Q10/06,G05B19/418 | G06Q10/06 | Y02P90/02,Resources|Total factory control|Programme tota...,-
1,AT523245A3,NOT_RELEVANT,2021,1032,Applications,N,"Heizgerät, insbesondere Gasbrennwertgerät","The invention relates to a heating device, in ...",-,F24H9/06 | F24H9/14 | F24H9/20,F24H9/06 | F24H9/148 | F24H9/2035 | F24H8/00 |...,Fluid heaters|Sustainable buildings,Thermodynamics|Process engineering|Exhaust pip...
2,AT526185A2,HHP HPWH,2023,396,,Y,Wanddurchführungseinheit und Wanddurchführungs...,The invention relates to a wall feed-through u...,-,F24F13/02,F24F13/0209 | F24F2007/0025 | F16L5/12 | F24F1...,Ducting arrangements|Lighting and heating appa...,Home appliance|Heat pump|Physics|Hose support
3,AT526194A1,HHP HPWH,2023,395,,Y,Wärmepumpensystem,The invention relates to a heat pump system (1...,-,F25B30/02 | F25B49/02,F25B30/02 | F25B49/025 | F25B49/005 | F25B30/0...,Energy recovery in ventilation and heating|Hea...,Thermodynamics|Control power|Control cell|Mech...
4,AU2020205360A1,NOT_RELEVANT,2021,269,Applications,N,Cap and water-heating device including the same,The present disclosure provides a cap and a wa...,Te A cap for covering a top side of a case of ...,F24H9/02,F23J13/04 | F24H1/0072 | F24H9/02 | F23J2213/2...,Water heaters|Chimneys,Heat resistance|Exhaust gas emissions|Mechanic...


In [ ]:
set_2024 = set(df_combined_2024["Publication Number"].dropna())
set_2025 = set(df_combined_2025["Publication Number"].dropna())
set_ONE = set(df_DB_ONE_aggregated["Publication Number"].dropna())

In [ ]:
duplicates_2024_2025 = set_2024 & set_2025
duplicates_2024_ONE = set_2024 & set_ONE
duplicates_2025_ONE = set_2025 & set_ONE

In [ ]:
print("Duplicati esterni 2024 vs 2025:", len(duplicates_2024_2025))
print("Duplicati esterni 2024 vs DB_ONE:", len(duplicates_2024_ONE))
print("Duplicati esterni 2025 vs DB_ONE:", len(duplicates_2025_ONE))


Duplicati esterni 2024 vs 2025: 0
Duplicati esterni 2024 vs DB_ONE: 0
Duplicati esterni 2025 vs DB_ONE: 0


In [ ]:
label_column = "Piattaforma di Assegnazione"

labels = df_DB_ONE_aggregated[label_column].dropna().unique()

print(labels)

['NOT_RELEVANT' 'HHP HPWH' 'EWH GIWH' 'HHP' 'GAS BOILER HHP' 'GIWH' 'EWH'
 'HPWH' 'GAS BOILER' 'EWH GAS BOILER GIWH' 'EIWH EWH HHP' 'EIWH EWH HPWH'
 'EIWH EWH' 'EWH HHP' 'EWH HHP HPWH' 'EIWH EWH GAS BOILER GIWH'
 'GAS BOILER GIWH' 'EWH HPWH' 'EIWH EWH GAS BOILER'
 'GAHP GAS BOILER HHP HPWH' 'GAHP HHP' 'EIWH EWH GIWH'
 'EIWH EWH GAS BOILER HHP HPWH' 'EIWH EWH HHP HPWH' 'EWH GIWH HHP HPWH'
 'EWH GIWH HHP' 'EWH GAS BOILER' 'EWH GIWH HPWH' 'EWH GAS BOILER HPWH'
 'EIWH GAS BOILER GIWH HHP' 'GAHP HPWH' 'EIWH EWH GAHP GIWH HPWH'
 'EIWH EWH GAS BOILER HPWH' 'GAHP GAS BOILER HHP' 'EIWH EWH GIWH HHP HPWH'
 'EWH GAS BOILER GIWH HPWH' 'EIWH EWH GIWH HPWH' 'GAHP HHP HPWH'
 'EWH GAS BOILER GIWH HHP HPWH' 'GAS BOILER HHP HPWH'
 'EIWH EWH GAHP GAS BOILER GIWH HHP' 'GAS BOILER GIWH HHP' 'EIWH'
 'EWH GAHP' 'GAS BOILER GIWH HPWH' 'GAS BOILER HPWH' 'EIWH GAS BOILER'
 'EIWH HHP' 'EIWH EWH GIWH HHP' 'EIWH HHP HPWH'
 'EWH GAHP GAS BOILER HHP HPWH' 'GAHP' 'EWH GAS BOILER HHP'
 'EIWH GAHP GAS BOILER HHP' 'EIWH

In [ ]:
label_counts = df_DB_ONE_aggregated[label_column].value_counts()

print(label_counts)

Piattaforma di Assegnazione
NOT_RELEVANT                          7854
HHP                                   3346
HHP HPWH                              1182
HPWH                                   777
EWH                                    554
                                      ... 
EIWH GAS BOILER HPWH                     1
EWH GAHP HPWH                            1
EIWH EWH GAHP GAS BOILER GIWH HPWH       1
EIWH EWH GAHP HHP HPWH                   1
EWH GAHP GIWH HHP HPWH                   1
Name: count, Length: 75, dtype: int64


In [ ]:
def rename_db_one_columns(df):
    rename_map = {
        "Piattaforma di Assegnazione": "Manually assigned PLATFORM",
        "Title (Translated)(English)": "Title (Translated)(English)",
        "Abstract (Translated)(English)": "Abstract (Translated)(English)",
        "Independent Claims (Translated)(English)": "Independent Claims (Translated)(English)",
        "IPC": "IPC",
        "CPC": "CPC",
        "Application domain": "Application domain",
        "Technology topic": "Technology topic",
        "Technical Problem": "Technical Problem",
        "Technical Field": "Technical Field",
        "Std. Technical Benefit Phrase": "Std. Technical Benefit Phrase",
    }

    selected_columns = [
        "Manually assigned PLATFORM",
        "Title (Translated)(English)",
        "RELEVANT (Y/N)",
        "Abstract (Translated)(English)",
        "Independent Claims (Translated)(English)",
        "IPC",
        "CPC",
        "Application domain",
        "Technology topic",
        "Technical Problem",
        "Technical Field",
        "Std. Technical Benefit Phrase"
    ]

    df = df.rename(columns=rename_map)

    for col in selected_columns:
        if col not in df.columns:
            df[col] = ""

    df_selected = df[selected_columns].copy()

    return df_selected

In [ ]:
df_ONE_renamed = rename_db_one_columns(df_DB_ONE_aggregated)

In [ ]:
print(df_ONE_renamed.shape)
print(df_ONE_renamed.columns)
df_ONE_renamed.head()

(17178, 12)
Index(['Manually assigned PLATFORM', 'Title (Translated)(English)',
       'RELEVANT (Y/N)', 'Abstract (Translated)(English)',
       'Independent Claims (Translated)(English)', 'IPC', 'CPC',
       'Application domain', 'Technology topic', 'Technical Problem',
       'Technical Field', 'Std. Technical Benefit Phrase'],
      dtype='object')


,Manually assigned PLATFORM,Title (Translated)(English),RELEVANT (Y/N),Abstract (Translated)(English),Independent Claims (Translated)(English),IPC,CPC,Application domain,Technology topic,Technical Problem,Technical Field,Std. Technical Benefit Phrase
0,NOT_RELEVANT,-,N,Method for changing the configuration of a pro...,1. Claims 1. A method for changing the configu...,G05B19/418 | G06Q10/06,G05B19/418 | G06Q10/06 | Y02P90/02,Resources|Total factory control|Programme tota...,-,,,
1,NOT_RELEVANT,"Heizgerät, insbesondere Gasbrennwertgerät",N,"The invention relates to a heating device, in ...",-,F24H9/06 | F24H9/14 | F24H9/20,F24H9/06 | F24H9/148 | F24H9/2035 | F24H8/00 |...,Fluid heaters|Sustainable buildings,Thermodynamics|Process engineering|Exhaust pip...,,,
2,HHP HPWH,Wanddurchführungseinheit und Wanddurchführungs...,Y,The invention relates to a wall feed-through u...,-,F24F13/02,F24F13/0209 | F24F2007/0025 | F16L5/12 | F24F1...,Ducting arrangements|Lighting and heating appa...,Home appliance|Heat pump|Physics|Hose support,,,
3,HHP HPWH,Wärmepumpensystem,Y,The invention relates to a heat pump system (1...,-,F25B30/02 | F25B49/02,F25B30/02 | F25B49/025 | F25B49/005 | F25B30/0...,Energy recovery in ventilation and heating|Hea...,Thermodynamics|Control power|Control cell|Mech...,,,
4,NOT_RELEVANT,Cap and water-heating device including the same,N,The present disclosure provides a cap and a wa...,Te A cap for covering a top side of a case of ...,F24H9/02,F23J13/04 | F24H1/0072 | F24H9/02 | F23J2213/2...,Water heaters|Chimneys,Heat resistance|Exhaust gas emissions|Mechanic...,,,


In [ ]:
df_ONE_renamed["Manually assigned PLATFORM"] = (
    df_ONE_renamed["Manually assigned PLATFORM"]
    .apply(normalize_platform_label)
)

Contare i Nan

In [ ]:
selected_columns = [
    "Manually assigned PLATFORM",
    "RELEVANT (Y/N)",
    "Title (Translated)(English)",
    "Abstract (Translated)(English)",
    "Independent Claims (Translated)(English)",
    "IPC",
    "CPC",
    "Application domain",
    "Technology topic",
    "Technical Problem",
    "Technical Field",
    "Std. Technical Benefit Phrase"
]

df_selected = df_ONE_renamed[selected_columns].copy()

summary = pd.DataFrame({
    "Column": df_selected.columns,
    "NaN count": df_selected.isna().sum().values,
    "Empty string count": [
        df_selected[col].apply(lambda x: isinstance(x, str) and x.strip() == "").sum()
        for col in df_selected.columns
    ]
})

print(summary)


                                      Column  NaN count  Empty string count
0                 Manually assigned PLATFORM          0                   0
1                             RELEVANT (Y/N)          0                   0
2                Title (Translated)(English)          0                   0
3             Abstract (Translated)(English)          0                   0
4   Independent Claims (Translated)(English)          0                   0
5                                        IPC          0                   0
6                                        CPC          0                   0
7                         Application domain          0                   0
8                           Technology topic          0                   0
9                          Technical Problem          0               17178
10                           Technical Field          0               17178
11             Std. Technical Benefit Phrase          0               17178


Riempiamo i Nan con la stringa vuota

In [ ]:
df_ONE_renamed = df_ONE_renamed.fillna("")

In [ ]:
print(df_ONE_renamed[selected_columns].isna().sum())

Manually assigned PLATFORM                  0
RELEVANT (Y/N)                              0
Title (Translated)(English)                 0
Abstract (Translated)(English)              0
Independent Claims (Translated)(English)    0
IPC                                         0
CPC                                         0
Application domain                          0
Technology topic                            0
Technical Problem                           0
Technical Field                             0
Std. Technical Benefit Phrase               0
dtype: int64


Sistemiamo ora il dataset 2025

In [ ]:
rename_2025_to_2024 = {
    "#": "Number",
    "PLATFORM": "Manually assigned PLATFORM",
    "PUBLICATION NUMBER": "Publication Number",
    "PATENT TYPE": "Patent Type",
    "TITLE": "Title (Translated)(English)",
    "ABSTRACT": "Abstract (Translated)(English)",
    "INDEPENDENT CLAIMS": "Independent Claims (Translated)(English)",
    "Original Applicant": "Standardized Original Assignee(Applicant)",
    "Legal Status": "Simple Legal Status",
    "Priority Country": "Priority Country/Region",
    "Technical Benefit Phrase": "Std. Technical Benefit Phrase",
}

Andiamo a lavorare con la colonna RELEVANT Y/N

In [ ]:
df_ONE_renamed.head(100)

,Manually assigned PLATFORM,Title (Translated)(English),RELEVANT (Y/N),Abstract (Translated)(English),Independent Claims (Translated)(English),IPC,CPC,Application domain,Technology topic,Technical Problem,Technical Field,Std. Technical Benefit Phrase
0,NOT_RELEVANT,-,N,Method for changing the configuration of a pro...,1. Claims 1. A method for changing the configu...,G05B19/418 | G06Q10/06,G05B19/418 | G06Q10/06 | Y02P90/02,Resources|Total factory control|Programme tota...,-,,,
1,NOT_RELEVANT,"Heizgerät, insbesondere Gasbrennwertgerät",N,"The invention relates to a heating device, in ...",-,F24H9/06 | F24H9/14 | F24H9/20,F24H9/06 | F24H9/148 | F24H9/2035 | F24H8/00 |...,Fluid heaters|Sustainable buildings,Thermodynamics|Process engineering|Exhaust pip...,,,
2,HHP HPWH,Wanddurchführungseinheit und Wanddurchführungs...,Y,The invention relates to a wall feed-through u...,-,F24F13/02,F24F13/0209 | F24F2007/0025 | F16L5/12 | F24F1...,Ducting arrangements|Lighting and heating appa...,Home appliance|Heat pump|Physics|Hose support,,,
3,HHP HPWH,Wärmepumpensystem,Y,The invention relates to a heat pump system (1...,-,F25B30/02 | F25B49/02,F25B30/02 | F25B49/025 | F25B49/005 | F25B30/0...,Energy recovery in ventilation and heating|Hea...,Thermodynamics|Control power|Control cell|Mech...,,,
4,NOT_RELEVANT,Cap and water-heating device including the same,N,The present disclosure provides a cap and a wa...,Te A cap for covering a top side of a case of ...,F24H9/02,F23J13/04 | F24H1/0072 | F24H9/02 | F23J2213/2...,Water heaters|Chimneys,Heat resistance|Exhaust gas emissions|Mechanic...,,,
...,...,...,...,...,...,...,...,...,...,...,...,...
95,HHP,一种空气调节设备的联动控制方法及系统,Y,The invention relates to a linkage control met...,1. A linkage control method for air-conditio...,F24F11/67 | F24F11/64 | F24F110/10 | F24F110/12,F24F11/64 | F24F11/67 | F24F2110/10 | F24F2110/12,Mechanical apparatus|Lighting and heating appa...,Outdoor temperature|Control theory|Indoor air ...,,,
96,NOT_RELEVANT,"Control method, cooking appliance, cooking sys...",N,"The invention discloses a control method, a co...","1. A control method, It is characterized i...",F24C3/12 | F24C7/08 | A47J27/00 | A47J36/00 | ...,A47J27/00 | A47J36/00 | A47J36/24 | F24C3/126 ...,Domestic stoves or ranges|Lighting and heating...,Control manner|Process engineering|Cooked food...,,,
97,HHP HPWH,热泵系统和空调器,Y,The invention discloses a heat pump system and...,1. A heat pump system for an air conditioner...,F25B13/00 | F25B41/20 | F25B39/00 | F28G13/00,F25B13/00 | F25B39/00 | F28G13/005 | F25B2313/...,Mechanical apparatus|Cleaning heat-transfer de...,Refrigerant|Mechanical engineering|Heat pump|C...,,,
98,NOT_RELEVANT,燃烧器的火盖、燃烧器和燃气灶具,N,The invention discloses a fire cover of a burn...,"1. A fire cover for a burner, It is charac...",F23D14/02 | F23D14/46 | F23D14/60 | F23D14/62 ...,F23D14/02 | F23D14/46 | F23D14/60 | F23D14/62 ...,Domestic stoves or ranges|Gaseous fuel burner|...,Gas cooking stove|Fuel gas|Cooker|Physics,,,


## Estrazione dei dati passati di GAS BOILER e HPWH

In [ ]:
target_classes = ['GASBOILER', 'HPWH']

def filter_and_update_labels(label_string):
    if pd.isna(label_string) or not label_string:
        return None

    labels = label_string.split()
    filtered_labels = [label for label in labels if label in target_classes]

    if not filtered_labels:
        return None

    return " ".join(sorted(list(set(filtered_labels))))

# Apply the function to the 'Manually assigned PLATFORM' column
df_ONE_renamed['Manually assigned PLATFORM'] = df_ONE_renamed['Manually assigned PLATFORM'].apply(filter_and_update_labels)

# Remove rows where 'Manually assigned PLATFORM' became None
df_ONE_renamed = df_ONE_renamed.dropna(subset=['Manually assigned PLATFORM']).reset_index(drop=True)

print("Filtered df_ONE_renamed info:")
df_ONE_renamed.info()
print("\nValue counts for 'Manually assigned PLATFORM' after filtering:")
print(df_ONE_renamed['Manually assigned PLATFORM'].value_counts())
df_ONE_renamed.head()

Filtered df_ONE_renamed info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3804 entries, 0 to 3803
Data columns (total 12 columns):
 #   Column                                    Non-Null Count  Dtype 
---  ------                                    --------------  ----- 
 0   Manually assigned PLATFORM                3804 non-null   object
 1   Title (Translated)(English)               3804 non-null   object
 2   RELEVANT (Y/N)                            3804 non-null   object
 3   Abstract (Translated)(English)            3804 non-null   object
 4   Independent Claims (Translated)(English)  3804 non-null   object
 5   IPC                                       3804 non-null   object
 6   CPC                                       3804 non-null   object
 7   Application domain                        3804 non-null   object
 8   Technology topic                          3804 non-null   object
 9   Technical Problem                         3804 non-null   object
 10  Technical Field   

,Manually assigned PLATFORM,Title (Translated)(English),RELEVANT (Y/N),Abstract (Translated)(English),Independent Claims (Translated)(English),IPC,CPC,Application domain,Technology topic,Technical Problem,Technical Field,Std. Technical Benefit Phrase
0,HPWH,Wanddurchführungseinheit und Wanddurchführungs...,Y,The invention relates to a wall feed-through u...,-,F24F13/02,F24F13/0209 | F24F2007/0025 | F16L5/12 | F24F1...,Ducting arrangements|Lighting and heating appa...,Home appliance|Heat pump|Physics|Hose support,,,
1,HPWH,Wärmepumpensystem,Y,The invention relates to a heat pump system (1...,-,F25B30/02 | F25B49/02,F25B30/02 | F25B49/025 | F25B49/005 | F25B30/0...,Energy recovery in ventilation and heating|Hea...,Thermodynamics|Control power|Control cell|Mech...,,,
2,HPWH,Heat storage system,Y,- 39 Abstract A heat storage system includes a...,Claims _x000D_\n1. A heat storage system compr...,F24H4/02,F24H4/04 | F24H9/2021 | F24D2200/123 | F24D19/...,Heating fuel|Central heating storage tanks|Cen...,Thermodynamics|Thermal energy storage system|R...,,,
3,GASBOILER,MÉTODO E SISTEMA DE GERENCIAMENTO DE CONFORTO ...,N,THERMAL COMFORT MANAGEMENT METHOD AND SYSTEM I...,-,F24F11/00 | G06K9/62,G05B15/02 | F24F11/89 | F24F11/30 | F24F2110/2...,Programme control|Mechanical apparatus|Lightin...,Thermal comfort|Process engineering|Air condit...,,,
4,HPWH,Retrofit heat pump water heating systems,Y,A retrofit heat pump system for a water heater...,CLAIMS What is claimed is: 1. A heat pump sys...,F24H4/02 | F24F12/00 | F24H9/14,Y02B30/12 | F24H4/02 | F24F12/00 | F24D2200/12...,Fluid heaters|Energy recovery in ventilation a...,Refrigerant distribution|Cold air|Air conditio...,,,


# Uniamo i datasets

In [ ]:
df = pd.concat(
    [df_combined_2024, df_combined_2025, df_ONE_renamed],
    axis=0,
    ignore_index=True
)

Una volta creato il dataset, non dobbiamo più generarlo

In [ ]:
df = pd.read_excel("/content/ATG_SORV_ALLYEARS_combined .xlsx")

In [ ]:
print("Combined shape:", df.shape)
df.head()

Combined shape: (16871, 13)


,Publication Number,Manually assigned PLATFORM,Title (Translated)(English),Abstract (Translated)(English),Independent Claims (Translated)(English),IPC,CPC,Application domain,Technology topic,Technical Problem,Technical Field,Std. Technical Benefit Phrase,RELEVANT (Y/N)
0,AT526453A2,HHP,Wall duct unit and external wall grille with a...,Wall feed-through unit (100) for a connection ...,-,F24F13/02 | F24F13/08,F24F13/0272 | F24F13/082 | F24F2007/0025 | F24...,Ducting arrangements|Mechanical apparatus|Ligh...,Home appliance|Mechanics|Waste gas|Heat pump|H...,-,-,-,NaN
1,AU2023203051B2,ESWH HPWH,Foaming method and device for water tank of wa...,The present invention relates to the technical...,1. A foaming method for a water tank of water ...,F24H1/18 | B29C44/18,B29C44/02 | B29C44/35,Water heaters,Foaming agent|Injection port|Heat conservation...,The existing foaming method for water tanks in...,-,-,NaN
2,AU2023206174A1,ESWH,Water heater sensor,Abstract A temperature sensing device 10 for a...,1. A water heater including:_x000D_\na water s...,F24H15/225 | F24H9/20 | F24H15/37 | G01K1/14 |...,F24H15/225 | F24H9/20 | G05D23/1931 | F24H15/3...,Thermometer details|Thermometers using electri...,Sensing data|Control cell|Process engineering|...,Water heaters with storage tanks experience te...,-,-,NaN
3,AU2024203072A1,ESWH,Program executed on mobile communication termi...,[Problem] To provide a technology for pairing ...,1. A program executed on a mobile communicatio...,H04Q9/00 | F24H1/00 | F24H15/45 | G05B19/02 | ...,H04W12/06 | H04W76/10 | H04W12/50,Telemetry/telecontrol selection arrangements|T...,Data matching|Telecommunications|Display devic...,In scenarios where multiple water heaters are ...,-,-,NaN
4,BR202022015721U2,GSWH,UNIFIED FILTERING DEVICE WITH OVER PRESSURE RE...,Unified filtering device with overpressure rel...,-,F24H9/00 | F24H9/20 | B01D35/02,F24H9/00 | F24H9/20 | F24H9/0042 | B01D35/02,Fluid heaters|Filtration separation,Environmental engineering|Patent application|F...,Existing gas water heaters have complex design...,-,-,NaN


Check finale

In [ ]:
print("Combined shape:", df.shape)
print("Missing labels:", df["Manually assigned PLATFORM"].isna().sum())
print("Empty labels:", (df["Manually assigned PLATFORM"].astype(str).str.strip() == "").sum())

Combined shape: (16871, 13)
Missing labels: 0
Empty labels: 0


In [ ]:
# 1. Identificare la colonna target
label_column = "Manually assigned PLATFORM"

# 2. Rimuovere righe senza label
df = df.dropna(subset=[label_column])

df = df[
    df[label_column].astype(str).str.strip() != ""
]

df = df.reset_index(drop=True)

In [ ]:
print("Combined shape:", df.shape)
print("Missing labels:", df["Manually assigned PLATFORM"].isna().sum())
print("Empty labels:", (df["Manually assigned PLATFORM"].astype(str).str.strip() == "").sum())


Combined shape: (16871, 13)
Missing labels: 0
Empty labels: 0


In [ ]:
df["Manually assigned PLATFORM"].value_counts()

,count
Manually assigned PLATFORM,
HHP,7300
HPWH,3351
NOT_RELEVANT,1903
GASBOILER,1473
ESWH,806
...,...
GASBOILER EIWH,1
ESWH EIWH GSWH GIWH HPWH,1
ESWH GSWH EIWH GIWH GASBOILER,1


Creiamo un file excel dei due file comibinati

In [ ]:
df.to_excel("ATG_SORV_ALLYEARS_combined.xlsx", index=False)
df.to_excel("ATG_SORV_ALLYEARS_combined.xlsx", index=False)
files.download("ATG_SORV_ALLYEARS_combined.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Creiamo un file excel dei due file comibinati senza la colonna delle label

In [ ]:
df_no_labels = df.drop(columns=["Manually assigned PLATFORM", "RELEVANT (Y/N)"], errors='ignore')
df_no_labels.to_excel("ATG_SORV_ALLYEARS_combined_no_labels.xlsx", index=False)
files.download("ATG_SORV_ALLYEARS_combined_no_labels.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Caricamento Combined DF (una volta che la fusione e l'ordinamento è stato fatto)

In [ ]:
uploaded = files.upload()

Saving TOTAL raw_no dupl 3270.xlsx to TOTAL raw_no dupl 3270.xlsx


In [ ]:
df = pd.read_excel("/content/TOTAL raw_no dupl 3270.xlsx")

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3270 entries, 0 to 3269
Data columns (total 25 columns):
 #   Column                                     Non-Null Count  Dtype         
---  ------                                     --------------  -----         
 0   source pltf                                3270 non-null   object        
 1   Publication Number                         3270 non-null   object        
 2   Patent Type                                3270 non-null   object        
 3   Grouped Assignee                           3270 non-null   object        
 4   Standardized Original Assignee(Applicant)  3270 non-null   object        
 5   Title (Translated)(English)                3270 non-null   object        
 6   Abstract (Translated)(English)             3270 non-null   object        
 7   Independent Claims (Translated)(English)   3270 non-null   object        
 8   IPC                                        3270 non-null   object        
 9   Application domain 

# Prepariamo il combinedDataset per l'apprendimento


In [ ]:
# Identificare la colonna target (label)
label_column = 'Manually assigned PLATFORM'
y = df[label_column]



# Identificare le colonne features.
# Per l'apprendimento della rete neurale. Tutte le altre colonne (incluse quelle di predizione
# e metadata) non saranno usate direttamente come input per questo modello testuale.

feature_columns = [
    "Title (Translated)(English)",
    "Abstract (Translated)(English)",
    "Independent Claims (Translated)(English)",
    "IPC",
    "CPC",
    "Application domain",
    "Technology topic",
    "Technical Problem",
    "Technical Field",
    "Std. Technical Benefit Phrase"
]

X = df[feature_columns].fillna("")


# Multi-label binarization
# Ogni brevetto può appartenere a più classi contemporaneamente.
# Esempio: "GASBOILER GSWH HPWH" -> [GASBOILER, GSWH, HPWH]

y_raw = df[label_column]

y_split = y_raw.apply(lambda x: x.split())

mlb = MultiLabelBinarizer()
y_multi = mlb.fit_transform(y_split)

num_classes = len(mlb.classes_)

print(f"Numero di classi reali: {num_classes}")
print(f"Classi: {mlb.classes_}")


# Distribuzione delle classi

class_counts = y_multi.sum(axis=0)

class_distribution = pd.DataFrame({
    "Classe": mlb.classes_,
    "Numero di sample": class_counts
}).sort_values(by="Numero di sample", ascending=False)

print(class_distribution)

Numero di classi reali: 9
Classi: ['EIWH' 'ESWH' 'GAHP' 'GASBOILER' 'GIWH' 'GSWH' 'GWSH' 'HHP' 'HPWH']
      Classe  Numero di sample
7        HHP              8029
8       HPWH              1832
1       ESWH              1517
3  GASBOILER              1205
4       GIWH               709
5       GSWH               528
0       EIWH               506
6       GWSH               473
2       GAHP               448


Controlliamo come è strutturato il dataset X

In [ ]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11986 entries, 0 to 11985
Data columns (total 10 columns):
 #   Column                                    Non-Null Count  Dtype 
---  ------                                    --------------  ----- 
 0   Title (Translated)(English)               11986 non-null  object
 1   Abstract (Translated)(English)            11986 non-null  object
 2   Independent Claims (Translated)(English)  11986 non-null  object
 3   IPC                                       11986 non-null  object
 4   CPC                                       11986 non-null  object
 5   Application domain                        11986 non-null  object
 6   Technology topic                          11986 non-null  object
 7   Technical Problem                         11986 non-null  object
 8   Technical Field                           11986 non-null  object
 9   Std. Technical Benefit Phrase             11986 non-null  object
dtypes: object(10)
memory usage: 936.5+ KB


Prepariamo il dataset X per l'apprendimento della rete neurale

In [ ]:
X = X.copy()

X["COMBINED_TEXT"] = (
    X["Title (Translated)(English)"].astype(str) + " " +
    X["Abstract (Translated)(English)"].astype(str) + " " +
    X["Independent Claims (Translated)(English)"].astype(str) + " " +
    X["IPC"].astype(str) + " " +
    X["CPC"].astype(str) + " " +
    X["Application domain"].astype(str) + " " +
    X["Technology topic"].astype(str) + " " +
    X["Technical Problem"].astype(str) + " " +
    X["Technical Field"].astype(str) + " " +
    X["Std. Technical Benefit Phrase"].astype(str)
)

X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11986 entries, 0 to 11985
Data columns (total 11 columns):
 #   Column                                    Non-Null Count  Dtype 
---  ------                                    --------------  ----- 
 0   Title (Translated)(English)               11986 non-null  object
 1   Abstract (Translated)(English)            11986 non-null  object
 2   Independent Claims (Translated)(English)  11986 non-null  object
 3   IPC                                       11986 non-null  object
 4   CPC                                       11986 non-null  object
 5   Application domain                        11986 non-null  object
 6   Technology topic                          11986 non-null  object
 7   Technical Problem                         11986 non-null  object
 8   Technical Field                           11986 non-null  object
 9   Std. Technical Benefit Phrase             11986 non-null  object
 10  COMBINED_TEXT                             1198

In [ ]:
x = X['COMBINED_TEXT']
x.info()

<class 'pandas.core.series.Series'>
RangeIndex: 11986 entries, 0 to 11985
Series name: COMBINED_TEXT
Non-Null Count  Dtype 
--------------  ----- 
11986 non-null  object
dtypes: object(1)
memory usage: 93.8+ KB


In [ ]:
x.head()

,COMBINED_TEXT
0,Wall duct unit and external wall grille with a...
1,Foaming method and device for water tank of wa...
2,Water heater sensor Abstract A temperature sen...
3,Program executed on mobile communication termi...
4,UNIFIED FILTERING DEVICE WITH OVER PRESSURE RE...


# Let's code the Transformer and the Vectorizer

Shuffle the dataset

In [ ]:
# Shuffle iniziale
x_shuffled, y_shuffled = shuffle(x[:len(y_multi)], y_multi, random_state=42)

# Primo split: train+val / test
X_temp, X_test_text, Y_temp, Y_test = train_test_split(
    x_shuffled,
    y_shuffled,
    test_size=0.15,
    random_state=42
)

# Secondo split: train / validation
X_train_text, X_val_text, Y_train, Y_val = train_test_split(
    X_temp,
    Y_temp,
    test_size=0.15,
    random_state=42
)

print("Train size:", len(X_train_text))
print("Validation size:", len(X_val_text))
print("Test size:", len(X_test_text))

Train size: 8659
Validation size: 1529
Test size: 1798


Let's check the numbers of the samples!!

In [ ]:
print("Numero di campioni di addestramento:", len(X_train_text))
print("Numero di campioni di validazione:", len(X_val_text))
print("Numero di campioni di test:", len(X_test_text))
print("Numero di classi:", num_classes)

print("\nNumero di campioni per classe di addestramento:")
print(np.sum(Y_train, axis=0))

print("\nNumero di campioni per classe di validazione:")
print(np.sum(Y_val, axis=0))

print("\nNumero di campioni per classe di test:")
print(np.sum(Y_test, axis=0))

print("\nClassi:")
print(mlb.classes_)


Numero di campioni di addestramento: 8659
Numero di campioni di validazione: 1529
Numero di campioni di test: 1798
Numero di classi: 9

Numero di campioni per classe di addestramento:
[ 363 1100  312  883  517  374  351 5815 1307]

Numero di campioni per classe di validazione:
[ 66 213  69 153  75  73  48 990 259]

Numero di campioni per classe di test:
[  77  204   67  169  117   81   74 1224  266]

Classi:
['EIWH' 'ESWH' 'GAHP' 'GASBOILER' 'GIWH' 'GSWH' 'GWSH' 'HHP' 'HPWH']


Define the vectorizer

In [ ]:
tfidf = TfidfVectorizer(
    sublinear_tf=True,
    max_features=25000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    stop_words='english'
)


def clean_text(text):
    text = str(text).upper()
    text = re.sub(r'[^A-Z0-9\s/.\-]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Pulizia testi
X_train_text_clean = [clean_text(text) for text in X_train_text]
X_val_text_clean   = [clean_text(text) for text in X_val_text]
X_test_text_clean  = [clean_text(text) for text in X_test_text]

# TF-IDF: fit solo su train
X_train_tfidf = tfidf.fit_transform(X_train_text_clean)
X_val_tfidf   = tfidf.transform(X_val_text_clean)
X_test_tfidf  = tfidf.transform(X_test_text_clean)

# Salvataggio vettorizzatore
with open("ORM_tfidf_vectorizer.pkl", "wb") as file:
    pickle.dump(tfidf, file)

print("Vettorizzatore TF-IDF salvato con successo!")

# Conversione in array densi per JAX
X_train = jnp.array(X_train_tfidf.toarray(), dtype=jnp.float32)
X_val   = jnp.array(X_val_tfidf.toarray(), dtype=jnp.float32)
X_test  = jnp.array(X_test_tfidf.toarray(), dtype=jnp.float32)

Y_train = jnp.array(Y_train, dtype=jnp.float32)
Y_val   = jnp.array(Y_val, dtype=jnp.float32)
Y_test  = jnp.array(Y_test, dtype=jnp.float32)

print("\nShape finali:")
print("X_train:", X_train.shape)
print("X_val:  ", X_val.shape)
print("X_test: ", X_test.shape)
print("Y_train:", Y_train.shape)
print("Y_val:  ", Y_val.shape)
print("Y_test: ", Y_test.shape)


Vettorizzatore TF-IDF salvato con successo!

Shape finali:
X_train: (8659, 25000)
X_val:   (1529, 25000)
X_test:  (1798, 25000)
Y_train: (8659, 9)
Y_val:   (1529, 9)
Y_test:  (1798, 9)


Numero feature TF-IDF effettive

In [ ]:
print("Numero effettivo di feature TF-IDF:", len(tfidf.get_feature_names_out()))

Numero effettivo di feature TF-IDF: 25000


#ORM's Customized ANN Function Definition


## Let's check the size !!

In [ ]:
X_train.shape

NameError: name 'X_train' is not defined

In [ ]:
X_test.shape

In [ ]:
X_val.shape

In [ ]:
Y_train.shape

In [ ]:
Y_test.shape

In [ ]:
Y_val.shape

In [ ]:
X_train = jnp.asarray(X_train, dtype=jnp.float32)
X_test = jnp.asarray(X_test, dtype=jnp.float32)
X_val = jnp.asarray(X_val, dtype=jnp.float32)
Y_train = jnp.asarray(Y_train, dtype=jnp.float32)
Y_test = jnp.asarray(Y_test, dtype=jnp.float32)
Y_val = jnp.asarray(Y_val, dtype=jnp.float32)

print(X_train.shape)
print(X_test.shape)
print(X_val.shape)
print(Y_train.shape)
print(Y_test.shape)
print(Y_val.shape)

## Define the main function of the ANN, the training and testing phases

In [ ]:
def oversample_minority_multilabel(X, Y, min_count=300):
    """
    Oversampling semplice:
    replica i campioni che contengono classi con supporto < min_count.
    """
    class_counts = np.sum(np.array(Y), axis=0)

    rare_classes = np.where(class_counts < min_count)[0]
    print("Rare classes idx:", rare_classes)
    print("Rare classes counts:", class_counts[rare_classes])

    X_np = np.array(X)
    Y_np = np.array(Y)

    extra_X = []
    extra_Y = []

    for class_idx in rare_classes:
        sample_idxs = np.where(Y_np[:, class_idx] == 1)[0]

        if len(sample_idxs) == 0:
            continue

        n_to_add = min_count - len(sample_idxs)

        chosen = np.random.choice(sample_idxs, size=n_to_add, replace=True)

        extra_X.append(X_np[chosen])
        extra_Y.append(Y_np[chosen])

    if len(extra_X) > 0:
        X_aug = np.concatenate([X_np] + extra_X, axis=0)
        Y_aug = np.concatenate([Y_np] + extra_Y, axis=0)
    else:
        X_aug = X_np
        Y_aug = Y_np

    idx = np.random.permutation(len(X_aug))
    X_aug = X_aug[idx]
    Y_aug = Y_aug[idx]

    return jnp.array(X_aug, dtype=jnp.float32), jnp.array(Y_aug, dtype=jnp.float32)

In [ ]:
def initialize_params(layers_size, key):
    params = []
    keys = jax.random.split(key, len(layers_size) - 1)

    for i in range(1, len(layers_size)):
        fan_in = layers_size[i - 1]
        fan_out = layers_size[i]

        std = jnp.sqrt(2.0 / (fan_in + fan_out))

        W = jax.random.normal(keys[i - 1], shape=(fan_in, fan_out)) * std
        b = jnp.zeros((1, fan_out))

        params.append((W, b))

    return params


In [ ]:
def ANN_logits(x, params):
    layer = x

    for i, (W, b) in enumerate(params):
        layer = jnp.dot(layer, W) + b

        if i < len(params) - 1:
            layer = jnp.tanh(layer)

    return layer


def ANN(x, params):
    logits = ANN_logits(x, params)
    return jax.nn.sigmoid(logits)

In [ ]:
def compute_pos_weights(y, max_weight=20.0):
    positives = jnp.sum(y, axis=0)
    negatives = y.shape[0] - positives

    pos_weights = negatives / (positives + 1e-8)
    pos_weights = jnp.clip(pos_weights, 1.0, max_weight)

    return pos_weights


In [ ]:
def focal_loss_logits(x, y, params, alpha=0.75, gamma=2.0):
    logits = ANN_logits(x, params)

    prob = jax.nn.sigmoid(logits)

    ce_positive = y * jax.nn.softplus(-logits)
    ce_negative = (1.0 - y) * jax.nn.softplus(logits)

    focal_positive = alpha * y * ((1.0 - prob) ** gamma) * ce_positive
    focal_negative = (1.0 - alpha) * (1.0 - y) * (prob ** gamma) * ce_negative

    return jnp.mean(focal_positive + focal_negative)


In [ ]:
def weighted_binary_cross_entropy_logits(x, y, params, pos_weights):
    logits = ANN_logits(x, params)

    loss_positive = pos_weights * y * jax.nn.softplus(-logits)
    loss_negative = (1.0 - y) * jax.nn.softplus(logits)

    return jnp.mean(loss_positive + loss_negative)


def loss_function(params, x, y, pos_weights):
    return weighted_binary_cross_entropy_logits(x, y, params, pos_weights)


In [ ]:
def clip_gradients(grads, max_norm=5.0):
    total_norm = 0.0

    for dW, db in grads:
        total_norm += jnp.sum(dW ** 2) + jnp.sum(db ** 2)

    total_norm = jnp.sqrt(total_norm)

    scale = jnp.minimum(1.0, max_norm / (total_norm + 1e-8))

    clipped_grads = []

    for dW, db in grads:
        clipped_grads.append((dW * scale, db * scale))

    return clipped_grads


In [ ]:
@jax.jit
def rmsprop_update(params, x_batch, y_batch, pos_weights, learning_rate, decay_rate, delta, cumulated_sqr_grad):
    loss_value, grads = jax.value_and_grad(loss_function)(
        params,
        x_batch,
        y_batch,
        pos_weights
    )

    grads = clip_gradients(grads, max_norm=5.0)

    new_params = []
    new_cumulated_sqr_grad = []

    for (W, b), (dW, db), (cW, cb) in zip(params, grads, cumulated_sqr_grad):
        cW = decay_rate * cW + (1.0 - decay_rate) * (dW ** 2)
        cb = decay_rate * cb + (1.0 - decay_rate) * (db ** 2)

        W = W - learning_rate * dW / (jnp.sqrt(cW) + delta)
        b = b - learning_rate * db / (jnp.sqrt(cb) + delta)

        new_params.append((W, b))
        new_cumulated_sqr_grad.append((cW, cb))

    return new_params, new_cumulated_sqr_grad, loss_value

Metrics and Loss functions

In [ ]:
def find_best_thresholds_per_label(y_true, y_prob, beta=2.0):
    num_labels = y_true.shape[1]
    best_thresholds = []

    for j in range(num_labels):
        y_true_j = y_true[:, j]
        y_prob_j = y_prob[:, j]

        best_t = 0.5
        best_score = -1.0

        for t in np.arange(0.10, 0.91, 0.05):
            y_pred_j = (y_prob_j >= t).astype(jnp.float32)

            TP = jnp.sum((y_pred_j == 1) & (y_true_j == 1))
            FP = jnp.sum((y_pred_j == 1) & (y_true_j == 0))
            FN = jnp.sum((y_pred_j == 0) & (y_true_j == 1))

            precision = TP / (TP + FP + 1e-8)
            recall = TP / (TP + FN + 1e-8)

            beta_sq = beta ** 2
            score = (1 + beta_sq) * precision * recall / (beta_sq * precision + recall + 1e-8)

            if score > best_score:
                best_score = float(score)
                best_t = float(t)

        best_thresholds.append(best_t)

    return jnp.array(best_thresholds, dtype=jnp.float32)


In [ ]:
def predict_with_thresholds(y_prob, thresholds_per_label):
    return (y_prob >= thresholds_per_label).astype(jnp.float32)


In [ ]:
def multilabel_fbeta_score_with_thresholds(y_true, y_prob, thresholds_per_label, beta=1.0):
    y_pred = predict_with_thresholds(y_prob, thresholds_per_label)

    TP = jnp.sum((y_pred == 1) & (y_true == 1))
    FP = jnp.sum((y_pred == 1) & (y_true == 0))
    FN = jnp.sum((y_pred == 0) & (y_true == 1))

    precision = TP / (TP + FP + 1e-8)
    recall = TP / (TP + FN + 1e-8)

    beta_sq = beta ** 2
    return (1 + beta_sq) * precision * recall / (beta_sq * precision + recall + 1e-8)


def multilabel_accuracy_with_thresholds(y_true, y_prob, thresholds_per_label):
    y_pred = predict_with_thresholds(y_prob, thresholds_per_label)
    return jnp.mean(y_pred == y_true)


In [ ]:
def weighted_binary_cross_entropy(x, y, params, pos_weights):
    y_pred = ANN(x, params)

    eps = 1e-8
    y_pred = jnp.clip(y_pred, eps, 1.0 - eps)

    loss_positive = -pos_weights * y * jnp.log(y_pred)
    loss_negative = -(1.0 - y) * jnp.log(1.0 - y_pred)

    loss = loss_positive + loss_negative

    return jnp.mean(loss)


In [ ]:
def multilabel_accuracy(x, y, params, threshold=0.5):
    y_pred = ANN(x, params)
    y_pred_labels = (y_pred >= threshold).astype(jnp.float32)

    return jnp.mean(y_pred_labels == y)


In [ ]:
def multilabel_accuracy_from_probs(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(jnp.float32)
    return jnp.mean(y_pred == y_true)


In [ ]:
def multilabel_precision(x, y, params, threshold=0.5):
    y_pred = ANN(x, params)
    y_pred_labels = (y_pred >= threshold).astype(jnp.float32)

    TP = jnp.sum((y_pred_labels == 1) & (y == 1))
    FP = jnp.sum((y_pred_labels == 1) & (y == 0))

    return TP / (TP + FP + 1e-8)


In [ ]:
def multilabel_recall(x, y, params, threshold=0.5):
    y_pred = ANN(x, params)
    y_pred_labels = (y_pred >= threshold).astype(jnp.float32)

    TP = jnp.sum((y_pred_labels == 1) & (y == 1))
    FN = jnp.sum((y_pred_labels == 0) & (y == 1))

    return TP / (TP + FN + 1e-8)


In [ ]:
def multilabel_f1_score(x, y, params, threshold=0.5):
    precision = multilabel_precision(x, y, params, threshold)
    recall = multilabel_recall(x, y, params, threshold)

    return 2.0 * precision * recall / (precision + recall + 1e-8)


In [ ]:
def multilabel_f1_score_from_probs(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(jnp.float32)

    TP = jnp.sum((y_pred == 1) & (y_true == 1))
    FP = jnp.sum((y_pred == 1) & (y_true == 0))
    FN = jnp.sum((y_pred == 0) & (y_true == 1))

    precision = TP / (TP + FP + 1e-8)
    recall = TP / (TP + FN + 1e-8)

    f1 = 2.0 * precision * recall / (precision + recall + 1e-8)

    return f1


In [ ]:
def multilabel_precision_with_thresholds(y_true, y_prob, thresholds_per_label):
    y_pred = predict_with_thresholds(y_prob, thresholds_per_label)

    TP = jnp.sum((y_pred == 1) & (y_true == 1))
    FP = jnp.sum((y_pred == 1) & (y_true == 0))

    return TP / (TP + FP + 1e-8)


def multilabel_recall_with_thresholds(y_true, y_prob, thresholds_per_label):
    y_pred = predict_with_thresholds(y_prob, thresholds_per_label)

    TP = jnp.sum((y_pred == 1) & (y_true == 1))
    FN = jnp.sum((y_pred == 0) & (y_true == 1))

    return TP / (TP + FN + 1e-8)


In [ ]:
def conf_matrix_multilabel(x, y, params, threshold=0.5):
    y_pred = ANN(x, params)
    y_pred_labels = (y_pred >= threshold).astype(jnp.float32)

    TP = jnp.sum((y_pred_labels == 1) & (y == 1))
    TN = jnp.sum((y_pred_labels == 0) & (y == 0))
    FP = jnp.sum((y_pred_labels == 1) & (y == 0))
    FN = jnp.sum((y_pred_labels == 0) & (y == 1))

    total = TP + TN + FP + FN

    print("CONFUSION MATRIX MULTI-LABEL:")
    print()
    print("                         True 1       True 0")
    print()
    print(f"Predicted 1              {TP} TP       {FP} FP")
    print("                         -------------------")
    print(f"Predicted 0              {FN} FN       {TN} TN")
    print()
    print(f"False Negatives = {(FN / total) * 100:.1f} %")
    print(f"False Positives = {(FP / total) * 100:.1f} %")


In [ ]:
def conf_matrix_per_label(x, y, params, threshold=0.5):
    y_prob = ANN(x, params)
    y_pred = (y_prob >= threshold).astype(jnp.int32)
    y_true = y.astype(jnp.int32)

    num_labels = y_true.shape[1]

    for label_idx in range(num_labels):
        true_label = y_true[:, label_idx]
        pred_label = y_pred[:, label_idx]

        TP = jnp.sum((pred_label == 1) & (true_label == 1))
        TN = jnp.sum((pred_label == 0) & (true_label == 0))
        FP = jnp.sum((pred_label == 1) & (true_label == 0))
        FN = jnp.sum((pred_label == 0) & (true_label == 1))

        precision = TP / (TP + FP + 1e-8)
        recall = TP / (TP + FN + 1e-8)
        f1 = 2.0 * precision * recall / (precision + recall + 1e-8)

        support = TP + FN

        print(f"Label {label_idx}")
        print(f"TP: {int(TP)}, TN: {int(TN)}, FP: {int(FP)}, FN: {int(FN)}")
        print(f"Support:   {int(support)}")
        print(f"Precision: {float(precision):.4f}")
        print(f"Recall:    {float(recall):.4f}")
        print(f"F1-score:  {float(f1):.4f}")
        print("-" * 40)


In [ ]:
def multilabel_fbeta_score_from_probs(y_true, y_prob, threshold=0.5, beta=0.5):
    y_pred = (y_prob >= threshold).astype(jnp.float32)

    TP = jnp.sum((y_pred == 1) & (y_true == 1))
    FP = jnp.sum((y_pred == 1) & (y_true == 0))
    FN = jnp.sum((y_pred == 0) & (y_true == 1))

    precision = TP / (TP + FP + 1e-8)
    recall = TP / (TP + FN + 1e-8)

    beta_squared = beta ** 2

    fbeta = (1.0 + beta_squared) * precision * recall / (
        beta_squared * precision + recall + 1e-8
    )

    return fbeta


In [ ]:
def multilabel_f2_score_from_probs(y_true, y_prob, threshold=0.5):
    return multilabel_fbeta_score_from_probs(
        y_true,
        y_prob,
        threshold=threshold,
        beta=2.0
    )

In [ ]:
def conf_matrix_per_label_with_thresholds(y_true, y_prob, thresholds_per_label, class_names=None):
    y_pred = predict_with_thresholds(y_prob, thresholds_per_label).astype(jnp.int32)
    y_true = y_true.astype(jnp.int32)

    num_labels = y_true.shape[1]

    for label_idx in range(num_labels):
        true_label = y_true[:, label_idx]
        pred_label = y_pred[:, label_idx]

        TP = jnp.sum((pred_label == 1) & (true_label == 1))
        TN = jnp.sum((pred_label == 0) & (true_label == 0))
        FP = jnp.sum((pred_label == 1) & (true_label == 0))
        FN = jnp.sum((pred_label == 0) & (true_label == 1))

        precision = TP / (TP + FP + 1e-8)
        recall = TP / (TP + FN + 1e-8)
        f1 = 2.0 * precision * recall / (precision + recall + 1e-8)

        support = TP + FN

        label_name = class_names[label_idx] if class_names is not None else f"Label {label_idx}"

        print(label_name)
        print(f"Threshold: {float(thresholds_per_label[label_idx]):.2f}")
        print(f"TP: {int(TP)}, TN: {int(TN)}, FP: {int(FP)}, FN: {int(FN)}")
        print(f"Support:   {int(support)}")
        print(f"Precision: {float(precision):.4f}")
        print(f"Recall:    {float(recall):.4f}")
        print(f"F1-score:  {float(f1):.4f}")
        print("-" * 40)


In [ ]:
def conf_matrix_multilabel_with_thresholds(y_true, y_prob, thresholds_per_label):
    y_pred = predict_with_thresholds(y_prob, thresholds_per_label)

    TP = jnp.sum((y_pred == 1) & (y_true == 1))
    TN = jnp.sum((y_pred == 0) & (y_true == 0))
    FP = jnp.sum((y_pred == 1) & (y_true == 0))
    FN = jnp.sum((y_pred == 0) & (y_true == 1))

    total = TP + TN + FP + FN

    print("CONFUSION MATRIX MULTI-LABEL:")
    print()
    print("                         True 1       True 0")
    print()
    print(f"Predicted 1              {int(TP)} TP       {int(FP)} FP")
    print("                         -------------------")
    print(f"Predicted 0              {int(FN)} FN       {int(TN)} TN")
    print()
    print(f"False Negatives = {(float(FN) / float(total)) * 100:.1f} %")
    print(f"False Positives = {(float(FP) / float(total)) * 100:.1f} %")


## GRID SEARCH FOR BEST PARAMETERS AND BEST RESULTS

Applichiamo l'oversampling solo su TRAINING SET per le classe minoritaria

In [ ]:
class_counts_train = np.sum(np.array(Y_train), axis=0)

print("Classi:", mlb.classes_)
print("Campioni per classe nel training:")
print(class_counts_train)

X_train_os, Y_train_os = oversample_minority_multilabel(
    X_train,
    Y_train,
    min_count=800
)

print("\nOriginal train shape:", X_train.shape, Y_train.shape)
print("Oversampled train shape:", X_train_os.shape, Y_train_os.shape)

print("\nCampioni per classe dopo oversampling:")
print(np.sum(np.array(Y_train_os), axis=0))


NameError: name 'Y_train' is not defined

Calcolo pos_weights

In [ ]:
pos_weights = compute_pos_weights(Y_train)

print(pos_weights)

LOOP to tune the parameters through the optimization algorithm RMSProp

In [ ]:
# GRID SEARCH per ANN multi-label

input_dim = X_train_os.shape[1]
output_dim = Y_train_os.shape[1]

layers_sizes = [
    [input_dim, 64, output_dim],
    [input_dim, 128, output_dim],
]

nums_epochs = [100]
batch_sizes = [64]
learning_rates = [1e-4]
decay_rates = [0.95]
deltas = [1e-8]
max_weights = [2.0, 3.0, 5.0]

best_parameters = None
best_score = -1.0
best_params = None
best_thresholds_per_label = None

n_training_points = X_train_os.shape[0]
search_counter = 0

for layers_size in layers_sizes:
    for num_epochs in nums_epochs:
        for batch_size in batch_sizes:
            for learning_rate in learning_rates:
                for decay_rate in decay_rates:
                    for delta in deltas:
                        for max_weight in max_weights:

                            search_counter += 1

                            key = jax.random.PRNGKey(search_counter)
                            params = initialize_params(layers_size, key)

                            pos_weights = compute_pos_weights(
                                Y_train_os,
                                max_weight=max_weight
                            )

                            cumulated_sqr_grad = [
                                (jnp.zeros_like(W), jnp.zeros_like(b))
                                for W, b in params
                            ]

                            for epoch in range(num_epochs):
                                permutation = np.random.permutation(n_training_points)

                                for start_idx in range(0, n_training_points, batch_size):
                                    end_idx = start_idx + batch_size
                                    batch_idxs = permutation[start_idx:end_idx]

                                    x_batch = X_train_os[batch_idxs, :]
                                    y_batch = Y_train_os[batch_idxs, :]

                                    params, cumulated_sqr_grad, train_loss = rmsprop_update(
                                        params,
                                        x_batch,
                                        y_batch,
                                        pos_weights,
                                        learning_rate,
                                        decay_rate,
                                        delta,
                                        cumulated_sqr_grad
                                    )

                            y_val_prob = ANN(X_val, params)

                            thresholds_per_label = find_best_thresholds_per_label(
                                Y_val,
                                y_val_prob,
                                beta=2.0
                            )

                            val_f1 = multilabel_fbeta_score_with_thresholds(
                                Y_val,
                                y_val_prob,
                                thresholds_per_label,
                                beta=1.0
                            )

                            val_f2 = multilabel_fbeta_score_with_thresholds(
                                Y_val,
                                y_val_prob,
                                thresholds_per_label,
                                beta=2.0
                            )

                            val_acc = multilabel_accuracy_with_thresholds(
                                Y_val,
                                y_val_prob,
                                thresholds_per_label
                            )

                            # scegli qui:
                            test_score = val_f1
                            # oppure:
                            # test_score = val_f2

                            if test_score > best_score:
                                best_score = float(test_score)
                                best_parameters = [
                                    layers_size,
                                    num_epochs,
                                    batch_size,
                                    learning_rate,
                                    decay_rate,
                                    delta,
                                    max_weight
                                ]
                                best_params = params
                                best_thresholds_per_label = thresholds_per_label

                                print("New best!")
                                print(f"Val F1: {float(val_f1):.4f}")
                                print(f"Val F2: {float(val_f2):.4f}")
                                print(f"Val Acc: {float(val_acc):.4f}")
                                print(f"Thresholds per label: {thresholds_per_label}")
                                print(f"Parameters: {best_parameters}")
                                print()

Best Parameters & Metrics Grid Search

In [ ]:
# Final evaluation after Grid Search

best_max_weight = best_parameters[6]

best_pos_weights = compute_pos_weights(
    Y_train_os,
    max_weight=best_max_weight
)

print("BEST PARAMETERS")
print(f"Layers size:   {best_parameters[0]}")
print(f"Epochs:        {best_parameters[1]}")
print(f"Batch size:    {best_parameters[2]}")
print(f"Learning rate: {best_parameters[3]}")
print(f"Decay rate:    {best_parameters[4]}")
print(f"Delta:         {best_parameters[5]}")
print(f"Max weight:    {best_max_weight}")
print(f"Thresholds per label: {best_thresholds_per_label}")
print()

# Valutazione sul train reale, non oversampled
train_loss = loss_function(best_params, X_train, Y_train, best_pos_weights)
val_loss   = loss_function(best_params, X_val, Y_val, best_pos_weights)
test_loss  = loss_function(best_params, X_test, Y_test, best_pos_weights)

y_train_prob = ANN(X_train, best_params)
y_val_prob   = ANN(X_val, best_params)
y_test_prob  = ANN(X_test, best_params)

train_acc = multilabel_accuracy_with_thresholds(Y_train, y_train_prob, best_thresholds_per_label)
val_acc   = multilabel_accuracy_with_thresholds(Y_val, y_val_prob, best_thresholds_per_label)
test_acc  = multilabel_accuracy_with_thresholds(Y_test, y_test_prob, best_thresholds_per_label)

train_precision = multilabel_precision_with_thresholds(Y_train, y_train_prob, best_thresholds_per_label)
val_precision   = multilabel_precision_with_thresholds(Y_val, y_val_prob, best_thresholds_per_label)
test_precision  = multilabel_precision_with_thresholds(Y_test, y_test_prob, best_thresholds_per_label)

train_recall = multilabel_recall_with_thresholds(Y_train, y_train_prob, best_thresholds_per_label)
val_recall   = multilabel_recall_with_thresholds(Y_val, y_val_prob, best_thresholds_per_label)
test_recall  = multilabel_recall_with_thresholds(Y_test, y_test_prob, best_thresholds_per_label)

train_f1 = multilabel_fbeta_score_with_thresholds(Y_train, y_train_prob, best_thresholds_per_label, beta=1.0)
val_f1   = multilabel_fbeta_score_with_thresholds(Y_val, y_val_prob, best_thresholds_per_label, beta=1.0)
test_f1  = multilabel_fbeta_score_with_thresholds(Y_test, y_test_prob, best_thresholds_per_label, beta=1.0)

train_f2 = multilabel_fbeta_score_with_thresholds(Y_train, y_train_prob, best_thresholds_per_label, beta=2.0)
val_f2   = multilabel_fbeta_score_with_thresholds(Y_val, y_val_prob, best_thresholds_per_label, beta=2.0)
test_f2  = multilabel_fbeta_score_with_thresholds(Y_test, y_test_prob, best_thresholds_per_label, beta=2.0)

print("FINAL METRICS")
print(f"Train loss:      {float(train_loss):.4f}")
print(f"Val loss:        {float(val_loss):.4f}")
print(f"Test loss:       {float(test_loss):.4f}")
print()

print(f"Train accuracy:  {float(train_acc):.4f}")
print(f"Val accuracy:    {float(val_acc):.4f}")
print(f"Test accuracy:   {float(test_acc):.4f}")
print()

print(f"Train precision: {float(train_precision):.4f}")
print(f"Val precision:   {float(val_precision):.4f}")
print(f"Test precision:  {float(test_precision):.4f}")
print()

print(f"Train recall:    {float(train_recall):.4f}")
print(f"Val recall:      {float(val_recall):.4f}")
print(f"Test recall:     {float(test_recall):.4f}")
print()

print(f"Train F1-score:  {float(train_f1):.4f}")
print(f"Val F1-score:    {float(val_f1):.4f}")
print(f"Test F1-score:   {float(test_f1):.4f}")
print()

print(f"Train F2-score:  {float(train_f2):.4f}")
print(f"Val F2-score:    {float(val_f2):.4f}")
print(f"Test F2-score:   {float(test_f2):.4f}")
print()

conf_matrix_multilabel_with_thresholds(
    Y_test,
    y_test_prob,
    best_thresholds_per_label
)

print()

conf_matrix_per_label_with_thresholds(
    Y_test,
    y_test_prob,
    best_thresholds_per_label,
    class_names=mlb.classes_
)


#TESTING DIFFERENT MODELS AND COMBINATIONS

## Funzione di Train-Test per le varie combinazioni

In [ ]:
def trainTestCombANN(
    df_work,
    feature_columns,
    label_column="Manually assigned PLATFORM",
    combo_name="FEATURE_COMBINATION",
    min_count_oversampling=800,
    thresholds_per_label=None,
    num_epochs=100,
    batch_size=64,
    learning_rate=1e-4,
    decay_rate=0.95,
    delta=1e-8,
    max_weight=5.0,
    random_state=42,
    save_vectorizer=True
):
    """
    Esegue training e testing della ANN multi-label usando una combinazione specifica di feature.

    Parametri:
    - df_work: dataset di lavoro
    - feature_columns: lista di colonne da usare per costruire COMBINED_TEXT
    - label_column: colonna target multi-label
    - combo_name: nome dell'esperimento
    """

    print("=" * 80)
    print(f"TRAIN/TEST ANN - {combo_name}")
    print("=" * 80)
    print("Feature usate:")
    for col in feature_columns:
        print("-", col)

    # Copia locale del dataset
    df_local = df_work.copy()

    # Controllo colonne
    required_columns = [label_column] + feature_columns
    missing_columns = [col for col in required_columns if col not in df_local.columns]

    if missing_columns:
        raise ValueError(f"Colonne mancanti nel dataset: {missing_columns}")

    # Rimuove righe senza label
    df_local = df_local.dropna(subset=[label_column]).copy()
    df_local = df_local[df_local[label_column].astype(str).str.strip() != ""].copy()
    df_local = df_local.reset_index(drop=True)

    # Features testuali
    X = df_local[feature_columns].fillna("").copy()

    # Costruzione dinamica di COMBINED_TEXT usando solo le feature date in input
    X["COMBINED_TEXT"] = X[feature_columns].astype(str).agg(" ".join, axis=1)

    x = X["COMBINED_TEXT"]

    # Multi-label binarization
    y_raw = df_local[label_column].astype(str)
    y_split = y_raw.apply(lambda value: value.split())

    mlb = MultiLabelBinarizer()
    y_multi = mlb.fit_transform(y_split)

    num_classes = len(mlb.classes_)

    print()
    print(f"Numero di classi reali: {num_classes}")
    print(f"Classi: {mlb.classes_}")

    class_distribution = pd.DataFrame({
        "Classe": mlb.classes_,
        "Numero di sample": y_multi.sum(axis=0)
    }).sort_values(by="Numero di sample", ascending=False)

    display(class_distribution)

    # Shuffle iniziale
    x_shuffled, y_shuffled = shuffle(
        x[:len(y_multi)],
        y_multi,
        random_state=random_state
    )

    # Primo split: train+val / test
    X_temp, X_test_text, Y_temp, Y_test = train_test_split(
        x_shuffled,
        y_shuffled,
        test_size=0.15,
        random_state=random_state
    )

    # Secondo split: train / validation
    X_train_text, X_val_text, Y_train, Y_val = train_test_split(
        X_temp,
        Y_temp,
        test_size=0.15,
        random_state=random_state
    )

    print()
    print("Train size:", len(X_train_text))
    print("Validation size:", len(X_val_text))
    print("Test size:", len(X_test_text))

    # TF-IDF
    tfidf = TfidfVectorizer(
    sublinear_tf=True,
    max_features=40000,
    ngram_range=(1, 3),
    min_df=3,
    max_df=0.95,
    stop_words=None,
    strip_accents="unicode"
)

    def clean_text(text):
        text = str(text).upper()
        text = re.sub(r"[^A-Z0-9\s/.\-]", " ", text)
        text = re.sub(r"\s+", " ", text).strip()
        return text

    # Pulizia testi
    X_train_text_clean = [clean_text(text) for text in X_train_text]
    X_val_text_clean   = [clean_text(text) for text in X_val_text]
    X_test_text_clean  = [clean_text(text) for text in X_test_text]

    # TF-IDF: fit solo su train
    X_train_tfidf = tfidf.fit_transform(X_train_text_clean)
    X_val_tfidf   = tfidf.transform(X_val_text_clean)
    X_test_tfidf  = tfidf.transform(X_test_text_clean)

    if save_vectorizer:
        vectorizer_filename = f"ORM_tfidf_vectorizer{combo_name}.pkl"
        with open(vectorizer_filename, "wb") as file:
            pickle.dump(tfidf, file)

        print()
        print(f"Vettorizzatore TF-IDF salvato: {vectorizer_filename}")

    # Conversione in array densi per JAX
    X_train = jnp.array(X_train_tfidf.toarray(), dtype=jnp.float32)
    X_val   = jnp.array(X_val_tfidf.toarray(), dtype=jnp.float32)
    X_test  = jnp.array(X_test_tfidf.toarray(), dtype=jnp.float32)

    Y_train = jnp.array(Y_train, dtype=jnp.float32)
    Y_val   = jnp.array(Y_val, dtype=jnp.float32)
    Y_test  = jnp.array(Y_test, dtype=jnp.float32)

    print()
    print("Shape finali:")
    print("X_train:", X_train.shape)
    print("X_val:  ", X_val.shape)
    print("X_test: ", X_test.shape)
    print("Y_train:", Y_train.shape)
    print("Y_val:  ", Y_val.shape)
    print("Y_test: ", Y_test.shape)

    print()
    print("Numero effettivo di feature TF-IDF:", len(tfidf.get_feature_names_out()))

    # Conversione esplicita
    X_train = jnp.asarray(X_train, dtype=jnp.float32)
    X_test  = jnp.asarray(X_test, dtype=jnp.float32)
    X_val   = jnp.asarray(X_val, dtype=jnp.float32)

    Y_train = jnp.asarray(Y_train, dtype=jnp.float32)
    Y_test  = jnp.asarray(Y_test, dtype=jnp.float32)
    Y_val   = jnp.asarray(Y_val, dtype=jnp.float32)

    # Oversampling solo sul training set
    class_counts_train = np.sum(np.array(Y_train), axis=0)

    print()
    print("Campioni per classe nel training:")
    print(pd.DataFrame({
        "Classe": mlb.classes_,
        "Train samples": class_counts_train
    }))

    X_train_os, Y_train_os = oversample_minority_multilabel(
        X_train,
        Y_train,
        min_count=min_count_oversampling
    )

    print()
    print("Original train shape:", X_train.shape, Y_train.shape)
    print("Oversampled train shape:", X_train_os.shape, Y_train_os.shape)

    print()
    print("Campioni per classe dopo oversampling:")
    print(np.sum(np.array(Y_train_os), axis=0))

    # Parametri fissi: no grid search
    input_dim = X_train_os.shape[1]
    output_dim = Y_train_os.shape[1]

    print(f"DEBUG: output_dim = {output_dim}") # Debug print

    layers_size = [input_dim, 128, output_dim]

    if thresholds_per_label is None:
        # Dynamically create thresholds based on output_dim
        thresholds_per_label = jnp.full(output_dim, 0.7, dtype=jnp.float32)
        print(f"DEBUG: thresholds_per_label (after jnp.full) length = {len(thresholds_per_label)}") # Debug print

    thresholds_per_label = jnp.asarray(thresholds_per_label, dtype=jnp.float32)
    print(f"DEBUG: thresholds_per_label (after jnp.asarray) length = {len(thresholds_per_label)}") # Debug print

    if len(thresholds_per_label) != output_dim:
        raise ValueError(
            f"Numero soglie ({len(thresholds_per_label)}) diverso dal numero classi ({output_dim})"
        )

    print()
    print("PARAMETRI USATI")
    print(f"Layers size:   {layers_size}")
    print(f"Epochs:        {num_epochs}")
    print(f"Batch size:    {batch_size}")
    print(f"Learning rate: {learning_rate}")
    print(f"Decay rate:    {decay_rate}")
    print(f"Delta:         {delta}")
    print(f"Max weight:    {max_weight}")
    print(f"Thresholds per label: {thresholds_per_label}")
    print()

    key = jax.random.PRNGKey(random_state)
    params = initialize_params(layers_size, key)

    pos_weights = compute_pos_weights(
        Y_train_os,
        max_weight=max_weight
    )

    cumulated_sqr_grad = [
        (jnp.zeros_like(W), jnp.zeros_like(b))
        for W, b in params
    ]

    n_training_points = X_train_os.shape[0]

    # Training senza grid search
    for epoch in range(num_epochs):
        permutation = np.random.permutation(n_training_points)

        epoch_losses = []

        for start_idx in range(0, n_training_points, batch_size):
            end_idx = start_idx + batch_size
            batch_idxs = permutation[start_idx:end_idx]

            x_batch = X_train_os[batch_idxs, :]
            y_batch = Y_train_os[batch_idxs, :]

            params, cumulated_sqr_grad, train_loss_batch = rmsprop_update(
                params,
                x_batch,
                y_batch,
                pos_weights,
                learning_rate,
                decay_rate,
                delta,
                cumulated_sqr_grad
            )

            epoch_losses.append(float(train_loss_batch))

        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(
                f"Epoch {epoch + 1:3d}/{num_epochs} - "
                f"Mean batch loss: {np.mean(epoch_losses):.6f}"
            )

    best_params = params
    best_thresholds_per_label = thresholds_per_label
    best_pos_weights = compute_pos_weights(
        Y_train_os,
        max_weight=max_weight
    )

    # Valutazione su train reale, validation e test
    train_loss = loss_function(best_params, X_train, Y_train, best_pos_weights)
    val_loss   = loss_function(best_params, X_val, Y_val, best_pos_weights)
    test_loss  = loss_function(best_params, X_test, Y_test, best_pos_weights)

    y_train_prob = ANN(X_train, best_params)
    y_val_prob   = ANN(X_val, best_params)
    y_test_prob  = ANN(X_test, best_params)

    train_acc = multilabel_accuracy_with_thresholds(
        Y_train, y_train_prob, best_thresholds_per_label
    )
    val_acc = multilabel_accuracy_with_thresholds(
        Y_val, y_val_prob, best_thresholds_per_label
    )
    test_acc = multilabel_accuracy_with_thresholds(
        Y_test, y_test_prob, best_thresholds_per_label
    )

    train_precision = multilabel_precision_with_thresholds(
        Y_train, y_train_prob, best_thresholds_per_label
    )
    val_precision = multilabel_precision_with_thresholds(
        Y_val, y_val_prob, best_thresholds_per_label
    )
    test_precision = multilabel_precision_with_thresholds(
        Y_test, y_test_prob, best_thresholds_per_label
    )

    train_recall = multilabel_recall_with_thresholds(
        Y_train, y_train_prob, best_thresholds_per_label
    )
    val_recall = multilabel_recall_with_thresholds(
        Y_val, y_val_prob, best_thresholds_per_label
    )
    test_recall = multilabel_recall_with_thresholds(
        Y_test, y_test_prob, best_thresholds_per_label
    )

    train_f1 = multilabel_fbeta_score_with_thresholds(
        Y_train, y_train_prob, best_thresholds_per_label, beta=1.0
    )
    val_f1 = multilabel_fbeta_score_with_thresholds(
        Y_val, y_val_prob, best_thresholds_per_label, beta=1.0
    )
    test_f1 = multilabel_fbeta_score_with_thresholds(
        Y_test, y_test_prob, best_thresholds_per_label, beta=1.0
    )

    train_f2 = multilabel_fbeta_score_with_thresholds(
        Y_train, y_train_prob, best_thresholds_per_label, beta=2.0
    )
    val_f2 = multilabel_fbeta_score_with_thresholds(
        Y_val, y_val_prob, best_thresholds_per_label, beta=2.0
    )
    test_f2 = multilabel_fbeta_score_with_thresholds(
        Y_test, y_test_prob, best_thresholds_per_label, beta=2.0
    )

    print()
    print("FINAL METRICS")
    print(f"Train loss:      {float(train_loss):.4f}")
    print(f"Val loss:        {float(val_loss):.4f}")
    print(f"Test loss:       {float(test_loss):.4f}")
    print()

    print(f"Train accuracy:  {float(train_acc):.4f}")
    print(f"Val accuracy:    {float(val_acc):.4f}")
    print(f"Test accuracy:   {float(test_acc):.4f}")
    print()

    print(f"Train precision: {float(train_precision):.4f}")
    print(f"Val precision:   {float(val_precision):.4f}")
    print(f"Test precision:  {float(test_precision):.4f}")
    print()

    print(f"Train recall:    {float(train_recall):.4f}")
    print(f"Val recall:      {float(val_recall):.4f}")
    print(f"Test recall:     {float(test_recall):.4f}")
    print()

    print(f"Train F1-score:  {float(train_f1):.4f}")
    print(f"Val F1-score:    {float(val_f1):.4f}")
    print(f"Test F1-score:   {float(test_f1):.4f}")
    print()

    print(f"Train F2-score:  {float(train_f2):.4f}")
    print(f"Val F2-score:    {float(val_f2):.4f}")
    print(f"Test F2-score:   {float(test_f2):.4f}")
    print()

    conf_matrix_multilabel_with_thresholds(
        Y_test,
        y_test_prob,
        best_thresholds_per_label
    )

    print()

    conf_matrix_per_label_with_thresholds(
        Y_test,
        y_test_prob,
        best_thresholds_per_label,
        class_names=mlb.classes_
    )

    # Dizionario finale utile per confrontare più combinazioni
    result = {
        "combo_name": combo_name,
        "features": feature_columns,
        "num_input_columns": len(feature_columns),
        "num_tfidf_features": len(tfidf.get_feature_names_out()),
        "num_classes": num_classes,

        "train_loss": float(train_loss),
        "val_loss": float(val_loss),
        "test_loss": float(test_loss),

        "train_accuracy": float(train_acc),
        "val_accuracy": float(val_acc),
        "test_accuracy": float(test_acc),

        "train_precision": float(train_precision),
        "val_precision": float(val_precision),
        "test_precision": float(test_precision),

        "train_recall": float(train_recall),
        "val_recall": float(val_recall),
        "test_recall": float(test_recall),

        "train_f1": float(train_f1),
        "val_f1": float(val_f1),
        "test_f1": float(test_f1),

        "train_f2": float(train_f2),
        "val_f2": float(val_f2),
        "test_f2": float(test_f2),

        "thresholds_per_label": np.array(best_thresholds_per_label),
        "classes": mlb.classes_,
        "params": best_params,
        "tfidf": tfidf,
        "mlb": mlb
    }

    return result

## Funzioni migliorate con migliore feature's combination e modelli differenti

### Funzioni helper

In [ ]:

def optimize_per_label_ensemble(
    Y_val,
    prob_logreg,
    prob_svc,
    prob_sgd,
    beta=1.0,
    min_precision=0.55,
    threshold_grid=np.arange(0.10, 0.91, 0.02),
    weight_grid=(0.0, 0.25, 0.5, 0.75, 1.0)
):
    Y_val = np.array(Y_val)
    n_labels = Y_val.shape[1]

    best_weights = []
    best_thresholds = []

    for j in range(n_labels):
        y_true = Y_val[:, j]

        best_score = -1.0
        best_w = (1.0, 0.0, 0.0)
        best_t = 0.5

        fallback_score = -1.0
        fallback_w = (1.0, 0.0, 0.0)
        fallback_t = 0.5

        for w_lr in weight_grid:
            for w_svc in weight_grid:
                for w_sgd in weight_grid:
                    weight_sum = w_lr + w_svc + w_sgd

                    if weight_sum == 0:
                        continue

                    w_lr_n = w_lr / weight_sum
                    w_svc_n = w_svc / weight_sum
                    w_sgd_n = w_sgd / weight_sum

                    y_prob = (
                        w_lr_n * prob_logreg[:, j]
                        + w_svc_n * prob_svc[:, j]
                        + w_sgd_n * prob_sgd[:, j]
                    )

                    for t in threshold_grid:
                        y_pred = (y_prob >= t).astype(int)

                        TP = np.sum((y_pred == 1) & (y_true == 1))
                        FP = np.sum((y_pred == 1) & (y_true == 0))
                        FN = np.sum((y_pred == 0) & (y_true == 1))

                        precision = TP / (TP + FP + 1e-8)
                        recall = TP / (TP + FN + 1e-8)

                        beta_sq = beta ** 2
                        score = (1 + beta_sq) * precision * recall / (
                            beta_sq * precision + recall + 1e-8
                        )

                        if score > fallback_score:
                            fallback_score = score
                            fallback_w = (w_lr_n, w_svc_n, w_sgd_n)
                            fallback_t = t

                        if precision >= min_precision and score > best_score:
                            best_score = score
                            best_w = (w_lr_n, w_svc_n, w_sgd_n)
                            best_t = t

        if best_score < 0:
            best_w = fallback_w
            best_t = fallback_t

        best_weights.append(best_w)
        best_thresholds.append(best_t)

    return np.array(best_weights), np.array(best_thresholds)


def apply_per_label_ensemble(prob_logreg, prob_svc, prob_sgd, weights):
    n_labels = prob_logreg.shape[1]
    blended = np.zeros_like(prob_logreg)

    for j in range(n_labels):
        w_lr, w_svc, w_sgd = weights[j]
        blended[:, j] = (
            w_lr * prob_logreg[:, j]
            + w_svc * prob_svc[:, j]
            + w_sgd * prob_sgd[:, j]
        )

    return blended

def ANN_logits_dropout(x, params, key=None, dropout_rate=0.0, training=False):
    layer = x
    keys = None

    if training and dropout_rate > 0:
        keys = jax.random.split(key, len(params) - 1)

    for i, (W, b) in enumerate(params):
        layer = jnp.dot(layer, W) + b

        if i < len(params) - 1:
            layer = jax.nn.relu(layer)

            if training and dropout_rate > 0:
                keep_prob = 1.0 - dropout_rate
                mask = jax.random.bernoulli(keys[i], keep_prob, shape=layer.shape)
                layer = jnp.where(mask, layer / keep_prob, 0.0)

    return layer


def ANN_dropout(x, params):
    return jax.nn.sigmoid(ANN_logits_dropout(x, params, training=False))


def loss_function_adamw(params, x, y, pos_weights, key, dropout_rate=0.2, l2_lambda=1e-5):
    logits = ANN_logits_dropout(
        x,
        params,
        key=key,
        dropout_rate=dropout_rate,
        training=True
    )

    loss_positive = pos_weights * y * jax.nn.softplus(-logits)
    loss_negative = (1.0 - y) * jax.nn.softplus(logits)
    bce = jnp.mean(loss_positive + loss_negative)

    l2 = 0.0
    for W, b in params:
        l2 += jnp.sum(W ** 2)

    return bce + l2_lambda * l2


def initialize_adam_state(params):
    m = []
    v = []

    for W, b in params:
        m.append((jnp.zeros_like(W), jnp.zeros_like(b)))
        v.append((jnp.zeros_like(W), jnp.zeros_like(b)))

    return m, v


def _adamw_update(
    params,
    m,
    v,
    x_batch,
    y_batch,
    pos_weights,
    key,
    step,
    learning_rate,
    beta1,
    beta2,
    eps,
    weight_decay,
    dropout_rate,
    l2_lambda
):
    loss_value, grads = jax.value_and_grad(loss_function_adamw)(
        params,
        x_batch,
        y_batch,
        pos_weights,
        key,
        dropout_rate,
        l2_lambda
    )

    grads = clip_gradients(grads, max_norm=5.0)

    new_params = []
    new_m = []
    new_v = []

    for (W, b), (dW, db), (mW, mb), (vW, vb) in zip(params, grads, m, v):
        mW = beta1 * mW + (1.0 - beta1) * dW
        mb = beta1 * mb + (1.0 - beta1) * db

        vW = beta2 * vW + (1.0 - beta2) * (dW ** 2)
        vb = beta2 * vb + (1.0 - beta2) * (db ** 2)

        mW_hat = mW / (1.0 - beta1 ** step)
        mb_hat = mb / (1.0 - beta1 ** step)

        vW_hat = vW / (1.0 - beta2 ** step)
        vb_hat = vb / (1.0 - beta2 ** step)

        W = W - learning_rate * (mW_hat / (jnp.sqrt(vW_hat) + eps) + weight_decay * W)
        b = b - learning_rate * (mb_hat / (jnp.sqrt(vb_hat) + eps))

        new_params.append((W, b))
        new_m.append((mW, mb))
        new_v.append((vW, vb))

    return new_params, new_m, new_v, loss_value

# Apply jax.jit explicitly after definition
adamw_update = jax.jit(_adamw_update, static_argnums=(13, 14))


def compute_extended_metrics(y_true, y_prob, thresholds):
    y_pred = np.array(y_prob >= thresholds).astype(int)
    y_true = np.array(y_true).astype(int)

    micro_p, micro_r, micro_f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="micro",
        zero_division=0
    )

    macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

    micro_f2 = fbeta_score(y_true, y_pred, beta=2.0, average="micro", zero_division=0)
    macro_f2 = fbeta_score(y_true, y_pred, beta=2.0, average="macro", zero_division=0)

    subset_accuracy = np.mean(np.all(y_pred == y_true, axis=1))
    label_accuracy = np.mean(y_pred == y_true)

    return {
        "micro_precision": micro_p,
        "micro_recall": micro_r,
        "micro_f1": micro_f1,
        "micro_f2": micro_f2,
        "macro_precision": macro_p,
        "macro_recall": macro_r,
        "macro_f1": macro_f1,
        "macro_f2": macro_f2,
        "subset_accuracy": subset_accuracy,
        "label_accuracy": label_accuracy
    }


def parse_multilabel_string(value):
    if pd.isna(value):
        return []
    return [x.strip().upper() for x in str(value).split() if x.strip()]

def build_prefixed_text(df_features, feature_columns):
    """
    Costruisce un'unica stringa testuale per ogni riga, mantenendo il nome
    della feature come prefisso.

    Esempio:
    TITLE: ... ABSTRACT: ... IPC: ... CPC: ...

    Questo aiuta il vettorizzatore a distinguere la provenienza delle parole.
    """

    parts = []

    for col in feature_columns:
        prefix = (
            col.replace(" (Translated)(English)", "")
               .replace("Std. ", "")
               .replace(".", "")
               .replace("/", "_")
               .replace(" ", "_")
               .upper()
        )

        field_text = df_features[col].fillna("").astype(str)

        parts.append(prefix + ": " + field_text)

    return pd.concat(parts, axis=1).agg(" ".join, axis=1)


def build_prefixed_text_local(df_features, feature_columns):
    parts = []

    for col in feature_columns:
        prefix = (
            col.replace(" (Translated)(English)", "")
               .replace("Std. ", "")
               .replace(".", "")
               .replace("/", "_")
               .replace(" ", "_")
               .upper()
        )

        parts.append(prefix + ": " + df_features[col].fillna("").astype(str))

    return pd.concat(parts, axis=1).agg(" ".join, axis=1)


def clean_text(text):
    text = str(text).upper()
    text = re.sub(r"[^A-Z0-9\s/.:-]", " ", text) # Corrected regex for literal hyphen at the end
    text = re.sub(r"\s+", " ", text).strip() # Corrected regex for whitespace
    return text

def clean_text_ann(text):
    text = str(text).upper()
    text = re.sub(r"[^A-Z0-9\s/.\-:]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def multilabel_train_val_test_split(x, y, test_size=0.30, val_size=0.15, random_state=42):
    try:
        from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

        msss_test = MultilabelStratifiedShuffleSplit(
            n_splits=1,
            test_size=test_size,
            random_state=random_state
        )
        train_val_idx, test_idx = next(msss_test.split(np.zeros(len(y)), y))

        x_train_val = x.iloc[train_val_idx].reset_index(drop=True)
        y_train_val = y[train_val_idx]
        x_test = x.iloc[test_idx].reset_index(drop=True)
        y_test = y[test_idx]

        relative_val_size = val_size / (1.0 - test_size)

        msss_val = MultilabelStratifiedShuffleSplit(
            n_splits=1,
            test_size=relative_val_size,
            random_state=random_state
        )
        train_idx, val_idx = next(msss_val.split(np.zeros(len(y_train_val)), y_train_val))

        return (
            x_train_val.iloc[train_idx].reset_index(drop=True),
            x_train_val.iloc[val_idx].reset_index(drop=True),
            x_test,
            y_train_val[train_idx],
            y_train_val[val_idx],
            y_test
        )

    except Exception as exc:
        print("Iterative stratification non disponibile, uso train_test_split standard.")
        print("Motivo:", exc)

        X_temp, X_test, Y_temp, Y_test = train_test_split(
            x,
            y,
            test_size=test_size,
            random_state=random_state
        )

        X_train, X_val, Y_train, Y_val = train_test_split(
            X_temp,
            Y_temp,
            test_size=val_size / (1.0 - test_size),
            random_state=random_state
        )

        return X_train, X_val, X_test, Y_train, Y_val, Y_test


def copy_params(params):
    return [(jnp.array(W), jnp.array(b)) for W, b in params]

def find_best_thresholds_sklearn_constrained(
    y_true,
    y_prob,
    beta=1.0,
    min_precision=0.55,
    min_threshold=0.10,
    max_threshold=0.90,
    step=0.02
):
    y_true = np.array(y_true)
    y_prob = np.array(y_prob)

    thresholds = []

    for j in range(y_true.shape[1]):
        y_true_j = y_true[:, j]
        y_prob_j = y_prob[:, j]

        best_t = 0.5
        best_score = -1.0

        fallback_t = 0.5
        fallback_score = -1.0

        for t in np.arange(min_threshold, max_threshold + 1e-9, step):
            y_pred_j = (y_prob_j >= t).astype(int)

            TP = np.sum((y_pred_j == 1) & (y_true_j == 1))
            FP = np.sum((y_pred_j == 1) & (y_true_j == 0))
            FN = np.sum((y_pred_j == 0) & (y_true_j == 1))

            precision = TP / (TP + FP + 1e-8)
            recall = TP / (TP + FN + 1e-8)

            beta_sq = beta ** 2
            score = (1 + beta_sq) * precision * recall / (
                beta_sq * precision + recall + 1e-8
            )

            if score > fallback_score:
                fallback_score = score
                fallback_t = t

            if precision >= min_precision and score > best_score:
                best_score = score
                best_t = t

        if best_score < 0:
            best_t = fallback_t

        thresholds.append(best_t)

    return np.array(thresholds, dtype=float)


def compute_multilabel_metrics_sklearn_extended(y_true, y_prob, thresholds):
    y_true = np.array(y_true).astype(int)
    y_pred = (np.array(y_prob) >= thresholds).astype(int)

    micro_p, micro_r, micro_f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="micro",
        zero_division=0
    )

    macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

    micro_f2 = fbeta_score(
        y_true,
        y_pred,
        beta=2.0,
        average="micro",
        zero_division=0
    )

    macro_f2 = fbeta_score(
        y_true,
        y_pred,
        beta=2.0,
        average="macro",
        zero_division=0
    )

    label_accuracy = np.mean(y_pred == y_true)
    subset_accuracy = np.mean(np.all(y_pred == y_true, axis=1))

    return {
        "micro_precision": micro_p,
        "micro_recall": micro_r,
        "micro_f1": micro_f1,
        "micro_f2": micro_f2,
        "macro_precision": macro_p,
        "macro_recall": macro_r,
        "macro_f1": macro_f1,
        "macro_f2": macro_f2,
        "label_accuracy": label_accuracy,
        "subset_accuracy": subset_accuracy
    }


def predict_with_thresholds_sklearn(y_prob, thresholds):
    return (y_prob >= thresholds).astype(int)


def multilabel_metrics_sklearn(y_true, y_prob, thresholds):
    y_pred = predict_with_thresholds_sklearn(y_prob, thresholds)

    TP = np.sum((y_pred == 1) & (y_true == 1))
    FP = np.sum((y_pred == 1) & (y_true == 0))
    FN = np.sum((y_pred == 0) & (y_true == 1))

    precision = TP / (TP + FP + 1e-8)
    recall = TP / (TP + FN + 1e-8)
    f1 = 2 * precision * recall / (precision + recall + 1e-8)
    f2 = 5 * precision * recall / (4 * precision + recall + 1e-8)
    accuracy = np.mean(y_pred == y_true)

    return accuracy, precision, recall, f1, f2


def find_best_thresholds_per_label_constrained(
    y_true,
    y_prob,
    beta=1.0,
    min_precision=None,
    min_threshold=0.10,
    max_threshold=0.90,
    step=0.02
):
    y_true = np.array(y_true)
    y_prob = np.array(y_prob)

    best_thresholds = []

    for j in range(y_true.shape[1]):
        y_true_j = y_true[:, j]
        y_prob_j = y_prob[:, j]

        best_t = 0.5
        best_score = -1.0
        fallback_t = 0.5
        fallback_score = -1.0

        for t in np.arange(min_threshold, max_threshold + 1e-9, step):
            y_pred_j = (y_prob_j >= t).astype(int)

            TP = np.sum((y_pred_j == 1) & (y_true_j == 1))
            FP = np.sum((y_pred_j == 1) & (y_true_j == 0))
            FN = np.sum((y_pred_j == 0) & (y_true_j == 1))

            precision = TP / (TP + FP + 1e-8)
            recall = TP / (TP + FN + 1e-8)

            beta_sq = beta ** 2
            score = (1.0 + beta_sq) * precision * recall / (
                beta_sq * precision + recall + 1e-8
            )

            if score > fallback_score:
                fallback_score = score
                fallback_t = t

            if min_precision is None:
                if score > best_score:
                    best_score = score
                    best_t = t
            else:
                if precision >= min_precision and score > best_score:
                    best_score = score
                    best_t = t

        if best_score < 0:
            best_t = fallback_t

        best_thresholds.append(best_t)

    return jnp.array(best_thresholds, dtype=jnp.float32)




### Funzione migliorata V1

In [ ]:
def trainTestCombANNV1(
    df_work,
    feature_columns,
    label_column="Manually assigned PLATFORM",
    combo_name="FEATURE_COMBINATION",
    publication_col="Publication Number",

    min_count_oversampling=800,
    use_oversampling=True,
    use_pos_weights=False,

    thresholds_per_label=None,
    threshold_beta=1.0,
    min_precision_threshold=0.55,

    num_epochs=120,
    batch_size=64,
    learning_rate=7e-5,
    decay_rate=0.95,
    delta=1e-8,
    max_weight=3.0,
    hidden_units=256,
    patience=20,

    random_state=42,
    save_vectorizer=True
):
    print("=" * 80)
    print(f"TRAIN/TEST ANN V1 IMPROVED - {combo_name}")
    print("=" * 80)

    np.random.seed(random_state)

    df_local = df_work.copy()

    required_columns = [label_column] + feature_columns
    missing_columns = [col for col in required_columns if col not in df_local.columns]

    if missing_columns:
        raise ValueError(f"Colonne mancanti nel dataset: {missing_columns}")

    df_local = df_local.dropna(subset=[label_column]).copy()
    df_local = df_local[df_local[label_column].astype(str).str.strip() != ""].copy()

    if publication_col in df_local.columns:
        before = len(df_local)
        df_local[publication_col] = (
            df_local[publication_col]
            .astype(str)
            .str.strip()
            .str.upper()
        )

        df_local = df_local.drop_duplicates(
            subset=[publication_col, label_column]
        ).copy()

        print(f"Deduplica brevetti/label: {before} -> {len(df_local)} righe")

    df_local = df_local.reset_index(drop=True)

    X_features = df_local[feature_columns].fillna("").copy()
    x = build_prefixed_text(X_features, feature_columns)

    y_split = df_local[label_column].apply(parse_multilabel_string)

    mlb = MultiLabelBinarizer()
    y_multi = mlb.fit_transform(y_split)

    num_classes = len(mlb.classes_)

    print(f"\nNumero classi: {num_classes}")
    print("Classi:", mlb.classes_)

    class_distribution = pd.DataFrame({
        "Classe": mlb.classes_,
        "Numero di sample": y_multi.sum(axis=0)
    }).sort_values("Numero di sample", ascending=False)

    display(class_distribution)

    X_train_text, X_val_text, X_test_text, Y_train, Y_val, Y_test = multilabel_train_val_test_split(
        x,
        y_multi,
        test_size=0.15,
        val_size=0.15,
        random_state=random_state
    )

    print("\nSplit:")
    print("Train size:", len(X_train_text))
    print("Validation size:", len(X_val_text))
    print("Test size:", len(X_test_text))

    tfidf = TfidfVectorizer(
        sublinear_tf=True,
        max_features=50000,
        ngram_range=(1, 3),
        min_df=2,
        max_df=0.97,
        stop_words=None,
        strip_accents="unicode",
        token_pattern=r"(?u)\b[A-Z0-9][A-Z0-9/.\-:]+\b|\b[A-Z0-9]\b"
    )

    X_train_text_clean = [clean_text(text) for text in X_train_text]
    X_val_text_clean = [clean_text(text) for text in X_val_text]
    X_test_text_clean = [clean_text(text) for text in X_test_text]

    X_train_tfidf = tfidf.fit_transform(X_train_text_clean)
    X_val_tfidf = tfidf.transform(X_val_text_clean)
    X_test_tfidf = tfidf.transform(X_test_text_clean)

    if save_vectorizer:
        vectorizer_filename = f"ORM_tfidf_vectorizer_V1_IMPROVED_{combo_name}.pkl"
        with open(vectorizer_filename, "wb") as file:
            pickle.dump(tfidf, file)
        print(f"\nVettorizzatore TF-IDF salvato: {vectorizer_filename}")

    X_train = jnp.array(X_train_tfidf.toarray(), dtype=jnp.float32)
    X_val = jnp.array(X_val_tfidf.toarray(), dtype=jnp.float32)
    X_test = jnp.array(X_test_tfidf.toarray(), dtype=jnp.float32)

    Y_train = jnp.array(Y_train, dtype=jnp.float32)
    Y_val = jnp.array(Y_val, dtype=jnp.float32)
    Y_test = jnp.array(Y_test, dtype=jnp.float32)

    print("\nShape finali:")
    print("X_train:", X_train.shape)
    print("X_val:  ", X_val.shape)
    print("X_test: ", X_test.shape)
    print("Y_train:", Y_train.shape)
    print("Y_val:  ", Y_val.shape)
    print("Y_test: ", Y_test.shape)
    print("Numero TF-IDF feature:", len(tfidf.get_feature_names_out()))

    if use_oversampling:
        X_train_fit, Y_train_fit = oversample_minority_multilabel(
            X_train,
            Y_train,
            min_count=min_count_oversampling
        )
    else:
        X_train_fit, Y_train_fit = X_train, Y_train

    print("\nTraining class counts:")
    print(pd.DataFrame({
        "Classe": mlb.classes_,
        "Train samples": np.sum(np.array(Y_train_fit), axis=0)
    }))

    input_dim = X_train_fit.shape[1]
    output_dim = Y_train_fit.shape[1]
    layers_size = [input_dim, hidden_units, output_dim]

    key = jax.random.PRNGKey(random_state)
    params = initialize_params(layers_size, key)

    if use_pos_weights:
        pos_weights = compute_pos_weights(Y_train_fit, max_weight=max_weight)
    else:
        pos_weights = jnp.ones((output_dim,), dtype=jnp.float32)

    cumulated_sqr_grad = [
        (jnp.zeros_like(W), jnp.zeros_like(b))
        for W, b in params
    ]

    n_training_points = X_train_fit.shape[0]

    best_params = None
    best_thresholds_per_label = None
    best_val_score = -1.0
    best_epoch = 0
    epochs_without_improvement = 0

    print("\nPARAMETRI USATI")
    print(f"Layers size:             {layers_size}")
    print(f"Epochs max:              {num_epochs}")
    print(f"Batch size:              {batch_size}")
    print(f"Learning rate:           {learning_rate}")
    print(f"Max weight:              {max_weight}")
    print(f"Oversampling:            {use_oversampling}")
    print(f"Pos weights:             {use_pos_weights}")
    print(f"Threshold beta:          {threshold_beta}")
    print(f"Min precision threshold: {min_precision_threshold}")
    print(f"Early stopping patience: {patience}")

    for epoch in range(num_epochs):
        permutation = np.random.permutation(n_training_points)
        epoch_losses = []

        for start_idx in range(0, n_training_points, batch_size):
            end_idx = start_idx + batch_size
            batch_idxs = permutation[start_idx:end_idx]

            x_batch = X_train_fit[batch_idxs, :]
            y_batch = Y_train_fit[batch_idxs, :]

            params, cumulated_sqr_grad, train_loss_batch = rmsprop_update(
                params,
                x_batch,
                y_batch,
                pos_weights,
                learning_rate,
                decay_rate,
                delta,
                cumulated_sqr_grad
            )

            epoch_losses.append(float(train_loss_batch))

        y_val_prob_epoch = ANN(X_val, params)

        if thresholds_per_label is None:
            epoch_thresholds = find_best_thresholds_per_label_constrained(
                Y_val,
                y_val_prob_epoch,
                beta=threshold_beta,
                min_precision=min_precision_threshold,
                min_threshold=0.20,
                max_threshold=0.90,
                step=0.02
            )
        else:
            epoch_thresholds = jnp.asarray(thresholds_per_label, dtype=jnp.float32)

        val_score = multilabel_fbeta_score_with_thresholds(
            Y_val,
            y_val_prob_epoch,
            epoch_thresholds,
            beta=threshold_beta
        )

        val_precision = multilabel_precision_with_thresholds(
            Y_val,
            y_val_prob_epoch,
            epoch_thresholds
        )

        val_recall = multilabel_recall_with_thresholds(
            Y_val,
            y_val_prob_epoch,
            epoch_thresholds
        )

        if float(val_score) > best_val_score:
            best_val_score = float(val_score)
            best_params = copy_params(params)
            best_thresholds_per_label = jnp.asarray(epoch_thresholds, dtype=jnp.float32)
            best_epoch = epoch + 1
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(
                f"Epoch {epoch + 1:3d}/{num_epochs} - "
                f"loss={np.mean(epoch_losses):.6f} - "
                f"val_F{threshold_beta}={float(val_score):.4f} - "
                f"val_precision={float(val_precision):.4f} - "
                f"val_recall={float(val_recall):.4f} - "
                f"best={best_val_score:.4f}"
            )

        if epochs_without_improvement >= patience:
            print(f"Early stopping alla epoch {epoch + 1}. Best epoch: {best_epoch}")
            break

    train_loss = loss_function(best_params, X_train, Y_train, pos_weights)
    val_loss = loss_function(best_params, X_val, Y_val, pos_weights)
    test_loss = loss_function(best_params, X_test, Y_test, pos_weights)

    y_train_prob = ANN(X_train, best_params)
    y_val_prob = ANN(X_val, best_params)
    y_test_prob = ANN(X_test, best_params)

    train_acc = multilabel_accuracy_with_thresholds(Y_train, y_train_prob, best_thresholds_per_label)
    val_acc = multilabel_accuracy_with_thresholds(Y_val, y_val_prob, best_thresholds_per_label)
    test_acc = multilabel_accuracy_with_thresholds(Y_test, y_test_prob, best_thresholds_per_label)

    train_precision = multilabel_precision_with_thresholds(Y_train, y_train_prob, best_thresholds_per_label)
    val_precision = multilabel_precision_with_thresholds(Y_val, y_val_prob, best_thresholds_per_label)
    test_precision = multilabel_precision_with_thresholds(Y_test, y_test_prob, best_thresholds_per_label)

    train_recall = multilabel_recall_with_thresholds(Y_train, y_train_prob, best_thresholds_per_label)
    val_recall = multilabel_recall_with_thresholds(Y_val, y_val_prob, best_thresholds_per_label)
    test_recall = multilabel_recall_with_thresholds(Y_test, y_test_prob, best_thresholds_per_label)

    train_f1 = multilabel_fbeta_score_with_thresholds(Y_train, y_train_prob, best_thresholds_per_label, beta=1.0)
    val_f1 = multilabel_fbeta_score_with_thresholds(Y_val, y_val_prob, best_thresholds_per_label, beta=1.0)
    test_f1 = multilabel_fbeta_score_with_thresholds(Y_test, y_test_prob, best_thresholds_per_label, beta=1.0)

    train_f2 = multilabel_fbeta_score_with_thresholds(Y_train, y_train_prob, best_thresholds_per_label, beta=2.0)
    val_f2 = multilabel_fbeta_score_with_thresholds(Y_val, y_val_prob, best_thresholds_per_label, beta=2.0)
    test_f2 = multilabel_fbeta_score_with_thresholds(Y_test, y_test_prob, best_thresholds_per_label, beta=2.0)

    y_test_pred = np.array(predict_with_thresholds(y_test_prob, best_thresholds_per_label))
    y_test_true = np.array(Y_test)

    per_label_precision, per_label_recall, per_label_f1, per_label_support = precision_recall_fscore_support(
        y_test_true,
        y_test_pred,
        average=None,
        zero_division=0
    )

    per_label_metrics = pd.DataFrame({
        "Classe": mlb.classes_,
        "Support": per_label_support,
        "Precision": per_label_precision,
        "Recall": per_label_recall,
        "F1": per_label_f1,
        "Threshold": np.array(best_thresholds_per_label)
    })

    print("\nFINAL METRICS")
    print(f"Best epoch:      {best_epoch}")
    print(f"Train loss:      {float(train_loss):.4f}")
    print(f"Val loss:        {float(val_loss):.4f}")
    print(f"Test loss:       {float(test_loss):.4f}")
    print(f"Train accuracy:  {float(train_acc):.4f}")
    print(f"Val accuracy:    {float(val_acc):.4f}")
    print(f"Test accuracy:   {float(test_acc):.4f}")
    print(f"Train precision: {float(train_precision):.4f}")
    print(f"Val precision:   {float(val_precision):.4f}")
    print(f"Test precision:  {float(test_precision):.4f}")
    print(f"Train recall:    {float(train_recall):.4f}")
    print(f"Val recall:      {float(val_recall):.4f}")
    print(f"Test recall:     {float(test_recall):.4f}")
    print(f"Train F1-score:  {float(train_f1):.4f}")
    print(f"Val F1-score:    {float(val_f1):.4f}")
    print(f"Test F1-score:   {float(test_f1):.4f}")
    print(f"Train F2-score:  {float(train_f2):.4f}")
    print(f"Val F2-score:    {float(val_f2):.4f}")
    print(f"Test F2-score:   {float(test_f2):.4f}")
    print(f"Thresholds:      {best_thresholds_per_label}")

    print("\nPer-label metrics:")
    display(per_label_metrics)

    conf_matrix_multilabel_with_thresholds(
        Y_test,
        y_test_prob,
        best_thresholds_per_label
    )

    print()

    conf_matrix_per_label_with_thresholds(
        Y_test,
        y_test_prob,
        best_thresholds_per_label,
        class_names=mlb.classes_
    )

    result = {
        "combo_name": combo_name,
        "features": feature_columns,
        "num_input_columns": len(feature_columns),
        "num_tfidf_features": len(tfidf.get_feature_names_out()),
        "num_classes": num_classes,
        "best_epoch": best_epoch,

        "train_loss": float(train_loss),
        "val_loss": float(val_loss),
        "test_loss": float(test_loss),

        "train_accuracy": float(train_acc),
        "val_accuracy": float(val_acc),
        "test_accuracy": float(test_acc),

        "train_precision": float(train_precision),
        "val_precision": float(val_precision),
        "test_precision": float(test_precision),

        "train_recall": float(train_recall),
        "val_recall": float(val_recall),
        "test_recall": float(test_recall),

        "train_f1": float(train_f1),
        "val_f1": float(val_f1),
        "test_f1": float(test_f1),

        "train_f2": float(train_f2),
        "val_f2": float(val_f2),
        "test_f2": float(test_f2),

        "thresholds_per_label": np.array(best_thresholds_per_label),
        "classes": mlb.classes_,
        "per_label_metrics": per_label_metrics,
        "params": best_params,
        "tfidf": tfidf,
        "mlb": mlb
    }

    return result


### Funzione migliorata V2

In [ ]:
def trainTestCombANNV3(
    df_work,
    feature_columns,
    label_column="Manually assigned PLATFORM",
    combo_name="FEATURE_COMBINATION",
    publication_col="Publication Number",

    min_count_oversampling=800,
    use_oversampling=True,
    use_pos_weights=False,

    thresholds_per_label=None,
    threshold_beta=1.0,
    min_precision_threshold=0.55,

    num_epochs=120,
    batch_size=64,
    learning_rate=1e-4,
    hidden_units=256,
    dropout_rate=0.10,
    l2_lambda=1e-6,
    weight_decay=1e-5,
    patience=20,

    random_state=42,
    save_vectorizer=True,
    save_bundle=True
):
    print("=" * 80)
    print(f"TRAIN/TEST ANN V3 - {combo_name}")
    print("=" * 80)

    np.random.seed(random_state)

    df_local = df_work.copy()

    required_columns = [label_column] + feature_columns
    missing_columns = [col for col in required_columns if col not in df_local.columns]

    if missing_columns:
        raise ValueError(f"Colonne mancanti nel dataset: {missing_columns}")

    df_local = df_local.dropna(subset=[label_column]).copy()
    df_local = df_local[df_local[label_column].astype(str).str.strip() != ""].copy()

    if publication_col in df_local.columns:
        before = len(df_local)
        df_local[publication_col] = (
            df_local[publication_col]
            .astype(str)
            .str.strip()
            .str.upper()
        )

        df_local = df_local.drop_duplicates(
            subset=[publication_col, label_column]
        ).copy()

        print(f"Deduplica brevetti/label: {before} -> {len(df_local)} righe")

    df_local = df_local.reset_index(drop=True)

    X_features = df_local[feature_columns].fillna("").copy()
    x = build_prefixed_text(X_features, feature_columns)

    y_split = df_local[label_column].apply(parse_multilabel_string)

    mlb = MultiLabelBinarizer()
    y_multi = mlb.fit_transform(y_split)

    num_classes = len(mlb.classes_)

    print(f"\nNumero classi: {num_classes}")
    print("Classi:", mlb.classes_)

    class_distribution = pd.DataFrame({
        "Classe": mlb.classes_,
        "Numero di sample": y_multi.sum(axis=0)
    }).sort_values("Numero di sample", ascending=False)

    display(class_distribution)

    X_train_text, X_val_text, X_test_text, Y_train, Y_val, Y_test = multilabel_train_val_test_split(
        x,
        y_multi,
        test_size=0.15,
        val_size=0.15,
        random_state=random_state
    )

    print("\nSplit:")
    print("Train size:", len(X_train_text))
    print("Validation size:", len(X_val_text))
    print("Test size:", len(X_test_text))

    tfidf = TfidfVectorizer(
        sublinear_tf=True,
        max_features=50000,
        ngram_range=(1, 3),
        min_df=2,
        max_df=0.95,
        stop_words=None,
        strip_accents="unicode",
        token_pattern=r"(?u)\b[A-Z0-9][A-Z0-9/.\-:]+\b|\b[A-Z0-9]\b"
    )

    X_train_text_clean = [clean_text(text) for text in X_train_text]
    X_val_text_clean = [clean_text(text) for text in X_val_text]
    X_test_text_clean = [clean_text(text) for text in X_test_text]

    X_train_tfidf = tfidf.fit_transform(X_train_text_clean)
    X_val_tfidf = tfidf.transform(X_val_text_clean)
    X_test_tfidf = tfidf.transform(X_test_text_clean)

    X_train = jnp.array(X_train_tfidf.toarray(), dtype=jnp.float32)
    X_val = jnp.array(X_val_tfidf.toarray(), dtype=jnp.float32)
    X_test = jnp.array(X_test_tfidf.toarray(), dtype=jnp.float32)

    Y_train = jnp.array(Y_train, dtype=jnp.float32)
    Y_val = jnp.array(Y_val, dtype=jnp.float32)
    Y_test = jnp.array(Y_test, dtype=jnp.float32)

    print("\nShape finali:")
    print("X_train:", X_train.shape)
    print("X_val:  ", X_val.shape)
    print("X_test: ", X_test.shape)
    print("Y_train:", Y_train.shape)
    print("Y_val:  ", Y_val.shape)
    print("Y_test: ", Y_test.shape)
    print("Numero TF-IDF feature:", len(tfidf.get_feature_names_out()))

    if use_oversampling:
        X_train_fit, Y_train_fit = oversample_minority_multilabel(
            X_train,
            Y_train,
            min_count=min_count_oversampling
        )
    else:
        X_train_fit, Y_train_fit = X_train, Y_train

    print("\nTraining class counts:")
    print(pd.DataFrame({
        "Classe": mlb.classes_,
        "Train samples": np.sum(np.array(Y_train_fit), axis=0)
    }))

    input_dim = X_train_fit.shape[1]
    output_dim = Y_train_fit.shape[1]
    layers_size = [input_dim, hidden_units, output_dim]

    key = jax.random.PRNGKey(random_state)
    params = initialize_params(layers_size, key)

    if use_pos_weights:
        pos_weights = compute_pos_weights(Y_train_fit, max_weight=5.0)
    else:
        pos_weights = jnp.ones((output_dim,), dtype=jnp.float32)

    m, v = initialize_adam_state(params)

    best_params = None
    best_val_score = -1.0
    best_epoch = 0
    epochs_without_improvement = 0

    n_training_points = X_train_fit.shape[0]
    step = 0

    print("\nPARAMETRI USATI")
    print(f"Layers size:             {layers_size}")
    print(f"Epochs max:              {num_epochs}")
    print(f"Batch size:              {batch_size}")
    print(f"Learning rate:           {learning_rate}")
    print(f"Hidden units:            {hidden_units}")
    print(f"Dropout rate:            {dropout_rate}")
    print(f"L2 lambda:               {l2_lambda}")
    print(f"Weight decay:            {weight_decay}")
    print(f"Oversampling:            {use_oversampling}")
    print(f"Pos weights:             {use_pos_weights}")
    print(f"Threshold beta:          {threshold_beta}")
    print(f"Min precision threshold: {min_precision_threshold}")
    print(f"Early patience:          {patience}")

    for epoch in range(num_epochs):
        permutation = np.random.permutation(n_training_points)
        epoch_losses = []

        for start_idx in range(0, n_training_points, batch_size):
            step += 1

            end_idx = start_idx + batch_size
            batch_idxs = permutation[start_idx:end_idx]

            x_batch = X_train_fit[batch_idxs, :]
            y_batch = Y_train_fit[batch_idxs, :]

            key, subkey = jax.random.split(key)

            params, m, v, train_loss_batch = adamw_update(
                params,
                m,
                v,
                x_batch,
                y_batch,
                pos_weights,
                subkey,
                step,
                learning_rate,
                0.9,
                0.999,
                1e-8,
                weight_decay,
                dropout_rate,
                l2_lambda
            )

            epoch_losses.append(float(train_loss_batch))

        y_val_prob_epoch = ANN_dropout(X_val, params)

        temp_thresholds = find_best_thresholds_per_label_constrained(
            Y_val,
            y_val_prob_epoch,
            beta=threshold_beta,
            min_precision=min_precision_threshold,
            min_threshold=0.20,
            max_threshold=0.90,
            step=0.02
        )

        val_score = multilabel_fbeta_score_with_thresholds(
            Y_val,
            y_val_prob_epoch,
            temp_thresholds,
            beta=threshold_beta
        )

        if float(val_score) > best_val_score:
            best_val_score = float(val_score)
            best_params = copy_params(params)
            best_epoch = epoch + 1
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if (epoch + 1) % 5 == 0 or epoch == 0:
            val_metrics_tmp = compute_extended_metrics(
                Y_val,
                y_val_prob_epoch,
                np.array(temp_thresholds)
            )

            print(
                f"Epoch {epoch + 1:3d}/{num_epochs} - "
                f"train_loss={np.mean(epoch_losses):.6f} - "
                f"val_F{threshold_beta}={float(val_score):.4f} - "
                f"val_precision={val_metrics_tmp['micro_precision']:.4f} - "
                f"val_recall={val_metrics_tmp['micro_recall']:.4f} - "
                f"best_epoch={best_epoch}"
            )

        if epochs_without_improvement >= patience:
            print(f"Early stopping alla epoch {epoch + 1}. Best epoch: {best_epoch}")
            break

    y_val_prob = ANN_dropout(X_val, best_params)

    if thresholds_per_label is None:
        best_thresholds_per_label = find_best_thresholds_per_label_constrained(
            Y_val,
            y_val_prob,
            beta=threshold_beta,
            min_precision=min_precision_threshold,
            min_threshold=0.20,
            max_threshold=0.90,
            step=0.02
        )
    else:
        best_thresholds_per_label = jnp.asarray(
            thresholds_per_label,
            dtype=jnp.float32
        )

    y_train_prob = ANN_dropout(X_train, best_params)
    y_test_prob = ANN_dropout(X_test, best_params)

    train_loss = loss_function(best_params, X_train, Y_train, pos_weights)
    val_loss = loss_function(best_params, X_val, Y_val, pos_weights)
    test_loss = loss_function(best_params, X_test, Y_test, pos_weights)

    train_metrics = compute_extended_metrics(
        Y_train,
        y_train_prob,
        np.array(best_thresholds_per_label)
    )

    val_metrics = compute_extended_metrics(
        Y_val,
        y_val_prob,
        np.array(best_thresholds_per_label)
    )

    test_metrics = compute_extended_metrics(
        Y_test,
        y_test_prob,
        np.array(best_thresholds_per_label)
    )

    y_test_pred = np.array(
        predict_with_thresholds(y_test_prob, best_thresholds_per_label)
    )

    y_test_true = np.array(Y_test)

    per_label_precision, per_label_recall, per_label_f1, per_label_support = precision_recall_fscore_support(
        y_test_true,
        y_test_pred,
        average=None,
        zero_division=0
    )

    per_label_metrics = pd.DataFrame({
        "Classe": mlb.classes_,
        "Support": per_label_support,
        "Precision": per_label_precision,
        "Recall": per_label_recall,
        "F1": per_label_f1,
        "Threshold": np.array(best_thresholds_per_label)
    })

    print("\nFINAL METRICS")
    print(f"Best epoch:       {best_epoch}")
    print(f"Train loss:       {float(train_loss):.4f}")
    print(f"Val loss:         {float(val_loss):.4f}")
    print(f"Test loss:        {float(test_loss):.4f}")
    print()
    print(f"Test micro F1:    {test_metrics['micro_f1']:.4f}")
    print(f"Test micro F2:    {test_metrics['micro_f2']:.4f}")
    print(f"Test macro F1:    {test_metrics['macro_f1']:.4f}")
    print(f"Test macro F2:    {test_metrics['macro_f2']:.4f}")
    print(f"Test precision:   {test_metrics['micro_precision']:.4f}")
    print(f"Test recall:      {test_metrics['micro_recall']:.4f}")
    print(f"Label accuracy:   {test_metrics['label_accuracy']:.4f}")
    print(f"Subset accuracy:  {test_metrics['subset_accuracy']:.4f}")
    print(f"Thresholds:       {best_thresholds_per_label}")

    print("\nPer-label metrics:")
    display(per_label_metrics)

    conf_matrix_multilabel_with_thresholds(
        Y_test,
        y_test_prob,
        best_thresholds_per_label
    )

    if save_vectorizer:
        vectorizer_filename = f"ORM_tfidf_vectorizer_{combo_name}.pkl"
        with open(vectorizer_filename, "wb") as file:
            pickle.dump(tfidf, file)
        print(f"\nVettorizzatore TF-IDF salvato: {vectorizer_filename}")

    if save_bundle:
        bundle_filename = f"ORM_ANN_bundle_{combo_name}.pkl"

        bundle = {
            "params": best_params,
            "tfidf": tfidf,
            "mlb": mlb,
            "thresholds_per_label": np.array(best_thresholds_per_label),
            "feature_columns": feature_columns,
            "classes": mlb.classes_,
            "combo_name": combo_name
        }

        with open(bundle_filename, "wb") as file:
            pickle.dump(bundle, file)

        print(f"Bundle modello salvato: {bundle_filename}")

    result = {
        "model_type": "ANN_TFIDF_AdamW_Dropout_V3",
        "combo_name": combo_name,
        "features": feature_columns,
        "num_input_columns": len(feature_columns),
        "num_tfidf_features": len(tfidf.get_feature_names_out()),
        "num_classes": num_classes,
        "best_epoch": best_epoch,

        "train_loss": float(train_loss),
        "val_loss": float(val_loss),
        "test_loss": float(test_loss),

        "train_precision": train_metrics["micro_precision"],
        "val_precision": val_metrics["micro_precision"],
        "test_precision": test_metrics["micro_precision"],

        "train_recall": train_metrics["micro_recall"],
        "val_recall": val_metrics["micro_recall"],
        "test_recall": test_metrics["micro_recall"],

        "train_f1": train_metrics["micro_f1"],
        "val_f1": val_metrics["micro_f1"],
        "test_f1": test_metrics["micro_f1"],

        "train_f2": train_metrics["micro_f2"],
        "val_f2": val_metrics["micro_f2"],
        "test_f2": test_metrics["micro_f2"],

        "test_accuracy": test_metrics["label_accuracy"],
        "test_subset_accuracy": test_metrics["subset_accuracy"],

        "test_macro_precision": test_metrics["macro_precision"],
        "test_macro_recall": test_metrics["macro_recall"],
        "test_macro_f1": test_metrics["macro_f1"],
        "test_macro_f2": test_metrics["macro_f2"],

        "thresholds_per_label": np.array(best_thresholds_per_label),
        "classes": mlb.classes_,
        "per_label_metrics": per_label_metrics,
        "params": best_params,
        "tfidf": tfidf,
        "mlb": mlb
    }

    return result


### Funzione con Logistic Regression + SVM per confronto

In [ ]:
def trainTestCombLinearEnsembleStrong(
    df_work,
    feature_columns,
    label_column="Manually assigned PLATFORM",
    combo_name="FEATURE_COMBINATION_LINEAR_ENSEMBLE",
    publication_col="Publication Number",

    threshold_beta=1.0,
    min_precision_threshold=0.55,

    logreg_C_values=(0.5, 1.0, 2.0, 4.0),
    svc_C_values=(0.25, 0.5, 1.0, 2.0),
    ensemble_weights=(0.25, 0.5, 0.75),

    random_state=42,
    save_bundle=True
):
    print("=" * 80)
    print(f"TRAIN/TEST LINEAR ENSEMBLE STRONG - {combo_name}")
    print("=" * 80)

    df_local = df_work.copy()

    required_columns = [label_column] + feature_columns
    missing_columns = [col for col in required_columns if col not in df_local.columns]

    if missing_columns:
        raise ValueError(f"Colonne mancanti nel dataset: {missing_columns}")

    df_local = df_local.dropna(subset=[label_column]).copy()
    df_local = df_local[df_local[label_column].astype(str).str.strip() != ""].copy()

    if publication_col in df_local.columns:
        before = len(df_local)
        df_local[publication_col] = (
            df_local[publication_col]
            .astype(str)
            .str.strip()
            .str.upper()
        )

        df_local = df_local.drop_duplicates(
            subset=[publication_col, label_column]
        ).copy()

        print(f"Deduplica brevetti/label: {before} -> {len(df_local)} righe")

    df_local = df_local.reset_index(drop=True)

    X_features = df_local[feature_columns].fillna("").copy()
    x = build_prefixed_text(X_features, feature_columns)

    y_split = df_local[label_column].apply(parse_multilabel_string)

    mlb = MultiLabelBinarizer()
    y_multi = mlb.fit_transform(y_split)

    print(f"\nNumero classi: {len(mlb.classes_)}")
    print("Classi:", mlb.classes_)

    class_distribution = pd.DataFrame({
        "Classe": mlb.classes_,
        "Numero di sample": y_multi.sum(axis=0)
    }).sort_values("Numero di sample", ascending=False)

    display(class_distribution)

    X_train_text, X_val_text, X_test_text, Y_train, Y_val, Y_test = multilabel_train_val_test_split(
        x,
        y_multi,
        test_size=0.15,
        val_size=0.15,
        random_state=random_state
    )

    print("\nSplit:")
    print("Train size:", len(X_train_text))
    print("Validation size:", len(X_val_text))
    print("Test size:", len(X_test_text))

    X_train_text_clean = [clean_text(text) for text in X_train_text]
    X_val_text_clean = [clean_text(text) for text in X_val_text]
    X_test_text_clean = [clean_text(text) for text in X_test_text]

    word_tfidf = TfidfVectorizer(
        analyzer="word",
        sublinear_tf=True,
        max_features=80000,
        ngram_range=(1, 3),
        min_df=2,
        max_df=0.97,
        stop_words=None,
        strip_accents="unicode",
        token_pattern=r"(?u)\b[A-Z0-9][A-Z0-9/.\-:]+\b|\b[A-Z0-9]\b"
    )

    char_tfidf = TfidfVectorizer(
        analyzer="char_wb",
        sublinear_tf=True,
        max_features=70000,
        ngram_range=(3, 7),
        min_df=2,
        max_df=0.97,
        strip_accents="unicode"
    )

    print("\nFitting TF-IDF word + char...")

    X_train_word = word_tfidf.fit_transform(X_train_text_clean)
    X_val_word = word_tfidf.transform(X_val_text_clean)
    X_test_word = word_tfidf.transform(X_test_text_clean)

    X_train_char = char_tfidf.fit_transform(X_train_text_clean)
    X_val_char = char_tfidf.transform(X_val_text_clean)
    X_test_char = char_tfidf.transform(X_test_text_clean)

    X_train = hstack([X_train_word, X_train_char]).tocsr()
    X_val = hstack([X_val_word, X_val_char]).tocsr()
    X_test = hstack([X_test_word, X_test_char]).tocsr()

    print("\nShape finali sparse:")
    print("X_train:", X_train.shape)
    print("X_val:  ", X_val.shape)
    print("X_test: ", X_test.shape)

    search_rows = []

    best_logreg = None
    best_logreg_prob_val = None
    best_logreg_prob_test = None
    best_logreg_prob_train = None
    best_logreg_params = None
    best_logreg_score = -1.0

    print("\nSearch Logistic Regression...")

    for C in logreg_C_values:
        for class_weight in ("balanced", None):
            print(f"Training LogReg C={C}, class_weight={class_weight}")

            model = OneVsRestClassifier(
                LogisticRegression(
                    solver="liblinear",
                    penalty="l2",
                    class_weight=class_weight,
                    C=C,
                    max_iter=4000,
                    random_state=random_state
                )
            )

            model.fit(X_train, Y_train)

            y_val_prob = model.predict_proba(X_val)

            thresholds = find_best_thresholds_sklearn_constrained(
                Y_val,
                y_val_prob,
                beta=threshold_beta,
                min_precision=min_precision_threshold,
                min_threshold=0.10,
                max_threshold=0.90,
                step=0.02
            )

            metrics = compute_multilabel_metrics_sklearn_extended(
                Y_val,
                y_val_prob,
                thresholds
            )

            score = metrics["micro_f1"]

            search_rows.append({
                "model": "LogReg",
                "C": C,
                "class_weight": class_weight,
                "ensemble_weight_logreg": 1.0,
                "val_precision": metrics["micro_precision"],
                "val_recall": metrics["micro_recall"],
                "val_f1": metrics["micro_f1"],
                "val_f2": metrics["micro_f2"],
                "val_macro_f1": metrics["macro_f1"],
                "val_macro_f2": metrics["macro_f2"]
            })

            if score > best_logreg_score:
                best_logreg_score = score
                best_logreg = model
                best_logreg_prob_val = y_val_prob
                best_logreg_prob_train = model.predict_proba(X_train)
                best_logreg_prob_test = model.predict_proba(X_test)
                best_logreg_params = {
                    "C": C,
                    "class_weight": class_weight
                }

    best_svc = None
    best_svc_prob_val = None
    best_svc_prob_test = None
    best_svc_prob_train = None
    best_svc_params = None
    best_svc_score = -1.0

    print("\nSearch Calibrated LinearSVC...")

    for C in svc_C_values:
        for class_weight in ("balanced", None):
            print(f"Training LinearSVC C={C}, class_weight={class_weight}")

            base_svc = LinearSVC(
                C=C,
                class_weight=class_weight,
                max_iter=6000,
                random_state=random_state
            )

            calibrated_svc = OneVsRestClassifier(
                CalibratedClassifierCV(
                    estimator=base_svc,
                    method="sigmoid",
                    cv=3
                )
            )

            calibrated_svc.fit(X_train, Y_train)

            y_val_prob = calibrated_svc.predict_proba(X_val)

            thresholds = find_best_thresholds_sklearn_constrained(
                Y_val,
                y_val_prob,
                beta=threshold_beta,
                min_precision=min_precision_threshold,
                min_threshold=0.10,
                max_threshold=0.90,
                step=0.02
            )

            metrics = compute_multilabel_metrics_sklearn_extended(
                Y_val,
                y_val_prob,
                thresholds
            )

            score = metrics["micro_f1"]

            search_rows.append({
                "model": "CalibratedLinearSVC",
                "C": C,
                "class_weight": class_weight,
                "ensemble_weight_logreg": 0.0,
                "val_precision": metrics["micro_precision"],
                "val_recall": metrics["micro_recall"],
                "val_f1": metrics["micro_f1"],
                "val_f2": metrics["micro_f2"],
                "val_macro_f1": metrics["macro_f1"],
                "val_macro_f2": metrics["macro_f2"]
            })

            if score > best_svc_score:
                best_svc_score = score
                best_svc = calibrated_svc
                best_svc_prob_val = y_val_prob
                best_svc_prob_train = calibrated_svc.predict_proba(X_train)
                best_svc_prob_test = calibrated_svc.predict_proba(X_test)
                best_svc_params = {
                    "C": C,
                    "class_weight": class_weight
                }

    best_model_type = None
    best_thresholds = None
    best_weight = None
    best_val_score = -1.0
    best_val_prob = None
    best_train_prob = None
    best_test_prob = None

    print("\nSearch ensemble weights...")

    candidates = []

    candidates.append({
        "type": "LogReg",
        "weight": 1.0,
        "train_prob": best_logreg_prob_train,
        "val_prob": best_logreg_prob_val,
        "test_prob": best_logreg_prob_test
    })

    candidates.append({
        "type": "CalibratedLinearSVC",
        "weight": 0.0,
        "train_prob": best_svc_prob_train,
        "val_prob": best_svc_prob_val,
        "test_prob": best_svc_prob_test
    })

    for w in ensemble_weights:
        candidates.append({
            "type": "Ensemble",
            "weight": w,
            "train_prob": w * best_logreg_prob_train + (1.0 - w) * best_svc_prob_train,
            "val_prob": w * best_logreg_prob_val + (1.0 - w) * best_svc_prob_val,
            "test_prob": w * best_logreg_prob_test + (1.0 - w) * best_svc_prob_test
        })

    for candidate in candidates:
        thresholds = find_best_thresholds_sklearn_constrained(
            Y_val,
            candidate["val_prob"],
            beta=threshold_beta,
            min_precision=min_precision_threshold,
            min_threshold=0.10,
            max_threshold=0.90,
            step=0.02
        )

        metrics = compute_multilabel_metrics_sklearn_extended(
            Y_val,
            candidate["val_prob"],
            thresholds
        )

        score = metrics["micro_f1"]

        search_rows.append({
            "model": candidate["type"],
            "C": None,
            "class_weight": None,
            "ensemble_weight_logreg": candidate["weight"],
            "val_precision": metrics["micro_precision"],
            "val_recall": metrics["micro_recall"],
            "val_f1": metrics["micro_f1"],
            "val_f2": metrics["micro_f2"],
            "val_macro_f1": metrics["macro_f1"],
            "val_macro_f2": metrics["macro_f2"]
        })

        if score > best_val_score:
            best_val_score = score
            best_model_type = candidate["type"]
            best_weight = candidate["weight"]
            best_thresholds = thresholds
            best_train_prob = candidate["train_prob"]
            best_val_prob = candidate["val_prob"]
            best_test_prob = candidate["test_prob"]

    search_results_df = pd.DataFrame(search_rows).sort_values(
        by=["val_f1", "val_f2", "val_precision", "val_recall"],
        ascending=False
    )

    print("\nBest model type:", best_model_type)
    print("Best ensemble weight LogReg:", best_weight)
    print("Best LogReg params:", best_logreg_params)
    print("Best SVC params:", best_svc_params)

    print("\nSearch results:")
    display(search_results_df)

    train_metrics = compute_multilabel_metrics_sklearn_extended(
        Y_train,
        best_train_prob,
        best_thresholds
    )

    val_metrics = compute_multilabel_metrics_sklearn_extended(
        Y_val,
        best_val_prob,
        best_thresholds
    )

    test_metrics = compute_multilabel_metrics_sklearn_extended(
        Y_test,
        best_test_prob,
        best_thresholds
    )

    y_test_pred = (best_test_prob >= best_thresholds).astype(int)

    per_label_precision, per_label_recall, per_label_f1, per_label_support = precision_recall_fscore_support(
        Y_test,
        y_test_pred,
        average=None,
        zero_division=0
    )

    per_label_metrics = pd.DataFrame({
        "Classe": mlb.classes_,
        "Support": per_label_support,
        "Precision": per_label_precision,
        "Recall": per_label_recall,
        "F1": per_label_f1,
        "Threshold": best_thresholds
    })

    print("\nFINAL METRICS - LINEAR ENSEMBLE STRONG")
    print(f"Best model type:       {best_model_type}")
    print(f"Best weight LogReg:    {best_weight}")
    print()
    print(f"Test micro F1:         {test_metrics['micro_f1']:.4f}")
    print(f"Test micro F2:         {test_metrics['micro_f2']:.4f}")
    print(f"Test macro F1:         {test_metrics['macro_f1']:.4f}")
    print(f"Test macro F2:         {test_metrics['macro_f2']:.4f}")
    print(f"Test precision:        {test_metrics['micro_precision']:.4f}")
    print(f"Test recall:           {test_metrics['micro_recall']:.4f}")
    print(f"Label accuracy:        {test_metrics['label_accuracy']:.4f}")
    print(f"Subset accuracy:       {test_metrics['subset_accuracy']:.4f}")
    print("Thresholds:", best_thresholds)

    print("\nPer-label metrics:")
    display(per_label_metrics)

    if save_bundle:
        bundle_filename = f"ORM_LINEAR_ENSEMBLE_STRONG_bundle_{combo_name}.pkl"

        bundle = {
            "best_model_type": best_model_type,
            "best_weight_logreg": best_weight,
            "logreg_model": best_logreg,
            "svc_model": best_svc,
            "word_tfidf": word_tfidf,
            "char_tfidf": char_tfidf,
            "mlb": mlb,
            "thresholds_per_label": best_thresholds,
            "feature_columns": feature_columns,
            "classes": mlb.classes_,
            "best_logreg_params": best_logreg_params,
            "best_svc_params": best_svc_params
        }

        with open(bundle_filename, "wb") as file:
            pickle.dump(bundle, file)

        print(f"\nBundle Linear Ensemble salvato: {bundle_filename}")

    result = {
        "model_type": "LinearEnsemble_LogReg_CalibratedSVC_WordChar_TFIDF",
        "combo_name": combo_name,
        "features": feature_columns,
        "num_input_columns": len(feature_columns),
        "num_tfidf_features": X_train.shape[1],
        "num_classes": len(mlb.classes_),

        "best_model_type": best_model_type,
        "best_weight_logreg": best_weight,
        "best_logreg_params": best_logreg_params,
        "best_svc_params": best_svc_params,
        "search_results": search_results_df,

        "train_precision": train_metrics["micro_precision"],
        "val_precision": val_metrics["micro_precision"],
        "test_precision": test_metrics["micro_precision"],

        "train_recall": train_metrics["micro_recall"],
        "val_recall": val_metrics["micro_recall"],
        "test_recall": test_metrics["micro_recall"],

        "train_f1": train_metrics["micro_f1"],
        "val_f1": val_metrics["micro_f1"],
        "test_f1": test_metrics["micro_f1"],

        "train_f2": train_metrics["micro_f2"],
        "val_f2": val_metrics["micro_f2"],
        "test_f2": test_metrics["micro_f2"],

        "test_accuracy": test_metrics["label_accuracy"],
        "test_subset_accuracy": test_metrics["subset_accuracy"],

        "test_macro_precision": test_metrics["macro_precision"],
        "test_macro_recall": test_metrics["macro_recall"],
        "test_macro_f1": test_metrics["macro_f1"],
        "test_macro_f2": test_metrics["macro_f2"],

        "thresholds_per_label": best_thresholds,
        "classes": mlb.classes_,
        "per_label_metrics": per_label_metrics,

        "logreg_model": best_logreg,
        "svc_model": best_svc,
        "word_tfidf": word_tfidf,
        "char_tfidf": char_tfidf,
        "mlb": mlb
    }

    return result


### Funzione con Logistic Regression + SVM Ensemble Ultra

In [ ]:
def trainTestCombLinearEnsembleUltra(
    df_work,
    feature_columns,
    label_column="Manually assigned PLATFORM",
    combo_name="FEATURE_COMBINATION_LINEAR_ENSEMBLE_ULTRA",
    publication_col="Publication Number",

    threshold_beta=1.0,
    min_precision_threshold=0.55,

    logreg_C_values=(0.5, 1.0, 2.0, 4.0, 8.0),
    svc_C_values=(0.5, 1.0, 2.0),
    sgd_alpha_values=(1e-5, 3e-5, 1e-4),

    random_state=42,
    save_bundle=True
):
    print("=" * 80)
    print(f"TRAIN/TEST LINEAR ENSEMBLE ULTRA - {combo_name}")
    print("=" * 80)

    df_local = df_work.copy()

    required_columns = [label_column] + feature_columns
    missing_columns = [col for col in required_columns if col not in df_local.columns]

    if missing_columns:
        raise ValueError(f"Colonne mancanti nel dataset: {missing_columns}")

    df_local = df_local.dropna(subset=[label_column]).copy()
    df_local = df_local[df_local[label_column].astype(str).str.strip() != ""].copy()

    if publication_col in df_local.columns:
        before = len(df_local)
        df_local[publication_col] = (
            df_local[publication_col]
            .astype(str)
            .str.strip()
            .str.upper()
        )

        df_local = df_local.drop_duplicates(
            subset=[publication_col, label_column]
        ).copy()

        print(f"Deduplica brevetti/label: {before} -> {len(df_local)} righe")

    df_local = df_local.reset_index(drop=True)

    X_features = df_local[feature_columns].fillna("").copy()
    x = build_prefixed_text(X_features, feature_columns)

    y_split = df_local[label_column].apply(parse_multilabel_string)

    mlb = MultiLabelBinarizer()
    y_multi = mlb.fit_transform(y_split)

    print(f"\nNumero classi: {len(mlb.classes_)}")
    print("Classi:", mlb.classes_)

    class_distribution = pd.DataFrame({
        "Classe": mlb.classes_,
        "Numero di sample": y_multi.sum(axis=0)
    }).sort_values("Numero di sample", ascending=False)

    display(class_distribution)

    X_train_text, X_val_text, X_test_text, Y_train, Y_val, Y_test = multilabel_train_val_test_split(
        x,
        y_multi,
        test_size=0.30,
        val_size=0.15,
        random_state=random_state
    )

    print("\nSplit:")
    print("Train size:", len(X_train_text))
    print("Validation size:", len(X_val_text))
    print("Test size:", len(X_test_text))

    X_train_text_clean = [clean_text(text) for text in X_train_text]
    X_val_text_clean = [clean_text(text) for text in X_val_text]
    X_test_text_clean = [clean_text(text) for text in X_test_text]

    word_tfidf = TfidfVectorizer(
        analyzer="word",
        sublinear_tf=True,
        max_features=100000,
        ngram_range=(1, 4),
        min_df=2,
        max_df=0.98,
        stop_words=None,
        strip_accents="unicode",
        token_pattern=r"(?u)\b[A-Z0-9][A-Z0-9/.\-:]+\b|\b[A-Z0-9]\b"
    )

    char_tfidf = TfidfVectorizer(
        analyzer="char_wb",
        sublinear_tf=True,
        max_features=90000,
        ngram_range=(3, 7),
        min_df=2,
        max_df=0.98,
        strip_accents="unicode"
    )

    print("\nFitting TF-IDF word + char...")

    X_train_word = word_tfidf.fit_transform(X_train_text_clean)
    X_val_word = word_tfidf.transform(X_val_text_clean)
    X_test_word = word_tfidf.transform(X_test_text_clean)

    X_train_char = char_tfidf.fit_transform(X_train_text_clean)
    X_val_char = char_tfidf.transform(X_val_text_clean)
    X_test_char = char_tfidf.transform(X_test_text_clean)

    X_train = hstack([X_train_word, X_train_char]).tocsr()
    X_val = hstack([X_val_word, X_val_char]).tocsr()
    X_test = hstack([X_test_word, X_test_char]).tocsr()

    print("\nShape finali sparse:")
    print("X_train:", X_train.shape)
    print("X_val:  ", X_val.shape)
    print("X_test: ", X_test.shape)

    search_rows = []

    best_logreg = None
    best_logreg_score = -1.0

    print("\nSearch Logistic Regression...")

    for C in logreg_C_values:
        for class_weight in ("balanced", None):
            print(f"Training LogReg C={C}, class_weight={class_weight}")

            model = OneVsRestClassifier(
                LogisticRegression(
                    solver="liblinear",
                    penalty="l2",
                    class_weight=class_weight,
                    C=C,
                    max_iter=5000,
                    random_state=random_state
                )
            )

            model.fit(X_train, Y_train)

            y_val_prob = model.predict_proba(X_val)

            thresholds = find_best_thresholds_sklearn_constrained(
                Y_val,
                y_val_prob,
                beta=threshold_beta,
                min_precision=min_precision_threshold,
                min_threshold=0.10,
                max_threshold=0.90,
                step=0.02
            )

            metrics = compute_multilabel_metrics_sklearn_extended(
                Y_val,
                y_val_prob,
                thresholds
            )

            score = metrics["micro_f1"]

            search_rows.append({
                "model": "LogReg",
                "params": f"C={C}, class_weight={class_weight}",
                "val_precision": metrics["micro_precision"],
                "val_recall": metrics["micro_recall"],
                "val_f1": metrics["micro_f1"],
                "val_f2": metrics["micro_f2"],
                "val_macro_f1": metrics["macro_f1"],
                "val_macro_f2": metrics["macro_f2"]
            })

            if score > best_logreg_score:
                best_logreg_score = score
                best_logreg = model
                best_logreg_params = {
                    "C": C,
                    "class_weight": class_weight
                }

    best_svc = None
    best_svc_score = -1.0

    print("\nSearch Calibrated LinearSVC...")

    for C in svc_C_values:
        for class_weight in ("balanced", None):
            print(f"Training LinearSVC C={C}, class_weight={class_weight}")

            base_svc = LinearSVC(
                C=C,
                class_weight=class_weight,
                max_iter=7000,
                random_state=random_state
            )

            model = OneVsRestClassifier(
                CalibratedClassifierCV(
                    estimator=base_svc,
                    method="sigmoid",
                    cv=3
                )
            )

            model.fit(X_train, Y_train)

            y_val_prob = model.predict_proba(X_val)

            thresholds = find_best_thresholds_sklearn_constrained(
                Y_val,
                y_val_prob,
                beta=threshold_beta,
                min_precision=min_precision_threshold,
                min_threshold=0.10,
                max_threshold=0.90,
                step=0.02
            )

            metrics = compute_multilabel_metrics_sklearn_extended(
                Y_val,
                y_val_prob,
                thresholds
            )

            score = metrics["micro_f1"]

            search_rows.append({
                "model": "CalibratedLinearSVC",
                "params": f"C={C}, class_weight={class_weight}",
                "val_precision": metrics["micro_precision"],
                "val_recall": metrics["micro_recall"],
                "val_f1": metrics["micro_f1"],
                "val_f2": metrics["micro_f2"],
                "val_macro_f1": metrics["macro_f1"],
                "val_macro_f2": metrics["macro_f2"]
            })

            if score > best_svc_score:
                best_svc_score = score
                best_svc = model
                best_svc_params = {
                    "C": C,
                    "class_weight": class_weight
                }

    best_sgd = None
    best_sgd_score = -1.0

    print("\nSearch SGD Logistic...")

    for alpha in sgd_alpha_values:
        for class_weight in ("balanced", None):
            print(f"Training SGD alpha={alpha}, class_weight={class_weight}")

            model = OneVsRestClassifier(
                SGDClassifier(
                    loss="log_loss",
                    penalty="elasticnet",
                    alpha=alpha,
                    l1_ratio=0.15,
                    class_weight=class_weight,
                    max_iter=3000,
                    tol=1e-4,
                    random_state=random_state
                )
            )

            model.fit(X_train, Y_train)

            y_val_prob = model.predict_proba(X_val)

            thresholds = find_best_thresholds_sklearn_constrained(
                Y_val,
                y_val_prob,
                beta=threshold_beta,
                min_precision=min_precision_threshold,
                min_threshold=0.10,
                max_threshold=0.90,
                step=0.02
            )

            metrics = compute_multilabel_metrics_sklearn_extended(
                Y_val,
                y_val_prob,
                thresholds
            )

            score = metrics["micro_f1"]

            search_rows.append({
                "model": "SGDLogistic",
                "params": f"alpha={alpha}, class_weight={class_weight}",
                "val_precision": metrics["micro_precision"],
                "val_recall": metrics["micro_recall"],
                "val_f1": metrics["micro_f1"],
                "val_f2": metrics["micro_f2"],
                "val_macro_f1": metrics["macro_f1"],
                "val_macro_f2": metrics["macro_f2"]
            })

            if score > best_sgd_score:
                best_sgd_score = score
                best_sgd = model
                best_sgd_params = {
                    "alpha": alpha,
                    "class_weight": class_weight
                }

    print("\nComputing best model probabilities...")

    logreg_train_prob = best_logreg.predict_proba(X_train)
    logreg_val_prob = best_logreg.predict_proba(X_val)
    logreg_test_prob = best_logreg.predict_proba(X_test)

    svc_train_prob = best_svc.predict_proba(X_train)
    svc_val_prob = best_svc.predict_proba(X_val)
    svc_test_prob = best_svc.predict_proba(X_test)

    sgd_train_prob = best_sgd.predict_proba(X_train)
    sgd_val_prob = best_sgd.predict_proba(X_val)
    sgd_test_prob = best_sgd.predict_proba(X_test)

    print("\nOptimizing per-label ensemble weights and thresholds...")

    best_weights, best_thresholds = optimize_per_label_ensemble(
        Y_val,
        logreg_val_prob,
        svc_val_prob,
        sgd_val_prob,
        beta=threshold_beta,
        min_precision=min_precision_threshold
    )

    best_train_prob = apply_per_label_ensemble(
        logreg_train_prob,
        svc_train_prob,
        sgd_train_prob,
        best_weights
    )

    best_val_prob = apply_per_label_ensemble(
        logreg_val_prob,
        svc_val_prob,
        sgd_val_prob,
        best_weights
    )

    best_test_prob = apply_per_label_ensemble(
        logreg_test_prob,
        svc_test_prob,
        sgd_test_prob,
        best_weights
    )

    train_metrics = compute_multilabel_metrics_sklearn_extended(
        Y_train,
        best_train_prob,
        best_thresholds
    )

    val_metrics = compute_multilabel_metrics_sklearn_extended(
        Y_val,
        best_val_prob,
        best_thresholds
    )

    test_metrics = compute_multilabel_metrics_sklearn_extended(
        Y_test,
        best_test_prob,
        best_thresholds
    )

    search_rows.append({
        "model": "PerLabelEnsemble",
        "params": "LogReg + CalibratedLinearSVC + SGDLogistic",
        "val_precision": val_metrics["micro_precision"],
        "val_recall": val_metrics["micro_recall"],
        "val_f1": val_metrics["micro_f1"],
        "val_f2": val_metrics["micro_f2"],
        "val_macro_f1": val_metrics["macro_f1"],
        "val_macro_f2": val_metrics["macro_f2"]
    })

    search_results_df = pd.DataFrame(search_rows).sort_values(
        by=["val_f1", "val_f2", "val_precision", "val_recall"],
        ascending=False
    )

    print("\nSearch results:")
    display(search_results_df)

    y_test_pred = (best_test_prob >= best_thresholds).astype(int)

    per_label_precision, per_label_recall, per_label_f1, per_label_support = precision_recall_fscore_support(
        Y_test,
        y_test_pred,
        average=None,
        zero_division=0
    )

    per_label_metrics = pd.DataFrame({
        "Classe": mlb.classes_,
        "Support": per_label_support,
        "Precision": per_label_precision,
        "Recall": per_label_recall,
        "F1": per_label_f1,
        "Threshold": best_thresholds,
        "Weight_LogReg": best_weights[:, 0],
        "Weight_SVC": best_weights[:, 1],
        "Weight_SGD": best_weights[:, 2]
    })

    print("\nFINAL METRICS - LINEAR ENSEMBLE ULTRA")
    print(f"Best LogReg params: {best_logreg_params}")
    print(f"Best SVC params:    {best_svc_params}")
    print(f"Best SGD params:    {best_sgd_params}")
    print()
    print(f"Test micro F1:      {test_metrics['micro_f1']:.4f}")
    print(f"Test micro F2:      {test_metrics['micro_f2']:.4f}")
    print(f"Test macro F1:      {test_metrics['macro_f1']:.4f}")
    print(f"Test macro F2:      {test_metrics['macro_f2']:.4f}")
    print(f"Test precision:     {test_metrics['micro_precision']:.4f}")
    print(f"Test recall:        {test_metrics['micro_recall']:.4f}")
    print(f"Label accuracy:     {test_metrics['label_accuracy']:.4f}")
    print(f"Subset accuracy:    {test_metrics['subset_accuracy']:.4f}")

    print("\nPer-label metrics:")
    display(per_label_metrics)

    if save_bundle:
        bundle_filename = f"ORM_LINEAR_ENSEMBLE_ULTRA_bundle_{combo_name}.pkl"

        bundle = {
            "logreg_model": best_logreg,
            "svc_model": best_svc,
            "sgd_model": best_sgd,
            "word_tfidf": word_tfidf,
            "char_tfidf": char_tfidf,
            "mlb": mlb,
            "thresholds_per_label": best_thresholds,
            "weights_per_label": best_weights,
            "feature_columns": feature_columns,
            "classes": mlb.classes_,
            "best_logreg_params": best_logreg_params,
            "best_svc_params": best_svc_params,
            "best_sgd_params": best_sgd_params
        }

        with open(bundle_filename, "wb") as file:
            pickle.dump(bundle, file)

        print(f"\nBundle Linear Ensemble Ultra salvato: {bundle_filename}")

    result = {
        "model_type": "LinearEnsembleUltra_LogReg_CalibratedSVC_SGD_WordChar_TFIDF",
        "combo_name": combo_name,
        "features": feature_columns,
        "num_input_columns": len(feature_columns),
        "num_tfidf_features": X_train.shape[1],
        "num_classes": len(mlb.classes_),

        "best_logreg_params": best_logreg_params,
        "best_svc_params": best_svc_params,
        "best_sgd_params": best_sgd_params,
        "search_results": search_results_df,

        "train_precision": train_metrics["micro_precision"],
        "val_precision": val_metrics["micro_precision"],
        "test_precision": test_metrics["micro_precision"],

        "train_recall": train_metrics["micro_recall"],
        "val_recall": val_metrics["micro_recall"],
        "test_recall": test_metrics["micro_recall"],

        "train_f1": train_metrics["micro_f1"],
        "val_f1": val_metrics["micro_f1"],
        "test_f1": test_metrics["micro_f1"],

        "train_f2": train_metrics["micro_f2"],
        "val_f2": val_metrics["micro_f2"],
        "test_f2": test_metrics["micro_f2"],

        "test_accuracy": test_metrics["label_accuracy"],
        "test_subset_accuracy": test_metrics["subset_accuracy"],

        "test_macro_precision": test_metrics["macro_precision"],
        "test_macro_recall": test_metrics["macro_recall"],
        "test_macro_f1": test_metrics["macro_f1"],
        "test_macro_f2": test_metrics["macro_f2"],

        "thresholds_per_label": best_thresholds,
        "weights_per_label": best_weights,
        "classes": mlb.classes_,
        "per_label_metrics": per_label_metrics,

        "logreg_model": best_logreg,
        "svc_model": best_svc,
        "sgd_model": best_sgd,
        "word_tfidf": word_tfidf,
        "char_tfidf": char_tfidf,
        "mlb": mlb
    }

    return result


### Funzione con ANN standard Improved

In [ ]:
def trainTestCombANNImproved(
    df_work,
    feature_columns,
    label_column="Manually assigned PLATFORM",
    combo_name="FEATURE_COMBINATION",

    publication_col="Publication Number",

    min_count_oversampling=800,
    use_oversampling=True,
    use_pos_weights=False,

    thresholds_per_label=None,
    threshold_beta=1.0,
    min_precision_threshold=None,

    num_epochs=120,
    batch_size=64,
    learning_rate=1e-4,
    decay_rate=0.95,
    delta=1e-8,
    max_weight=3.0,
    hidden_units=128,
    patience=15,

    random_state=42,
    save_vectorizer=True
):
    print("=" * 80)
    print(f"TRAIN/TEST ANN IMPROVED - {combo_name}")
    print("=" * 80)

    np.random.seed(random_state)

    print("Feature usate:")
    for col in feature_columns:
        print("-", col)

    df_local = df_work.copy()

    required_columns = [label_column] + feature_columns
    missing_columns = [col for col in required_columns if col not in df_local.columns]

    if missing_columns:
        raise ValueError(f"Colonne mancanti nel dataset: {missing_columns}")

    df_local = df_local.dropna(subset=[label_column]).copy()
    df_local = df_local[df_local[label_column].astype(str).str.strip() != ""].copy()

    if publication_col in df_local.columns:
        before = len(df_local)
        df_local[publication_col] = (
            df_local[publication_col]
            .astype(str)
            .str.strip()
            .str.upper()
        )

        df_local = df_local.drop_duplicates(
            subset=[publication_col, label_column]
        ).copy()

        print(f"Deduplica brevetti/label: {before} -> {len(df_local)} righe")

    df_local = df_local.reset_index(drop=True)

    X = df_local[feature_columns].fillna("").copy()
    x = build_prefixed_text_local(X, feature_columns)

    y_raw = df_local[label_column].astype(str)
    y_split = y_raw.apply(lambda value: [token for token in value.split() if token.strip() != ""])

    mlb = MultiLabelBinarizer()
    y_multi = mlb.fit_transform(y_split)

    num_classes = len(mlb.classes_)

    print()
    print(f"Numero di classi reali: {num_classes}")
    print(f"Classi: {mlb.classes_}")

    class_distribution = pd.DataFrame({
        "Classe": mlb.classes_,
        "Numero di sample": y_multi.sum(axis=0)
    }).sort_values(by="Numero di sample", ascending=False)

    display(class_distribution)

    try:
        from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

        msss_test = MultilabelStratifiedShuffleSplit(
            n_splits=1,
            test_size=0.15,
            random_state=random_state
        )

        train_val_idx, test_idx = next(
            msss_test.split(np.zeros(len(y_multi)), y_multi)
        )

        x_train_val = x.iloc[train_val_idx].reset_index(drop=True)
        y_train_val = y_multi[train_val_idx]

        X_test_text = x.iloc[test_idx].reset_index(drop=True)
        Y_test = y_multi[test_idx]

        msss_val = MultilabelStratifiedShuffleSplit(
            n_splits=1,
            test_size=0.15 / 0.85,
            random_state=random_state
        )

        train_idx, val_idx = next(
            msss_val.split(np.zeros(len(y_train_val)), y_train_val)
        )

        X_train_text = x_train_val.iloc[train_idx].reset_index(drop=True)
        Y_train = y_train_val[train_idx]

        X_val_text = x_train_val.iloc[val_idx].reset_index(drop=True)
        Y_val = y_train_val[val_idx]

        print("\nSplit multi-label stratificato: OK")

    except Exception as exc:
        print("\nSplit multi-label stratificato non disponibile.")
        print("Uso train_test_split standard.")
        print("Motivo:", exc)

        x_shuffled, y_shuffled = shuffle(
            x[:len(y_multi)],
            y_multi,
            random_state=random_state
        )

        X_temp, X_test_text, Y_temp, Y_test = train_test_split(
            x_shuffled,
            y_shuffled,
            test_size=0.15,
            random_state=random_state
        )

        X_train_text, X_val_text, Y_train, Y_val = train_test_split(
            X_temp,
            Y_temp,
            test_size=0.15 / 0.85,
            random_state=random_state
        )

    print()
    print("Train size:", len(X_train_text))
    print("Validation size:", len(X_val_text))
    print("Test size:", len(X_test_text))

    tfidf = TfidfVectorizer(
        sublinear_tf=True,
        max_features=50000,
        ngram_range=(1, 3),
        min_df=2,
        max_df=0.97,
        stop_words=None,
        strip_accents="unicode",
        token_pattern=r"(?u)\b[A-Z0-9][A-Z0-9/.\-:]+\b|\b[A-Z0-9]\b"
    )

    X_train_text_clean = [clean_text_ann(text) for text in X_train_text]
    X_val_text_clean = [clean_text_ann(text) for text in X_val_text]
    X_test_text_clean = [clean_text_ann(text) for text in X_test_text]

    X_train_tfidf = tfidf.fit_transform(X_train_text_clean)
    X_val_tfidf = tfidf.transform(X_val_text_clean)
    X_test_tfidf = tfidf.transform(X_test_text_clean)

    if save_vectorizer:
        vectorizer_filename = f"ORM_tfidf_vectorizer_IMPROVED_{combo_name}.pkl"
        with open(vectorizer_filename, "wb") as file:
            pickle.dump(tfidf, file)

        print()
        print(f"Vettorizzatore TF-IDF salvato: {vectorizer_filename}")

    X_train = jnp.array(X_train_tfidf.toarray(), dtype=jnp.float32)
    X_val = jnp.array(X_val_tfidf.toarray(), dtype=jnp.float32)
    X_test = jnp.array(X_test_tfidf.toarray(), dtype=jnp.float32)

    Y_train = jnp.array(Y_train, dtype=jnp.float32)
    Y_val = jnp.array(Y_val, dtype=jnp.float32)
    Y_test = jnp.array(Y_test, dtype=jnp.float32)

    print()
    print("Shape finali:")
    print("X_train:", X_train.shape)
    print("X_val:  ", X_val.shape)
    print("X_test: ", X_test.shape)
    print("Y_train:", Y_train.shape)
    print("Y_val:  ", Y_val.shape)
    print("Y_test: ", Y_test.shape)

    print()
    print("Numero effettivo di feature TF-IDF:", len(tfidf.get_feature_names_out()))

    print()
    print("Campioni per classe nel training reale:")
    print(pd.DataFrame({
        "Classe": mlb.classes_,
        "Train samples": np.sum(np.array(Y_train), axis=0)
    }))

    if use_oversampling:
        X_train_fit, Y_train_fit = oversample_minority_multilabel(
            X_train,
            Y_train,
            min_count=min_count_oversampling
        )
    else:
        X_train_fit, Y_train_fit = X_train, Y_train

    print()
    print("Training shape effettiva:", X_train_fit.shape, Y_train_fit.shape)
    print("Campioni per classe nel training effettivo:")
    print(np.sum(np.array(Y_train_fit), axis=0))

    input_dim = X_train_fit.shape[1]
    output_dim = Y_train_fit.shape[1]

    layers_size = [input_dim, hidden_units, output_dim]

    if thresholds_per_label is not None:
        thresholds_per_label = jnp.asarray(thresholds_per_label, dtype=jnp.float32)

        if len(thresholds_per_label) != output_dim:
            raise ValueError(
                f"Numero soglie ({len(thresholds_per_label)}) diverso dal numero classi ({output_dim})"
            )

    print()
    print("PARAMETRI USATI")
    print(f"Layers size:             {layers_size}")
    print(f"Epochs max:              {num_epochs}")
    print(f"Batch size:              {batch_size}")
    print(f"Learning rate:           {learning_rate}")
    print(f"Decay rate:              {decay_rate}")
    print(f"Delta:                   {delta}")
    print(f"Max weight:              {max_weight}")
    print(f"Oversampling:            {use_oversampling}")
    print(f"Pos weights:             {use_pos_weights}")
    print(f"Threshold beta:          {threshold_beta}")
    print(f"Min precision threshold: {min_precision_threshold}")
    print(f"Early stopping patience: {patience}")
    print()

    key = jax.random.PRNGKey(random_state)
    params = initialize_params(layers_size, key)

    if use_pos_weights:
        pos_weights = compute_pos_weights(
            Y_train_fit,
            max_weight=max_weight
        )
    else:
        pos_weights = jnp.ones((output_dim,), dtype=jnp.float32)

    cumulated_sqr_grad = [
        (jnp.zeros_like(W), jnp.zeros_like(b))
        for W, b in params
    ]

    n_training_points = X_train_fit.shape[0]

    best_params = None
    best_thresholds_per_label = None
    best_score = -1.0
    best_epoch = 0
    epochs_without_improvement = 0

    for epoch in range(num_epochs):
        permutation = np.random.permutation(n_training_points)
        epoch_losses = []

        for start_idx in range(0, n_training_points, batch_size):
            end_idx = start_idx + batch_size
            batch_idxs = permutation[start_idx:end_idx]

            x_batch = X_train_fit[batch_idxs, :]
            y_batch = Y_train_fit[batch_idxs, :]

            params, cumulated_sqr_grad, train_loss_batch = rmsprop_update(
                params,
                x_batch,
                y_batch,
                pos_weights,
                learning_rate,
                decay_rate,
                delta,
                cumulated_sqr_grad
            )

            epoch_losses.append(float(train_loss_batch))

        y_val_prob = ANN(X_val, params)

        if thresholds_per_label is None:
            epoch_thresholds = find_best_thresholds_per_label_constrained(
                Y_val,
                y_val_prob,
                beta=threshold_beta,
                min_precision=min_precision_threshold,
                min_threshold=0.10,
                max_threshold=0.90,
                step=0.02
            )
        else:
            epoch_thresholds = thresholds_per_label

        val_f = multilabel_fbeta_score_with_thresholds(
            Y_val,
            y_val_prob,
            epoch_thresholds,
            beta=threshold_beta
        )

        val_precision = multilabel_precision_with_thresholds(
            Y_val,
            y_val_prob,
            epoch_thresholds
        )

        val_recall = multilabel_recall_with_thresholds(
            Y_val,
            y_val_prob,
            epoch_thresholds
        )

        if float(val_f) > best_score:
            best_score = float(val_f)
            best_params = copy_params(params)
            best_thresholds_per_label = jnp.asarray(epoch_thresholds, dtype=jnp.float32)
            best_epoch = epoch + 1
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(
                f"Epoch {epoch + 1:3d}/{num_epochs} - "
                f"loss={np.mean(epoch_losses):.6f} - "
                f"val_F{threshold_beta}={float(val_f):.4f} - "
                f"val_precision={float(val_precision):.4f} - "
                f"val_recall={float(val_recall):.4f} - "
                f"best_epoch={best_epoch}"
            )

        if epochs_without_improvement >= patience:
            print(f"Early stopping alla epoch {epoch + 1}. Best epoch: {best_epoch}")
            break

    best_pos_weights = pos_weights

    train_loss = loss_function(best_params, X_train, Y_train, best_pos_weights)
    val_loss = loss_function(best_params, X_val, Y_val, best_pos_weights)
    test_loss = loss_function(best_params, X_test, Y_test, best_pos_weights)

    y_train_prob = ANN(X_train, best_params)
    y_val_prob = ANN(X_val, best_params)
    y_test_prob = ANN(X_test, best_params)

    train_acc = multilabel_accuracy_with_thresholds(Y_train, y_train_prob, best_thresholds_per_label)
    val_acc = multilabel_accuracy_with_thresholds(Y_val, y_val_prob, best_thresholds_per_label)
    test_acc = multilabel_accuracy_with_thresholds(Y_test, y_test_prob, best_thresholds_per_label)

    train_precision = multilabel_precision_with_thresholds(Y_train, y_train_prob, best_thresholds_per_label)
    val_precision = multilabel_precision_with_thresholds(Y_val, y_val_prob, best_thresholds_per_label)
    test_precision = multilabel_precision_with_thresholds(Y_test, y_test_prob, best_thresholds_per_label)

    train_recall = multilabel_recall_with_thresholds(Y_train, y_train_prob, best_thresholds_per_label)
    val_recall = multilabel_recall_with_thresholds(Y_val, y_val_prob, best_thresholds_per_label)
    test_recall = multilabel_recall_with_thresholds(Y_test, y_test_prob, best_thresholds_per_label)

    train_f1 = multilabel_fbeta_score_with_thresholds(Y_train, y_train_prob, best_thresholds_per_label, beta=1.0)
    val_f1 = multilabel_fbeta_score_with_thresholds(Y_val, y_val_prob, best_thresholds_per_label, beta=1.0)
    test_f1 = multilabel_fbeta_score_with_thresholds(Y_test, y_test_prob, best_thresholds_per_label, beta=1.0)

    train_f2 = multilabel_fbeta_score_with_thresholds(Y_train, y_train_prob, best_thresholds_per_label, beta=2.0)
    val_f2 = multilabel_fbeta_score_with_thresholds(Y_val, y_val_prob, best_thresholds_per_label, beta=2.0)
    test_f2 = multilabel_fbeta_score_with_thresholds(Y_test, y_test_prob, best_thresholds_per_label, beta=2.0)

    y_test_pred = np.array(
        predict_with_thresholds(y_test_prob, best_thresholds_per_label)
    )

    y_test_true = np.array(Y_test)

    per_label_precision, per_label_recall, per_label_f1, per_label_support = precision_recall_fscore_support(
        y_test_true,
        y_test_pred,
        average=None,
        zero_division=0
    )

    per_label_metrics = pd.DataFrame({
        "Classe": mlb.classes_,
        "Support": per_label_support,
        "Precision": per_label_precision,
        "Recall": per_label_recall,
        "F1": per_label_f1,
        "Threshold": np.array(best_thresholds_per_label)
    })

    print()
    print("FINAL METRICS")
    print(f"Best epoch:      {best_epoch}")
    print(f"Best val F{threshold_beta}: {best_score:.4f}")
    print(f"Train loss:      {float(train_loss):.4f}")
    print(f"Val loss:        {float(val_loss):.4f}")
    print(f"Test loss:       {float(test_loss):.4f}")
    print()
    print(f"Train accuracy:  {float(train_acc):.4f}")
    print(f"Val accuracy:    {float(val_acc):.4f}")
    print(f"Test accuracy:   {float(test_acc):.4f}")
    print()
    print(f"Train precision: {float(train_precision):.4f}")
    print(f"Val precision:   {float(val_precision):.4f}")
    print(f"Test precision:  {float(test_precision):.4f}")
    print()
    print(f"Train recall:    {float(train_recall):.4f}")
    print(f"Val recall:      {float(val_recall):.4f}")
    print(f"Test recall:     {float(test_recall):.4f}")
    print()
    print(f"Train F1-score:  {float(train_f1):.4f}")
    print(f"Val F1-score:    {float(val_f1):.4f}")
    print(f"Test F1-score:   {float(test_f1):.4f}")
    print()
    print(f"Train F2-score:  {float(train_f2):.4f}")
    print(f"Val F2-score:    {float(val_f2):.4f}")
    print(f"Test F2-score:   {float(test_f2):.4f}")
    print()
    print(f"Thresholds:      {best_thresholds_per_label}")

    print()
    print("Per-label metrics:")
    display(per_label_metrics)

    conf_matrix_multilabel_with_thresholds(
        Y_test,
        y_test_prob,
        best_thresholds_per_label
    )

    print()

    conf_matrix_per_label_with_thresholds(
        Y_test,
        y_test_prob,
        best_thresholds_per_label,
        class_names=mlb.classes_
    )

    result = {
        "combo_name": combo_name,
        "features": feature_columns,
        "num_input_columns": len(feature_columns),
        "num_tfidf_features": len(tfidf.get_feature_names_out()),
        "num_classes": num_classes,
        "best_epoch": best_epoch,
        "best_val_score": best_score,

        "train_loss": float(train_loss),
        "val_loss": float(val_loss),
        "test_loss": float(test_loss),

        "train_accuracy": float(train_acc),
        "val_accuracy": float(val_acc),
        "test_accuracy": float(test_acc),

        "train_precision": float(train_precision),
        "val_precision": float(val_precision),
        "test_precision": float(test_precision),

        "train_recall": float(train_recall),
        "val_recall": float(val_recall),
        "test_recall": float(test_recall),

        "train_f1": float(train_f1),
        "val_f1": float(val_f1),
        "test_f1": float(test_f1),

        "train_f2": float(train_f2),
        "val_f2": float(val_f2),
        "test_f2": float(test_f2),

        "thresholds_per_label": np.array(best_thresholds_per_label),
        "classes": mlb.classes_,
        "per_label_metrics": per_label_metrics,
        "params": best_params,
        "tfidf": tfidf,
        "mlb": mlb
    }

    return result


### SentenceTransformer Embeddings + OneVsRest LogisticRegression

In [ ]:
!pip install -q sentence-transformers

from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_recall_fscore_support, fbeta_score
import numpy as np
import pandas as pd
import pickle


def trainTestCombSentenceTransformerFast(
    df_work,
    feature_columns,
    label_column="Manually assigned PLATFORM",
    combo_name="SENTENCE_TRANSFORMER_FAST",
    publication_col="Publication Number",

    embedding_model_name="sentence-transformers/all-MiniLM-L6-v2",

    threshold_beta=1.0,
    min_precision_threshold=0.55,

    logreg_C_values=(1.0, 2.0),
    sgd_alpha_values=(3e-5, 1e-4),

    batch_size=128,
    random_state=42,
    save_bundle=True
):
    print("=" * 80)
    print(f"TRAIN/TEST SENTENCE TRANSFORMER FAST - {combo_name}")
    print("=" * 80)

    df_local = df_work.copy()

    required_columns = [label_column] + feature_columns
    missing_columns = [col for col in required_columns if col not in df_local.columns]
    if missing_columns:
        raise ValueError(f"Colonne mancanti nel dataset: {missing_columns}")

    df_local = df_local.dropna(subset=[label_column]).copy()
    df_local = df_local[df_local[label_column].astype(str).str.strip() != ""].copy()

    if publication_col in df_local.columns:
        before = len(df_local)
        df_local[publication_col] = (
            df_local[publication_col]
            .astype(str)
            .str.strip()
            .str.upper()
        )
        df_local = df_local.drop_duplicates(
            subset=[publication_col, label_column]
        ).copy()
        print(f"Deduplica brevetti/label: {before} -> {len(df_local)} righe")

    df_local = df_local.reset_index(drop=True)

    x = build_prefixed_text(
        df_local[feature_columns].fillna(""),
        feature_columns
    )

    y_split = df_local[label_column].apply(parse_multilabel_string)

    mlb = MultiLabelBinarizer()
    y_multi = mlb.fit_transform(y_split)

    print(f"\nNumero classi: {len(mlb.classes_)}")
    print("Classi:", mlb.classes_)

    class_distribution = pd.DataFrame({
        "Classe": mlb.classes_,
        "Numero di sample": y_multi.sum(axis=0)
    }).sort_values("Numero di sample", ascending=False)
    display(class_distribution)

    X_train_text, X_val_text, X_test_text, Y_train, Y_val, Y_test = multilabel_train_val_test_split(
        x,
        y_multi,
        test_size=0.15,
        val_size=0.15,
        random_state=random_state
    )

    print("\nSplit:")
    print("Train size:", len(X_train_text))
    print("Validation size:", len(X_val_text))
    print("Test size:", len(X_test_text))

    X_train_text_clean = [clean_text(text) for text in X_train_text]
    X_val_text_clean = [clean_text(text) for text in X_val_text]
    X_test_text_clean = [clean_text(text) for text in X_test_text]

    print(f"\nCaricamento embedding model: {embedding_model_name}")
    embedding_model = SentenceTransformer(embedding_model_name)

    print("\nEncoding embeddings...")

    X_train_emb = embedding_model.encode(
        X_train_text_clean,
        batch_size=batch_size,
        show_progress_bar=True,
        normalize_embeddings=True
    )

    X_val_emb = embedding_model.encode(
        X_val_text_clean,
        batch_size=batch_size,
        show_progress_bar=True,
        normalize_embeddings=True
    )

    X_test_emb = embedding_model.encode(
        X_test_text_clean,
        batch_size=batch_size,
        show_progress_bar=True,
        normalize_embeddings=True
    )

    print("\nShape embeddings:")
    print("X_train_emb:", X_train_emb.shape)
    print("X_val_emb:  ", X_val_emb.shape)
    print("X_test_emb: ", X_test_emb.shape)

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train_emb)
    X_val = scaler.transform(X_val_emb)
    X_test = scaler.transform(X_test_emb)

    search_rows = []

    best_model = None
    best_model_name = None
    best_params = None
    best_thresholds = None
    best_val_score = -1.0

    print("\nSearch Logistic Regression...")

    for C in logreg_C_values:
        for class_weight in ("balanced", None):
            print(f"Training LogReg C={C}, class_weight={class_weight}")

            model = OneVsRestClassifier(
                LogisticRegression(
                    solver="liblinear",
                    penalty="l2",
                    C=C,
                    class_weight=class_weight,
                    max_iter=2000,
                    random_state=random_state
                )
            )

            model.fit(X_train, Y_train)
            y_val_prob = model.predict_proba(X_val)

            thresholds = find_best_thresholds_sklearn_constrained(
                Y_val,
                y_val_prob,
                beta=threshold_beta,
                min_precision=min_precision_threshold,
                min_threshold=0.10,
                max_threshold=0.90,
                step=0.03
            )

            metrics = compute_multilabel_metrics_sklearn_extended(
                Y_val,
                y_val_prob,
                thresholds
            )

            score = metrics["micro_f1"]

            search_rows.append({
                "model": "LogReg",
                "params": f"C={C}, class_weight={class_weight}",
                "val_precision": metrics["micro_precision"],
                "val_recall": metrics["micro_recall"],
                "val_f1": metrics["micro_f1"],
                "val_f2": metrics["micro_f2"],
                "val_macro_f1": metrics["macro_f1"],
                "val_macro_f2": metrics["macro_f2"]
            })

            if score > best_val_score:
                best_val_score = score
                best_model = model
                best_model_name = "LogReg"
                best_params = {
                    "C": C,
                    "class_weight": class_weight
                }
                best_thresholds = thresholds

    print("\nSearch SGD Logistic...")

    for alpha in sgd_alpha_values:
        for class_weight in ("balanced", None):
            print(f"Training SGD alpha={alpha}, class_weight={class_weight}")

            model = OneVsRestClassifier(
                SGDClassifier(
                    loss="log_loss",
                    penalty="elasticnet",
                    alpha=alpha,
                    l1_ratio=0.15,
                    class_weight=class_weight,
                    max_iter=1500,
                    tol=1e-4,
                    random_state=random_state
                )
            )

            model.fit(X_train, Y_train)
            y_val_prob = model.predict_proba(X_val)

            thresholds = find_best_thresholds_sklearn_constrained(
                Y_val,
                y_val_prob,
                beta=threshold_beta,
                min_precision=min_precision_threshold,
                min_threshold=0.10,
                max_threshold=0.90,
                step=0.03
            )

            metrics = compute_multilabel_metrics_sklearn_extended(
                Y_val,
                y_val_prob,
                thresholds
            )

            score = metrics["micro_f1"]

            search_rows.append({
                "model": "SGDLogistic",
                "params": f"alpha={alpha}, class_weight={class_weight}",
                "val_precision": metrics["micro_precision"],
                "val_recall": metrics["micro_recall"],
                "val_f1": metrics["micro_f1"],
                "val_f2": metrics["micro_f2"],
                "val_macro_f1": metrics["macro_f1"],
                "val_macro_f2": metrics["macro_f2"]
            })

            if score > best_val_score:
                best_val_score = score
                best_model = model
                best_model_name = "SGDLogistic"
                best_params = {
                    "alpha": alpha,
                    "class_weight": class_weight
                }
                best_thresholds = thresholds

    search_results_df = pd.DataFrame(search_rows).sort_values(
        by=["val_f1", "val_f2", "val_precision", "val_recall"],
        ascending=False
    )

    print("\nBest model:", best_model_name)
    print("Best params:", best_params)

    print("\nSearch results:")
    display(search_results_df)

    y_train_prob = best_model.predict_proba(X_train)
    y_val_prob = best_model.predict_proba(X_val)
    y_test_prob = best_model.predict_proba(X_test)

    train_metrics = compute_multilabel_metrics_sklearn_extended(
        Y_train,
        y_train_prob,
        best_thresholds
    )

    val_metrics = compute_multilabel_metrics_sklearn_extended(
        Y_val,
        y_val_prob,
        best_thresholds
    )

    test_metrics = compute_multilabel_metrics_sklearn_extended(
        Y_test,
        y_test_prob,
        best_thresholds
    )

    y_test_pred = (y_test_prob >= best_thresholds).astype(int)

    per_label_precision, per_label_recall, per_label_f1, per_label_support = precision_recall_fscore_support(
        Y_test,
        y_test_pred,
        average=None,
        zero_division=0
    )

    per_label_f2 = fbeta_score(
        Y_test,
        y_test_pred,
        beta=2.0,
        average=None,
        zero_division=0
    )

    per_label_metrics = pd.DataFrame({
        "Classe": mlb.classes_,
        "Support": per_label_support,
        "Precision": per_label_precision,
        "Recall": per_label_recall,
        "F1": per_label_f1,
        "F2": per_label_f2,
        "Threshold": best_thresholds
    }).sort_values("F1", ascending=False)

    print("\nFINAL METRICS - SENTENCE TRANSFORMER FAST")
    print(f"Embedding model: {embedding_model_name}")
    print(f"Best model:      {best_model_name}")
    print(f"Best params:     {best_params}")
    print()
    print(f"Test micro F1:   {test_metrics['micro_f1']:.4f}")
    print(f"Test micro F2:   {test_metrics['micro_f2']:.4f}")
    print(f"Test macro F1:   {test_metrics['macro_f1']:.4f}")
    print(f"Test macro F2:   {test_metrics['macro_f2']:.4f}")
    print(f"Test precision:  {test_metrics['micro_precision']:.4f}")
    print(f"Test recall:     {test_metrics['micro_recall']:.4f}")
    print(f"Label accuracy:  {test_metrics['label_accuracy']:.4f}")
    print(f"Subset accuracy: {test_metrics['subset_accuracy']:.4f}")

    print("\nPer-label metrics:")
    display(per_label_metrics)

    if save_bundle:
        bundle_filename = f"ORM_SENTENCE_TRANSFORMER_FAST_bundle_{combo_name}.pkl"

        bundle = {
            "embedding_model_name": embedding_model_name,
            "scaler": scaler,
            "classifier": best_model,
            "classifier_name": best_model_name,
            "mlb": mlb,
            "thresholds_per_label": best_thresholds,
            "feature_columns": feature_columns,
            "classes": mlb.classes_,
            "best_params": best_params
        }

        with open(bundle_filename, "wb") as file:
            pickle.dump(bundle, file)

        print(f"\nBundle SentenceTransformer Fast salvato: {bundle_filename}")

    result = {
        "model_type": "SentenceTransformerFast",
        "combo_name": combo_name,
        "features": feature_columns,
        "embedding_model_name": embedding_model_name,
        "embedding_dim": X_train.shape[1],
        "num_classes": len(mlb.classes_),

        "best_model": best_model_name,
        "best_params": best_params,
        "search_results": search_results_df,

        "train_precision": train_metrics["micro_precision"],
        "val_precision": val_metrics["micro_precision"],
        "test_precision": test_metrics["micro_precision"],

        "train_recall": train_metrics["micro_recall"],
        "val_recall": val_metrics["micro_recall"],
        "test_recall": test_metrics["micro_recall"],

        "train_f1": train_metrics["micro_f1"],
        "val_f1": val_metrics["micro_f1"],
        "test_f1": test_metrics["micro_f1"],

        "train_f2": train_metrics["micro_f2"],
        "val_f2": val_metrics["micro_f2"],
        "test_f2": test_metrics["micro_f2"],

        "test_accuracy": test_metrics["label_accuracy"],
        "test_subset_accuracy": test_metrics["subset_accuracy"],

        "test_macro_precision": test_metrics["macro_precision"],
        "test_macro_recall": test_metrics["macro_recall"],
        "test_macro_f1": test_metrics["macro_f1"],
        "test_macro_f2": test_metrics["macro_f2"],

        "thresholds_per_label": best_thresholds,
        "classes": mlb.classes_,
        "per_label_metrics": per_label_metrics,
        "classifier": best_model,
        "scaler": scaler,
        "mlb": mlb
    }

    return result

In [ ]:
feature_names_list = ['Abstract (Translated)(English)', 'Independent Claims (Translated)(English)', 'IPC', 'CPC', 'Application domain', 'Technology topic', 'Technical Problem', 'Technical Field', 'Std. Technical Benefit Phrase']


sentence_transformer_fast_result = trainTestCombSentenceTransformerFast(
    df_work=df,
    feature_columns=feature_names_list,
    combo_name="10-FEATURES-SENTENCE-TRANSFORMER-FAST",
    embedding_model_name="sentence-transformers/all-MiniLM-L6-v2",
    threshold_beta=1.0,
    min_precision_threshold=0.55,
    logreg_C_values=(1.0, 2.0),
    sgd_alpha_values=(3e-5, 1e-4),
    batch_size=128
)

TRAIN/TEST SENTENCE TRANSFORMER FAST - 10-FEATURES-SENTENCE-TRANSFORMER-FAST
Deduplica brevetti/label: 16871 -> 13070 righe

Numero classi: 9
Classi: ['EIWH' 'ESWH' 'GAHP' 'GASBOILER' 'GIWH' 'GSWH' 'HHP' 'HPWH'
 'NOT_RELEVANT']


,Classe,Numero di sample
6,HHP,7731
8,NOT_RELEVANT,1903
7,HPWH,1311
1,ESWH,1156
5,GSWH,1025
3,GASBOILER,593
4,GIWH,291
0,EIWH,177
2,GAHP,102



Split:
Train size: 9127
Validation size: 1979
Test size: 1964

Caricamento embedding model: sentence-transformers/all-MiniLM-L6-v2


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

...


Best model: LogReg
Best params: {'C': 1.0, 'class_weight': None}

Search results:


,model,params,val_precision,val_recall,val_f1,val_f2,val_macro_f1,val_macro_f2
1,LogReg,"C=1.0, class_weight=None",0.755495,0.769949,0.762653,0.767014,0.591287,0.580817
3,LogReg,"C=2.0, class_weight=None",0.745072,0.776015,0.760229,0.769622,0.605191,0.588399
0,LogReg,"C=1.0, class_weight=balanced",0.693919,0.788147,0.738038,0.767309,0.580024,0.618185
7,SGDLogistic,"alpha=0.0001, class_weight=None",0.719086,0.748950,0.733714,0.742780,0.528522,0.535412
2,LogReg,"C=2.0, class_weight=balanced",0.684937,0.787214,0.732523,0.764386,0.569589,0.612764
5,SGDLogistic,"alpha=3e-05, class_weight=None",0.679599,0.727485,0.702727,0.717375,0.510267,0.527514
6,SGDLogistic,"alpha=0.0001, class_weight=balanced",0.562889,0.843677,0.675257,0.767142,0.499236,0.621551
4,SGDLogistic,"alpha=3e-05, class_weight=balanced",0.528742,0.845544,0.650628,0.755063,0.485212,0.616795



FINAL METRICS - SENTENCE TRANSFORMER FAST
Embedding model: sentence-transformers/all-MiniLM-L6-v2
Best model:      LogReg
Best params:     {'C': 1.0, 'class_weight': None}

Test micro F1:   0.7575
Test micro F2:   0.7634
Test macro F1:   0.5718
Test macro F2:   0.5626
Test precision:  0.7478
Test recall:     0.7674
Label accuracy:  0.9404
Subset accuracy: 0.6619

Per-label metrics:


,Classe,Support,Precision,Recall,F1,F2,Threshold
6,HHP,1160,0.860129,0.922414,0.890183,0.909245,0.40
1,ESWH,173,0.655172,0.768786,0.707447,0.743017,0.31
5,GSWH,154,0.595238,0.649351,0.621118,0.637755,0.34
8,NOT_RELEVANT,286,0.598513,0.562937,0.580180,0.569710,0.37
7,HPWH,197,0.586387,0.568528,0.577320,0.572012,0.40
3,GASBOILER,89,0.614286,0.483146,0.540881,0.504695,0.52
2,GAHP,15,0.636364,0.466667,0.538462,0.492958,0.40
0,EIWH,27,0.366667,0.407407,0.385965,0.398551,0.37
4,GIWH,44,0.600000,0.204545,0.305085,0.235602,0.85



Bundle SentenceTransformer Fast salvato: ORM_SENTENCE_TRANSFORMER_FAST_bundle_10-FEATURES-SENTENCE-TRANSFORMER-FAST.pkl


## Eseguiamo il loop per le varie combinazioni standard

Creare automaticamente feature_combinations

In [ ]:
all_features = [
    "Title (Translated)(English)",
    "Abstract (Translated)(English)",
    "Independent Claims (Translated)(English)",
    "IPC",
    "CPC",
    "Application domain",
    "Technology topic",
    "Technical Problem",
    "Technical Field",
    "Std. Technical Benefit Phrase"
]

feature_combinations = {}

for r in range(1, len(all_features) + 1):
    for combo in combinations(all_features, r):
        combo_name = f"{r}_FEATURES__" + "__".join([
            col
            .replace(" (Translated)(English)", "")
            .replace("Std. ", "")
            .replace(" ", "_")
            .replace(".", "")
            .replace("(", "")
            .replace(")", "")
            .replace("/", "_")
            for col in combo
        ])

        feature_combinations[combo_name] = list(combo)

print("Numero totale combinazioni:", len(feature_combinations))

Numero totale combinazioni: 1023


Selezioniamo le combinazioni con 10 feutures

In [ ]:
feature_combinations_10_to_10 = {
    name: features
    for name, features in feature_combinations.items()
    if 10 <= len(features) <= 10
}

print("Numero totale combinazioni:", len(feature_combinations_10_to_10))

Numero totale combinazioni: 1


Applichiamo finalmente il loop

In [ ]:
results = []
count = 0

for combo_name, features in feature_combinations_10_to_10.items():

    print()
    print(f"[{count + 1}/{len(feature_combinations_10_to_10)}] {combo_name}")

    result = trainTestCombANN(
        df_work=df,
        feature_columns=features,
        combo_name=combo_name
    )
    results.append(result)

    count = count + 1



[1/1] 10_FEATURES__Title__Abstract__Independent_Claims__IPC__CPC__Application_domain__Technology_topic__Technical_Problem__Technical_Field__Technical_Benefit_Phrase
TRAIN/TEST ANN - 10_FEATURES__Title__Abstract__Independent_Claims__IPC__CPC__Application_domain__Technology_topic__Technical_Problem__Technical_Field__Technical_Benefit_Phrase
Feature usate:
- Title (Translated)(English)
- Abstract (Translated)(English)
- Independent Claims (Translated)(English)
- IPC
- CPC
- Application domain
- Technology topic
- Technical Problem
- Technical Field
- Std. Technical Benefit Phrase

Numero di classi reali: 10
Classi: ['EIWH' 'ESWH' 'EWH' 'GAHP' 'GASBOILER' 'GIWH' 'GSWH' 'HHP' 'HPWH'
 'NOT_RELEVANT']


,Classe,Numero di sample
7,HHP,13356
9,NOT_RELEVANT,9757
8,HPWH,3997
2,EWH,2720
5,GIWH,1897
4,GASBOILER,1841
1,ESWH,1156
6,GSWH,1025
0,EIWH,968
3,GAHP,668



Train size: 21851
Validation size: 3857
Test size: 4537

Vettorizzatore TF-IDF salvato: ORM_tfidf_vectorizer10_FEATURES__Title__Abstract__Independent_Claims__IPC__CPC__Application_domain__Technology_topic__Technical_Problem__Technical_Field__Technical_Benefit_Phrase.pkl

Shape finali:
X_train: (21851, 40000)
X_val:   (3857, 40000)
X_test:  (4537, 40000)
Y_train: (21851, 10)
Y_val:   (3857, 10)
Y_test:  (4537, 10)

Numero effettivo di feature TF-IDF: 40000

Campioni per classe nel training:
         Classe  Train samples
0          EIWH          722.0
1          ESWH          834.0
2           EWH         1998.0
3          GAHP          493.0
4     GASBOILER         1352.0
5          GIWH         1403.0
6          GSWH          747.0
7           HHP         9638.0
8          HPWH         2865.0
9  NOT_RELEVANT         7034.0


NameError: name 'oversample_minority_multilabel' is not defined

### Tabella comparativa

In [ ]:
comparison_df = pd.DataFrame([
    {
        "combo_name": r["combo_name"],
        "num_input_columns": r["num_input_columns"],
        "num_tfidf_features": r["num_tfidf_features"],
        "train_precision": r["train_precision"],
        "test_precision": r["test_precision"],
        "train_recall": r["train_recall"],
        "test_recall": r["test_recall"],
        "train_f1": r["train_f1"],
        "test_f1": r["test_f1"],
        "train_f2": r["train_f2"],
        "test_f2": r["test_f2"],
        "test_accuracy": r["test_accuracy"]
    }
    for r in results
])

comparison_df.sort_values("test_f2", ascending=False)


Let's sort the dataset of results

In [ ]:
comparison_df_sorted = comparison_df.sort_values(
    by=[
        "test_accuracy",   # priorità principale
        "test_f2",
        "test_f1",
        "test_recall",
        "test_precision"
    ],
    ascending=[False, False, False, False, False]
)

comparison_df_sorted

In [ ]:
best_combo_name = comparison_df_sorted.iloc[0]['combo_name']
features_string = best_combo_name.split('__', 2)[2] # Split twice to get rid of 'X_FEATURES__'
feature_names_list = features_string.replace('__', ',').replace('_', ' ').split(',')
feature_names_list = [name.strip() for name in feature_names_list]

# Adjust names to match original column names
for i, name in enumerate(feature_names_list):
    if name == 'Title':
        feature_names_list[i] = 'Title (Translated)(English)'
    elif name == 'Abstract':
        feature_names_list[i] = 'Abstract (Translated)(English)'
    elif name == 'Independent Claims':
        feature_names_list[i] = 'Independent Claims (Translated)(English)'
    elif name == 'Technical Problem':
        feature_names_list[i] = 'Technical Problem'
    elif name == 'Technical Field':
        feature_names_list[i] = 'Technical Field'
    elif name == 'Technical Benefit Phrase':
        feature_names_list[i] = 'Std. Technical Benefit Phrase'


print("The features for the best performing model are:")
print(feature_names_list)

Con le features migliori andiamo ad applicare delle migliorie alla pipeline !!!

## Risultati del LOOP standard di combinazione di Training e Test

### 1) LOOP con vettorizzatore a 1 o 2 termini rilevanti
Il loop di training è stato eseguito su **56 combinazioni di feature** (selezionate tra quelle con 8, 9 e 10 feature), utilizzando la funzione `trainTestCombANN`.
Per ogni combinazione è stato addestrato un modello di **Artificial Neural Network multi-label**, mantenendo costanti i principali iperparametri (100 epoche, batch size = 64, learning rate = 1e-4) e applicando **TF-IDF (max 25.000 feature)** come rappresentazione testuale.

### Strategia di valutazione

Le performance sono state valutate su:

* **Training set**
* **Test set (hold-out)**

Le metriche considerate includono:

* Accuracy
* Precision
* Recall
* F1-score
* F2-score (metrica principale per il confronto)

Il confronto finale tra le combinazioni è stato effettuato ordinando i risultati per **test F2-score**, in quanto più sensibile al **recall** e quindi più adatto a valutare la capacità del modello di identificare correttamente anche le classi meno rappresentate.

---

### Risultati principali

L’analisi della tabella comparativa (`comparison_df`) evidenzia che:

* Le combinazioni con **numero maggiore di feature (9–10)** tendono a ottenere:

  * **maggiore recall**
  * **F2-score più elevato**
* Tuttavia, questo avviene spesso a scapito di:

  * una leggera riduzione della **precision**
  * un aumento dei **falsi positivi**

In generale, i modelli mostrano:

* **Test accuracy elevata**, tipicamente superiore a **0.90**
* **Test precision** generalmente compresa tra **0.75 e 0.92**
* **Test recall** variabile, tipicamente tra **0.60 e 0.82**
* **Test F1-score** nell’intervallo **0.73 – 0.79**
* **Test F2-score** nell’intervallo **0.80 – 0.86** per le migliori combinazioni

---

### Osservazioni tecniche

* Le combinazioni con **tutte le feature (10)** o quasi complete risultano tra le migliori in termini di:

  * **stabilità delle metriche**
  * **capacità predittiva complessiva**

* L’introduzione di più feature:

  * aumenta la **ricchezza informativa del testo combinato**
  * migliora la rappresentazione TF-IDF
  * consente una migliore separazione delle classi nello spazio delle feature

* Tuttavia:

  * l’elevata dimensionalità (**fino a 25.000 feature TF-IDF**) può introdurre:

    * maggiore complessità del modello
    * rischio di **overfitting**

---

### Trade-off Precision vs Recall

Dai risultati emerge chiaramente un trade-off:

* Configurazioni più conservative (threshold più alto):

  * **alta precision**
  * **basso recall**

* Configurazioni più permissive:

  * **maggiore recall**
  * **riduzione della precision**

L’utilizzo del **F2-score come metrica di ranking** permette di privilegiare modelli con:

* maggiore capacità di recuperare le classi minoritarie
* minore numero di **false negatives**

---

### Conclusione

Il loop di training ha permesso di:

* confrontare sistematicamente diverse combinazioni di feature
* identificare le configurazioni più performanti
* evidenziare l’importanza di:

  * **feature engineering**
  * **bilanciamento tra precision e recall**

Le migliori combinazioni risultano essere quelle con **9–10 feature**, che garantiscono il miglior compromesso tra:

* accuratezza globale
* capacità di generalizzazione
* recupero delle classi meno rappresentate


### 2) LOOP con vettorizzatore a 1, 2 o 3 termini rilevanti

Il loop sperimentale ha confrontato diverse combinazioni di feature testuali per il modello **ANN multi-label**.  
Tutte le configurazioni riportate utilizzano una rappresentazione **TF-IDF con 40.000 feature** e sono state valutate confrontando precision, recall, F1-score, F2-score e accuracy sul test set.

La migliore configurazione ordinata secondo le metriche di test utilizza **9 feature in input** e raggiunge le seguenti performance:

| Metrica | Train | Test |
|---|---:|---:|
| Precision | 0.9996 | 0.8856 |
| Recall | 0.9996 | 0.7306 |
| F1-score | 0.9996 | 0.8007 |
| F2-score | 0.9996 | 0.7571 |
| Accuracy | - | 0.9488 |

Nel complesso, le migliori configurazioni presentano risultati molto vicini tra loro.  
La **test accuracy** rimane nell’intervallo circa **0.946-0.949**, mentre il **test F1-score** varia tra circa **0.791 e 0.801**. Il **test recall** si colloca intorno a **0.720-0.731**, indicando che il modello recupera una buona parte delle label corrette, ma lascia ancora spazio di miglioramento nella riduzione dei falsi negativi.

Un aspetto evidente è il forte divario tra performance di training e test: le metriche di training sono prossime a **0.999**, mentre le metriche di test sono sensibilmente inferiori. Questo suggerisce una possibile tendenza all’**overfitting**, anche se il modello mantiene una buona capacità di generalizzazione sul test set.

Dal confronto tra combinazioni emerge inoltre che l’uso di **9 feature** risulta competitivo, e in alcuni casi migliore, rispetto all’uso di tutte le **10 feature**. Questo indica che alcune colonne aggiuntive potrebbero introdurre rumore informativo invece di migliorare la predizione.

In sintesi, il modello mostra prestazioni complessivamente solide per una classificazione multi-label basata su TF-IDF e ANN. Tuttavia, per migliorare ulteriormente la robustezza predittiva, sarebbe opportuno lavorare su regolarizzazione, calibrazione delle soglie per label, controllo dell’overfitting e confronto con baseline lineari come `OneVsRestClassifier(LogisticRegression)`.



## Eseguiamo il confronto con le versioni migliorate di ANN e la Logistic Regression nel caso della combinazione migliore

### LinearEnsemble Model

In [ ]:
feature_names_list = ['Abstract (Translated)(English)', 'Independent Claims (Translated)(English)', 'IPC', 'CPC', 'Application domain', 'Technology topic', 'Technical Problem', 'Technical Field', 'Std. Technical Benefit Phrase']

linear_ensemble_result = trainTestCombLinearEnsembleStrong(
    df_work=df,
    feature_columns=feature_names_list,
    combo_name="10-FEATURES-LINEAR-ENSEMBLE",

    threshold_beta=1.0,
    min_precision_threshold=0.55,

    logreg_C_values=(0.5, 1.0, 2.0, 4.0),
    svc_C_values=(0.25, 0.5, 1.0, 2.0),
    ensemble_weights=(0.25, 0.5, 0.75)
)

TRAIN/TEST LINEAR ENSEMBLE STRONG - 10-FEATURES-LINEAR-ENSEMBLE
Deduplica brevetti/label: 16871 -> 13070 righe

Numero classi: 9
Classi: ['EIWH' 'ESWH' 'GAHP' 'GASBOILER' 'GIWH' 'GSWH' 'HHP' 'HPWH'
 'NOT_RELEVANT']


,Classe,Numero di sample
6,HHP,7731
8,NOT_RELEVANT,1903
7,HPWH,1311
1,ESWH,1156
5,GSWH,1025
3,GASBOILER,593
4,GIWH,291
0,EIWH,177
2,GAHP,102



Split:
Train size: 9127
Validation size: 1979
Test size: 1964

Fitting TF-IDF word + char...

Shape finali sparse:
X_train: (9127, 92616)
X_val:   (1979, 92616)
X_test:  (1964, 92616)

Search Logistic Regression...
Training LogReg C=0.5, class_weight=balanced
Training LogReg C=0.5, class_weight=None
Training LogReg C=1.0, class_weight=balanced
Training LogReg C=1.0, class_weight=None
Training LogReg C=2.0, class_weight=balanced
Training LogReg C=2.0, class_weight=None
Training LogReg C=4.0, class_weight=balanced
Training LogReg C=4.0, class_weight=None

Search Calibrated LinearSVC...
Training LinearSVC C=0.25, class_weight=balanced
Training LinearSVC C=0.25, class_weight=None
Training LinearSVC C=0.5, class_weight=balanced
Training LinearSVC C=0.5, class_weight=None
Training LinearSVC C=1.0, class_weight=balanced
Training LinearSVC C=1.0, class_weight=None
Training LinearSVC C=2.0, class_weight=balanced
Training LinearSVC C=2.0, class_weight=None

Search ensemble weights...

Best mode

,model,C,class_weight,ensemble_weight_logreg,val_precision,val_recall,val_f1,val_f2,val_macro_f1,val_macro_f2
11,CalibratedLinearSVC,0.50,None,0.00,0.850024,0.835744,0.842824,0.838562,0.702352,0.668733
17,CalibratedLinearSVC,NaN,None,0.00,0.850024,0.835744,0.842824,0.838562,0.702352,0.668733
18,Ensemble,NaN,None,0.25,0.847498,0.837611,0.842525,0.839570,0.702174,0.670021
13,CalibratedLinearSVC,1.00,None,0.00,0.854127,0.830611,0.842205,0.835210,0.700209,0.662311
19,Ensemble,NaN,None,0.50,0.838515,0.843210,0.840856,0.842267,0.711864,0.688895
15,CalibratedLinearSVC,2.00,None,0.00,0.858463,0.823612,0.840676,0.830354,0.700770,0.664896
7,LogReg,4.00,None,1.00,0.841486,0.834811,0.838135,0.836138,0.703846,0.682377
16,LogReg,NaN,None,1.00,0.841486,0.834811,0.838135,0.836138,0.703846,0.682377
20,Ensemble,NaN,None,0.75,0.828545,0.847877,0.838100,0.843939,0.713402,0.700962
6,LogReg,4.00,balanced,1.00,0.845606,0.830611,0.838041,0.833567,0.713296,0.691464



FINAL METRICS - LINEAR ENSEMBLE STRONG
Best model type:       CalibratedLinearSVC
Best weight LogReg:    0.0

Test micro F1:         0.8480
Test micro F2:         0.8410
Test macro F1:         0.6782
Test macro F2:         0.6502
Test precision:        0.8600
Test recall:           0.8364
Label accuracy:        0.9636
Subset accuracy:       0.7643
Thresholds: [0.56 0.38 0.44 0.36 0.32 0.42 0.48 0.32 0.34]

Per-label metrics:


,Classe,Support,Precision,Recall,F1,Threshold
0,EIWH,27,1.000000,0.148148,0.258065,0.56
1,ESWH,173,0.785714,0.763006,0.774194,0.38
2,GAHP,15,0.846154,0.733333,0.785714,0.44
3,GASBOILER,89,0.746032,0.528090,0.618421,0.36
4,GIWH,44,0.629630,0.386364,0.478873,0.32
5,GSWH,154,0.798561,0.720779,0.757679,0.42
6,HHP,1160,0.931761,0.953448,0.942480,0.48
7,HPWH,197,0.685567,0.675127,0.680307,0.32
8,NOT_RELEVANT,286,0.800687,0.814685,0.807626,0.34



Bundle Linear Ensemble salvato: ORM_LINEAR_ENSEMBLE_STRONG_bundle_10-FEATURES-LINEAR-ENSEMBLE.pkl


### LinearEnsemble Model Ultra

In [ ]:
feature_names_list = ['Abstract (Translated)(English)', 'Independent Claims (Translated)(English)', 'IPC', 'CPC', 'Application domain', 'Technology topic', 'Technical Problem', 'Technical Field', 'Std. Technical Benefit Phrase']


linear_ensemble_ultra_result = trainTestCombLinearEnsembleUltra(
    df_work=df,
    feature_columns=feature_names_list,
    combo_name="10-FEATURES-LINEAR-ENSEMBLE-ULTRA",

    threshold_beta=1.0,
    min_precision_threshold=0.55,

    logreg_C_values=(0.5, 1.0, 2.0, 4.0, 8.0),
    svc_C_values=(0.5, 1.0, 2.0),
    sgd_alpha_values=(1e-5, 3e-5, 1e-4)
)


TRAIN/TEST LINEAR ENSEMBLE ULTRA - 10-FEATURES-LINEAR-ENSEMBLE-ULTRA
Deduplica brevetti/label: 16871 -> 13070 righe

Numero classi: 9
Classi: ['EIWH' 'ESWH' 'GAHP' 'GASBOILER' 'GIWH' 'GSWH' 'HHP' 'HPWH'
 'NOT_RELEVANT']


,Classe,Numero di sample
6,HHP,7731
8,NOT_RELEVANT,1903
7,HPWH,1311
1,ESWH,1156
5,GSWH,1025
3,GASBOILER,593
4,GIWH,291
0,EIWH,177
2,GAHP,102



Split:
Train size: 7191
Validation size: 1969
Test size: 3910

Fitting TF-IDF word + char...

Shape finali sparse:
X_train: (7191, 115508)
X_val:   (1969, 115508)
X_test:  (3910, 115508)

Search Logistic Regression...
Training LogReg C=0.5, class_weight=balanced
Training LogReg C=0.5, class_weight=None
Training LogReg C=1.0, class_weight=balanced
Training LogReg C=1.0, class_weight=None
Training LogReg C=2.0, class_weight=balanced
Training LogReg C=2.0, class_weight=None
Training LogReg C=4.0, class_weight=balanced
Training LogReg C=4.0, class_weight=None
Training LogReg C=8.0, class_weight=balanced
Training LogReg C=8.0, class_weight=None

Search Calibrated LinearSVC...
Training LinearSVC C=0.5, class_weight=balanced
Training LinearSVC C=0.5, class_weight=None
Training LinearSVC C=1.0, class_weight=balanced
Training LinearSVC C=1.0, class_weight=None
Training LinearSVC C=2.0, class_weight=balanced
Training LinearSVC C=2.0, class_weight=None

Search SGD Logistic...
Training SGD alpha=

,model,params,val_precision,val_recall,val_f1,val_f2,val_macro_f1,val_macro_f2
22,PerLabelEnsemble,LogReg + CalibratedLinearSVC + SGDLogistic,0.837965,0.829757,0.833841,0.831386,0.702836,0.678356
13,CalibratedLinearSVC,"C=1.0, class_weight=None",0.823095,0.831157,0.827106,0.829532,0.690578,0.669167
17,SGDLogistic,"alpha=1e-05, class_weight=None",0.833333,0.820896,0.827068,0.823353,0.687650,0.661311
19,SGDLogistic,"alpha=3e-05, class_weight=None",0.835558,0.817631,0.826497,0.821154,0.676193,0.646462
12,CalibratedLinearSVC,"C=1.0, class_weight=balanced",0.830500,0.820429,0.825434,0.822424,0.685075,0.663438
18,SGDLogistic,"alpha=3e-05, class_weight=balanced",0.828477,0.822295,0.825375,0.823524,0.678815,0.656111
11,CalibratedLinearSVC,"C=0.5, class_weight=None",0.824709,0.825093,0.824901,0.825016,0.687851,0.670369
15,CalibratedLinearSVC,"C=2.0, class_weight=None",0.822656,0.826493,0.824570,0.825722,0.687364,0.668052
10,CalibratedLinearSVC,"C=0.5, class_weight=balanced",0.822955,0.826026,0.824488,0.825410,0.681987,0.662261
7,LogReg,"C=4.0, class_weight=None",0.829003,0.818563,0.823750,0.820630,0.681919,0.653857



FINAL METRICS - LINEAR ENSEMBLE ULTRA
Best LogReg params: {'C': 4.0, 'class_weight': None}
Best SVC params:    {'C': 1.0, 'class_weight': None}
Best SGD params:    {'alpha': 1e-05, 'class_weight': None}

Test micro F1:      0.8361
Test micro F2:      0.8339
Test macro F1:      0.6836
Test macro F2:      0.6647
Test precision:     0.8397
Test recall:        0.8325
Label accuracy:     0.9602
Subset accuracy:    0.7535

Per-label metrics:


,Classe,Support,Precision,Recall,F1,Threshold,Weight_LogReg,Weight_SVC,Weight_SGD
0,EIWH,53,0.709677,0.415094,0.523810,0.18,1.000000,0.000000,0.000000
1,ESWH,347,0.796117,0.708934,0.750000,0.40,0.571429,0.285714,0.142857
2,GAHP,31,0.689655,0.645161,0.666667,0.24,0.000000,1.000000,0.000000
3,GASBOILER,178,0.653595,0.561798,0.604230,0.24,0.000000,0.666667,0.333333
4,GIWH,87,0.521127,0.425287,0.468354,0.20,0.166667,0.666667,0.166667
5,GSWH,307,0.742604,0.817590,0.778295,0.22,0.600000,0.000000,0.400000
6,HHP,2319,0.907830,0.959897,0.933138,0.38,0.500000,0.500000,0.000000
7,HPWH,393,0.709091,0.595420,0.647303,0.34,0.500000,0.166667,0.333333
8,NOT_RELEVANT,571,0.805970,0.756567,0.780488,0.36,0.000000,0.400000,0.600000



Bundle Linear Ensemble Ultra salvato: ORM_LINEAR_ENSEMBLE_ULTRA_bundle_10-FEATURES-LINEAR-ENSEMBLE-ULTRA.pkl


### Risultati Best Model

#### Risultati ottenuti andando ad utilizzare il Linear Ensemble Model Ultra con il dataset 2024 e 2025:

In [ ]:
result_LEMU = pd.DataFrame(
    {
        "model": ["Linear Ensemble Ultra"],
        "test_precision": [linear_ensemble_ultra_result["test_precision"]],
        "test_recall": [linear_ensemble_ultra_result["test_recall"]],
        "test_f1": [linear_ensemble_ultra_result["test_f1"]],
        "test_f2": [linear_ensemble_ultra_result["test_f2"]],
        "test_accuracy": [linear_ensemble_ultra_result["test_accuracy"]],
        "test_macro_f1": [linear_ensemble_ultra_result["test_macro_f1"]],
        "test_macro_f2": [linear_ensemble_ultra_result["test_macro_f2"]]
    }
)

result_LEMU_sorted = result_LEMU.sort_values(
    by=["test_f2", "test_f1", "test_recall", "test_precision"],
    ascending=False
)

result_LEMU_sorted

,model,test_precision,test_recall,test_f1,test_f2,test_accuracy,test_macro_f1,test_macro_f2
0,Linear Ensemble Ultra,0.839727,0.832478,0.836087,0.833918,0.960244,0.683587,0.664684


#### Risultati ottenuti andando ad utilizzare il Linear Ensemble Model Ultra con il dataset 2024, 2025 e i soli GAS BOILER e HPWH degli anni passati:

In [ ]:
result_LEMU = pd.DataFrame([
    {
        "model": "Linear Ensemble Ultra",
        "test_precision": linear_ensemble_ultra_result["test_precision"],
        "test_recall": linear_ensemble_ultra_result["test_recall"],
        "test_f1": linear_ensemble_ultra_result["test_f1"],
        "test_f2": linear_ensemble_ultra_result["test_f2"],
        "test_accuracy": linear_ensemble_ultra_result["test_accuracy"],
        "test_macro_f1": linear_ensemble_ultra_result["test_macro_f1"],
        "test_macro_f2": linear_ensemble_ultra_result["test_macro_f2"]
    }]
)

result_LEMU_sorted = result_LEMU.sort_values(
    by=["test_f2", "test_f1", "test_recall", "test_precision"],
    ascending=False
)

result_LEMU_sorted

,model,test_precision,test_recall,test_f1,test_f2,test_accuracy,test_macro_f1,test_macro_f2
0,Linear Ensemble Ultra,0.839727,0.832478,0.836087,0.833918,0.960244,0.683587,0.664684


### ANN Standard

In [ ]:
ann_result = trainTestCombANN(
    df_work=df,
    feature_columns=feature_names_list,
    combo_name="10-FEATURES-ANN",
    min_count_oversampling=800,
    thresholds_per_label=None,
    num_epochs=100,
    batch_size=64,
    learning_rate=1e-4,
    decay_rate=0.95,
    delta=1e-8,
    max_weight=5.0,
    random_state=42,
    save_vectorizer=True
)

TRAIN/TEST ANN - 10-FEATURES-ANN
Feature usate:
- Abstract (Translated)(English)
- Independent Claims (Translated)(English)
- IPC
- CPC
- Application domain
- Technology topic
- Technical Problem
- Technical Field
- Std. Technical Benefit Phrase

Numero di classi reali: 9
Classi: ['EIWH' 'ESWH' 'GAHP' 'GASBOILER' 'GIWH' 'GSWH' 'HHP' 'HPWH'
 'NOT_RELEVANT']


,Classe,Numero di sample
6,HHP,7731
7,HPWH,3997
8,NOT_RELEVANT,1903
3,GASBOILER,1841
1,ESWH,1156
5,GSWH,1025
4,GIWH,291
0,EIWH,177
2,GAHP,102



Train size: 12189
Validation size: 2151
Test size: 2531

Vettorizzatore TF-IDF salvato: ORM_tfidf_vectorizer10-FEATURES-ANN.pkl

Shape finali:
X_train: (12189, 40000)
X_val:   (2151, 40000)
X_test:  (2531, 40000)
Y_train: (12189, 9)
Y_val:   (2151, 9)
Y_test:  (2531, 9)

Numero effettivo di feature TF-IDF: 40000

Campioni per classe nel training:
         Classe  Train samples
0          EIWH          136.0
1          ESWH          835.0
2          GAHP           76.0
3     GASBOILER         1338.0
4          GIWH          209.0
5          GSWH          736.0
6           HHP         5591.0
7          HPWH         2861.0
8  NOT_RELEVANT         1358.0
Rare classes idx: [0 2 4 5]
Rare classes counts: [136.  76. 209. 736.]

Original train shape: (12189, 40000) (12189, 9)
Oversampled train shape: (14232, 40000) (14232, 9)

Campioni per classe dopo oversampling:
[ 819. 1061.  800. 1413.  834. 1203. 5923. 3000. 1358.]
DEBUG: output_dim = 9
DEBUG: thresholds_per_label (after jnp.full) length

### ANN V1

In [ ]:
ann_V1_result = trainTestCombANNV1(
    df_work=df,
    feature_columns=feature_names_list,
    combo_name="10-FEATURES-ANN-V1-IMPROVED",

    use_oversampling=True,
    use_pos_weights=False,

    threshold_beta=1.0,
    min_precision_threshold=0.55,

    num_epochs=120,
    batch_size=64,
    learning_rate=7e-5,
    hidden_units=256,
    patience=20
)

TRAIN/TEST ANN V1 IMPROVED - 10-FEATURES-ANN-V1-IMPROVED
Deduplica brevetti/label: 16871 -> 13070 righe

Numero classi: 9
Classi: ['EIWH' 'ESWH' 'GAHP' 'GASBOILER' 'GIWH' 'GSWH' 'HHP' 'HPWH'
 'NOT_RELEVANT']


,Classe,Numero di sample
6,HHP,7731
8,NOT_RELEVANT,1903
7,HPWH,1311
1,ESWH,1156
5,GSWH,1025
3,GASBOILER,593
4,GIWH,291
0,EIWH,177
2,GAHP,102



Split:
Train size: 9127
Validation size: 1979
Test size: 1964

Vettorizzatore TF-IDF salvato: ORM_tfidf_vectorizer_V1_IMPROVED_10-FEATURES-ANN-V1-IMPROVED.pkl

Shape finali:
X_train: (9127, 22616)
X_val:   (1979, 22616)
X_test:  (1964, 22616)
Y_train: (9127, 9)
Y_val:   (1979, 9)
Y_test:  (1964, 9)
Numero TF-IDF feature: 22616
Rare classes idx: [0 2 3 4 5]
Rare classes counts: [124.  72. 415. 203. 717.]

Training class counts:
         Classe  Train samples
0          EIWH          825.0
1          ESWH         1074.0
2          GAHP          800.0
3     GASBOILER          885.0
4          GIWH          875.0
5          GSWH         1374.0
6           HHP         5785.0
7          HPWH         1143.0
8  NOT_RELEVANT         1332.0

PARAMETRI USATI
Layers size:             [22616, 256, 9]
Epochs max:              120
Batch size:              64
Learning rate:           7e-05
Max weight:              3.0
Oversampling:            True
Pos weights:             False
Threshold beta:       

,Classe,Support,Precision,Recall,F1,Threshold
0,EIWH,27,0.666667,0.074074,0.133333,0.88
1,ESWH,173,0.550296,0.537572,0.543860,0.28
2,GAHP,15,0.833333,0.333333,0.476190,0.40
3,GASBOILER,89,0.769231,0.112360,0.196078,0.80
4,GIWH,44,0.500000,0.068182,0.120000,0.90
5,GSWH,154,0.678571,0.370130,0.478992,0.48
6,HHP,1160,0.776163,0.920690,0.842271,0.20
7,HPWH,197,0.522124,0.299492,0.380645,0.40
8,NOT_RELEVANT,286,0.611111,0.538462,0.572491,0.28


CONFUSION MATRIX MULTI-LABEL:

                         True 1       True 0

Predicted 1              1451 TP       571 FP
                         -------------------
Predicted 0              694 FN       14960 TN

False Negatives = 3.9 %
False Positives = 3.2 %

EIWH
Threshold: 0.88
TP: 2, TN: 1936, FP: 1, FN: 25
Support:   27
Precision: 0.6667
Recall:    0.0741
F1-score:  0.1333
----------------------------------------
ESWH
Threshold: 0.28
TP: 93, TN: 1715, FP: 76, FN: 80
Support:   173
Precision: 0.5503
Recall:    0.5376
F1-score:  0.5439
----------------------------------------
GAHP
Threshold: 0.40
TP: 5, TN: 1948, FP: 1, FN: 10
Support:   15
Precision: 0.8333
Recall:    0.3333
F1-score:  0.4762
----------------------------------------
GASBOILER
Threshold: 0.80
TP: 10, TN: 1872, FP: 3, FN: 79
Support:   89
Precision: 0.7692
Recall:    0.1124
F1-score:  0.1961
----------------------------------------
GIWH
Threshold: 0.90
TP: 3, TN: 1917, FP: 3, FN: 41
Support:   44
Precision: 0.500

### ANN V3

In [ ]:
ann_v3_result = trainTestCombANNV3(
    df_work=df,
    feature_columns=feature_names_list,
    combo_name="10-FEATURES-ANN-V3",

    use_oversampling=True,
    use_pos_weights=False,

    threshold_beta=1.0,
    min_precision_threshold=0.55,

    num_epochs=120,
    batch_size=64,
    learning_rate=1e-4,
    hidden_units=256,
    dropout_rate=0.10,
    l2_lambda=1e-6,
    weight_decay=1e-5,
    patience=20
)

TRAIN/TEST ANN V3 - 10-FEATURES-ANN-V3
Deduplica brevetti/label: 16871 -> 13070 righe

Numero classi: 9
Classi: ['EIWH' 'ESWH' 'GAHP' 'GASBOILER' 'GIWH' 'GSWH' 'HHP' 'HPWH'
 'NOT_RELEVANT']


,Classe,Numero di sample
6,HHP,7731
8,NOT_RELEVANT,1903
7,HPWH,1311
1,ESWH,1156
5,GSWH,1025
3,GASBOILER,593
4,GIWH,291
0,EIWH,177
2,GAHP,102



Split:
Train size: 9127
Validation size: 1979
Test size: 1964

Shape finali:
X_train: (9127, 22616)
X_val:   (1979, 22616)
X_test:  (1964, 22616)
Y_train: (9127, 9)
Y_val:   (1979, 9)
Y_test:  (1964, 9)
Numero TF-IDF feature: 22616
Rare classes idx: [0 2 3 4 5]
Rare classes counts: [124.  72. 415. 203. 717.]

Training class counts:
         Classe  Train samples
0          EIWH          825.0
1          ESWH         1074.0
2          GAHP          800.0
3     GASBOILER          885.0
4          GIWH          875.0
5          GSWH         1374.0
6           HHP         5785.0
7          HPWH         1143.0
8  NOT_RELEVANT         1332.0

PARAMETRI USATI
Layers size:             [22616, 256, 9]
Epochs max:              120
Batch size:              64
Learning rate:           0.0001
Hidden units:            256
Dropout rate:            0.1
L2 lambda:               1e-06
Weight decay:            1e-05
Oversampling:            True
Pos weights:             False
Threshold beta:          1.

,Classe,Support,Precision,Recall,F1,Threshold
0,EIWH,27,0.666667,0.074074,0.133333,0.74
1,ESWH,173,0.624161,0.537572,0.577640,0.26
2,GAHP,15,0.833333,0.333333,0.476190,0.42
3,GASBOILER,89,0.900000,0.101124,0.181818,0.74
4,GIWH,44,0.400000,0.045455,0.081633,0.82
5,GSWH,154,0.540984,0.428571,0.478261,0.28
6,HHP,1160,0.779240,0.918966,0.843354,0.22
7,HPWH,197,0.632911,0.253807,0.362319,0.44
8,NOT_RELEVANT,286,0.652174,0.472028,0.547667,0.34


CONFUSION MATRIX MULTI-LABEL:

                         True 1       True 0

Predicted 1              1428 TP       521 FP
                         -------------------
Predicted 0              717 FN       15010 TN

False Negatives = 4.1 %
False Positives = 2.9 %

Vettorizzatore TF-IDF salvato: ORM_tfidf_vectorizer_10-FEATURES-ANN-V3.pkl
Bundle modello salvato: ORM_ANN_bundle_10-FEATURES-ANN-V3.pkl


### ANN Standard Improved

In [ ]:
ann_improved_result = trainTestCombANNImproved(
    df_work=df,
    feature_columns=feature_names_list,
    combo_name="10-FEATURES-ANN-IMPROVED",

    use_oversampling=True,
    use_pos_weights=False,

    threshold_beta=2.0,
    min_precision_threshold=0.55,

    num_epochs=120,
    batch_size=64,
    learning_rate=1e-4,
    max_weight=3.0,
    hidden_units=128,
    patience=15
)

TRAIN/TEST ANN IMPROVED - 10-FEATURES-ANN-IMPROVED
Feature usate:
- Abstract (Translated)(English)
- Independent Claims (Translated)(English)
- IPC
- CPC
- Application domain
- Technology topic
- Technical Problem
- Technical Field
- Std. Technical Benefit Phrase
Deduplica brevetti/label: 16871 -> 13070 righe

Numero di classi reali: 9
Classi: ['EIWH' 'ESWH' 'GAHP' 'GASBOILER' 'GIWH' 'GSWH' 'HHP' 'HPWH'
 'NOT_RELEVANT']


,Classe,Numero di sample
6,HHP,7731
8,NOT_RELEVANT,1903
7,HPWH,1311
1,ESWH,1156
5,GSWH,1025
3,GASBOILER,593
4,GIWH,291
0,EIWH,177
2,GAHP,102



Split multi-label stratificato: OK

Train size: 9127
Validation size: 1979
Test size: 1964

Vettorizzatore TF-IDF salvato: ORM_tfidf_vectorizer_IMPROVED_10-FEATURES-ANN-IMPROVED.pkl

Shape finali:
X_train: (9127, 22616)
X_val:   (1979, 22616)
X_test:  (1964, 22616)
Y_train: (9127, 9)
Y_val:   (1979, 9)
Y_test:  (1964, 9)

Numero effettivo di feature TF-IDF: 22616

Campioni per classe nel training reale:
         Classe  Train samples
0          EIWH          124.0
1          ESWH          810.0
2          GAHP           72.0
3     GASBOILER          415.0
4          GIWH          203.0
5          GSWH          717.0
6           HHP         5411.0
7          HPWH          917.0
8  NOT_RELEVANT         1332.0
Rare classes idx: [0 2 3 4 5]
Rare classes counts: [124.  72. 415. 203. 717.]

Training shape effettiva: (11596, 22616) (11596, 9)
Campioni per classe nel training effettivo:
[ 825. 1074.  800.  885.  875. 1374. 5785. 1143. 1332.]

PARAMETRI USATI
Layers size:             [22616, 1

,Classe,Support,Precision,Recall,F1,Threshold
0,EIWH,27,0.666667,0.074074,0.133333,0.88
1,ESWH,173,0.546512,0.543353,0.544928,0.26
2,GAHP,15,0.833333,0.333333,0.476190,0.44
3,GASBOILER,89,0.769231,0.112360,0.196078,0.72
4,GIWH,44,0.545455,0.136364,0.218182,0.76
5,GSWH,154,0.496296,0.435065,0.463668,0.28
6,HHP,1160,0.734127,0.956897,0.830838,0.12
7,HPWH,197,0.469697,0.314721,0.376900,0.34
8,NOT_RELEVANT,286,0.604563,0.555944,0.579235,0.26


CONFUSION MATRIX MULTI-LABEL:

                         True 1       True 0

Predicted 1              1515 TP       732 FP
                         -------------------
Predicted 0              630 FN       14799 TN

False Negatives = 3.6 %
False Positives = 4.1 %

EIWH
Threshold: 0.88
TP: 2, TN: 1936, FP: 1, FN: 25
Support:   27
Precision: 0.6667
Recall:    0.0741
F1-score:  0.1333
----------------------------------------
ESWH
Threshold: 0.26
TP: 94, TN: 1713, FP: 78, FN: 79
Support:   173
Precision: 0.5465
Recall:    0.5434
F1-score:  0.5449
----------------------------------------
GAHP
Threshold: 0.44
TP: 5, TN: 1948, FP: 1, FN: 10
Support:   15
Precision: 0.8333
Recall:    0.3333
F1-score:  0.4762
----------------------------------------
GASBOILER
Threshold: 0.72
TP: 10, TN: 1872, FP: 3, FN: 79
Support:   89
Precision: 0.7692
Recall:    0.1124
F1-score:  0.1961
----------------------------------------
GIWH
Threshold: 0.76
TP: 6, TN: 1915, FP: 5, FN: 38
Support:   44
Precision: 0.545

### Eseguiamo il vero confronto

In [ ]:
comparison_extended = pd.DataFrame([
    {
        "model": "ANN",
        "test_precision": ann_result["test_precision"],
        "test_recall": ann_result["test_recall"],
        "test_f1": ann_result["test_f1"],
        "test_f2": ann_result["test_f2"],
        "test_accuracy": ann_result["test_accuracy"],
        "test_macro_f1": np.nan,
        "test_macro_f2": np.nan
    },
    {
        "model": "ANN V1",
        "test_precision": ann_V1_result["test_precision"],
        "test_recall": ann_V1_result["test_recall"],
        "test_f1": ann_V1_result["test_f1"],
        "test_f2": ann_V1_result["test_f2"],
        "test_accuracy": ann_V1_result["test_accuracy"],
        "test_macro_f1": np.nan,
        "test_macro_f2": np.nan
    },
    {
        "model": "ANN Improved",
        "test_precision": ann_improved_result["test_precision"],
        "test_recall": ann_improved_result["test_recall"],
        "test_f1": ann_improved_result["test_f1"],
        "test_f2": ann_improved_result["test_f2"],
        "test_accuracy": ann_improved_result["test_accuracy"],
        "test_macro_f1": np.nan,
        "test_macro_f2": np.nan
    },
    {
        "model": "ANN V3",
        "test_precision": ann_v3_result["test_precision"],
        "test_recall": ann_v3_result["test_recall"],
        "test_f1": ann_v3_result["test_f1"],
        "test_f2": ann_v3_result["test_f2"],
        "test_accuracy": ann_v3_result["test_accuracy"],
        "test_macro_f1": ann_v3_result["test_macro_f1"],
        "test_macro_f2": ann_v3_result["test_macro_f2"]
    },
    {
        "model": "LogReg + SVM model",
        "test_precision": linear_ensemble_result["test_precision"],
        "test_recall": linear_ensemble_result["test_recall"],
        "test_f1": linear_ensemble_result["test_f1"],
        "test_f2": linear_ensemble_result["test_f2"],
        "test_accuracy": linear_ensemble_result["test_accuracy"],
        "test_macro_f1": linear_ensemble_result["test_macro_f1"],
        "test_macro_f2": linear_ensemble_result["test_macro_f2"]
    },
    {
        "model": "Linear Ensemble Ultra",
        "test_precision": linear_ensemble_ultra_result["test_precision"],
        "test_recall": linear_ensemble_ultra_result["test_recall"],
        "test_f1": linear_ensemble_ultra_result["test_f1"],
        "test_f2": linear_ensemble_ultra_result["test_f2"],
        "test_accuracy": linear_ensemble_ultra_result["test_accuracy"],
        "test_macro_f1": linear_ensemble_ultra_result["test_macro_f1"],
        "test_macro_f2": linear_ensemble_ultra_result["test_macro_f2"]
    }
])

comparison_extended_sorted = comparison_extended.sort_values(
    by=["test_f2", "test_f1", "test_recall", "test_precision"],
    ascending=False
)

comparison_extended_sorted


,model,test_precision,test_recall,test_f1,test_f2,test_accuracy,test_macro_f1,test_macro_f2
4,LogReg + SVM model,0.860019,0.836364,0.848026,0.840990,0.963623,0.678151,0.650242
5,Linear Ensemble Ultra,0.839727,0.832478,0.836087,0.833918,0.960244,0.683587,0.664684
0,ANN,0.894142,0.777374,0.831679,0.798222,0.962026,NaN,NaN
2,ANN Improved,0.674232,0.706294,0.689891,0.699640,0.922946,NaN,NaN
1,ANN V1,0.717606,0.676457,0.696424,0.684305,0.928434,NaN,NaN
3,ANN V3,0.732683,0.665734,0.697606,0.678127,0.929962,0.409135,0.369142


# RISULTATI CONFRONTO MODELLI

## Performance modelli

La tabella confronta le performance dei modelli sul test set usando **precision**, **recall**, **F1-score**, **F2-score**, **accuracy** e, dove disponibili, le metriche **macro F1** e **macro F2**.

Il modello migliore risulta essere **LogReg + SVM model**, che ottiene le prestazioni complessive più alte: **precision = 0.860**, **recall = 0.836**, **F1 = 0.848**, **F2 = 0.841** e **accuracy = 0.964**. Questo indica un buon equilibrio tra capacità di individuare correttamente le label positive e contenimento dei falsi positivi.

Il modello **Linear Ensemble Ultra** mostra risultati molto vicini, con **F1 = 0.836** e **F2 = 0.834**. Pur avendo una precision leggermente inferiore rispetto al modello LogReg + SVM, mantiene buone metriche macro (**macro F1 = 0.684**, **macro F2 = 0.665**), suggerendo una performance più stabile anche sulle classi meno rappresentate.

Il modello **Sentence Transformer Fast** utilizza embedding semantici generati con `sentence-transformers/all-MiniLM-L6-v2` e un classificatore **Logistic Regression** come best model. Le sue performance sono inferiori rispetto ai modelli TF-IDF lineari: ottiene **precision = 0.748**, **recall = 0.767**, **F1 = 0.758**, **F2 = 0.763** e **accuracy = 0.940**. Il modello mostra quindi un recall leggermente superiore alla precision, indicando una maggiore tendenza a recuperare label positive, ma con più falsi positivi rispetto a LogReg + SVM.

Le metriche macro del Sentence Transformer Fast sono **macro F1 = 0.572** e **macro F2 = 0.563**. Questi valori sono migliori rispetto alle ANN più deboli, ma restano sotto quelli del Linear Ensemble Ultra e del LogReg + SVM model. A livello per-label, il modello performa molto bene sulla classe **HHP** (**F1 = 0.890**) e discretamente su **ESWH** (**F1 = 0.707**), mentre mostra difficoltà su classi meno rappresentate come **EIWH**, **GIWH** e **GAHP**.

Tra i modelli ANN, la migliore configurazione è la **ANN base**, che raggiunge la precision più alta (**0.894**) e un buon **F1 = 0.832**, ma con recall inferiore (**0.777**) e F2 più basso (**0.798**). Questo suggerisce che la rete è più conservativa: commette meno falsi positivi, ma perde più label corrette rispetto ai modelli lineari.

Le versioni **ANN Improved**, **ANN V1** e **ANN V3** ottengono risultati inferiori, con F1 intorno a **0.69-0.70** e accuracy più bassa. In particolare, **ANN V3** dispone anche di metriche macro, ma queste risultano contenute (**macro F1 = 0.409**, **macro F2 = 0.369**), indicando difficoltà sulle classi minoritarie.

Nel complesso, i risultati confermano che i modelli lineari basati su **TF-IDF + LogReg/SVM** sono ancora i più efficaci per questo task multi-label. Il **Sentence Transformer Fast** introduce una rappresentazione semantica più avanzata rispetto al TF-IDF, ma in questa configurazione non supera i modelli TF-IDF: probabilmente perché codici tecnici, sigle e keyword specifiche dei brevetti vengono catturati meglio da TF-IDF word/char n-grams rispetto agli embedding semantici generali.

## Performance dei modelli ANN

Tra i soli modelli **ANN**, la configurazione migliore è la **ANN base**, che ottiene **precision = 0.894**, **recall = 0.777**, **F1 = 0.832**, **F2 = 0.798** e **accuracy = 0.962**. Questo indica che il modello è molto preciso: quando assegna una label, nella maggior parte dei casi la predizione è corretta. Tuttavia, il recall è più basso della precision, quindi la rete tende a essere conservativa: predice meno label positive, riducendo i falsi positivi, ma lasciando fuori alcune label corrette.

La **ANN Improved** ottiene **precision = 0.674**, **recall = 0.706**, **F1 = 0.690** e **F2 = 0.700**. Rispetto alla ANN base, recupera leggermente più label positive rispetto ad ANN V1/V3, ma perde molta precision. Questo suggerisce che il modello è più permissivo e genera più falsi positivi, risultando complessivamente meno affidabile.

La **ANN V1** raggiunge **precision = 0.718**, **recall = 0.676**, **F1 = 0.696** e **F2 = 0.684**. Le metriche sono più bilanciate rispetto ad ANN Improved, ma restano sensibilmente inferiori alla ANN base. Il modello non riesce né a mantenere una precision elevata né a recuperare abbastanza label vere.

La **ANN V3** ottiene **precision = 0.733**, **recall = 0.666**, **F1 = 0.698**, **F2 = 0.678** e **accuracy = 0.930**. Le metriche macro disponibili sono basse (**macro F1 = 0.409**, **macro F2 = 0.369**), indicando difficoltà sulle classi minoritarie. Questo suggerisce che il modello performa meglio sulle classi frequenti, ma fatica a generalizzare sulle piattaforme meno rappresentate.

Nel complesso, la **ANN base risulta nettamente la migliore tra le reti neurali testate**. Le versioni successive non migliorano le performance finali: alcune aumentano leggermente il recall ma riducono troppo la precision, mentre altre non riescono a gestire efficacemente le classi meno rappresentate. Per questo dataset, la configurazione ANN più semplice si dimostra la più efficace.



## Performance Best Model LEMU


Il miglior modello tra le reti neurali testate è la **ANN base**, che ottiene sul test set **precision = 0.894**, **recall = 0.777**, **F1-score = 0.832**, **F2-score = 0.798** e **accuracy = 0.962**.

La precision molto alta indica che il modello è affidabile quando assegna una label: la maggior parte delle piattaforme predette risulta corretta. Questo significa che la rete commette relativamente pochi falsi positivi.

Il recall, invece, è più basso della precision. Questo suggerisce che il modello è piuttosto conservativo: tende a predire solo le label di cui è più sicuro, ma in alcuni casi non recupera tutte le piattaforme realmente presenti. Di conseguenza, il modello perde alcune label vere, aumentando i falsi negativi.

Il valore di **F1 = 0.832** mostra comunque un buon equilibrio complessivo tra precision e recall. Tuttavia, il valore di **F2 = 0.798**, più sensibile al recall, conferma che il principale limite del modello è la mancata identificazione di alcune classi positive.

In sintesi, la **ANN base è il miglior modello neurale testato** perché garantisce predizioni molto precise e una buona performance complessiva. È particolarmente adatta se l’obiettivo è ridurre i falsi positivi; se invece si vuole massimizzare il recupero di tutte le label corrette, sarebbe necessario lavorare ulteriormente sul recall, ad esempio tramite soglie per classe o strategie mirate sulle classi meno rappresentate.

##  Results First Trainings: ANN Multi-label Classification




### 1) ALL FEATURES' TRAINING (10 Features)

The Artificial Neural Network was evaluated as a **multi-label classification model**.  
The best model was selected through Grid Search using **F0.5-score**, giving more importance to precision and therefore penalizing false positives.

### Best hyperparameters

| Parameter | Value |
|---|---:|
| Layers size | `[20000, 64, 9]` |
| Epochs | `100` |
| Batch size | `64` |
| Learning rate | `0.0001` |
| Decay rate | `0.95` |
| Delta | `1e-08` |
| Threshold | `0.7` |
| Max weight | `2.0` |

### Overall performance

| Metric | Train | Test |
|---|---:|---:|
| Loss | 0.0002 | 0.0433 |
| Accuracy | 0.9996 | 0.9464 |
| Precision | 0.9999 | 0.9152 |
| Recall | 0.9971 | 0.6823 |
| F1-score | 0.9985 | 0.7818 |
| F0.5-score | 0.9994 | 0.8567 |

### Multi-label confusion matrix

|  | True 1 | True 0 |
|---|---:|---:|
| Predicted 1 | 1555 TP | 144 FP |
| Predicted 0 | 724 FN | 13759 TN |

| Error type | Percentage |
|---|---:|
| False positives | 0.9% |
| False negatives | 4.5% |

### Per-label performance

| Label | Support | Precision | Recall | F1-score |
|---|---:|---:|---:|---:|
| EIWH | 77 | 0.7692 | 0.1299 | 0.2222 |
| ESWH | 204 | 0.8120 | 0.5294 | 0.6409 |
| EWH | 67 | 0.8125 | 0.1940 | 0.3133 |
| GAHP | 169 | 0.7763 | 0.3491 | 0.4816 |
| GASBOILER | 117 | 0.8955 | 0.5128 | 0.6522 |
| GIWH | 81 | 0.8065 | 0.3086 | 0.4464 |
| GSWH | 74 | 0.8929 | 0.3378 | 0.4902 |
| HHP | 1224 | 0.9598 | 0.9158 | 0.9373 |
| HPWH | 266 | 0.8024 | 0.5038 | 0.6189 |

### Technical interpretation

The model achieved **good overall test performance**, with **test accuracy = 0.9464**, **test precision = 0.9152**, **test recall = 0.6823**, **test F1-score = 0.7818**, and **test F0.5-score = 0.8567**.

The model remains **precision-oriented**, producing relatively few incorrect positive predictions. This is confirmed by the confusion matrix, where false positives account for only **0.9%** of all label decisions. However, recall is still lower than precision, meaning that the network misses a non-negligible number of true labels, as shown by the **4.5% false negative rate**.

The selected threshold of **0.7** provides a compromise between precision and recall: it is less conservative than higher thresholds, but the model still favors precision, consistently with the use of **F0.5-score** as the optimization criterion.

The training and test losses are both **finite and stable** (`0.0002` and `0.0433`), indicating that the loss computation is numerically stable. At the same time, the large gap between train and test metrics suggests some degree of **overfitting**, although the generalization performance on the test set remains acceptable.

From the per-label analysis, **HHP** is clearly the best predicted class, with very high precision and recall (**F1-score = 0.9373**). In contrast, minority or more difficult classes such as **EIWH**, **EWH**, **GIWH**, and **GSWH** show much lower recall, indicating that the model struggles to recover these labels reliably.

### Overall interpretation

Overall, the proposed multi-label ANN shows a solid predictive capability, especially in terms of precision, indicating that most assigned labels are reliable. The model is particularly effective in recognizing the dominant class **HHP**, while performance on less represented classes remains more limited, mainly due to lower recall. This suggests that the network is suitable for supporting patent classification as a high-precision screening tool, although further improvements are still needed to better recover minority classes and reduce the number of missed true labels.


### 2) 9 FEATURES' TRAINING (WOUT 'Abstract...' 9 Features)



### Best hyperparameters

| Parameter | Value |
|---|---:|
| Layers size | `[20000, 64, 9]` |
| Epochs | `100` |
| Batch size | `64` |
| Learning rate | `0.0001` |
| Decay rate | `0.95` |
| Delta | `1e-08` |
| Threshold | `0.8` |
| Max weight | `2.0` |

### Overall performance

| Metric | Train | Test |
|---|---:|---:|
| Loss | 0.0013 | 0.0455 |
| Accuracy | 0.9840 | 0.9288 |
| Precision | 0.9984 | 0.9038 |
| Recall | 0.9005 | 0.6241 |
| F1-score | 0.9469 | 0.7383 |
| F0.5-score | 0.9772 | 0.8294 |

### Multi-label confusion matrix

|  | True 1 | True 0 |
|---|---:|---:|
| Predicted 1 | 4000 TP | 426 FP |
| Predicted 0 | 2409 FN | 32999 TN |

| Error type | Percentage |
|---|---:|
| False positives | 1.1% |
| False negatives | 6.0% |

### Per-label performance

| Label | Support | Precision | Recall | F1-score |
|---|---:|---:|---:|---:|
| EIWH | 219 | 0.8556 | 0.3516 | 0.4984 |
| ESWH | 173 | 0.9178 | 0.3873 | 0.5447 |
| EWH | 675 | 0.8711 | 0.5007 | 0.6359 |
| GAHP | 138 | 0.6923 | 0.2609 | 0.3789 |
| GASBOILER | 574 | 0.8279 | 0.4861 | 0.6125 |
| GIWH | 367 | 0.7937 | 0.2725 | 0.4057 |
| GSWH | 669 | 0.8270 | 0.4215 | 0.5584 |
| HHP | 2728 | 0.9606 | 0.8669 | 0.9114 |
| HPWH | 866 | 0.8187 | 0.5266 | 0.6409 |

### Interpretation

The model achieved reasonable test performance, with **test accuracy = 0.9288**, **test precision = 0.9038**, **test recall = 0.6241**, **test F1-score = 0.7383**, and **test F0.5-score = 0.8294**.

The model remains quite precise, meaning that most predicted positive labels are correct. This is confirmed by the confusion matrix, where false positives represent only **1.1%** of all label decisions. However, recall is substantially lower, which means that the model misses a relatively large number of true labels. This is also reflected by the **6.0% false negative rate**.

The selected threshold of **0.8** makes the model conservative and reduces false positives, but it also lowers recall. This behavior is coherent with the use of **F0.5-score** as the optimization criterion, which favors precision more than recall.

The training and test losses are finite and stable (`0.0013` and `0.0455`), so the loss computation is numerically stable. However, the gap between training and test performance suggests some overfitting, and the relatively low recall indicates that the model may be too strict in assigning labels.

### 3) ALL FEATURES' TRAINING (10 Features) + OVERSAMPLING + THRESHOLDS PER LABEL

The Artificial Neural Network was evaluated as a **multi-label classification model**.  
The best model was selected through Grid Search using **F2-score**, in order to give more importance to recall and improve the detection of minority classes. In addition, **class-wise thresholds** and **oversampling of rare classes** were introduced to increase sensitivity on underrepresented labels.

### Best hyperparameters

| Parameter | Value |
|---|---:|
| Layers size | `[25000, 128, 9]` |
| Epochs | `100` |
| Batch size | `64` |
| Learning rate | `0.0001` |
| Decay rate | `0.95` |
| Delta | `1e-08` |
| Max weight | `5.0` |
| Thresholds per label | `[0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.2, 0.1, 0.1]` |

### Overall performance

| Metric | Train | Validation | Test |
|---|---:|---:|---:|
| Loss | 0.0016 | 0.9216 | 0.8745 |
| Accuracy | 0.9991 | 0.9359 | 0.9386 |
| Precision | 0.9933 | 0.7521 | 0.7619 |
| Recall | 1.0000 | 0.8155 | 0.8201 |
| F1-score | 0.9967 | 0.7825 | 0.7899 |
| F2-score | 0.9987 | 0.8020 | 0.8078 |

### Multi-label confusion matrix

|  | True 1 | True 0 |
|---|---:|---:|
| Predicted 1 | 1869 TP | 584 FP |
| Predicted 0 | 410 FN | 13319 TN |

| Error type | Percentage |
|---|---:|
| False positives | 3.6% |
| False negatives | 2.5% |

### Per-label performance

| Label | Support | Threshold | Precision | Recall | F1-score |
|---|---:|---:|---:|---:|---:|
| EIWH | 77 | 0.10 | 0.5714 | 0.4156 | 0.4812 |
| ESWH | 204 | 0.10 | 0.6145 | 0.7500 | 0.6755 |
| GAHP | 67 | 0.10 | 0.4792 | 0.3433 | 0.4000 |
| GASBOILER | 169 | 0.10 | 0.5914 | 0.6509 | 0.6197 |
| GIWH | 117 | 0.10 | 0.7143 | 0.7692 | 0.7407 |
| GSWH | 81 | 0.10 | 0.6364 | 0.6914 | 0.6627 |
| GWSH | 74 | 0.20 | 0.6444 | 0.3919 | 0.4874 |
| HHP | 1224 | 0.10 | 0.8974 | 0.9649 | 0.9299 |
| HPWH | 266 | 0.10 | 0.5752 | 0.7331 | 0.6446 |

### Technical interpretation

The model achieved **balanced overall performance**, with **test accuracy = 0.9386**, **test precision = 0.7619**, **test recall = 0.8201**, **test F1-score = 0.7899**, and **test F2-score = 0.8078**.

Compared with the previous precision-oriented configurations, this model is clearly more **recall-oriented**. The lower class-wise thresholds and the oversampling strategy increased the model’s ability to recover true positive labels, especially for minority classes. This is confirmed by the relatively low **false negative rate (2.5%)**, which is now lower than the **false positive rate (3.6%)**.

This behavior indicates a deliberate trade-off: the model now captures more relevant labels, but at the cost of more false positive assignments. Such a trade-off is coherent with the use of **F2-score** as the optimization criterion, which emphasizes recall more strongly than precision.

The training metrics remain extremely high, while validation and test scores are lower, suggesting that some degree of overfitting is still present. However, the validation and test results remain aligned, which indicates that the model is still generalizing reasonably well.

From the per-label analysis, **HHP** remains the strongest class, with very high precision and recall (**F1-score = 0.9299**). At the same time, several minority classes such as **EIWH**, **GAHP**, and **GWSH** still show limited recall or moderate precision, indicating that these labels remain the most challenging and may require additional feature engineering or more targeted balancing strategies.

### Overall interpretation

Overall, the proposed multi-label ANN shows a stronger ability to identify minority labels than the previous configurations. The introduction of **oversampling**, **class weighting**, and **thresholds per label** improved recall and reduced missed true labels, making the model more suitable for applications in which sensitivity is more important than a very conservative prediction policy. Although precision decreased compared with earlier settings, the resulting trade-off appears more balanced and potentially more useful for patent screening tasks where missing relevant labels is costly.


# TAKE A LOOK MODEL

## Uploading the datasets

### 2024

2024_Estrazione.xlsx

In [ ]:
uploaded = files.upload()

Saving 2024_Estrazione.xlsx to 2024_Estrazione.xlsx


In [ ]:
df_2024_estrazione = pd.read_excel("/content/2024_Estrazione.xlsx")

In [ ]:
df_2024_estrazione.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8683 entries, 0 to 8682
Data columns (total 30 columns):
 #   Column                                     Non-Null Count  Dtype 
---  ------                                     --------------  ----- 
 0   Piattaforma Estrazione                     8683 non-null   object
 1   Number                                     8683 non-null   int64 
 2   Publication Number                         8683 non-null   object
 3   Patent Type                                8683 non-null   object
 4   Title (Translated)(English)                8683 non-null   object
 5   Abstract (Translated)(English)             8683 non-null   object
 6   Independent Claims (Translated)(English)   8683 non-null   object
 7   IPC                                        8683 non-null   object
 8   CPC                                        8683 non-null   object
 9   Application domain                         8683 non-null   object
 10  Technology topic                    

2024_Assegnazione.xlsx

In [ ]:
uploaded = files.upload()

Saving 2024_Assegnazione.xlsx to 2024_Assegnazione.xlsx


In [ ]:
df_2024_assegnazione = pd.read_excel("/content/2024_Assegnazione.xlsx")

In [ ]:
df_2024_assegnazione.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5264 entries, 0 to 5263
Data columns (total 48 columns):
 #   Column                                     Non-Null Count  Dtype  
---  ------                                     --------------  -----  
 0   Number                                     5264 non-null   int64  
 1   Piattaforma Assegnazione                   5264 non-null   object 
 2   Predicted Neural Network
PLATFORM          4843 non-null   object 
 3   Publication Number                         5264 non-null   object 
 4   Patent Type                                5264 non-null   object 
 5   Title (Translated)(English)                5264 non-null   object 
 6   Abstract (Translated)(English)             5264 non-null   object 
 7   Independent Claims (Translated)(English)   5264 non-null   object 
 8   IPC                                        5264 non-null   object 
 9   CPC                                        5264 non-null   object 
 10  Application domain      

Creazione dataset 2024 combined

In [ ]:
def normalize_platform_label(label):
    if pd.isna(label):
        return None

    label = str(label).upper()
    label = re.sub(r"GAS\s+BOILER", "GASBOILER", label)
    label = label.replace("/", " ")
    label = re.sub(r"\s+", " ", label).strip()

    # Convert 'NON' and 'RELEVANT' to 'NOT_RELEVANT'
    if label == "NON" or label == "RELEVANT":
        return "NOT_RELEVANT"

    return label if label != "" else None

def combine_labels(series):
    labels = []
    for value in series.dropna():
        for token in str(value).split():
            if token not in labels:
                labels.append(token)
    return " ".join(labels)


common_cols_estrazione = [
    'Abstract (Translated)(English)',
    'Independent Claims (Translated)(English)',
    'IPC',
    'CPC',
    'Application domain',
    'Technology topic',
    'Technical Problem',
    'Technical Field',
    'Std. Technical Benefit Phrase',
    'Title (Translated)(English)'
]

# Step 1: Select and rename columns from df_2024_estrazione
df_estrazione_processed = df_2024_estrazione[
    common_cols_estrazione + ['Piattaforma Estrazione', 'Publication Number']
].copy()

df_estrazione_processed = df_estrazione_processed.rename(columns={
    'Piattaforma Estrazione': 'Piattaforma di Estrazione',
    'Publication Number': 'Pubblication Number'
})

# Normalize the 'Piattaforma di Estrazione' column from the original df_2024_estrazione
df_estrazione_processed['Piattaforma di Estrazione'] = df_estrazione_processed['Piattaforma di Estrazione'].apply(normalize_platform_label)

# Step 2: Select and rename columns from df_2024_assegnazione and handle multi-labels
df_assegnazione_temp = df_2024_assegnazione[['Piattaforma Assegnazione', 'Publication Number']].copy()

df_assegnazione_temp['Piattaforma Assegnazione'] = df_assegnazione_temp['Piattaforma Assegnazione'].apply(normalize_platform_label)
df_assegnazione_temp['Publication Number'] = df_assegnazione_temp['Publication Number'].astype(str).str.strip().str.upper()

# Aggregate 'Piattaforma Assegnazione' for each 'Publication Number' to handle multi-labels
df_assegnazione_processed = df_assegnazione_temp.groupby('Publication Number', as_index=False).agg({
    'Piattaforma Assegnazione': combine_labels
})

df_assegnazione_processed = df_assegnazione_processed.rename(columns={
    'Publication Number': 'Pubblication Number'
})

# Step 3: Perform a left merge to add 'Piattaforma Assegnazione' to the base dataframe
df_2024_combined = pd.merge(
    df_estrazione_processed,
    df_assegnazione_processed,
    on='Pubblication Number',
    how='left'
)

# Step 4: Fill NaN values in 'Piattaforma Assegnazione' with 'NOT_RELEVANT'
df_2024_combined['Piattaforma Assegnazione'] = df_2024_combined['Piattaforma Assegnazione'].fillna('NOT_RELEVANT')


# Display info and head of the combined dataframe
print("Combined 2024 dataset info:")
df_2024_combined.info()
print("\nCombined 2024 dataset head:")
display(df_2024_combined.head())

Combined 2024 dataset info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8683 entries, 0 to 8682
Data columns (total 13 columns):
 #   Column                                    Non-Null Count  Dtype 
---  ------                                    --------------  ----- 
 0   Abstract (Translated)(English)            8683 non-null   object
 1   Independent Claims (Translated)(English)  8683 non-null   object
 2   IPC                                       8683 non-null   object
 3   CPC                                       8683 non-null   object
 4   Application domain                        8683 non-null   object
 5   Technology topic                          8683 non-null   object
 6   Technical Problem                         8683 non-null   object
 7   Technical Field                           8683 non-null   object
 8   Std. Technical Benefit Phrase             8683 non-null   object
 9   Title (Translated)(English)               8683 non-null   object
 10  Piattaforma di Estra

,Abstract (Translated)(English),Independent Claims (Translated)(English),IPC,CPC,Application domain,Technology topic,Technical Problem,Technical Field,Std. Technical Benefit Phrase,Title (Translated)(English),Piattaforma di Estrazione,Pubblication Number,Piattaforma Assegnazione
0,A water heating system is disclosed. The syste...,1. A water heating system comprising:_x000D_\n...,F24H15/168 | F24H15/108 | F24H15/219 | F24H15/...,F24H15/168 | F24H15/108 | F24H15/219 | F24H15/...,Electric power|Control of fluid heaters|Load s...,Electrical battery|Process engineering|Environ...,Tankless water heating systems face installati...,[0002]The present disclosure relates to water ...,-,Tankless Water Heating Systems and Methods tha...,EWH,US20240271826A1,EIWH HPWH
1,"Systems, devices and methods for wirelessly ch...",1. A system for wirelessly charging a water he...,F24H1/00 | H02J50/12 | H02J50/20 | H02J50/30 |...,F24H1/0018 | H02J50/12 | H02J50/20 | H02J50/30...,Water heaters|Induction current sources|Ac net...,Transmitter|Water heating|Physics|Receiver|Hea...,Water heating devices often face challenges in...,[0002]The present disclosure is generally rela...,-,Systems and methods for powering water heating...,EWH,US20240159427A1,ESWH EIWH HPWH
2,Systems and methods are provided for wireless ...,1. A method for controlling a wireless water h...,F24H15/464 | G05B19/4155,F24H15/464 | G05B19/4155 | G05B2219/50333 | F2...,Programme control|Computer control|Simulator c...,Environmental engineering|Thermal water|User e...,Existing cascading water heater systems are vu...,[0001]The present disclosure is generally in t...,-,Systems and Methods for a Wireless Cascading W...,EWH,US20240410623A1,GIWH HPWH GASBOILER
3,Abstract A temperature sensing device 10 for a...,1. A water heater including:_x000D_\na water s...,F24H15/225 | F24H9/20 | F24H15/37 | G01K1/14 |...,F24H15/225 | F24H9/20 | G05D23/1931 | F24H15/3...,Thermometer details|Thermometers using electri...,Sensing data|Control cell|Process engineering|...,Water heaters with storage tanks experience te...,-,-,Water heater sensor,EWH,AU2023206174A1,ESWH
4,A water heating system can include a water hea...,"21. A method for controlling water heaters, th...",F24H9/20 | F24D19/00 | F24D19/10 | F24H1/20 | ...,F24H9/2021 | F24D19/1051 | F24H1/202 | F24H15/...,Heating fuel|Temperature control using digital...,Environmental engineering|Thermal water|Heatin...,Storage-type water heaters often face challeng...,[0002]The present disclosure relates generally...,-,Water heaters with real-time hot water supply ...,EWH,US20240280293A1,ESWH


In [ ]:
df_2024_combined.to_excel("df_2024_combined.xlsx", index=False)
files.download("df_2024_combined.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Sanity Checks

In [ ]:
print("\n--- df_2024_combined Info ---")
df_2024_combined.info()

print("\n--- df_2024_combined Head ---")
display(df_2024_combined.head())

print("\n--- Multi-label Analysis ---")
y_raw_2024_combined = df_2024_combined['Piattaforma Assegnazione']

y_split_2024_combined = y_raw_2024_combined.apply(lambda x: x.split())

mlb_2024_combined = MultiLabelBinarizer()
y_multi_2024_combined = mlb_2024_combined.fit_transform(y_split_2024_combined)

num_classes_2024_combined = len(mlb_2024_combined.classes_)

print(f"Number of unique multi-labels: {num_classes_2024_combined}")
print(f"Multi-labels: {mlb_2024_combined.classes_}")

class_counts_2024_combined = y_multi_2024_combined.sum(axis=0)

class_distribution_2024_combined = pd.DataFrame({
    "Classe": mlb_2024_combined.classes_,
    "Numero di sample": class_counts_2024_combined
}).sort_values(by="Numero di sample", ascending=False)

print("\nDistribution of multi-labels in df_2024_combined:")
display(class_distribution_2024_combined)



--- df_2024_combined Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8683 entries, 0 to 8682
Data columns (total 13 columns):
 #   Column                                    Non-Null Count  Dtype 
---  ------                                    --------------  ----- 
 0   Abstract (Translated)(English)            8683 non-null   object
 1   Independent Claims (Translated)(English)  8683 non-null   object
 2   IPC                                       8683 non-null   object
 3   CPC                                       8683 non-null   object
 4   Application domain                        8683 non-null   object
 5   Technology topic                          8683 non-null   object
 6   Technical Problem                         8683 non-null   object
 7   Technical Field                           8683 non-null   object
 8   Std. Technical Benefit Phrase             8683 non-null   object
 9   Title (Translated)(English)               8683 non-null   object
 10  Piattaforma di Es

,Abstract (Translated)(English),Independent Claims (Translated)(English),IPC,CPC,Application domain,Technology topic,Technical Problem,Technical Field,Std. Technical Benefit Phrase,Title (Translated)(English),Piattaforma di Estrazione,Pubblication Number,Piattaforma Assegnazione
0,A water heating system is disclosed. The syste...,1. A water heating system comprising:_x000D_\n...,F24H15/168 | F24H15/108 | F24H15/219 | F24H15/...,F24H15/168 | F24H15/108 | F24H15/219 | F24H15/...,Electric power|Control of fluid heaters|Load s...,Electrical battery|Process engineering|Environ...,Tankless water heating systems face installati...,[0002]The present disclosure relates to water ...,-,Tankless Water Heating Systems and Methods tha...,EWH,US20240271826A1,EIWH HPWH
1,"Systems, devices and methods for wirelessly ch...",1. A system for wirelessly charging a water he...,F24H1/00 | H02J50/12 | H02J50/20 | H02J50/30 |...,F24H1/0018 | H02J50/12 | H02J50/20 | H02J50/30...,Water heaters|Induction current sources|Ac net...,Transmitter|Water heating|Physics|Receiver|Hea...,Water heating devices often face challenges in...,[0002]The present disclosure is generally rela...,-,Systems and methods for powering water heating...,EWH,US20240159427A1,ESWH EIWH HPWH
2,Systems and methods are provided for wireless ...,1. A method for controlling a wireless water h...,F24H15/464 | G05B19/4155,F24H15/464 | G05B19/4155 | G05B2219/50333 | F2...,Programme control|Computer control|Simulator c...,Environmental engineering|Thermal water|User e...,Existing cascading water heater systems are vu...,[0001]The present disclosure is generally in t...,-,Systems and Methods for a Wireless Cascading W...,EWH,US20240410623A1,GIWH HPWH GASBOILER
3,Abstract A temperature sensing device 10 for a...,1. A water heater including:_x000D_\na water s...,F24H15/225 | F24H9/20 | F24H15/37 | G01K1/14 |...,F24H15/225 | F24H9/20 | G05D23/1931 | F24H15/3...,Thermometer details|Thermometers using electri...,Sensing data|Control cell|Process engineering|...,Water heaters with storage tanks experience te...,-,-,Water heater sensor,EWH,AU2023206174A1,ESWH
4,A water heating system can include a water hea...,"21. A method for controlling water heaters, th...",F24H9/20 | F24D19/00 | F24D19/10 | F24H1/20 | ...,F24H9/2021 | F24D19/1051 | F24H1/202 | F24H15/...,Heating fuel|Temperature control using digital...,Environmental engineering|Thermal water|Heatin...,Storage-type water heaters often face challeng...,[0002]The present disclosure relates generally...,-,Water heaters with real-time hot water supply ...,EWH,US20240280293A1,ESWH



--- Multi-label Analysis ---
Number of unique multi-labels: 9
Multi-labels: ['EIWH' 'ESWH' 'GAHP' 'GASBOILER' 'GIWH' 'GSWH' 'HHP' 'HPWH'
 'NOT_RELEVANT']

Distribution of multi-labels in df_2024_combined:


,Classe,Numero di sample
6,HHP,4274
8,NOT_RELEVANT,1306
5,GSWH,1215
1,ESWH,1058
7,HPWH,942
3,GASBOILER,669
4,GIWH,443
0,EIWH,219
2,GAHP,145


In [ ]:
print("\n--- Verifica Piattaforma di Estrazione (singole label) ---")
# Count entries in 'Piattaforma di Estrazione' that contain spaces, indicating multiple labels
multi_labels_estrazione = df_2024_combined['Piattaforma di Estrazione'].apply(lambda x: ' ' in str(x)).sum()

if multi_labels_estrazione == 0:
    print("Tutte le etichette in 'Piattaforma di Estrazione' sono singole.")
else:
    print(f"Trovate {multi_labels_estrazione} etichette multiple in 'Piattaforma di Estrazione'.")

print("\n--- Verifica Piattaforma Assegnazione (multi-label) ---")
# Count entries in 'Piattaforma Assegnazione' that contain spaces, indicating multiple labels
multi_labels_assegnazione = df_2024_combined['Piattaforma Assegnazione'].apply(lambda x: ' ' in str(x)).sum()

if multi_labels_assegnazione > 0:
    print(f"Trovate {multi_labels_assegnazione} etichette multiple in 'Piattaforma Assegnazione'.")
else:
    print("Nessuna etichetta multipla trovata in 'Piattaforma Assegnazione'.")



--- Verifica Piattaforma di Estrazione (singole label) ---
Tutte le etichette in 'Piattaforma di Estrazione' sono singole.

--- Verifica Piattaforma Assegnazione (multi-label) ---
Trovate 1380 etichette multiple in 'Piattaforma Assegnazione'.


In [ ]:
print("\n--- Verification for 'NOT_RELEVANT' in Piattaforma Assegnazione (df_2024_combined) ---")

# Filter rows where 'Piattaforma Assegnazione' contains 'NOT_RELEVANT'
not_relevant_samples_2024 = df_2024_combined[
    df_2024_combined['Piattaforma Assegnazione'].str.contains('NOT_RELEVANT', na=False)
].copy()

# Check if any of these samples have other labels besides 'NOT_RELEVANT'
# This implies checking if there is more than one word in the string after splitting, or if there is a space.
contradictory_samples_2024 = not_relevant_samples_2024[
    not_relevant_samples_2024['Piattaforma Assegnazione'].apply(lambda x: len(x.split()) > 1)
]

if not contradictory_samples_2024.empty:
    print(f"Found {len(contradictory_samples_2024)} samples in df_2024_combined with 'NOT_RELEVANT' and other labels:")
    display(contradictory_samples_2024[['Pubblication Number', 'Piattaforma Assegnazione']].head())
else:
    print("No samples found in df_2024_combined where 'Piattaforma Assegnazione' contains 'NOT_RELEVANT' alongside other labels.")


--- Verification for 'NOT_RELEVANT' in Piattaforma Assegnazione (df_2024_combined) ---
No samples found in df_2024_combined where 'Piattaforma Assegnazione' contains 'NOT_RELEVANT' alongside other labels.


### 2025

2025_Estrazione.xlsx

In [ ]:
uploaded = files.upload()

Saving 2025_Estrazione.xlsx to 2025_Estrazione.xlsx


In [ ]:
df_2025_estrazione = pd.read_excel("/content/2025_Estrazione.xlsx")

In [ ]:
df_2025_estrazione.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9335 entries, 0 to 9334
Data columns (total 25 columns):
 #   Column                                     Non-Null Count  Dtype 
---  ------                                     --------------  ----- 
 0   Number                                     9335 non-null   int64 
 1   Piattaforma Estrazione                     9335 non-null   object
 2   Publication Number                         9335 non-null   object
 3   text merger                                9335 non-null   object
 4   Patent Type                                9335 non-null   object
 5   Grouped Assignee                           9335 non-null   object
 6   Standardized Original Assignee(Applicant)  9335 non-null   object
 7   Title (Translated)(English)                9335 non-null   object
 8   Abstract (Translated)(English)             9335 non-null   object
 9   Independent Claims (Translated)(English)   9335 non-null   object
 10  IPC                                 

2025_Assegnazione.xlsx

In [ ]:
uploaded = files.upload()

Saving 2025_Assegnazione.xlsx to 2025_Assegnazione.xlsx


In [ ]:
df_2025_assegnazione = pd.read_excel("/content/2025_Assegnazione.xlsx")

In [ ]:
df_2025_assegnazione.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5903 entries, 0 to 5902
Data columns (total 48 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   #                             5903 non-null   int64  
 1   Piattaforma Assegnazione      5902 non-null   object 
 2   ESWH                          5903 non-null   int64  
 3   GSWH                          5903 non-null   int64  
 4   EIWH                          5903 non-null   int64  
 5   GIWH                          5903 non-null   int64  
 6   GAS BOILER                    5903 non-null   int64  
 7   HPWH                          5903 non-null   int64  
 8   HHP                           5903 non-null   int64  
 9   GAHP                          5903 non-null   int64  
 10  TAGS                          5903 non-null   object 
 11  MAIN TAG (Gemini)             16 non-null     object 
 12  SECONDARY TAGs (Gemini)       16 non-null     object 
 13  Key

Create 2025 dataset combined

In [ ]:
common_cols_estrazione = [
    'Abstract (Translated)(English)',
    'Independent Claims (Translated)(English)',
    'IPC',
    'CPC',
    'Application domain',
    'Technology topic',
    'Technical Problem',
    'Technical Field',
    'Std. Technical Benefit Phrase',
    'Title (Translated)(English)'
]

def normalize_platform_label(label):
    if pd.isna(label):
        return None

    label = str(label).upper()
    label = re.sub(r"GAS\s+BOILER", "GASBOILER", label)
    label = label.replace("/", " ")
    label = re.sub(r"\s+", " ", label).strip()

    # Convert 'NON' and 'RELEVANT' to 'NOT_RELEVANT'
    if label == "NON" or label == "RELEVANT":
        return "NOT_RELEVANT"

    return label if label != "" else None

def combine_labels(series):
    labels = set() # Use a set to automatically handle uniqueness and then convert to list for sorting
    for value in series.dropna():
        # Apply the normalize_platform_label function to the entire value first.
        # This will convert "NON" or "RELEVANT" if they are standalone values.
        # For multi-word values like "HHP NON", it will leave them as is at this step.
        cleaned_value = normalize_platform_label(value)

        if cleaned_value is None:
            continue

        # Split the (potentially already normalized) string into individual tokens
        tokens = cleaned_value.split()

        for token in tokens:
            # Now, normalize each individual token. This will catch "NON" or "RELEVANT"
            # that were part of a multi-word string.
            normalized_token = normalize_platform_label(token)
            if normalized_token is not None and normalized_token != "":
                labels.add(normalized_token)
    return " ".join(sorted(list(labels)))


# Ensure all common_cols_estrazione are present in df_2025_estrazione
for col in common_cols_estrazione:
    if col not in df_2025_estrazione.columns:
        df_2025_estrazione[col] = ""

# Step 1: Select and rename columns from df_2025_estrazione
df_estrazione_2025_processed = df_2025_estrazione[
    common_cols_estrazione + ['Piattaforma Estrazione', 'Publication Number']
].copy()

df_estrazione_2025_processed = df_estrazione_2025_processed.rename(columns={
    'Piattaforma Estrazione': 'Piattaforma di Estrazione',
    'Publication Number': 'Pubblication Number'
})

# Normalize the 'Piattaforma di Estrazione' column
df_estrazione_2025_processed['Piattaforma di Estrazione'] = df_estrazione_2025_processed['Piattaforma di Estrazione'].apply(normalize_platform_label)

# Step 2: Select and rename columns from df_2025_assegnazione and handle multi-labels
df_assegnazione_2025_temp = df_2025_assegnazione[['Piattaforma Assegnazione', 'Publication Number']].copy()

df_assegnazione_2025_temp['Piattaforma Assegnazione'] = df_assegnazione_2025_temp['Piattaforma Assegnazione'].apply(normalize_platform_label)
df_assegnazione_2025_temp['Publication Number'] = df_assegnazione_2025_temp['Publication Number'].astype(str).str.strip().str.upper()

# Aggregate 'Piattaforma Assegnazione' for each 'Publication Number' to handle multi-labels
df_assegnazione_2025_processed = df_assegnazione_2025_temp.groupby('Publication Number', as_index=False).agg({
    'Piattaforma Assegnazione': combine_labels
})

df_assegnazione_2025_processed = df_assegnazione_2025_processed.rename(columns={
    'Publication Number': 'Pubblication Number'
})

# Step 3: Perform a left merge to add 'Piattaforma Assegnazione' to the base dataframe
df_2025_combined = pd.merge(
    df_estrazione_2025_processed,
    df_assegnazione_2025_processed,
    on='Pubblication Number',
    how='left'
)

# Step 4: Fill NaN values in 'Piattaforma Assegnazione' with 'NOT_RELEVANT'
df_2025_combined['Piattaforma Assegnazione'] = df_2025_combined['Piattaforma Assegnazione'].fillna('NOT_RELEVANT')


# Display info and head of the combined dataframe
print("Combined 2025 dataset info:")
df_2025_combined.info()
print("\nCombined 2025 dataset head:")
display(df_2025_combined.head())

Combined 2025 dataset info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9335 entries, 0 to 9334
Data columns (total 13 columns):
 #   Column                                    Non-Null Count  Dtype 
---  ------                                    --------------  ----- 
 0   Abstract (Translated)(English)            9335 non-null   object
 1   Independent Claims (Translated)(English)  9335 non-null   object
 2   IPC                                       9335 non-null   object
 3   CPC                                       9335 non-null   object
 4   Application domain                        9335 non-null   object
 5   Technology topic                          9335 non-null   object
 6   Technical Problem                         9335 non-null   object
 7   Technical Field                           9335 non-null   object
 8   Std. Technical Benefit Phrase             9335 non-null   object
 9   Title (Translated)(English)               9335 non-null   object
 10  Piattaforma di Estra

,Abstract (Translated)(English),Independent Claims (Translated)(English),IPC,CPC,Application domain,Technology topic,Technical Problem,Technical Field,Std. Technical Benefit Phrase,Title (Translated)(English),Piattaforma di Estrazione,Pubblication Number,Piattaforma Assegnazione
0,Process for ammonia synthesis using a reaction...,1. A process for ammonia synthesis with a reac...,C01C1/04 | B01D5/00 | B01J8/02 | B01J23/755 | ...,,Additive manufacturing apparatus|Vapor condens...,Ptru catalyst|Nitrogen gas|Reagent|Chemical en...,Current ammonia synthesis processes are ineffi...,,,Process for ammonia synthesis using a reaction...,GASBOILER,AU2024259712A1,NOT_RELEVANT
1,An air conditioner includes a rotary shaft mem...,[Claim 1] An air conditioner including:_...,F24F1/028 | F24F13/14 | F24F13/20,,Lighting and heating apparatus|Heating and ven...,Rotational axis|Air conditioning|Physics,Existing air conditioners suffer from air leak...,,,Air conditioner,HHP,AU2025201295A1,HHP
2,The vertical boiler with bottom water supply a...,-,F28D20/00 | F24H1/18 | F24H1/20 | F24H9/13,,Mechanical apparatus|Heat storage plants|Water...,Water storage|Mechanical engineering|Water pip...,-,,,Vertical boiler with bottom water supply,ESWH,BG67821B1,ESWH
3,-,-,F23D14/10,,Gaseous fuel burner,-,Existing gas water heaters face the issue of e...,,,GAS WATER HEATER,EIWH,BR102024024922A2,NOT_RELEVANT
4,-,-,F23D14/10,,Gaseous fuel burner,-,Existing gas water heaters face the issue of e...,,,GAS WATER HEATER,ESWH,BR102024024922A2,NOT_RELEVANT


Sanity Checks

In [ ]:
print("\n--- df_2025_combined Info ---")
df_2025_combined.info()

print("\n--- df_2025_combined Head ---")
display(df_2025_combined.head())

print("\n--- Multi-label Analysis for df_2025_combined ---")
y_raw_2025_combined = df_2025_combined['Piattaforma Assegnazione']

y_split_2025_combined = y_raw_2025_combined.apply(lambda x: x.split())

mlb_2025_combined = MultiLabelBinarizer()
y_multi_2025_combined = mlb_2025_combined.fit_transform(y_split_2025_combined)

num_classes_2025_combined = len(mlb_2025_combined.classes_)

print(f"Number of unique multi-labels: {num_classes_2025_combined}")
print(f"Multi-labels: {mlb_2025_combined.classes_}")

class_counts_2025_combined = y_multi_2025_combined.sum(axis=0)

class_distribution_2025_combined = pd.DataFrame({
    "Classe": mlb_2025_combined.classes_,
    "Numero di sample": class_counts_2025_combined
}).sort_values(by="Numero di sample", ascending=False)

print("\nDistribution of multi-labels in df_2025_combined:")
display(class_distribution_2025_combined)

print("\n--- Verifica Piattaforma di Estrazione (singole label) ---")
# Count entries in 'Piattaforma di Estrazione' that contain spaces, indicating multiple labels
multi_labels_estrazione_2025 = df_2025_combined['Piattaforma di Estrazione'].apply(lambda x: ' ' in str(x)).sum()

if multi_labels_estrazione_2025 == 0:
    print("Tutte le etichette in 'Piattaforma di Estrazione' sono singole.")
else:
    print(f"Trovate {multi_labels_estrazione_2025} etichette multiple in 'Piattaforma di Estrazione'.")

print("\n--- Verifica Piattaforma Assegnazione (multi-label) ---")
# Count entries in 'Piattaforma Assegnazione' that contain spaces, indicating multiple labels
multi_labels_assegnazione_2025 = df_2025_combined['Piattaforma Assegnazione'].apply(lambda x: ' ' in str(x)).sum()

if multi_labels_assegnazione_2025 > 0:
    print(f"Trovate {multi_labels_assegnazione_2025} etichette multiple in 'Piattaforma Assegnazione'.")
else:
    print("Nessuna etichetta multipla trovata in 'Piattaforma Assegnazione'.")


--- df_2025_combined Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9335 entries, 0 to 9334
Data columns (total 13 columns):
 #   Column                                    Non-Null Count  Dtype 
---  ------                                    --------------  ----- 
 0   Abstract (Translated)(English)            9335 non-null   object
 1   Independent Claims (Translated)(English)  9335 non-null   object
 2   IPC                                       9335 non-null   object
 3   CPC                                       9335 non-null   object
 4   Application domain                        9335 non-null   object
 5   Technology topic                          9335 non-null   object
 6   Technical Problem                         9335 non-null   object
 7   Technical Field                           9335 non-null   object
 8   Std. Technical Benefit Phrase             9335 non-null   object
 9   Title (Translated)(English)               9335 non-null   object
 10  Piattaforma di Es

,Abstract (Translated)(English),Independent Claims (Translated)(English),IPC,CPC,Application domain,Technology topic,Technical Problem,Technical Field,Std. Technical Benefit Phrase,Title (Translated)(English),Piattaforma di Estrazione,Pubblication Number,Piattaforma Assegnazione
0,Process for ammonia synthesis using a reaction...,1. A process for ammonia synthesis with a reac...,C01C1/04 | B01D5/00 | B01J8/02 | B01J23/755 | ...,,Additive manufacturing apparatus|Vapor condens...,Ptru catalyst|Nitrogen gas|Reagent|Chemical en...,Current ammonia synthesis processes are ineffi...,,,Process for ammonia synthesis using a reaction...,GASBOILER,AU2024259712A1,NOT_RELEVANT
1,An air conditioner includes a rotary shaft mem...,[Claim 1] An air conditioner including:_...,F24F1/028 | F24F13/14 | F24F13/20,,Lighting and heating apparatus|Heating and ven...,Rotational axis|Air conditioning|Physics,Existing air conditioners suffer from air leak...,,,Air conditioner,HHP,AU2025201295A1,HHP
2,The vertical boiler with bottom water supply a...,-,F28D20/00 | F24H1/18 | F24H1/20 | F24H9/13,,Mechanical apparatus|Heat storage plants|Water...,Water storage|Mechanical engineering|Water pip...,-,,,Vertical boiler with bottom water supply,ESWH,BG67821B1,ESWH
3,-,-,F23D14/10,,Gaseous fuel burner,-,Existing gas water heaters face the issue of e...,,,GAS WATER HEATER,EIWH,BR102024024922A2,NOT_RELEVANT
4,-,-,F23D14/10,,Gaseous fuel burner,-,Existing gas water heaters face the issue of e...,,,GAS WATER HEATER,ESWH,BR102024024922A2,NOT_RELEVANT



--- Multi-label Analysis for df_2025_combined ---
Number of unique multi-labels: 9
Multi-labels: ['EIWH' 'ESWH' 'GAHP' 'GASBOILER' 'GIWH' 'GSWH' 'HHP' 'HPWH'
 'NOT_RELEVANT']

Distribution of multi-labels in df_2025_combined:


,Classe,Numero di sample
6,HHP,4978
5,GSWH,1273
7,HPWH,1238
8,NOT_RELEVANT,1131
1,ESWH,1085
3,GASBOILER,544
4,GIWH,162
0,EIWH,146
2,GAHP,49



--- Verifica Piattaforma di Estrazione (singole label) ---
Tutte le etichette in 'Piattaforma di Estrazione' sono singole.

--- Verifica Piattaforma Assegnazione (multi-label) ---
Trovate 1073 etichette multiple in 'Piattaforma Assegnazione'.


In [ ]:
print("\n--- Verification for 'NOT_RELEVANT' in Piattaforma Assegnazione ---")

# Filter rows where 'Piattaforma Assegnazione' contains 'NOT_RELEVANT'
not_relevant_samples = df_2025_combined[
    df_2025_combined['Piattaforma Assegnazione'].str.contains('NOT_RELEVANT', na=False)
].copy()

# Check if any of these samples have other labels besides 'NOT_RELEVANT'
# This implies checking if there is more than one word in the string after splitting, or if there is a space.
contradictory_samples = not_relevant_samples[
    not_relevant_samples['Piattaforma Assegnazione'].apply(lambda x: len(x.split()) > 1)
]

if not contradictory_samples.empty:
    print(f"Found {len(contradictory_samples)} samples with 'NOT_RELEVANT' and other labels:")
    display(contradictory_samples[['Pubblication Number', 'Piattaforma Assegnazione']].head())
else:
    print("No samples found where 'Piattaforma Assegnazione' contains 'NOT_RELEVANT' alongside other labels.")


--- Verification for 'NOT_RELEVANT' in Piattaforma Assegnazione ---
No samples found where 'Piattaforma Assegnazione' contains 'NOT_RELEVANT' alongside other labels.


In [ ]:
df_2025_combined.to_excel("df_2025_combined.xlsx", index=False)
files.download("df_2025_combined.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Union Datasets

In [ ]:
df_combined_final = pd.concat([df_2024_combined, df_2025_combined], ignore_index=True)

# Ensure the label column is clean in the final combined dataset
df_combined_final['Piattaforma Assegnazione'] = df_combined_final['Piattaforma Assegnazione'].fillna('NOT_RELEVANT')
df_combined_final = df_combined_final[df_combined_final['Piattaforma Assegnazione'].astype(str).str.strip() != ''].copy()

print("Combined dataset info:")
df_combined_final.info()

print("\nCombined dataset head:")
display(df_combined_final.head())

print("\n--- Multi-label Analysis for df_combined_final ---")
y_raw_combined_final = df_combined_final['Piattaforma Assegnazione']

y_split_combined_final = y_raw_combined_final.apply(lambda x: x.split() if isinstance(x, str) else ['NOT_RELEVANT'])

mlb_combined_final = MultiLabelBinarizer()
y_multi_combined_final = mlb_combined_final.fit_transform(y_split_combined_final)

num_classes_combined_final = len(mlb_combined_final.classes_)

print(f"Number of unique multi-labels: {num_classes_combined_final}")
print(f"Multi-labels: {mlb_combined_final.classes_}")

class_counts_combined_final = y_multi_combined_final.sum(axis=0)

class_distribution_combined_final = pd.DataFrame({
    "Classe": mlb_combined_final.classes_,
    "Numero di sample": class_counts_combined_final
}).sort_values(by="Numero di sample", ascending=False)

print("\nDistribution of multi-labels in df_combined_final:")
display(class_distribution_combined_final)


df_combined_final.to_excel("df_combined_2024_2025.xlsx", index=False)
files.download("df_combined_2024_2025.xlsx")

Combined dataset info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18018 entries, 0 to 18017
Data columns (total 13 columns):
 #   Column                                    Non-Null Count  Dtype 
---  ------                                    --------------  ----- 
 0   Abstract (Translated)(English)            18018 non-null  object
 1   Independent Claims (Translated)(English)  18018 non-null  object
 2   IPC                                       18018 non-null  object
 3   CPC                                       18018 non-null  object
 4   Application domain                        18018 non-null  object
 5   Technology topic                          18018 non-null  object
 6   Technical Problem                         18018 non-null  object
 7   Technical Field                           18018 non-null  object
 8   Std. Technical Benefit Phrase             18018 non-null  object
 9   Title (Translated)(English)               18018 non-null  object
 10  Piattaforma di Estrazio

,Abstract (Translated)(English),Independent Claims (Translated)(English),IPC,CPC,Application domain,Technology topic,Technical Problem,Technical Field,Std. Technical Benefit Phrase,Title (Translated)(English),Piattaforma di Estrazione,Pubblication Number,Piattaforma Assegnazione
0,A water heating system is disclosed. The syste...,1. A water heating system comprising:_x000D_\n...,F24H15/168 | F24H15/108 | F24H15/219 | F24H15/...,F24H15/168 | F24H15/108 | F24H15/219 | F24H15/...,Electric power|Control of fluid heaters|Load s...,Electrical battery|Process engineering|Environ...,Tankless water heating systems face installati...,[0002]The present disclosure relates to water ...,-,Tankless Water Heating Systems and Methods tha...,EWH,US20240271826A1,EIWH HPWH
1,"Systems, devices and methods for wirelessly ch...",1. A system for wirelessly charging a water he...,F24H1/00 | H02J50/12 | H02J50/20 | H02J50/30 |...,F24H1/0018 | H02J50/12 | H02J50/20 | H02J50/30...,Water heaters|Induction current sources|Ac net...,Transmitter|Water heating|Physics|Receiver|Hea...,Water heating devices often face challenges in...,[0002]The present disclosure is generally rela...,-,Systems and methods for powering water heating...,EWH,US20240159427A1,ESWH EIWH HPWH
2,Systems and methods are provided for wireless ...,1. A method for controlling a wireless water h...,F24H15/464 | G05B19/4155,F24H15/464 | G05B19/4155 | G05B2219/50333 | F2...,Programme control|Computer control|Simulator c...,Environmental engineering|Thermal water|User e...,Existing cascading water heater systems are vu...,[0001]The present disclosure is generally in t...,-,Systems and Methods for a Wireless Cascading W...,EWH,US20240410623A1,GIWH HPWH GASBOILER
3,Abstract A temperature sensing device 10 for a...,1. A water heater including:_x000D_\na water s...,F24H15/225 | F24H9/20 | F24H15/37 | G01K1/14 |...,F24H15/225 | F24H9/20 | G05D23/1931 | F24H15/3...,Thermometer details|Thermometers using electri...,Sensing data|Control cell|Process engineering|...,Water heaters with storage tanks experience te...,-,-,Water heater sensor,EWH,AU2023206174A1,ESWH
4,A water heating system can include a water hea...,"21. A method for controlling water heaters, th...",F24H9/20 | F24D19/00 | F24D19/10 | F24H1/20 | ...,F24H9/2021 | F24D19/1051 | F24H1/202 | F24H15/...,Heating fuel|Temperature control using digital...,Environmental engineering|Thermal water|Heatin...,Storage-type water heaters often face challeng...,[0002]The present disclosure relates generally...,-,Water heaters with real-time hot water supply ...,EWH,US20240280293A1,ESWH



--- Multi-label Analysis for df_combined_final ---
Number of unique multi-labels: 9
Multi-labels: ['EIWH' 'ESWH' 'GAHP' 'GASBOILER' 'GIWH' 'GSWH' 'HHP' 'HPWH'
 'NOT_RELEVANT']

Distribution of multi-labels in df_combined_final:


,Classe,Numero di sample
6,HHP,9252
5,GSWH,2488
8,NOT_RELEVANT,2437
7,HPWH,2180
1,ESWH,2143
3,GASBOILER,1213
4,GIWH,605
0,EIWH,365
2,GAHP,194


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Sanity Checks final

In [ ]:
print("DataFrame Info (including non-null counts):")
df_combined_final.info()

print("\nNull counts per column:")
print(df_combined_final.isnull().sum())

DataFrame Info (including non-null counts):
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18018 entries, 0 to 18017
Data columns (total 13 columns):
 #   Column                                    Non-Null Count  Dtype 
---  ------                                    --------------  ----- 
 0   Abstract (Translated)(English)            18018 non-null  object
 1   Independent Claims (Translated)(English)  18018 non-null  object
 2   IPC                                       18018 non-null  object
 3   CPC                                       18018 non-null  object
 4   Application domain                        18018 non-null  object
 5   Technology topic                          18018 non-null  object
 6   Technical Problem                         18018 non-null  object
 7   Technical Field                           18018 non-null  object
 8   Std. Technical Benefit Phrase             18018 non-null  object
 9   Title (Translated)(English)               18018 non-null  object
 10  Pi

## Upload Dataset when already created



In [ ]:
uploaded = files.upload()

Saving Copy of TOTAL raw_no dupl 3270.xlsx to Copy of TOTAL raw_no dupl 3270 (1).xlsx


In [ ]:
df_to_be_predicted = pd.read_excel("/content/Copy of TOTAL raw_no dupl 3270.xlsx")

df_to_be_predicted.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3270 entries, 0 to 3269
Data columns (total 25 columns):
 #   Column                                     Non-Null Count  Dtype         
---  ------                                     --------------  -----         
 0   Piattaforma di Estrazione                  3270 non-null   object        
 1   Publication Number                         3270 non-null   object        
 2   Patent Type                                3270 non-null   object        
 3   Grouped Assignee                           3270 non-null   object        
 4   Standardized Original Assignee(Applicant)  3270 non-null   object        
 5   Title (Translated)(English)                3270 non-null   object        
 6   Abstract (Translated)(English)             3270 non-null   object        
 7   Independent Claims (Translated)(English)   3270 non-null   object        
 8   IPC                                        3270 non-null   object        
 9   Application domain 

In [ ]:
df_to_be_predicted.info()
display(df_to_be_predicted.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3270 entries, 0 to 3269
Data columns (total 25 columns):
 #   Column                                     Non-Null Count  Dtype         
---  ------                                     --------------  -----         
 0   Piattaforma di Estrazione                  3270 non-null   object        
 1   Publication Number                         3270 non-null   object        
 2   Patent Type                                3270 non-null   object        
 3   Grouped Assignee                           3270 non-null   object        
 4   Standardized Original Assignee(Applicant)  3270 non-null   object        
 5   Title (Translated)(English)                3270 non-null   object        
 6   Abstract (Translated)(English)             3270 non-null   object        
 7   Independent Claims (Translated)(English)   3270 non-null   object        
 8   IPC                                        3270 non-null   object        
 9   Application domain 

,Piattaforma di Estrazione,Publication Number,Patent Type,Grouped Assignee,Standardized Original Assignee(Applicant),Title (Translated)(English),Abstract (Translated)(English),Independent Claims (Translated)(English),IPC,Application domain,...,Earliest Priority Date,Application Date,Publication Date,Application Number,Simple Legal Status,Authority,INPADOC Family,INPADOC Family Application Count,Priority Country/Region,Patent Valuation
0,HHP,US20260002708A1,Applications,DAIKIN,DAIKIN INDUSTRIES LTD | DAIKIN EURO,Heat source apparatus and refrigeration cycle ...,A heat source apparatus includes a refrigerant...,1. A heat source apparatus comprising:_x000D_\...,F25B13/00,Compression machines with reversible cycle|Ref...,...,2023-03-10,2025-09-09,2026-01-01,US19/322733,Pending,United States,CN120835976A | EP4679007A1 | EP4679007A4 | IN2...,6,-,"$ 15,000"
1,HHP / HPWH,US20260001382A1,Applications,MIDEA,MIDEA GROUP CO LTD,Recreational vehicle air conditioning system w...,A recreational vehicle air conditioning system...,1. A recreational vehicle air conditioning uni...,B60H1/00 | F25B49/02,Air-treating devices|Mechanical apparatus|Comp...,...,2021-09-30,2025-09-09,2026-01-01,US19/323055,Pending,United States,US11642936B2 | US12434533B2 | US20230098535A1 ...,3,-,"$ 140,000"
2,HPWH,CN121252034A,Applications,-,SHENZHEN WANRUN NEW ENERGY CO LTD,Falling film spiral tube integrated micro-pres...,The invention discloses a falling film spiral ...,1. A falling film spiral tube integrated micro...,F22G1/10 | F25B30/02 | F25B41/42 | F25B41/20 |...,Heat pumps|Mechanical apparatus|Fluid circulat...,...,2025-10-28,2025-10-28,2026-01-02,CN202511544134.4,Pending,China,CN121252034A,1,-,"$ 3,300"
3,ESWH / GIWH / GSWH / HPWH,CN121252192A,Applications,HAIER,QINGDAO HAIER WISDOM KITCHEN APPLIANCE CO LTD ...,One-way air supplement device,The invention discloses a one-way air suppleme...,"1. A unidirectional air supply device, charact...",F24F7/04 | F24F13/14 | F24F7/003,Lighting and heating apparatus|Air-flow contro...,...,2025-09-15,2025-09-15,2026-01-02,CN202511316467.1,Pending,China,CN121252192A,1,-,"$ 3,400"
4,ESWH / HPWH,CN121256371A,Applications,HAIER,QINGDAO HAIER TECH | QINGDAO HAIER INTELLIGENT...,Scene determination method and device based on...,The invention discloses a scene determination ...,1.A method for determining a scenario based on...,G06F18/22 | G06F18/26 | G06N3/045 | G05B15/02 ...,Computer control|Biological models|Total facto...,...,2025-07-17,2025-07-17,2026-01-02,CN202510989725.6,Pending,China,CN121256371A,1,-,"$ 3,300"


## Building the TAKE_A_LOOK Model

### Definition Model

Training phase

In [ ]:
import pickle
import numpy as np
import pandas as pd

from scipy.sparse import hstack

from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    precision_score,
    recall_score,
    fbeta_score,
    accuracy_score,
    precision_recall_fscore_support
)


def parse_multilabel_assignment_with_not_relevant_rule(
    value,
    not_relevant_label="NOT_RELEVANT"
):
    value = str(value).upper()
    value = value.replace("NOT RELEVANT", "NOT_RELEVANT")
    value = value.replace("NOT-RELEVANT", "NOT_RELEVANT")
    value = value.replace("NOT_REVELANT", "NOT_RELEVANT")
    value = value.replace("NON RELEVANT", "NOT_RELEVANT")
    value = value.replace("NON_RELEVANT", "NOT_RELEVANT")

    labels = parse_multilabel_string(value)

    normalized = []
    for label in labels:
        label = str(label).strip().upper()
        label = label.replace("NOT_REVELANT", "NOT_RELEVANT")

        if label:
            normalized.append(label)

    normalized = list(dict.fromkeys(normalized))

    if not_relevant_label in normalized:
        return [not_relevant_label]

    return normalized


def build_weighted_prefixed_text(
    df_features,
    feature_columns,
    extraction_platform_col="Piattaforma di Estrazione",
    extraction_weight=6
):
    parts = []

    for col in feature_columns:
        prefix = (
            col.replace(" (Translated)(English)", "")
               .replace("Std. ", "")
               .replace(".", "")
               .replace("/", "_")
               .replace(" ", "_")
               .upper()
        )

        col_text = df_features[col].fillna("").astype(str)

        if col == extraction_platform_col:
            for i in range(extraction_weight):
                parts.append(
                    f"EXTRACTION_PLATFORM_PRIORITY_{i}: " + col_text
                )
        else:
            parts.append(prefix + ": " + col_text)

    return pd.concat(parts, axis=1).agg(" ".join, axis=1)


def apply_not_relevant_exclusive_rule(
    probabilities,
    thresholds,
    classes,
    not_relevant_label="NOT_RELEVANT"
):
    y_pred = (probabilities >= thresholds).astype(int)
    classes = np.array(classes)

    if not_relevant_label not in classes:
        return y_pred

    nr_idx = np.where(classes == not_relevant_label)[0][0]

    for i in range(y_pred.shape[0]):
        if y_pred[i, nr_idx] == 1:
            y_pred[i, :] = 0
            y_pred[i, nr_idx] = 1

    return y_pred


def compute_metrics_with_fixed_predictions(y_true, y_pred):
    return {
        "micro_precision": precision_score(y_true, y_pred, average="micro", zero_division=0),
        "micro_recall": recall_score(y_true, y_pred, average="micro", zero_division=0),
        "micro_f1": fbeta_score(y_true, y_pred, beta=1.0, average="micro", zero_division=0),
        "micro_f2": fbeta_score(y_true, y_pred, beta=2.0, average="micro", zero_division=0),

        "macro_precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "macro_recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "macro_f1": fbeta_score(y_true, y_pred, beta=1.0, average="macro", zero_division=0),
        "macro_f2": fbeta_score(y_true, y_pred, beta=2.0, average="macro", zero_division=0),

        "label_accuracy": accuracy_score(
            np.asarray(y_true).reshape(-1),
            np.asarray(y_pred).reshape(-1)
        ),
        "subset_accuracy": accuracy_score(y_true, y_pred)
    }


def build_prediction_excel_output(
    df_input,
    feature_columns,
    label_column,
    extraction_platform_col,
    extraction_weight,
    word_tfidf,
    char_tfidf,
    best_logreg,
    best_svc,
    best_sgd,
    best_weights,
    best_thresholds,
    classes,
    not_relevant_label="NOT_RELEVANT",
    confidence_decimals=4
):
    df_predict = df_input.copy()
    feature_columns = list(feature_columns)

    X_features = df_predict[feature_columns].fillna("").copy()

    text_input = build_weighted_prefixed_text(
        X_features,
        feature_columns,
        extraction_platform_col=extraction_platform_col,
        extraction_weight=extraction_weight
    )

    texts_clean = [clean_text(text) for text in text_input]

    X_word = word_tfidf.transform(texts_clean)
    X_char = char_tfidf.transform(texts_clean)
    X_all = hstack([X_word, X_char]).tocsr()

    logreg_prob = best_logreg.predict_proba(X_all)
    svc_prob = best_svc.predict_proba(X_all)
    sgd_prob = best_sgd.predict_proba(X_all)

    probabilities = apply_per_label_ensemble(
        logreg_prob,
        svc_prob,
        sgd_prob,
        best_weights
    )

    y_pred = apply_not_relevant_exclusive_rule(
        probabilities,
        best_thresholds,
        classes,
        not_relevant_label=not_relevant_label
    )

    classes = np.array(classes)

    predicted_labels = [
        list(classes[np.where(row == 1)[0]])
        for row in y_pred
    ]

    df_predict["PREDICTED_PLATFORM"] = [
        "; ".join(labels) if len(labels) > 0 else ""
        for labels in predicted_labels
    ]

    if label_column in df_predict.columns:
        true_label_lists = df_predict[label_column].apply(
            lambda x: parse_multilabel_assignment_with_not_relevant_rule(
                x,
                not_relevant_label=not_relevant_label
            )
            if pd.notna(x) and str(x).strip() != ""
            else []
        )

        df_predict["TRUE_PLATFORM_NORMALIZED"] = [
            "; ".join(labels) for labels in true_label_lists
        ]

        df_predict["PREDICTION_EXACT_MATCH"] = [
            "SI" if set(true_labels) == set(pred_labels) else "NO"
            for true_labels, pred_labels in zip(true_label_lists, predicted_labels)
        ]

    for idx, class_name in enumerate(classes):
        safe_class_name = (
            str(class_name)
            .replace(" ", "_")
            .replace("/", "_")
            .replace("-", "_")
        )

        df_predict[f"PRED_{safe_class_name}"] = np.where(
            y_pred[:, idx] == 1,
            "SI",
            "NO"
        )

        df_predict[f"CONF_{safe_class_name}"] = np.round(
            probabilities[:, idx],
            confidence_decimals
        )

        df_predict[f"THRESHOLD_{safe_class_name}"] = np.round(
            best_thresholds[idx],
            confidence_decimals
        )

    return df_predict


def trainTestCombLinearEnsembleUltra(
    df_work,
    feature_columns,
    label_column="Piattaforma di Assegnazione",
    combo_name="FEATURE_COMBINATION_LINEAR_ENSEMBLE_ULTRA",
    publication_col="Publication Number",
    extraction_platform_col="Piattaforma di Estrazione",
    extraction_weight=6,
    not_relevant_label="NOT_RELEVANT",

    threshold_beta=1.0,
    min_precision_threshold=0.55,

    logreg_C_values=(0.5, 1.0, 2.0, 4.0, 8.0),
    svc_C_values=(0.5, 1.0, 2.0),
    sgd_alpha_values=(1e-5, 3e-5, 1e-4),

    random_state=42,
    save_bundle=True,
    save_predictions_excel=True,
    predictions_excel_filename=None
):
    print("=" * 80)
    print(f"TRAIN/TEST LINEAR ENSEMBLE ULTRA - {combo_name}")
    print("=" * 80)

    df_local = df_work.copy()
    feature_columns = list(feature_columns)

    feature_columns = [col for col in feature_columns if col != label_column]

    if extraction_platform_col not in feature_columns:
        feature_columns = [extraction_platform_col] + feature_columns

    required_columns = list(dict.fromkeys(
        [label_column, extraction_platform_col] + feature_columns
    ))

    missing_columns = [col for col in required_columns if col not in df_local.columns]

    if missing_columns:
        raise ValueError(f"Colonne mancanti nel dataset: {missing_columns}")

    df_local = df_local.dropna(subset=[label_column]).copy()
    df_local = df_local[df_local[label_column].astype(str).str.strip() != ""].copy()

    df_local["_LABEL_LIST"] = df_local[label_column].apply(
        lambda x: parse_multilabel_assignment_with_not_relevant_rule(
            x,
            not_relevant_label=not_relevant_label
        )
    )

    df_local["_LABEL_STR"] = df_local["_LABEL_LIST"].apply(lambda labels: " ".join(labels))

    if publication_col in df_local.columns:
        before = len(df_local)
        df_local[publication_col] = (
            df_local[publication_col]
            .astype(str)
            .str.strip()
            .str.upper()
        )

        df_local = df_local.drop_duplicates(
            subset=[publication_col, "_LABEL_STR"]
        ).copy()

        print(f"Deduplica brevetti/label: {before} -> {len(df_local)} righe")

    df_local = df_local.reset_index(drop=True)

    X_features = df_local[feature_columns].fillna("").copy()

    x = build_weighted_prefixed_text(
        X_features,
        feature_columns,
        extraction_platform_col=extraction_platform_col,
        extraction_weight=extraction_weight
    )

    y_split = df_local["_LABEL_LIST"]

    mlb = MultiLabelBinarizer()
    y_multi = mlb.fit_transform(y_split)

    print(f"\nNumero classi: {len(mlb.classes_)}")
    print("Classi:", mlb.classes_)

    class_distribution = pd.DataFrame({
        "Classe": mlb.classes_,
        "Numero di sample": y_multi.sum(axis=0)
    }).sort_values("Numero di sample", ascending=False)

    display(class_distribution)

    X_train_text, X_val_text, X_test_text, Y_train, Y_val, Y_test = multilabel_train_val_test_split(
        x,
        y_multi,
        test_size=0.20,
        val_size=0.15,
        random_state=random_state
    )

    print("\nSplit:")
    print("Train size:", len(X_train_text))
    print("Validation size:", len(X_val_text))
    print("Test size:", len(X_test_text))

    X_train_text_clean = [clean_text(text) for text in X_train_text]
    X_val_text_clean = [clean_text(text) for text in X_val_text]
    X_test_text_clean = [clean_text(text) for text in X_test_text]

    word_tfidf = TfidfVectorizer(
        analyzer="word",
        sublinear_tf=True,
        max_features=100000,
        ngram_range=(1, 4),
        min_df=2,
        max_df=0.98,
        stop_words=None,
        strip_accents="unicode",
        token_pattern=r"(?u)\b[A-Z0-9][A-Z0-9/.\-:]+\b|\b[A-Z0-9]\b"
    )

    char_tfidf = TfidfVectorizer(
        analyzer="char_wb",
        sublinear_tf=True,
        max_features=90000,
        ngram_range=(3, 7),
        min_df=2,
        max_df=0.98,
        strip_accents="unicode"
    )

    print("\nFitting TF-IDF word + char...")

    X_train_word = word_tfidf.fit_transform(X_train_text_clean)
    X_val_word = word_tfidf.transform(X_val_text_clean)
    X_test_word = word_tfidf.transform(X_test_text_clean)

    X_train_char = char_tfidf.fit_transform(X_train_text_clean)
    X_val_char = char_tfidf.transform(X_val_text_clean)
    X_test_char = char_tfidf.transform(X_test_text_clean)

    X_train = hstack([X_train_word, X_train_char]).tocsr()
    X_val = hstack([X_val_word, X_val_char]).tocsr()
    X_test = hstack([X_test_word, X_test_char]).tocsr()

    print("\nShape finali sparse:")
    print("X_train:", X_train.shape)
    print("X_val:  ", X_val.shape)
    print("X_test: ", X_test.shape)

    search_rows = []

    best_logreg = None
    best_logreg_score = -1.0
    best_logreg_params = None

    print("\nSearch Logistic Regression...")

    for C in logreg_C_values:
        for class_weight in ("balanced", None):
            print(f"Training LogReg C={C}, class_weight={class_weight}")

            model = OneVsRestClassifier(
                LogisticRegression(
                    solver="liblinear",
                    penalty="l2",
                    class_weight=class_weight,
                    C=C,
                    max_iter=5000,
                    random_state=random_state
                )
            )

            model.fit(X_train, Y_train)
            y_val_prob = model.predict_proba(X_val)

            thresholds = find_best_thresholds_sklearn_constrained(
                Y_val,
                y_val_prob,
                beta=threshold_beta,
                min_precision=min_precision_threshold,
                min_threshold=0.10,
                max_threshold=0.90,
                step=0.02
            )

            y_val_pred = apply_not_relevant_exclusive_rule(
                y_val_prob,
                thresholds,
                mlb.classes_,
                not_relevant_label=not_relevant_label
            )

            metrics = compute_metrics_with_fixed_predictions(Y_val, y_val_pred)
            score = metrics["micro_f1"]

            search_rows.append({
                "model": "LogReg",
                "params": f"C={C}, class_weight={class_weight}",
                "val_precision": metrics["micro_precision"],
                "val_recall": metrics["micro_recall"],
                "val_f1": metrics["micro_f1"],
                "val_f2": metrics["micro_f2"],
                "val_macro_f1": metrics["macro_f1"],
                "val_macro_f2": metrics["macro_f2"]
            })

            if score > best_logreg_score:
                best_logreg_score = score
                best_logreg = model
                best_logreg_params = {
                    "C": C,
                    "class_weight": class_weight
                }

    best_svc = None
    best_svc_score = -1.0
    best_svc_params = None

    print("\nSearch Calibrated LinearSVC...")

    for C in svc_C_values:
        for class_weight in ("balanced", None):
            print(f"Training LinearSVC C={C}, class_weight={class_weight}")

            base_svc = LinearSVC(
                C=C,
                class_weight=class_weight,
                max_iter=10000,
                random_state=random_state
            )

            model = OneVsRestClassifier(
                CalibratedClassifierCV(
                    estimator=base_svc,
                    method="sigmoid",
                    cv=3
                )
            )

            model.fit(X_train, Y_train)
            y_val_prob = model.predict_proba(X_val)

            thresholds = find_best_thresholds_sklearn_constrained(
                Y_val,
                y_val_prob,
                beta=threshold_beta,
                min_precision=min_precision_threshold,
                min_threshold=0.10,
                max_threshold=0.90,
                step=0.02
            )

            y_val_pred = apply_not_relevant_exclusive_rule(
                y_val_prob,
                thresholds,
                mlb.classes_,
                not_relevant_label=not_relevant_label
            )

            metrics = compute_metrics_with_fixed_predictions(Y_val, y_val_pred)
            score = metrics["micro_f1"]

            search_rows.append({
                "model": "CalibratedLinearSVC",
                "params": f"C={C}, class_weight={class_weight}",
                "val_precision": metrics["micro_precision"],
                "val_recall": metrics["micro_recall"],
                "val_f1": metrics["micro_f1"],
                "val_f2": metrics["micro_f2"],
                "val_macro_f1": metrics["macro_f1"],
                "val_macro_f2": metrics["macro_f2"]
            })

            if score > best_svc_score:
                best_svc_score = score
                best_svc = model
                best_svc_params = {
                    "C": C,
                    "class_weight": class_weight
                }

    best_sgd = None
    best_sgd_score = -1.0
    best_sgd_params = None

    print("\nSearch SGD Logistic...")

    for alpha in sgd_alpha_values:
        for class_weight in ("balanced", None):
            print(f"Training SGD alpha={alpha}, class_weight={class_weight}")

            model = OneVsRestClassifier(
                SGDClassifier(
                    loss="log_loss",
                    penalty="elasticnet",
                    alpha=alpha,
                    l1_ratio=0.15,
                    class_weight=class_weight,
                    max_iter=3000,
                    tol=1e-4,
                    random_state=random_state
                )
            )

            model.fit(X_train, Y_train)
            y_val_prob = model.predict_proba(X_val)

            thresholds = find_best_thresholds_sklearn_constrained(
                Y_val,
                y_val_prob,
                beta=threshold_beta,
                min_precision=min_precision_threshold,
                min_threshold=0.10,
                max_threshold=0.90,
                step=0.02
            )

            y_val_pred = apply_not_relevant_exclusive_rule(
                y_val_prob,
                thresholds,
                mlb.classes_,
                not_relevant_label=not_relevant_label
            )

            metrics = compute_metrics_with_fixed_predictions(Y_val, y_val_pred)
            score = metrics["micro_f1"]

            search_rows.append({
                "model": "SGDLogistic",
                "params": f"alpha={alpha}, class_weight={class_weight}",
                "val_precision": metrics["micro_precision"],
                "val_recall": metrics["micro_recall"],
                "val_f1": metrics["micro_f1"],
                "val_f2": metrics["micro_f2"],
                "val_macro_f1": metrics["macro_f1"],
                "val_macro_f2": metrics["macro_f2"]
            })

            if score > best_sgd_score:
                best_sgd_score = score
                best_sgd = model
                best_sgd_params = {
                    "alpha": alpha,
                    "class_weight": class_weight
                }

    print("\nComputing best model probabilities...")

    logreg_train_prob = best_logreg.predict_proba(X_train)
    logreg_val_prob = best_logreg.predict_proba(X_val)
    logreg_test_prob = best_logreg.predict_proba(X_test)

    svc_train_prob = best_svc.predict_proba(X_train)
    svc_val_prob = best_svc.predict_proba(X_val)
    svc_test_prob = best_svc.predict_proba(X_test)

    sgd_train_prob = best_sgd.predict_proba(X_train)
    sgd_val_prob = best_sgd.predict_proba(X_val)
    sgd_test_prob = best_sgd.predict_proba(X_test)

    print("\nOptimizing per-label ensemble weights and thresholds...")

    best_weights, best_thresholds = optimize_per_label_ensemble(
        Y_val,
        logreg_val_prob,
        svc_val_prob,
        sgd_val_prob,
        beta=threshold_beta,
        min_precision=min_precision_threshold
    )

    best_train_prob = apply_per_label_ensemble(
        logreg_train_prob,
        svc_train_prob,
        sgd_train_prob,
        best_weights
    )

    best_val_prob = apply_per_label_ensemble(
        logreg_val_prob,
        svc_val_prob,
        sgd_val_prob,
        best_weights
    )

    best_test_prob = apply_per_label_ensemble(
        logreg_test_prob,
        svc_test_prob,
        sgd_test_prob,
        best_weights
    )

    y_train_pred = apply_not_relevant_exclusive_rule(
        best_train_prob,
        best_thresholds,
        mlb.classes_,
        not_relevant_label=not_relevant_label
    )

    y_val_pred = apply_not_relevant_exclusive_rule(
        best_val_prob,
        best_thresholds,
        mlb.classes_,
        not_relevant_label=not_relevant_label
    )

    y_test_pred = apply_not_relevant_exclusive_rule(
        best_test_prob,
        best_thresholds,
        mlb.classes_,
        not_relevant_label=not_relevant_label
    )

    train_metrics = compute_metrics_with_fixed_predictions(Y_train, y_train_pred)
    val_metrics = compute_metrics_with_fixed_predictions(Y_val, y_val_pred)
    test_metrics = compute_metrics_with_fixed_predictions(Y_test, y_test_pred)

    search_rows.append({
        "model": "PerLabelEnsemble",
        "params": "LogReg + CalibratedLinearSVC + SGDLogistic",
        "val_precision": val_metrics["micro_precision"],
        "val_recall": val_metrics["micro_recall"],
        "val_f1": val_metrics["micro_f1"],
        "val_f2": val_metrics["micro_f2"],
        "val_macro_f1": val_metrics["macro_f1"],
        "val_macro_f2": val_metrics["macro_f2"]
    })

    search_results_df = pd.DataFrame(search_rows).sort_values(
        by=["val_f1", "val_f2", "val_precision", "val_recall"],
        ascending=False
    )

    print("\nSearch results:")
    display(search_results_df)

    per_label_precision, per_label_recall, per_label_f1, per_label_support = precision_recall_fscore_support(
        Y_test,
        y_test_pred,
        average=None,
        zero_division=0
    )

    per_label_f2 = fbeta_score(
        Y_test,
        y_test_pred,
        beta=2.0,
        average=None,
        zero_division=0
    )

    per_label_metrics = pd.DataFrame({
        "Classe": mlb.classes_,
        "Support": per_label_support,
        "Precision": per_label_precision,
        "Recall": per_label_recall,
        "F1": per_label_f1,
        "F2": per_label_f2,
        "Threshold": best_thresholds,
        "Weight_LogReg": best_weights[:, 0],
        "Weight_SVC": best_weights[:, 1],
        "Weight_SGD": best_weights[:, 2]
    })

    print("\nFINAL METRICS - LINEAR ENSEMBLE ULTRA")
    print(f"Best LogReg params: {best_logreg_params}")
    print(f"Best SVC params:    {best_svc_params}")
    print(f"Best SGD params:    {best_sgd_params}")
    print()
    print(f"Test micro F1:      {test_metrics['micro_f1']:.4f}")
    print(f"Test micro F2:      {test_metrics['micro_f2']:.4f}")
    print(f"Test macro F1:      {test_metrics['macro_f1']:.4f}")
    print(f"Test macro F2:      {test_metrics['macro_f2']:.4f}")
    print(f"Test precision:     {test_metrics['micro_precision']:.4f}")
    print(f"Test recall:        {test_metrics['micro_recall']:.4f}")
    print(f"Label accuracy:     {test_metrics['label_accuracy']:.4f}")
    print(f"Subset accuracy:    {test_metrics['subset_accuracy']:.4f}")
    print(f"Extraction weight:  {extraction_weight}")
    print(f"NOT_RELEVANT rule:  exclusive")

    print("\nPer-label metrics:")
    display(per_label_metrics)

    predictions_excel_df = None
    predictions_excel_path = None

    if save_predictions_excel:
        if predictions_excel_filename is None:
            predictions_excel_filename = f"ORM_PREDICTIONS_{combo_name}.xlsx"

        predictions_excel_df = build_prediction_excel_output(
            df_input=df_work,
            feature_columns=feature_columns,
            label_column=label_column,
            extraction_platform_col=extraction_platform_col,
            extraction_weight=extraction_weight,
            word_tfidf=word_tfidf,
            char_tfidf=char_tfidf,
            best_logreg=best_logreg,
            best_svc=best_svc,
            best_sgd=best_sgd,
            best_weights=best_weights,
            best_thresholds=best_thresholds,
            classes=mlb.classes_,
            not_relevant_label=not_relevant_label
        )

        predictions_excel_df.to_excel(predictions_excel_filename, index=False)
        predictions_excel_path = predictions_excel_filename

        print(f"\nFile Excel predizioni salvato: {predictions_excel_filename}")

    if save_bundle:
        bundle_filename = f"ORM_LINEAR_ENSEMBLE_ULTRA_bundle_{combo_name}.pkl"

        bundle = {
            "logreg_model": best_logreg,
            "svc_model": best_svc,
            "sgd_model": best_sgd,
            "word_tfidf": word_tfidf,
            "char_tfidf": char_tfidf,
            "mlb": mlb,
            "thresholds_per_label": best_thresholds,
            "weights_per_label": best_weights,
            "feature_columns": feature_columns,
            "features": feature_columns,
            "classes": mlb.classes_,
            "best_logreg_params": best_logreg_params,
            "best_svc_params": best_svc_params,
            "best_sgd_params": best_sgd_params,
            "label_column": label_column,
            "extraction_platform_col": extraction_platform_col,
            "extraction_weight": extraction_weight,
            "not_relevant_label": not_relevant_label,
            "not_relevant_rule": "exclusive",
            "predictions_excel_filename": predictions_excel_path
        }

        with open(bundle_filename, "wb") as file:
            pickle.dump(bundle, file)

        print(f"\nBundle Linear Ensemble Ultra salvato: {bundle_filename}")

    result = {
        "model_type": "LinearEnsembleUltra_WeightedExtraction_NotRelevantExclusive",
        "combo_name": combo_name,
        "features": feature_columns,
        "label_column": label_column,
        "extraction_platform_col": extraction_platform_col,
        "extraction_weight": extraction_weight,
        "not_relevant_label": not_relevant_label,

        "num_input_columns": len(feature_columns),
        "num_tfidf_features": X_train.shape[1],
        "num_classes": len(mlb.classes_),

        "best_logreg_params": best_logreg_params,
        "best_svc_params": best_svc_params,
        "best_sgd_params": best_sgd_params,
        "search_results": search_results_df,

        "train_precision": train_metrics["micro_precision"],
        "val_precision": val_metrics["micro_precision"],
        "test_precision": test_metrics["micro_precision"],

        "train_recall": train_metrics["micro_recall"],
        "val_recall": val_metrics["micro_recall"],
        "test_recall": test_metrics["micro_recall"],

        "train_f1": train_metrics["micro_f1"],
        "val_f1": val_metrics["micro_f1"],
        "test_f1": test_metrics["micro_f1"],

        "train_f2": train_metrics["micro_f2"],
        "val_f2": val_metrics["micro_f2"],
        "test_f2": test_metrics["micro_f2"],

        "test_accuracy": test_metrics["label_accuracy"],
        "test_subset_accuracy": test_metrics["subset_accuracy"],

        "test_macro_precision": test_metrics["macro_precision"],
        "test_macro_recall": test_metrics["macro_recall"],
        "test_macro_f1": test_metrics["macro_f1"],
        "test_macro_f2": test_metrics["macro_f2"],

        "thresholds_per_label": best_thresholds,
        "weights_per_label": best_weights,
        "classes": mlb.classes_,
        "per_label_metrics": per_label_metrics,

        "logreg_model": best_logreg,
        "svc_model": best_svc,
        "sgd_model": best_sgd,
        "word_tfidf": word_tfidf,
        "char_tfidf": char_tfidf,
        "mlb": mlb,

        "predictions_excel_filename": predictions_excel_path,
        "predictions_excel_df": predictions_excel_df
    }

    return result

Prediction

In [ ]:
def predict_with_trained_linear_ensemble_ultra(
    df_predict,
    trained_model,
    output_excel_filename="predizioni_finali.xlsx",
    label_column=None,
    save_excel=True,
    confidence_decimals=4
):
    """
    Esegue predizione usando il modello Linear Ensemble Ultra già addestrato.

    Parametri:
    - df_predict: dataset su cui fare predizione
    - trained_model: result restituito da trainTestCombLinearEnsembleUltra oppure path di un file .pkl
    - output_excel_filename: nome del file Excel finale
    - label_column: colonna true label, opzionale. Se presente, calcola TRUE_PLATFORM_NORMALIZED e PREDICTION_EXACT_MATCH
    - save_excel: se True salva il file Excel
    - confidence_decimals: numero decimali per le confidence

    Output:
    - DataFrame con predizioni, confidence e colonne SI/NO per ogni piattaforma
    """

    if isinstance(trained_model, str):
        with open(trained_model, "rb") as file:
            trained_model = pickle.load(file)

    df_output = df_predict.copy()

    feature_columns = trained_model.get("features", trained_model.get("feature_columns"))
    extraction_platform_col = trained_model.get("extraction_platform_col", "Piattaforma di Estrazione")
    extraction_weight = trained_model.get("extraction_weight", 6)
    not_relevant_label = trained_model.get("not_relevant_label", "NOT_RELEVANT")

    word_tfidf = trained_model["word_tfidf"]
    char_tfidf = trained_model["char_tfidf"]

    logreg_model = trained_model["logreg_model"]
    svc_model = trained_model["svc_model"]
    sgd_model = trained_model["sgd_model"]

    thresholds = np.array(trained_model["thresholds_per_label"])
    weights = np.array(trained_model["weights_per_label"])
    classes = np.array(trained_model["classes"])

    if label_column is None:
        label_column = trained_model.get("label_column", None)

    missing_columns = [col for col in feature_columns if col not in df_output.columns]

    if missing_columns:
        raise ValueError(f"Colonne mancanti nel dataset di predizione: {missing_columns}")

    X_features = df_output[feature_columns].fillna("").copy()

    prediction_text = build_weighted_prefixed_text(
        X_features,
        feature_columns,
        extraction_platform_col=extraction_platform_col,
        extraction_weight=extraction_weight
    )

    prediction_text_clean = [clean_text(text) for text in prediction_text]

    X_word = word_tfidf.transform(prediction_text_clean)
    X_char = char_tfidf.transform(prediction_text_clean)
    X_predict = hstack([X_word, X_char]).tocsr()

    logreg_prob = logreg_model.predict_proba(X_predict)
    svc_prob = svc_model.predict_proba(X_predict)
    sgd_prob = sgd_model.predict_proba(X_predict)

    probabilities = apply_per_label_ensemble(
        logreg_prob,
        svc_prob,
        sgd_prob,
        weights
    )

    y_pred = apply_not_relevant_exclusive_rule(
        probabilities,
        thresholds,
        classes,
        not_relevant_label=not_relevant_label
    )

    predicted_labels = [
        list(classes[np.where(row == 1)[0]])
        for row in y_pred
    ]

    df_output["PREDICTED_PLATFORM"] = [
        "; ".join(labels) if len(labels) > 0 else ""
        for labels in predicted_labels
    ]

    if label_column is not None and label_column in df_output.columns:
        true_label_lists = df_output[label_column].apply(
            lambda x: parse_multilabel_assignment_with_not_relevant_rule(
                x,
                not_relevant_label=not_relevant_label
            )
            if pd.notna(x) and str(x).strip() != ""
            else []
        )

        df_output["TRUE_PLATFORM_NORMALIZED"] = [
            "; ".join(labels) for labels in true_label_lists
        ]

        df_output["PREDICTION_EXACT_MATCH"] = [
            "SI" if set(true_labels) == set(pred_labels) else "NO"
            for true_labels, pred_labels in zip(true_label_lists, predicted_labels)
        ]

    for idx, class_name in enumerate(classes):
        safe_class_name = (
            str(class_name)
            .replace(" ", "_")
            .replace("/", "_")
            .replace("-", "_")
        )

        df_output[f"PRED_{safe_class_name}"] = np.where(
            y_pred[:, idx] == 1,
            "SI",
            "NO"
        )

        df_output[f"CONF_{safe_class_name}"] = np.round(
            probabilities[:, idx],
            confidence_decimals
        )

        df_output[f"THRESHOLD_{safe_class_name}"] = np.round(
            thresholds[idx],
            confidence_decimals
        )

    if save_excel:
        df_output.to_excel(output_excel_filename, index=False)
        print(f"File Excel predizioni salvato: {output_excel_filename}")

    return df_output

### Execution Model

#### Training

In [ ]:
feature_names_list = ['Piattaforma di Estrazione','Title (Translated)(English)','Abstract (Translated)(English)', 'Independent Claims (Translated)(English)', 'IPC', 'Application domain', 'Technology topic', 'Technical Problem', 'Technical Problem', 'Std. Technical Benefit Phrase']


linear_ensemble_result = trainTestCombLinearEnsembleUltra(
    df_work=df_combined_final ,
    feature_columns=feature_names_list,
    label_column="Piattaforma Assegnazione",
    extraction_platform_col="Piattaforma di Estrazione",
    extraction_weight=6,
    combo_name="WEIGHTED_EXTRACTION_ASSIGNMENT",
    save_predictions_excel=True,
    predictions_excel_filename="predizioni_finali_linear_ensemble.xlsx"
)

TRAIN/TEST LINEAR ENSEMBLE ULTRA - WEIGHTED_EXTRACTION_ASSIGNMENT

Numero classi: 9
Classi: ['EIWH' 'ESWH' 'GAHP' 'GASBOILER' 'GIWH' 'GSWH' 'HHP' 'HPWH'
 'NOT_RELEVANT']


,Classe,Numero di sample
6,HHP,9252
5,GSWH,2488
8,NOT_RELEVANT,2437
7,HPWH,2180
1,ESWH,2143
3,GASBOILER,1213
4,GIWH,605
0,EIWH,365
2,GAHP,194



Split:
Train size: 11717
Validation size: 2713
Test size: 3588

Fitting TF-IDF word + char...

Shape finali sparse:
X_train: (11717, 147055)
X_val:   (2713, 147055)
X_test:  (3588, 147055)

Search Logistic Regression...
Training LogReg C=0.5, class_weight=balanced
Training LogReg C=0.5, class_weight=None
Training LogReg C=1.0, class_weight=balanced
Training LogReg C=1.0, class_weight=None
Training LogReg C=2.0, class_weight=balanced
Training LogReg C=2.0, class_weight=None
Training LogReg C=4.0, class_weight=balanced
Training LogReg C=4.0, class_weight=None
Training LogReg C=8.0, class_weight=balanced
Training LogReg C=8.0, class_weight=None

Search Calibrated LinearSVC...
Training LinearSVC C=0.5, class_weight=balanced
Training LinearSVC C=0.5, class_weight=None
Training LinearSVC C=1.0, class_weight=balanced
Training LinearSVC C=1.0, class_weight=None
Training LinearSVC C=2.0, class_weight=balanced
Training LinearSVC C=2.0, class_weight=None

Search SGD Logistic...
Training SGD alph

,model,params,val_precision,val_recall,val_f1,val_f2,val_macro_f1,val_macro_f2
22,PerLabelEnsemble,LogReg + CalibratedLinearSVC + SGDLogistic,0.928978,0.885377,0.906654,0.893767,0.885199,0.864438
8,LogReg,"C=8.0, class_weight=balanced",0.924232,0.884100,0.903721,0.891845,0.881636,0.863953
14,CalibratedLinearSVC,"C=2.0, class_weight=balanced",0.922488,0.885377,0.903552,0.892558,0.877349,0.856657
12,CalibratedLinearSVC,"C=1.0, class_weight=balanced",0.925889,0.881545,0.903173,0.890071,0.875886,0.851078
13,CalibratedLinearSVC,"C=1.0, class_weight=None",0.919179,0.886015,0.902292,0.892455,0.876949,0.854883
15,CalibratedLinearSVC,"C=2.0, class_weight=None",0.918516,0.885377,0.901642,0.891812,0.877226,0.853408
16,SGDLogistic,"alpha=1e-05, class_weight=balanced",0.918713,0.884100,0.901074,0.890812,0.877948,0.864321
17,SGDLogistic,"alpha=1e-05, class_weight=None",0.913086,0.888889,0.900825,0.893625,0.875682,0.858126
6,LogReg,"C=4.0, class_weight=balanced",0.916089,0.885377,0.900471,0.891353,0.877479,0.859898
9,LogReg,"C=8.0, class_weight=None",0.913617,0.884738,0.898946,0.890367,0.870103,0.852205



FINAL METRICS - LINEAR ENSEMBLE ULTRA
Best LogReg params: {'C': 8.0, 'class_weight': 'balanced'}
Best SVC params:    {'C': 2.0, 'class_weight': 'balanced'}
Best SGD params:    {'alpha': 1e-05, 'class_weight': 'balanced'}

Test micro F1:      0.8911
Test micro F2:      0.8729
Test macro F1:      0.8608
Test macro F2:      0.8333
Test precision:     0.9233
Test recall:        0.8611
Label accuracy:     0.9728
Subset accuracy:    0.8381
Extraction weight:  6
NOT_RELEVANT rule:  exclusive

Per-label metrics:


,Classe,Support,Precision,Recall,F1,F2,Threshold,Weight_LogReg,Weight_SVC,Weight_SGD
0,EIWH,73,0.944444,0.698630,0.803150,0.736994,0.64,0.500000,0.500000,0.0
1,ESWH,429,0.921622,0.794872,0.853567,0.817354,0.60,0.500000,0.500000,0.0
2,GAHP,39,0.918919,0.871795,0.894737,0.880829,0.74,1.000000,0.000000,0.0
3,GASBOILER,243,0.938462,0.753086,0.835616,0.784062,0.72,0.200000,0.400000,0.4
4,GIWH,121,0.910714,0.842975,0.875536,0.855705,0.78,0.000000,0.000000,1.0
5,GSWH,498,0.952675,0.929719,0.941057,0.934221,0.58,0.000000,0.600000,0.4
6,HHP,1851,0.951381,0.930308,0.940727,0.934448,0.58,0.400000,0.600000,0.0
7,HPWH,436,0.895782,0.827982,0.860548,0.840708,0.50,0.333333,0.666667,0.0
8,NOT_RELEVANT,487,0.792541,0.698152,0.742358,0.715187,0.44,0.428571,0.571429,0.0



File Excel predizioni salvato: predizioni_finali_linear_ensemble.xlsx

Bundle Linear Ensemble Ultra salvato: ORM_LINEAR_ENSEMBLE_ULTRA_bundle_WEIGHTED_EXTRACTION_ASSIGNMENT.pkl


In [ ]:
# ============================================================
# SALVATAGGIO BEST MODEL / BEST PARAMETERS
# ============================================================

import pickle
import os

def save_linear_ensemble_ultra_bundle(
    trained_result,
    filename="best_linear_ensemble_ultra_bundle.pkl"
):
    bundle = {
        "model_type": trained_result["model_type"],
        "combo_name": trained_result["combo_name"],

        # Modelli addestrati
        "logreg_model": trained_result["logreg_model"],
        "svc_model": trained_result["svc_model"],
        "sgd_model": trained_result["sgd_model"],

        # Vettorizzatori TF-IDF
        "word_tfidf": trained_result["word_tfidf"],
        "char_tfidf": trained_result["char_tfidf"],

        # Parametri ottimali della fase di training
        "best_logreg_params": trained_result["best_logreg_params"],
        "best_svc_params": trained_result["best_svc_params"],
        "best_sgd_params": trained_result["best_sgd_params"],
        "thresholds_per_label": trained_result["thresholds_per_label"],
        "weights_per_label": trained_result["weights_per_label"],

        # Informazioni sulle classi
        "classes": trained_result["classes"],
        "mlb": trained_result["mlb"],

        # Colonne e preprocessing
        "features": trained_result["features"],
        "feature_columns": trained_result["features"],
        "label_column": trained_result["label_column"],
        "extraction_platform_col": trained_result["extraction_platform_col"],
        "extraction_weight": trained_result["extraction_weight"],
        "not_relevant_label": trained_result["not_relevant_label"],
        "not_relevant_rule": "exclusive",

        # Metriche finali
        "test_precision": trained_result["test_precision"],
        "test_recall": trained_result["test_recall"],
        "test_f1": trained_result["test_f1"],
        "test_f2": trained_result["test_f2"],
        "test_accuracy": trained_result["test_accuracy"],
        "test_subset_accuracy": trained_result["test_subset_accuracy"],
        "test_macro_precision": trained_result["test_macro_precision"],
        "test_macro_recall": trained_result["test_macro_recall"],
        "test_macro_f1": trained_result["test_macro_f1"],
        "test_macro_f2": trained_result["test_macro_f2"],
    }

    with open(filename, "wb") as file:
        pickle.dump(bundle, file)

    print(f"Bundle salvato correttamente: {filename}")
    print(f"Percorso completo: {os.path.abspath(filename)}")

    return filename

In [ ]:
bundle_path = save_linear_ensemble_ultra_bundle(
    trained_result=linear_ensemble_result,
    filename="best_linear_ensemble_ultra_bundle_TALM.pkl"
)

Bundle salvato correttamente: best_linear_ensemble_ultra_bundle_TALM.pkl
Percorso completo: /content/best_linear_ensemble_ultra_bundle_TALM.pkl


In [ ]:
from google.colab import files
files.download(bundle_path)

NameError: name 'bundle_path' is not defined

#### Predictions

In [ ]:
# ============================================================
# CARICAMENTO BEST MODEL / BEST PARAMETERS
# ============================================================

import pickle

with open("best_linear_ensemble_ultra_bundle_TALM.pkl", "rb") as file:
    loaded_linear_ensemble_bundle = pickle.load(file)

print("Bundle caricato correttamente")
print("Model type:", loaded_linear_ensemble_bundle["model_type"])
print("Classes:", loaded_linear_ensemble_bundle["classes"])
print("Features:", loaded_linear_ensemble_bundle["features"])

Bundle caricato correttamente
Model type: LinearEnsembleUltra_WeightedExtraction_NotRelevantExclusive
Classes: ['EIWH' 'ESWH' 'GAHP' 'GASBOILER' 'GIWH' 'GSWH' 'HHP' 'HPWH'
 'NOT_RELEVANT']
Features: ['Piattaforma di Estrazione', 'Title (Translated)(English)', 'Abstract (Translated)(English)', 'Independent Claims (Translated)(English)', 'IPC', 'Application domain', 'Technology topic', 'Technical Problem', 'Technical Problem', 'Std. Technical Benefit Phrase']


In [ ]:
predictions_df = predict_with_trained_linear_ensemble_ultra(
    df_predict=df_to_be_predicted,
    trained_model=loaded_linear_ensemble_bundle,
    output_excel_filename="predizioni_nuovi_brevetti.xlsx",
    save_excel=True
)

predictions_df.head()

File Excel predizioni salvato: predizioni_nuovi_brevetti.xlsx


,Piattaforma di Estrazione,Publication Number,Patent Type,Grouped Assignee,Standardized Original Assignee(Applicant),Title (Translated)(English),Abstract (Translated)(English),Independent Claims (Translated)(English),IPC,Application domain,...,THRESHOLD_GSWH,PRED_HHP,CONF_HHP,THRESHOLD_HHP,PRED_HPWH,CONF_HPWH,THRESHOLD_HPWH,PRED_NOT_RELEVANT,CONF_NOT_RELEVANT,THRESHOLD_NOT_RELEVANT
0,HHP,US20260002708A1,Applications,DAIKIN,DAIKIN INDUSTRIES LTD | DAIKIN EURO,Heat source apparatus and refrigeration cycle ...,A heat source apparatus includes a refrigerant...,1. A heat source apparatus comprising:_x000D_\...,F25B13/00,Compression machines with reversible cycle|Ref...,...,0.58,SI,0.9421,0.58,NO,0.2105,0.5,NO,0.1870,0.44
1,HHP / HPWH,US20260001382A1,Applications,MIDEA,MIDEA GROUP CO LTD,Recreational vehicle air conditioning system w...,A recreational vehicle air conditioning system...,1. A recreational vehicle air conditioning uni...,B60H1/00 | F25B49/02,Air-treating devices|Mechanical apparatus|Comp...,...,0.58,NO,0.4460,0.58,NO,0.1226,0.5,SI,0.5451,0.44
2,HPWH,CN121252034A,Applications,-,SHENZHEN WANRUN NEW ENERGY CO LTD,Falling film spiral tube integrated micro-pres...,The invention discloses a falling film spiral ...,1. A falling film spiral tube integrated micro...,F22G1/10 | F25B30/02 | F25B41/42 | F25B41/20 |...,Heat pumps|Mechanical apparatus|Fluid circulat...,...,0.58,NO,0.3665,0.58,NO,0.2922,0.5,SI,0.6852,0.44
3,ESWH / GIWH / GSWH / HPWH,CN121252192A,Applications,HAIER,QINGDAO HAIER WISDOM KITCHEN APPLIANCE CO LTD ...,One-way air supplement device,The invention discloses a one-way air suppleme...,"1. A unidirectional air supply device, charact...",F24F7/04 | F24F13/14 | F24F7/003,Lighting and heating apparatus|Air-flow contro...,...,0.58,NO,0.1513,0.58,NO,0.0116,0.5,NO,0.0852,0.44
4,ESWH / HPWH,CN121256371A,Applications,HAIER,QINGDAO HAIER TECH | QINGDAO HAIER INTELLIGENT...,Scene determination method and device based on...,The invention discloses a scene determination ...,1.A method for determining a scenario based on...,G06F18/22 | G06F18/26 | G06N3/045 | G05B15/02 ...,Computer control|Biological models|Total facto...,...,0.58,NO,0.2004,0.58,NO,0.0525,0.5,NO,0.4167,0.44


In [ ]:
print("Informazioni sul dataframe delle predizioni:")
predictions_df.info()

print("\nPrime 5 righe del dataframe delle predizioni:")
display(predictions_df.head())

print("\nDistribuzione delle piattaforme predette ('PREDICTED_PLATFORM'):")
print(predictions_df['PREDICTED_PLATFORM'].value_counts())

Informazioni sul dataframe delle predizioni:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3270 entries, 0 to 3269
Data columns (total 53 columns):
 #   Column                                     Non-Null Count  Dtype         
---  ------                                     --------------  -----         
 0   Piattaforma di Estrazione                  3270 non-null   object        
 1   Publication Number                         3270 non-null   object        
 2   Patent Type                                3270 non-null   object        
 3   Grouped Assignee                           3270 non-null   object        
 4   Standardized Original Assignee(Applicant)  3270 non-null   object        
 5   Title (Translated)(English)                3270 non-null   object        
 6   Abstract (Translated)(English)             3270 non-null   object        
 7   Independent Claims (Translated)(English)   3270 non-null   object        
 8   IPC                                        3270 non-n

,Piattaforma di Estrazione,Publication Number,Patent Type,Grouped Assignee,Standardized Original Assignee(Applicant),Title (Translated)(English),Abstract (Translated)(English),Independent Claims (Translated)(English),IPC,Application domain,...,THRESHOLD_GSWH,PRED_HHP,CONF_HHP,THRESHOLD_HHP,PRED_HPWH,CONF_HPWH,THRESHOLD_HPWH,PRED_NOT_RELEVANT,CONF_NOT_RELEVANT,THRESHOLD_NOT_RELEVANT
0,HHP,US20260002708A1,Applications,DAIKIN,DAIKIN INDUSTRIES LTD | DAIKIN EURO,Heat source apparatus and refrigeration cycle ...,A heat source apparatus includes a refrigerant...,1. A heat source apparatus comprising:_x000D_\...,F25B13/00,Compression machines with reversible cycle|Ref...,...,0.58,SI,0.9421,0.58,NO,0.2105,0.5,NO,0.1870,0.44
1,HHP / HPWH,US20260001382A1,Applications,MIDEA,MIDEA GROUP CO LTD,Recreational vehicle air conditioning system w...,A recreational vehicle air conditioning system...,1. A recreational vehicle air conditioning uni...,B60H1/00 | F25B49/02,Air-treating devices|Mechanical apparatus|Comp...,...,0.58,NO,0.4460,0.58,NO,0.1226,0.5,SI,0.5451,0.44
2,HPWH,CN121252034A,Applications,-,SHENZHEN WANRUN NEW ENERGY CO LTD,Falling film spiral tube integrated micro-pres...,The invention discloses a falling film spiral ...,1. A falling film spiral tube integrated micro...,F22G1/10 | F25B30/02 | F25B41/42 | F25B41/20 |...,Heat pumps|Mechanical apparatus|Fluid circulat...,...,0.58,NO,0.3665,0.58,NO,0.2922,0.5,SI,0.6852,0.44
3,ESWH / GIWH / GSWH / HPWH,CN121252192A,Applications,HAIER,QINGDAO HAIER WISDOM KITCHEN APPLIANCE CO LTD ...,One-way air supplement device,The invention discloses a one-way air suppleme...,"1. A unidirectional air supply device, charact...",F24F7/04 | F24F13/14 | F24F7/003,Lighting and heating apparatus|Air-flow contro...,...,0.58,NO,0.1513,0.58,NO,0.0116,0.5,NO,0.0852,0.44
4,ESWH / HPWH,CN121256371A,Applications,HAIER,QINGDAO HAIER TECH | QINGDAO HAIER INTELLIGENT...,Scene determination method and device based on...,The invention discloses a scene determination ...,1.A method for determining a scenario based on...,G06F18/22 | G06F18/26 | G06N3/045 | G05B15/02 ...,Computer control|Biological models|Total facto...,...,0.58,NO,0.2004,0.58,NO,0.0525,0.5,NO,0.4167,0.44



Distribuzione delle piattaforme predette ('PREDICTED_PLATFORM'):
PREDICTED_PLATFORM
HHP                            1747
NOT_RELEVANT                    430
                                338
GSWH                            194
ESWH                            179
HPWH                            144
GIWH; GSWH                       51
HHP; HPWH                        50
GASBOILER                        37
ESWH; HPWH                       21
GAHP                             21
GASBOILER; GSWH                  15
ESWH; GSWH                       10
GAHP; HHP                         5
GIWH                              5
EIWH                              3
ESWH; HHP                         3
GASBOILER; HHP                    3
GASBOILER; GIWH; GSWH             3
GSWH; HPWH                        3
GASBOILER; GIWH                   2
EIWH; ESWH                        2
GASBOILER; HPWH                   1
GAHP; GASBOILER                   1
ESWH; GASBOILER; GSWH; HPWH       1
ESWH; GASBOILER

In [ ]:
conf_columns = [col for col in predictions_df.columns if col.startswith('CONF_')]
print("Statistiche descrittive per le colonne di confidenza:")
display(predictions_df[conf_columns].describe())

print("\nPrime 5 righe delle colonne di confidenza:")
display(predictions_df[conf_columns].head())

Statistiche descrittive per le colonne di confidenza:


,CONF_EIWH,CONF_ESWH,CONF_GAHP,CONF_GASBOILER,CONF_GIWH,CONF_GSWH,CONF_HHP,CONF_HPWH,CONF_NOT_RELEVANT
count,3270.000000,3270.000000,3270.000000,3270.000000,3270.000000,3270.000000,3270.000000,3270.000000,3270.000000
mean,0.012529,0.101882,0.017218,0.067045,0.037643,0.107691,0.568444,0.105477,0.169419
std,0.047299,0.225150,0.095355,0.160184,0.152356,0.245588,0.409171,0.192269,0.233117
min,0.000100,0.000000,0.000000,0.000300,0.000000,0.000000,0.000300,0.000300,0.001200
25%,0.001400,0.002600,0.000500,0.003600,0.000000,0.003000,0.073175,0.008600,0.022200
50%,0.003000,0.009400,0.001100,0.009900,0.000300,0.009700,0.761900,0.024250,0.065050
75%,0.008000,0.052350,0.003600,0.039200,0.002000,0.044275,0.963875,0.087900,0.202475
max,0.807400,0.993400,0.995300,0.994200,0.999500,0.999100,1.000000,0.980600,0.987400



Prime 5 righe delle colonne di confidenza:


,CONF_EIWH,CONF_ESWH,CONF_GAHP,CONF_GASBOILER,CONF_GIWH,CONF_GSWH,CONF_HHP,CONF_HPWH,CONF_NOT_RELEVANT
0,0.0012,0.0072,0.2175,0.0051,0.0000,0.0000,0.9421,0.2105,0.1870
1,0.0039,0.0020,0.0065,0.0029,0.0001,0.0013,0.4460,0.1226,0.5451
2,0.0022,0.0020,0.0099,0.0088,0.0006,0.0014,0.3665,0.2922,0.6852
3,0.0041,0.0101,0.0010,0.1186,0.0008,0.2520,0.1513,0.0116,0.0852
4,0.0027,0.5956,0.0011,0.0170,0.0001,0.0194,0.2004,0.0525,0.4167


#### Post-Processing

In [ ]:
# ============================================================
# POST-PROCESSING RIGHE SENZA PREDIZIONE
# ============================================================

def postprocess_empty_predictions(
    predictions_df,
    classes=None,
    not_relevant_label="NOT_RELEVANT",
    uncertain_label="UNCERTAIN_REVIEW",
    min_absolute_confidence=0.35,
    min_threshold_ratio=0.80,
    confidence_decimals=4
):
    """
    Corregge in modo controllato le righe senza PREDICTED_PLATFORM.

    Logica:
    - se una riga non ha predizioni, cerca la classe con confidence più alta;
    - assegna quella classe solo se:
        1. confidence >= min_absolute_confidence
        oppure
        2. confidence >= threshold * min_threshold_ratio
    - se nessuna condizione è soddisfatta, assegna UNCERTAIN_REVIEW;
    - NOT_RELEVANT resta esclusiva.
    """

    df_out = predictions_df.copy()

    if classes is None:
        classes = [
            col.replace("CONF_", "")
            for col in df_out.columns
            if col.startswith("CONF_")
        ]

    classes = list(classes)

    conf_cols = [f"CONF_{cls}" for cls in classes]
    threshold_cols = [f"THRESHOLD_{cls}" for cls in classes]
    pred_cols = [f"PRED_{cls}" for cls in classes]

    missing_conf_cols = [col for col in conf_cols if col not in df_out.columns]
    missing_threshold_cols = [col for col in threshold_cols if col not in df_out.columns]
    missing_pred_cols = [col for col in pred_cols if col not in df_out.columns]

    if missing_conf_cols:
        raise ValueError(f"Colonne confidence mancanti: {missing_conf_cols}")

    if missing_threshold_cols:
        raise ValueError(f"Colonne threshold mancanti: {missing_threshold_cols}")

    if missing_pred_cols:
        raise ValueError(f"Colonne prediction mancanti: {missing_pred_cols}")

    if "PREDICTED_PLATFORM" not in df_out.columns:
        raise ValueError("Colonna mancante: PREDICTED_PLATFORM")

    empty_mask = (
        df_out["PREDICTED_PLATFORM"].isna()
        | (df_out["PREDICTED_PLATFORM"].astype(str).str.strip() == "")
    )

    df_out["POSTPROCESSING_ACTION"] = "UNCHANGED"
    df_out["TOP_1_PLATFORM"] = ""
    df_out["TOP_1_CONFIDENCE"] = np.nan
    df_out["TOP_1_THRESHOLD"] = np.nan
    df_out["TOP_1_THRESHOLD_RATIO"] = np.nan

    for idx in df_out.index[empty_mask]:
        conf_values = df_out.loc[idx, conf_cols].astype(float).values
        threshold_values = df_out.loc[idx, threshold_cols].astype(float).values

        best_idx = int(np.argmax(conf_values))

        best_class = classes[best_idx]
        best_conf = float(conf_values[best_idx])
        best_threshold = float(threshold_values[best_idx])

        threshold_ratio = (
            best_conf / best_threshold
            if best_threshold > 0
            else 0
        )

        df_out.at[idx, "TOP_1_PLATFORM"] = best_class
        df_out.at[idx, "TOP_1_CONFIDENCE"] = round(best_conf, confidence_decimals)
        df_out.at[idx, "TOP_1_THRESHOLD"] = round(best_threshold, confidence_decimals)
        df_out.at[idx, "TOP_1_THRESHOLD_RATIO"] = round(threshold_ratio, confidence_decimals)

        should_assign = (
            best_conf >= min_absolute_confidence
            or threshold_ratio >= min_threshold_ratio
        )

        if should_assign:
            for pred_col in pred_cols:
                df_out.at[idx, pred_col] = "NO"

            df_out.at[idx, f"PRED_{best_class}"] = "SI"
            df_out.at[idx, "PREDICTED_PLATFORM"] = best_class
            df_out.at[idx, "POSTPROCESSING_ACTION"] = "ASSIGNED_TOP_1"

            if best_class == not_relevant_label:
                for cls in classes:
                    pred_col = f"PRED_{cls}"
                    df_out.at[idx, pred_col] = "SI" if cls == not_relevant_label else "NO"

                df_out.at[idx, "PREDICTED_PLATFORM"] = not_relevant_label
                df_out.at[idx, "POSTPROCESSING_ACTION"] = "ASSIGNED_NOT_RELEVANT_EXCLUSIVE"

        else:
            for pred_col in pred_cols:
                df_out.at[idx, pred_col] = "NO"

            df_out.at[idx, "PREDICTED_PLATFORM"] = uncertain_label
            df_out.at[idx, "POSTPROCESSING_ACTION"] = "UNCERTAIN_REVIEW"

    print("Post-processing completato")
    print("Righe inizialmente senza predizione:", int(empty_mask.sum()))
    print()
    print("Distribuzione azioni:")
    display(df_out["POSTPROCESSING_ACTION"].value_counts())

    print()
    print("Distribuzione finale PREDICTED_PLATFORM:")
    display(df_out["PREDICTED_PLATFORM"].value_counts().head(30))

    return df_out

In [ ]:
predictions_df_fixed = postprocess_empty_predictions(
    predictions_df,
    min_absolute_confidence=0.35,
    min_threshold_ratio=0.80
)

predictions_df_fixed.head()

Post-processing completato
Righe inizialmente senza predizione: 338

Distribuzione azioni:


,count
POSTPROCESSING_ACTION,
UNCHANGED,2932
ASSIGNED_TOP_1,224
UNCERTAIN_REVIEW,88
ASSIGNED_NOT_RELEVANT_EXCLUSIVE,26



Distribuzione finale PREDICTED_PLATFORM:


,count
PREDICTED_PLATFORM,
HHP,1833
NOT_RELEVANT,456
ESWH,223
GSWH,218
HPWH,164
UNCERTAIN_REVIEW,88
GASBOILER,76
GIWH; GSWH,51
HHP; HPWH,50


,Piattaforma di Estrazione,Publication Number,Patent Type,Grouped Assignee,Standardized Original Assignee(Applicant),Title (Translated)(English),Abstract (Translated)(English),Independent Claims (Translated)(English),IPC,Application domain,...,CONF_HPWH,THRESHOLD_HPWH,PRED_NOT_RELEVANT,CONF_NOT_RELEVANT,THRESHOLD_NOT_RELEVANT,POSTPROCESSING_ACTION,TOP_1_PLATFORM,TOP_1_CONFIDENCE,TOP_1_THRESHOLD,TOP_1_THRESHOLD_RATIO
0,HHP,US20260002708A1,Applications,DAIKIN,DAIKIN INDUSTRIES LTD | DAIKIN EURO,Heat source apparatus and refrigeration cycle ...,A heat source apparatus includes a refrigerant...,1. A heat source apparatus comprising:_x000D_\...,F25B13/00,Compression machines with reversible cycle|Ref...,...,0.2105,0.5,NO,0.1870,0.44,UNCHANGED,,NaN,NaN,NaN
1,HHP / HPWH,US20260001382A1,Applications,MIDEA,MIDEA GROUP CO LTD,Recreational vehicle air conditioning system w...,A recreational vehicle air conditioning system...,1. A recreational vehicle air conditioning uni...,B60H1/00 | F25B49/02,Air-treating devices|Mechanical apparatus|Comp...,...,0.1226,0.5,SI,0.5451,0.44,UNCHANGED,,NaN,NaN,NaN
2,HPWH,CN121252034A,Applications,-,SHENZHEN WANRUN NEW ENERGY CO LTD,Falling film spiral tube integrated micro-pres...,The invention discloses a falling film spiral ...,1. A falling film spiral tube integrated micro...,F22G1/10 | F25B30/02 | F25B41/42 | F25B41/20 |...,Heat pumps|Mechanical apparatus|Fluid circulat...,...,0.2922,0.5,SI,0.6852,0.44,UNCHANGED,,NaN,NaN,NaN
3,ESWH / GIWH / GSWH / HPWH,CN121252192A,Applications,HAIER,QINGDAO HAIER WISDOM KITCHEN APPLIANCE CO LTD ...,One-way air supplement device,The invention discloses a one-way air suppleme...,"1. A unidirectional air supply device, charact...",F24F7/04 | F24F13/14 | F24F7/003,Lighting and heating apparatus|Air-flow contro...,...,0.0116,0.5,NO,0.0852,0.44,UNCERTAIN_REVIEW,GSWH,0.2520,0.58,0.4345
4,ESWH / HPWH,CN121256371A,Applications,HAIER,QINGDAO HAIER TECH | QINGDAO HAIER INTELLIGENT...,Scene determination method and device based on...,The invention discloses a scene determination ...,1.A method for determining a scenario based on...,G06F18/22 | G06F18/26 | G06N3/045 | G05B15/02 ...,Computer control|Biological models|Total facto...,...,0.0525,0.5,NO,0.4167,0.44,ASSIGNED_TOP_1,ESWH,0.5956,0.60,0.9927


In [ ]:
predictions_df_fixed.to_excel(
    "predizioni_nuovi_brevetti_postprocessed.xlsx",
    index=False
)

print("File salvato: predizioni_nuovi_brevetti_postprocessed.xlsx")

File salvato: predizioni_nuovi_brevetti_postprocessed.xlsx


## Legenda label assegnate

**HHP**  
Brevetti relativi a heat pump per riscaldamento ambienti.

**HPWH**  
Brevetti relativi a heat pump water heater, quindi pompe di calore per acqua calda sanitaria.

**GSWH**  
Brevetti relativi a gas storage water heater, cioè scaldacqua a gas con accumulo.

**GIWH**  
Brevetti relativi a gas instantaneous water heater, cioè scaldacqua a gas istantanei.

**ESWH**  
Brevetti relativi a electric storage water heater, cioè scaldacqua elettrici con accumulo.

**EIWH**  
Brevetti relativi a electric instantaneous water heater, cioè scaldacqua elettrici istantanei.

**GAHP**  
Brevetti relativi a gas absorption heat pump, cioè pompe di calore ad assorbimento a gas.

**GASBOILER**  
Brevetti relativi a caldaie a gas.

**NOT_RELEVANT**  
Brevetti non rilevanti per le piattaforme considerate. È una label esclusiva.

**UNCERTAIN_REVIEW**  
Casi in cui il modello non ha sufficiente confidenza. Richiedono revisione manuale.

## Interpretazione dei risultati Training Phase

### Come leggere le metriche del modello

Le metriche servono a capire quanto bene il modello predice le piattaforme assegnate ai brevetti. Poiché il problema è **multi-label**, ogni brevetto può avere una o più piattaforme. Quindi il modello non sceglie una sola classe, ma decide per ogni piattaforma se assegnarla oppure no.

La **precision** misura quanto sono affidabili le predizioni positive del modello. Se la precision è alta, significa che quando il modello assegna una piattaforma, di solito ha ragione. Una precision bassa indica invece che il modello produce troppi falsi positivi, cioè assegna piattaforme che non erano corrette.

La **recall** misura quante piattaforme corrette il modello riesce a recuperare. Se la recall è alta, il modello trova gran parte delle label vere. Se è bassa, significa che perde molte piattaforme corrette, cioè produce falsi negativi.

L’**F1-score** combina precision e recall in un unico valore, dando lo stesso peso a entrambe. È utile per valutare il bilanciamento generale del modello: un F1 alto indica che il modello è contemporaneamente preciso e capace di recuperare molte label vere.

L’**F2-score** è simile all’F1, ma dà più importanza alla recall. È utile quando si vuole penalizzare maggiormente il fatto di perdere una piattaforma corretta. In questo progetto è una metrica importante perché, in fase di supporto all’assegnazione, può essere preferibile mostrare una piattaforma potenzialmente corretta piuttosto che non segnalarla affatto.

La **label accuracy** misura quante decisioni singole sono corrette considerando ogni coppia brevetto-piattaforma come un sì/no. Ad esempio, per ogni brevetto il modello decide se assegnare `HHP`, se assegnare `GSWH`, se assegnare `HPWH`, e così via. Una label accuracy alta indica che la maggior parte di queste decisioni binarie è corretta, ma va interpretata insieme a precision e recall perché nei problemi multi-label molte decisioni sono negative.

La **subset accuracy** è una metrica più severa: considera corretta una predizione solo se il modello indovina esattamente tutto l’insieme di piattaforme del brevetto. Se manca anche una sola piattaforma, oppure se ne viene aggiunta una sbagliata, la predizione viene considerata errata. Per questo motivo, nel multi-label è normale che la subset accuracy sia più bassa della label accuracy.

Le metriche **micro** aggregano gli errori e le predizioni corrette su tutte le classi prima di calcolare precision, recall, F1 o F2. Di conseguenza, danno più peso alle classi più frequenti e descrivono bene la performance globale del modello sul volume totale dei dati.

Le metriche **macro**, invece, calcolano la metrica separatamente per ogni classe e poi fanno la media semplice. Questo significa che tutte le classi pesano allo stesso modo, anche quelle con pochi esempi. Sono quindi fondamentali per capire se il modello funziona bene anche sulle piattaforme meno rappresentate, come `GAHP`, `EIWH` o `GIWH`.

Le **per-label metrics** mostrano precision, recall, F1 e F2 per ogni singola piattaforma. Sono utili per capire quali classi il modello riconosce meglio e quali invece sono più problematiche. Una classe con precision alta e recall bassa viene predetta con prudenza: quando viene assegnata è quasi sempre corretta, ma alcuni casi veri vengono persi. Una classe con recall alta e precision più bassa viene recuperata spesso, ma con più falsi positivi.

Il **threshold** è la soglia di confidenza sopra la quale il modello assegna una piattaforma. Se la confidenza di una classe supera la sua soglia, la piattaforma viene predetta come presente. Soglie più alte tendono ad aumentare la precision ma possono ridurre la recall; soglie più basse aumentano la recall ma possono generare più falsi positivi.

La **confidence** indica quanto il modello ritiene probabile che una certa piattaforma sia corretta per un brevetto. Nel file Excel finale, per ogni piattaforma ci sono colonne come `CONF_HHP`, `CONF_GSWH` o `CONF_NOT_RELEVANT`, che mostrano la confidenza del modello per quella specifica classe.

Le colonne `PRED_<classe>` indicano se il modello ha assegnato o meno quella piattaforma. Ad esempio, `PRED_HHP = SI` significa che il modello ha assegnato `HHP`, mentre `PRED_HHP = NO` significa che non l’ha assegnata.

La colonna `PREDICTED_PLATFORM` contiene l’insieme finale di piattaforme predette dal modello per ciascun brevetto. La colonna `TRUE_PLATFORM_NORMALIZED` contiene invece le piattaforme vere, normalizzate e pulite. La colonna `PREDICTION_EXACT_MATCH` indica se la predizione del modello coincide esattamente con le label vere del brevetto.

Infine, la label `NOT_RELEVANT` viene gestita con una regola esclusiva: se il modello predice `NOT_RELEVANT`, allora non può predire contemporaneamente altre piattaforme. Questa regola è coerente con il significato della label, perché un brevetto non rilevante non dovrebbe essere assegnato anche a una piattaforma specifica.

### Linear Ensemble Ultra con peso sulla 'Piattaforma di Estrazione' Training Size = 70%, Test Size = 30% and Validation Size= 15% of Training Size



Il modello addestrato è una versione **Linear Ensemble Ultra** che combina tre classificatori lineari: **Logistic Regression**, **Calibrated LinearSVC** e **SGD Logistic**. La pipeline utilizza una rappresentazione testuale basata su **TF-IDF word + char n-grams** e assegna maggiore peso alla colonna **Piattaforma di Estrazione**, ripetendola nel testo di input con `extraction_weight = 6`. Inoltre, è stata applicata la regola secondo cui la label **NOT_RELEVANT** deve essere esclusiva: se viene predetta, non può comparire insieme ad altre label.

Il dataset contiene **9 classi**: `EIWH`, `ESWH`, `GAHP`, `GASBOILER`, `GIWH`, `GSWH`, `HHP`, `HPWH` e `NOT_RELEVANT`. La distribuzione è sbilanciata: la classe più frequente è **HHP** con 9252 campioni, seguita da **GSWH** con 2488 e **NOT_RELEVANT** con 2437. Le classi più rare sono invece **GAHP** con 194 campioni, **EIWH** con 365 e **GIWH** con 605. Questo sbilanciamento rende importanti le metriche macro, perché valutano meglio la capacità del modello di funzionare anche sulle classi minoritarie.

Lo split finale è composto da **9912 esempi di training**, **2723 di validation** e **5383 di test**. Dopo la vettorizzazione, il modello lavora su una matrice sparsa molto ampia, con **146211 feature TF-IDF**, ottenute combinando n-grammi di parole e caratteri. Questo permette al modello di catturare sia termini tecnici completi sia pattern parziali utili nei testi brevettuali.

Durante la ricerca dei modelli, il miglior risultato in validation è ottenuto dal **PerLabelEnsemble**, cioè dalla combinazione pesata per singola label di LogReg, LinearSVC calibrato e SGD Logistic. In validation raggiunge **precision = 0.914**, **recall = 0.876**, **F1 = 0.895** e **F2 = 0.883**. Anche le metriche macro sono alte (**macro F1 = 0.871**, **macro F2 = 0.848**), segnale positivo perché indica che il modello non sta performando bene solo sulle classi più frequenti.

Sul test set il modello ottiene risultati molto solidi: **micro F1 = 0.8894**, **micro F2 = 0.8801**, **macro F1 = 0.8615**, **macro F2 = 0.8460**, **precision = 0.9054**, **recall = 0.8740**, **label accuracy = 0.9719** e **subset accuracy = 0.8415**. La precision superiore alla recall indica che il modello è leggermente conservativo: quando assegna una piattaforma tende a essere molto affidabile, anche se può ancora perdere alcune label corrette.

Le metriche per classe sono generalmente buone. Le classi **GSWH** e **HHP** risultano molto ben riconosciute, con F1 rispettivamente circa **0.940** e **0.940**. Anche **GAHP**, pur essendo una classe rara, ottiene un F1 elevato (**0.879**), dato particolarmente positivo. **EIWH** ha precision molto alta (**0.957**) ma recall più bassa (**0.809**), quindi il modello è prudente su questa classe. **NOT_RELEVANT** ottiene F1 circa **0.757**, valore discreto considerando che viene gestita con una regola esclusiva più rigida.

Nel complesso, questo modello rappresenta un miglioramento importante: combina alta precision, buona recall e soprattutto metriche macro molto forti. Il fatto che `macro F1` sia vicino al `micro F1` indica che il modello sta lavorando bene anche sulle classi meno rappresentate, non solo sulle classi dominanti. La scelta di dare più peso alla **Piattaforma di Estrazione** sembra quindi utile per migliorare la predizione della **Piattaforma di Assegnazione**, mantenendo comunque un processo multilabel coerente e controllato.

### Linear Ensemble Ultra con peso sulla 'Piattaforma di Estrazione' Training Size = 80%, Test Size = 20% and Validation Size= 15% of Training Size.  



Il modello addestrato è una versione **Linear Ensemble Ultra** che combina tre classificatori lineari: **Logistic Regression**, **Calibrated LinearSVC** e **SGD Logistic**. La rappresentazione testuale è basata su **TF-IDF word + char n-grams**, con un peso maggiore assegnato alla colonna **Piattaforma di Estrazione** tramite `extraction_weight = 6`. Inoltre, è attiva la regola di esclusività per **NOT_RELEVANT**: se questa label viene predetta, non può essere assegnata insieme ad altre piattaforme.

Il dataset contiene **9 classi**: `EIWH`, `ESWH`, `GAHP`, `GASBOILER`, `GIWH`, `GSWH`, `HHP`, `HPWH` e `NOT_RELEVANT`. La distribuzione è sbilanciata: **HHP** è la classe dominante con **9252 sample**, seguita da **GSWH** con **2488**, **NOT_RELEVANT** con **2437**, **HPWH** con **2180** ed **ESWH** con **2143**. Le classi meno rappresentate sono **GAHP** con **194 sample**, **EIWH** con **365** e **GIWH** con **605**. Per questo motivo, oltre alle metriche micro, sono particolarmente importanti anche le metriche macro, che indicano quanto il modello sia stabile sulle classi minoritarie.

Lo split utilizzato è pari a **11717 esempi di training**, **2713 di validation** e **3588 di test**. Dopo la vettorizzazione, la matrice TF-IDF contiene **157347 feature**, ottenute combinando n-grammi di parole e caratteri. Questa rappresentazione consente al modello di catturare sia termini tecnici completi sia pattern testuali più brevi, utili nei testi brevettuali.

Durante la fase di ricerca, il miglior modello in validation è il **PerLabelEnsemble**, cioè la combinazione pesata per singola classe di LogReg, LinearSVC calibrato e SGD Logistic. In validation ottiene **precision = 0.926**, **recall = 0.897**, **F1 = 0.911** e **F2 = 0.903**. Anche le metriche macro sono elevate, con **macro F1 = 0.893** e **macro F2 = 0.874**, indicando che il modello non funziona bene solo sulle classi frequenti, ma mantiene buone prestazioni anche sulle classi meno rappresentate.

Sul test set il modello conferma prestazioni molto solide: **micro F1 = 0.898**, **micro F2 = 0.886**, **macro F1 = 0.871** e **macro F2 = 0.847**. La **precision = 0.918** è superiore alla **recall = 0.878**, quindi il modello tende a essere leggermente conservativo: quando assegna una piattaforma, nella maggior parte dei casi la predizione è corretta, anche se può ancora perdere alcune label vere. La **label accuracy = 0.974** indica un’elevata accuratezza a livello di singola label, mentre la **subset accuracy = 0.849** indica che circa l’85% dei brevetti ha l’intero insieme di label predetto correttamente.

A livello di singola classe, le performance sono complessivamente alte. **HHP** e **GSWH** sono le classi migliori, con F1 rispettivamente circa **0.947** e **0.937**. Anche **GIWH** e **GAHP**, pur essendo meno frequenti, ottengono risultati molto buoni, con F1 rispettivamente circa **0.921** e **0.907**. **EIWH** mostra precision molto alta (**0.945**) ma recall più bassa (**0.712**), quindi il modello è prudente su questa classe. **NOT_RELEVANT** ottiene F1 circa **0.761**, risultato discreto considerando che viene gestita con una regola più rigida di esclusività.

Nel complesso, questi risultati indicano che la nuova configurazione è molto efficace. L’aumento del peso della **Piattaforma di Estrazione** sembra aiutare il modello a predire meglio la **Piattaforma di Assegnazione**, mantenendo buone performance sia sulle classi principali sia su quelle minoritarie. Il modello è particolarmente forte in precisione e mostra metriche macro elevate, quindi rappresenta una pipeline robusta e più bilanciata rispetto alle versioni precedenti.

11:42

### Confronto tra split 70/30 e split 80/20 - Linear Ensemble Ultra pesato

Il confronto riguarda lo stesso modello, **Linear Ensemble Ultra con peso sulla Piattaforma di Estrazione**, valutato con due diversi split del dataset: una configurazione con **training size = 70% e test size = 30%**, e una configurazione con **training size = 80% e test size = 20%**. In entrambi i casi è stata mantenuta la validation pari al **15% del training set** ed è stata applicata la regola esclusiva per **NOT_RELEVANT**.

Con lo split **70/30**, il modello è stato addestrato su **9912 esempi**, validato su **2723** e testato su **5383**. Le performance finali sono molto solide: **precision = 0.905**, **recall = 0.874**, **micro F1 = 0.889**, **micro F2 = 0.880**, **macro F1 = 0.862**, **macro F2 = 0.846**, **label accuracy = 0.972** e **subset accuracy = 0.842**. Questo risultato mostra già un buon equilibrio tra precisione e capacità di recuperare le label corrette.

Con lo split **80/20**, il modello ha più dati a disposizione per l’addestramento: **11717 esempi di training**, **2713 di validation** e **3588 di test**. Le performance migliorano ulteriormente: **precision = 0.918**, **recall = 0.878**, **micro F1 = 0.898**, **micro F2 = 0.886**, **macro F1 = 0.871**, **macro F2 = 0.847**, **label accuracy = 0.974** e **subset accuracy = 0.849**.

Il miglioramento più evidente riguarda la **precision**, che passa da **0.905** a **0.918**. Questo indica che, con più dati di training, il modello diventa più affidabile quando assegna una piattaforma: diminuisce quindi il rischio di falsi positivi. Anche la **recall** aumenta leggermente, da **0.874** a **0.878**, quindi il modello non diventa solo più prudente, ma riesce anche a recuperare un numero leggermente maggiore di label corrette.

Anche le metriche aggregate migliorano: il **micro F1** passa da **0.889** a **0.898**, mentre il **micro F2** passa da **0.880** a **0.886**. Questo conferma un miglior bilanciamento complessivo tra precision e recall. Le metriche macro restano molto alte in entrambe le configurazioni: **macro F1** passa da **0.862** a **0.871**, mentre **macro F2** rimane sostanzialmente stabile, da **0.846** a **0.847**. Questo suggerisce che l’aumento del training set migliora soprattutto la stabilità generale del modello, senza penalizzare le classi minoritarie.

Anche la qualità della predizione multi-label completa migliora: la **subset accuracy** passa da **0.842** a **0.849**. Questo è un dato importante perché misura la percentuale di brevetti per cui l’intero insieme di label viene predetto correttamente. Il miglioramento non è enorme, ma è coerente con l’aumento delle altre metriche.

Nel complesso, lo split **80/20** risulta preferibile: fornisce più dati al modello, migliora precision, F1, F2, macro F1 e subset accuracy, mantenendo stabile la capacità di generalizzazione. Lo split **70/30** rimane comunque utile come valutazione più severa, perché testa il modello su un insieme più grande. Tuttavia, in termini di performance finali, la configurazione **80/20** è la più efficace tra le due.

## Confusion Matrix of the TAKE A LOOK MODEL Training Phase

In [ ]:
mlb = linear_ensemble_ultra_weighted_result['mlb']
feature_columns = linear_ensemble_ultra_weighted_result['features']
label_column = linear_ensemble_ultra_weighted_result['label_column']
extraction_platform_col = linear_ensemble_ultra_weighted_result['extraction_platform_col']
extraction_weight = linear_ensemble_ultra_weighted_result['extraction_weight']
not_relevant_label = linear_ensemble_ultra_weighted_result['not_relevant_label']
word_tfidf = linear_ensemble_ultra_weighted_result['word_tfidf']
char_tfidf = linear_ensemble_ultra_weighted_result['char_tfidf']
logreg_model = linear_ensemble_ultra_weighted_result['logreg_model']
svc_model = linear_ensemble_ultra_weighted_result['svc_model']
sgd_model = linear_ensemble_ultra_weighted_result['sgd_model']
best_weights = linear_ensemble_ultra_weighted_result['weights_per_label']
best_thresholds_per_label = linear_ensemble_ultra_weighted_result['thresholds_per_label']

# Recreate x and y_multi from df_combined_final
df_local = df_combined_final.copy()
df_local = df_local.dropna(subset=[label_column]).copy()
df_local = df_local[df_local[label_column].astype(str).str.strip() != ""].copy()
df_local["_LABEL_LIST"] = df_local[label_column].apply(
    lambda x: parse_multilabel_assignment_with_not_relevant_rule(
        x,
        not_relevant_label=not_relevant_label
    )
)
df_local["_LABEL_STR"] = df_local["_LABEL_LIST"].apply(lambda labels: " ".join(labels))

X_features = df_local[feature_columns].fillna("").copy()
x = build_weighted_prefixed_text(
    X_features,
    feature_columns,
    extraction_platform_col=extraction_platform_col,
    extraction_weight=extraction_weight
)
y_split = df_local["_LABEL_LIST"]
y_multi = mlb.fit_transform(y_split)

# Perform the split to get the test set
_, _, X_test_text, _, _, Y_test = multilabel_train_val_test_split(
    x,
    y_multi,
    test_size=0.20,
    val_size=0.15,
    random_state=42
)

# Clean and transform X_test_text
X_test_text_clean = [clean_text(text) for text in X_test_text]
X_test_word = word_tfidf.transform(X_test_text_clean)
X_test_char = char_tfidf.transform(X_test_text_clean)
X_test = hstack([X_test_word, X_test_char]).tocsr()

# Get probabilities from individual models
logreg_test_prob = logreg_model.predict_proba(X_test)
svc_test_prob = svc_model.predict_proba(X_test)
sgd_test_prob = sgd_model.predict_proba(X_test)

# Apply ensemble weights
best_test_prob = apply_per_label_ensemble(
    logreg_test_prob,
    svc_test_prob,
    sgd_test_prob,
    best_weights
)

# Apply the NOT_RELEVANT exclusive rule
y_test_pred_final = apply_not_relevant_exclusive_rule(
    best_test_prob,
    best_thresholds_per_label,
    mlb.classes_,
    not_relevant_label=not_relevant_label
)

print("\n--- Multi-label Confusion Matrix ---")
conf_matrix_multilabel_with_thresholds(
    Y_test,
    best_test_prob,
    best_thresholds_per_label
)

print("\n--- Per-label Confusion Matrix ---")
conf_matrix_per_label_with_thresholds(
    Y_test,
    best_test_prob,
    best_thresholds_per_label,
    class_names=mlb.classes_
)


--- Multi-label Confusion Matrix ---
CONFUSION MATRIX MULTI-LABEL:

                         True 1       True 0

Predicted 1              3704 TP       362 FP
                         -------------------
Predicted 0              473 FN       27753 TN

False Negatives = 1.5 %
False Positives = 1.1 %

--- Per-label Confusion Matrix ---
EIWH
Threshold: 0.68
TP: 53, TN: 3512, FP: 3, FN: 20
Support:   73
Precision: 0.9464
Recall:    0.7260
F1-score:  0.8217
----------------------------------------
ESWH
Threshold: 0.62
TP: 345, TN: 3132, FP: 27, FN: 84
Support:   429
Precision: 0.9274
Recall:    0.8042
F1-score:  0.8614
----------------------------------------
GAHP
Threshold: 0.72
TP: 35, TN: 3547, FP: 2, FN: 4
Support:   39
Precision: 0.9459
Recall:    0.8974
F1-score:  0.9211
----------------------------------------
GASBOILER
Threshold: 0.56
TP: 193, TN: 3319, FP: 26, FN: 50
Support:   243
Precision: 0.8813
Recall:    0.7942
F1-score:  0.8355
----------------------------------------
GIWH

### Interpretazione della confusion matrix multi-label

La confusion matrix multi-label riassume il comportamento complessivo del modello considerando tutte le label come decisioni binarie indipendenti. Il modello ottiene **3704 True Positives**, cioè label positive correttamente riconosciute, e **27753 True Negatives**, cioè label correttamente non assegnate. Gli errori sono relativamente contenuti: **473 False Negatives** e **362 False Positives**. In percentuale, i falsi negativi sono circa **1.5%**, mentre i falsi positivi circa **1.1%**. Questo indica che il modello è molto stabile: assegna poche label errate e perde una quota ridotta di label realmente corrette.

La presenza di più **False Negatives** rispetto ai **False Positives** suggerisce che il modello è leggermente conservativo: tende più spesso a non assegnare una label vera piuttosto che ad assegnarne una sbagliata. Questo comportamento è coerente con le metriche finali viste in precedenza, dove la precision è superiore alla recall. In un contesto di classificazione brevettuale, questo può essere positivo se si vuole privilegiare l’affidabilità delle label assegnate, ma può richiedere attenzione se l’obiettivo è massimizzare il recupero di tutte le piattaforme possibili.

A livello di singola classe, le performance sono generalmente buone. **HHP** è la classe più forte, con **TP = 1787**, **FP = 135**, **FN = 64**, **precision = 0.930**, **recall = 0.965** e **F1 = 0.947**. Questo indica che il modello riconosce molto bene questa piattaforma e perde pochi casi veri. Anche **GSWH** mostra ottime performance, con **precision = 0.926**, **recall = 0.956** e **F1 = 0.941**, quindi il modello è molto efficace sia nel riconoscere i casi positivi sia nel limitare gli errori.

Le classi **GAHP** e **GIWH**, pur avendo support più basso, ottengono risultati molto positivi. **GAHP** ha **precision = 0.946**, **recall = 0.897** e **F1 = 0.921**, con solo **2 falsi positivi** e **4 falsi negativi**. **GIWH** mostra una precision molto alta (**0.981**) e una buona recall (**0.868**), con **F1 = 0.921**. Questo è un segnale importante perché indica che il modello riesce a gestire bene anche classi meno frequenti.

Le classi **ESWH**, **GASBOILER** e **HPWH** hanno performance buone ma con margini di miglioramento soprattutto sulla recall. **ESWH** ottiene **F1 = 0.861**, con **84 falsi negativi**, quindi alcuni brevetti realmente ESWH non vengono riconosciuti. **GASBOILER** ha **F1 = 0.836**, con **50 falsi negativi**, mentre **HPWH** ha **F1 = 0.856**, ma presenta **77 falsi negativi**. In questi casi il modello è abbastanza preciso, ma tende a perdere una parte dei positivi reali.

La classe **EIWH** ha una precision molto alta (**0.946**) ma una recall più bassa (**0.726**), con **20 falsi negativi** su **73 casi reali**. Questo significa che quando il modello predice EIWH è quasi sempre corretto, ma non intercetta tutti i brevetti EIWH. È quindi una classe su cui il modello è prudente.

La label **NOT_RELEVANT** ottiene una performance discreta, con **precision = 0.805**, **recall = 0.721** e **F1 = 0.761**. Considerando che questa label è gestita con una regola esclusiva, il risultato è accettabile: il modello riesce a riconoscere una parte importante dei brevetti non rilevanti, ma presenta ancora falsi negativi, cioè casi realmente `NOT_RELEVANT` che vengono classificati come appartenenti ad altre piattaforme.

Nel complesso, la confusion matrix conferma che il modello ha un comportamento robusto e ben bilanciato. Le classi più forti sono **HHP**, **GSWH**, **GAHP** e **GIWH**, mentre le aree principali di miglioramento riguardano la recall di **EIWH**, **HPWH**, **GASBOILER** e **NOT_RELEVANT**. Il modello è quindi molto affidabile nelle predizioni positive, ma potrebbe beneficiare di un leggero aggiustamento delle soglie se l’obiettivo fosse aumentare ulteriormente il recupero delle label corrette.